# Dental Caries AI — Multi-CNN Cascade (Stage 1 → Stage 4) — FINAL
### Questionnaire → CNN 1 Healthy vs Caries → Stage 2 FDI tooth detection → cavity localization → lesion→tooth link → CNN 2 G.V. Black → CNN 3 ICDAS → report

## 1. Project overview

```
PATIENT INFORMATION + ORAL-HABITS QUESTIONNAIRE  (habit-based risk score — kept separate, never changes a CNN output)
        ↓
USER UPLOADS A FULL INTRAORAL IMAGE  (validated, EXIF-corrected, saved in UPLOAD_DIR)
        ↓
CNN MODEL 1  —  STAGE 1  Healthy vs Caries           (stage1_caries_model.keras)
        ↓  HEALTHY → STOP          UNCERTAIN → STOP (retake the photo)
        ↓  CARIES
STAGE 2  FDI tooth detection + numbering            (stage2_fdi_detector.keras, trained on caries_FDI_dataset_final.zip)
        ↓  tooth boxes + two-digit FDI numbers ('Unknown / uncertain' when not confident; UNAVAILABLE if not trained)
        ↓
CAVITY / LESION LOCALIZATION  (class-agnostic: WHERE is a lesion)   (cavity_localizer_model.keras)
        ↓  one or more lesion boxes → one crop set per lesion
LESION → TOOTH ASSOCIATION  (each lesion box → the detected tooth box covering most of it)
        ↓
CNN MODEL 2  —  STAGE 3  G.V. Black class of each lesion crop       (stage3_gvblack_model.keras)
        ↓
CNN MODEL 3  —  STAGE 4  ICDAS score of each lesion crop            (stage4_icdas_model.keras)
        ↓
FINAL REPORT  per lesion: FDI tooth, G.V. Black + confidence, ICDAS + confidence; questionnaire risk + advice; limitations
```
The three classifiers are **separate, independent CNNs** (separate training data, separate weights, separate files).
The localizer only answers *where is a lesion*; the Stage 2 detector only answers *where is which tooth*; the classifiers only answer *which class*. Lesions and teeth are joined explicitly by box overlap — a lesion box is never used as a tooth box.

### What changed in this FINAL version (Stage 3 and Stage 4 optimisation)
| Item | Before | Now |
|---|---|---|
| Architecture choice | 4 backbones, 3–8-epoch frozen fine-tune, one input size | **5 backbones** (+ ResNet50) × **2 input sizes** (× 3 context paddings for G.V. Black), frozen features + head trained to convergence, ranked on **validation** accuracy + macro-F1 (loss shown) |
| Class imbalance | capped class weights, always | none / class weights / training-set oversampling / both, and label smoothing 0 / 0.1 — **chosen on validation** |
| Fine-tuning | top 30 % unfrozen, LR 3e-5, ReduceLROnPlateau | **gradual** unfreezing (top 25 % → top 50 %), BatchNorm frozen, warm-up + **cosine** decay |
| Head / augmentation | GAP → Dropout → Dense | GAP → **BatchNorm** → Dropout → Dense; + slight **shear**, moderately stronger zoom / brightness / contrast (training only) |
| Inference | single view | **test-time augmentation** only if it helps on validation; the same variant is used for test and for every prediction |
| Honesty checks | — | majority-class and frozen-feature logistic-regression **baselines**; train–val gap; identical-crop check across splits |
| Error analysis | — | new sections for Stage 3 and Stage 4 (confident errors, confused pairs, adjacent ICDAS scores, recall vs data, crop size, view) |

**About the 80–90 % target.** Every lever above is legitimate and chosen on validation only; the test set is predicted **once**. Whether
80–90 % is reached depends on the data, and the notebook prints the reasons if it is not: G.V. Black Class IV / VI and ICDAS 4 have very
few samples, ICDAS 1 vs 2 is judged on dried vs wet enamel (often not visible in a photo), ICDAS 5 is under-represented because
`icdascl.zip` lacks 1,869 of its crops, and the majority-class baseline (always predicting ICDAS 3) already scores ≈ 40 %.

### Findings from inspecting `icdascl.zip` while building this notebook
These are **dataset facts** (counts of files), not model results. Sections 49–56 recompute all of them from the ZIP you upload.

* Structure: `icdascl/ICDAS_1_to_6_Classified/ICDAS_{1..6}/<photo>__toothNN.jpg` + `crop_manifest.csv` + `images_without_caries.txt`.
* **No ICDAS 0 folder** ("no ICDAS_0 by design" — sound photographs are listed in `images_without_caries.txt`).
  Stage 4 is therefore a **6-class** problem (ICDAS 1–6); "no caries" is the job of Stage 1.
* Crops on disk: ICDAS 1: 1,348 · 2: 343 · 3: 2,297 · **4: 31** · 5: 374 · 6: 1,143 (5,536 total), JPEG, none corrupted, no byte-identical duplicates.
* `crop_manifest.csv` lists **7,868** crops but only **5,536** are in the ZIP (ICDAS 5: 1,869 missing; ICDAS 6: 463 missing). The ZIP is not
  damaged — those files were never added to it. The notebook trains on the files that exist and reports the gap; a complete ZIP is used
  automatically.
* Each crop names its **source photograph** (3,509 photographs for the crops on disk, up to 7 crops per photo, 364 photos contain teeth with
  different scores) and lists **identical copies** of that photograph. All crops of a photograph and of its copies are kept in one split.
* Near-duplicate crops (dHash) occur only **between different source photographs** (same-mouth burst photos); some carry different labels.
  They are grouped into the same split; near-identical crops with conflicting labels are excluded.

### How to run
* `Runtime → Change runtime type → T4 GPU`, then **Run all**. Nothing is `pip install`ed.
* Put the ZIPs (including `caries_FDI_dataset_final.zip` for Stage 2) in `MyDrive/dentalcaries/` (Section 3) or upload them when asked.
* **Execution order (Run all):** Sections 2–6 (setup + functions) → 7–23 (data, split, Stage 1) → **24 (Stage 2 FDI)** → 25–27 (localizer) → 28–48 (Stage 3) → 49–74 (Stage 4) → 75 (load) → 76–77 (pipeline + test) → 78–81 (save, reload, limitations, summary) → **82 (questionnaire → upload → report)**.
* **Prediction only (models already saved):** run Sections 2–6, then Section 75, then Section 82. Section 24 needs to be run once to train Stage 2 (it only needs Sections 2–6 and the FDI ZIP).
* Research proof of concept — **not a clinical diagnosis**. No number in this notebook is pre-written: every metric is computed when you run it.

<br>

---
# PART 1 — IMPORTS

## 2. Imports
TensorFlow / Keras only (no YOLO, Ultralytics, PyTorch, KerasCV, Detectron). The YOLO-format `.txt` files are read as plain numbers.

In [ ]:
# ============================================================
# SECTION 2 — IMPORTS
# ============================================================
import os, sys, io, json, time, shutil, zipfile, hashlib, warnings, random, re, math, glob
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score, confusion_matrix,
                             classification_report, precision_recall_fscore_support, cohen_kappa_score)

AUTOTUNE = tf.data.AUTOTUNE
pd.set_option("display.width", 200)

FORBIDDEN = ("ultralytics", "yolo", "torch", "torchvision", "keras_cv", "detectron2")
loaded_forbidden = [m for m in sys.modules if m.split(".")[0].lower() in FORBIDDEN]
assert not loaded_forbidden, f"Forbidden framework imported: {loaded_forbidden}"
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| OpenCV", cv2.__version__, "| pandas", pd.__version__)
print("No YOLO / PyTorch / KerasCV / Detectron modules loaded.")

<br>

---
# PART 2 — CONFIGURATION

## 3. Configuration
Edit only this cell. Put the ZIPs on Google Drive once (or upload them when asked):
```
MyDrive/dentalcaries/
├── Comprehensive_visual_index_of_carious_lesions.zip   (or caries.zip)          → Stage 1 "Caries" class
├── healthy/  (shortcut to the healthy-dataset folder; packed to one zip on the first run)  → Stage 1 "Healthy" class
├── lowoccyolo (4).zip                                   (G.V. Black images + YOLO-format boxes) → localizer + Stage 3
├── icdascl.zip                                          (CORRECTED ICDAS dataset)             → Stage 4
└── caries_FDI_dataset_final.zip                         (tooth boxes with FDI numbers)        → Stage 2
```
`ICD2_ICDAS_0_to_6_Classified.zip` (the old, wrong ICDAS ZIP) is **refused** even if it is present.

`TRAIN_MODE`: `"auto"` = load the saved model if it exists **and** was trained on the current split, otherwise train ·
`"train"` = always retrain · `"load"` = never train.

In [ ]:
# ============================================================
# SECTION 3 — CONFIGURATION
# ============================================================
SEED = 42
DETERMINISTIC_OPS = False                # True = bit-reproducible GPU ops (slower)

# ---------- storage ----------
USE_GOOGLE_DRIVE = True
DRIVE_BASE = "/content/drive/MyDrive/dentalcaries"
HEALTHY_DRIVE_LINK = "https://drive.google.com/drive/folders/1MibDDokJMI8D96x2pM2EEIo6VgHox55Mos"

# ---------- data sources (first existing path wins) ----------
HEALTHY_DIR = None                       # set a folder path here to skip the automatic search
HEALTHY_DIR_CANDIDATES = [f"{DRIVE_BASE}/healthy", "/content/drive/MyDrive/healthy",
                          "/content/drive/MyDrive/Teeth or Dental Image Dataset", "/content/healthy"]
HEALTHY_ZIP_CANDIDATES = [f"{DRIVE_BASE}/healthy.zip", "/content/healthy.zip"]
CARIES_DIR_CANDIDATES = [f"{DRIVE_BASE}/caries"]
CARIES_ZIP_CANDIDATES = [f"{DRIVE_BASE}/Comprehensive_visual_index_of_carious_lesions.zip", f"{DRIVE_BASE}/caries.zip",
                         "/content/Comprehensive_visual_index_of_carious_lesions.zip", "/content/caries.zip"]
GVB_ZIP_CANDIDATES = [f"{DRIVE_BASE}/lowoccyolo (4).zip", f"{DRIVE_BASE}/lowoccyolo__4_.zip", f"{DRIVE_BASE}/lowoccyolo.zip",
                      "/content/lowoccyolo (4).zip", "/content/lowoccyolo__4_.zip", "/content/lowoccyolo.zip"]
# CORRECTED ICDAS dataset — icdascl.zip ONLY (glob patterns; a browser may rename a re-download to "icdascl (1).zip")
ICDAS_ZIP_PATTERNS = [f"{DRIVE_BASE}/icdascl.zip", f"{DRIVE_BASE}/icdascl*.zip", "/content/icdascl.zip", "/content/icdascl*.zip"]
REFUSED_ICDAS_ZIPS = ("ICD2_ICDAS_0_to_6_Classified.zip",)   # the OLD, wrong ICDAS ZIP — never used

DATA_ROOT = "/content/data"              # LOCAL Colab disk — all training reads from here
CACHE_DIR = "/content/cache"             # LOCAL decoded caches
ZIP_DIR = "/content/zips"
CKPT_DIR = "/content/checkpoints"
PREDICTION_DIR = "/content/predictions"

# ---------- speed: Drive is used only for a few BIG files ----------
PACKED_SUBDIR = "packed"                 # healthy.zip is created in DRIVE_BASE/packed on the first run
DRIVE_CACHE_SUBDIR = "cache"             # decoded Stage 1 cache is saved in DRIVE_BASE/cache
PERSIST_S1_CACHE_TO_DRIVE = True
COPY_WORKERS = 48
DECODE_WORKERS = 8

# ---------- outputs (Drive when mounted, local disk otherwise — resolved in Section 4) ----------
OUT_SUBDIR = "pipeline_stage1_to_4_corrected"

TRAIN_MODE = {"stage1": "auto", "localizer": "auto", "gvb": "auto", "icdas": "auto", "fdi": "auto"}
BACKBONE_WEIGHTS = "imagenet"
FINE_TUNE_ON_CPU = False                 # fine-tuning on CPU is impractically slow

# Backbones compared on VALIDATION data only (short frozen-backbone screening); the winner is fully trained.
# ResNet50 is not a default candidate: 25 M parameters + caffe-style BGR preprocessing, for no expected gain on data of this size.
BACKBONE_CANDIDATES = ["EfficientNetV2B0", "EfficientNetB0", "MobileNetV2", "DenseNet121"]      # Stage 1
# Stages 3 and 4: all five pretrained families are compared on VALIDATION (feature screening, Section 6.7)
STAGE34_CANDIDATES = ["EfficientNetV2B0", "EfficientNetB0", "DenseNet121", "ResNet50", "MobileNetV2"]

# ---------- G.V. Black mapping ----------
# Used ONLY if the G.V. Black ZIP contains no mapping file (classes.txt / data.yaml / notes.json).
# If a mapping file exists, Section 30 reads it and it wins. The mapping actually used is printed before training.
GVB_CONFIRMED_MAPPING = {0: "GVB_Class_I", 1: "GVB_Class_II", 2: "GVB_Class_III",
                         3: "GVB_Class_IV", 4: "GVB_Class_V", 5: "GVB_Class_VI"}
GVB_CLASS_NAMES = [GVB_CONFIRMED_MAPPING[i] for i in range(6)]
NUM_GVB = len(GVB_CLASS_NAMES)

# ---------- shared split + leakage control ----------
SPLIT_FRACS = (0.70, 0.15, 0.15)         # train / val / test, done ONCE for all stages, before any augmentation
DUP_HAMMING = 3                          # dHash distance treated as the same photo (dedup + label-conflict check)
S1_GROUP_HAMMING = 6                     # photos this close are kept in the same split (same-mouth bursts)
POOL_GROUP_HAMMING = 8                   # stricter grouping inside the G.V. Black photographs

# ---------- Stage 1 (CNN 1: healthy vs caries) ----------
S1 = dict(
    IMG_PX=224, BATCH=64,
    CANDIDATES=BACKBONE_CANDIDATES, SCREEN_EPOCHS=3, SCREEN_MAX_TRAIN=6000,
    EPOCHS_A=12, EPOCHS_B=15, LR_A=1e-3, LR_B=2e-5, WEIGHT_DECAY=1e-4,
    PATIENCE_A=4, PATIENCE_B=5, UNFREEZE_FRAC=0.30,
    DROPOUT=0.30, DENSE=256,
    AUG=dict(hflip=True, rotate=0.03, translate=0.06, zoom=0.10, brightness=0.15, contrast=0.15),
    CLASS_WEIGHTS="auto",                # "auto" = balanced weights if imbalance > 1.5 : 1
    # caries-archive photographs listed in icdascl.zip/images_without_caries.txt ("no carious tooth visible"):
    # "exclude" (default) = not used as Caries | "healthy" = relabelled Healthy | "keep" = left in Caries
    NO_CARIES_LIST_ACTION="exclude",
    UNCERTAIN_TARGET_ACC=0.95, UNCERTAIN_MAX_MARGIN=0.25,   # validation-derived uncertainty band around the threshold
    CONFIDENCE_THRESHOLD=0.60,           # additionally: confidence below this -> UNCERTAIN (configurable)
)

# ---------- cavity localizer (class-agnostic, architecture kept from your notebook) ----------
LOC = dict(
    INPUT_SIZE=416, STRIDE=16, BATCH=16, STEPS_PER_EPOCH=None,   # None = one pass over the training images
    EPOCHS_A=40, EPOCHS_B=30, LR_A=1e-3, LR_B=1e-4,
    UNFREEZE_FROM="block_10_expand", MIN_TRAIN_IMAGES_FOR_FINETUNE=30,
    CENTER_RADIUS=1.5, POS_VISIBLE_FRAC=0.6, HFLIP_PROB=0.5,
    FOCAL_ALPHA=0.25, FOCAL_GAMMA=2.0, BOX_LOSS_WEIGHT=2.0,
    NMS_IOU=0.40, MAX_DETECTIONS=10, AP_MIN_SCORE=0.01,
    DEFAULT_SCORE_THRESHOLD=0.30, MIN_BOX_PX=4, GEN_WORKERS=2,
)

# ---------- Stage 3 (CNN 2: G.V. Black on lesion crops) ----------
GVB = dict(
    IMG_SIZE=224, BATCH=32,
    SIZE_CANDIDATES=[224, 288],          # input resolutions compared on validation
    PAD_CANDIDATES=[0.0, 0.5, 1.0],      # context around the box, as a fraction of the box's longer side (each side)
    DEFAULT_PAD=0.5,                     # used only for the crop demo in Section 27, before Stage 3 has chosen its padding
    CANDIDATES=STAGE34_CANDIDATES,
    BALANCE_CANDIDATES=["none", "class_weight", "oversample", "class_weight+oversample"],
    LS_CANDIDATES=[0.0, 0.1],            # label smoothing values compared on validation
    OVERSAMPLE_POWER=0.5,                # n_c' = n_max^0.5 * n_c^0.5 (training rows only)
    EPOCHS_A=30, EPOCHS_B=25, LR_A=1e-3, LR_B=1e-4, WEIGHT_DECAY=1e-4,
    PATIENCE_A=7, PATIENCE_B=7, UNFREEZE_FRAC=0.30,
    UNFREEZE_STEPS=[0.25, 0.50],         # gradual unfreezing: top 25 %, then top 50 % (LR halved); BatchNorm stays frozen
    PHASE2_COSINE=True,                  # warm-up + cosine decay during fine-tuning
    HEAD_BN=True, DROPOUT=0.35, DENSE=256,
    MONITOR="val_macro_f1",              # checkpoint / early-stopping metric (robust under imbalance)
    JITTER_COPIES=2, JITTER_FRAC=0.10,   # extra TRAIN crops with ±10 % box jitter (mimics localizer error)
    MIN_CROP_PX=48, MAX_CLASS_WEIGHT=10.0,
    TTA_CANDIDATES=["none", "zoom"],     # no flip views: lesion position defines the class
    CONFIDENCE_THRESHOLD=0.50,           # below -> UNCERTAIN (configurable)
    MIN_RELIABLE_TEST=15,                # classes with fewer TEST crops are flagged as unreliable
    AUG=dict(hflip=False, rotate=0.04, translate=0.06, zoom=0.12, brightness=0.15, contrast=0.15, shear=0.05),
)

# ---------- Stage 4 (CNN 3: ICDAS on tooth / lesion crops from icdascl.zip) ----------
ICDAS = dict(
    IMG_SIZE=224, BATCH=32,              # crops are letterboxed to a square (aspect ratio kept); 224 = base cache / hashing size
    SIZE_CANDIDATES=[224, 288],          # input resolutions compared on validation
    CANDIDATES=STAGE34_CANDIDATES,
    BALANCE_CANDIDATES=["none", "class_weight", "oversample", "class_weight+oversample"],
    LS_CANDIDATES=[0.0, 0.1],
    OVERSAMPLE_POWER=0.5,
    EPOCHS_A=30, EPOCHS_B=25, LR_A=1e-3, LR_B=1e-4, WEIGHT_DECAY=1e-4,
    PATIENCE_A=7, PATIENCE_B=7, UNFREEZE_FRAC=0.30,
    UNFREEZE_STEPS=[0.25, 0.50], PHASE2_COSINE=True,
    HEAD_BN=True, DROPOUT=0.35, DENSE=256, MONITOR="val_macro_f1",
    MAX_CLASS_WEIGHT=10.0, CONFIDENCE_THRESHOLD=0.50,
    TTA_CANDIDATES=["none", "hflip", "hflip+zoom"],
    GROUP_HAMMING=6,                     # near-duplicate crops (different source photos) kept in the same split
    DUP_HAMMING=2,                       # near-identical crops ...
    DROP_CONFLICTING_DUPLICATES=True,    # ... with DIFFERENT ICDAS labels are excluded (contradictory labels, not "difficult" samples)
    MIN_RELIABLE_TRAIN=30,               # classes with fewer TRAINING crops are flagged as unreliable
    # Severity is mirror-invariant, so horizontal flips are allowed. No hue changes (discolouration is an ICDAS criterion).
    AUG=dict(hflip=True, rotate=0.05, translate=0.06, zoom=0.12, brightness=0.15, contrast=0.15, shear=0.05),
    # Pipeline: a localizer box marks a LESION, the ICDAS CNN was trained on TOOTH crops -> a tooth-scale window is cut
    # around each lesion: side = max(lesion side * mult, min_scale * median tooth-crop fraction of the photo).
    # (mult, min_scale) is chosen on VALIDATION lesions in the lesion-window section when links exist; this is the fallback.
    WINDOW_CANDIDATES=[(1.5, 0.0), (2.5, 0.0), (1.5, 0.5), (2.5, 0.5), (1.5, 1.0), (4.0, 0.5)],
    DEFAULT_WINDOW=(2.5, 0.5),
)

# ---------- Stage 2 (FDI tooth detection + numbering; TensorFlow/Keras detector trained on caries_FDI_dataset_final.zip) ----------
FDI_ZIP_PATTERNS = [f"{DRIVE_BASE}/caries_FDI_dataset_final.zip", f"{DRIVE_BASE}/caries_FDI_dataset_final*.zip",
                    f"{DRIVE_BASE}/caries_FDI*.zip", "/content/caries_FDI_dataset_final.zip", "/content/caries_FDI*.zip"]
FDI_DIR_CANDIDATES = [f"{DRIVE_BASE}/caries_FDI_dataset_final", f"{DRIVE_BASE}/caries_FDI_dataset",   # extracted folder on Drive
                      "/content/caries_FDI_dataset_final", "/content/caries_FDI_dataset"]
FDI = dict(
    ASK_UPLOAD=True,                     # ZIP not found -> ask for an upload (Cancel = Stage 2 stays UNAVAILABLE)
    INPUT_SIZE=512, STRIDE=16, BATCH=8, STEPS_PER_EPOCH=None, MIN_STEPS_PER_EPOCH=50,   # None = one pass (at least MIN_STEPS)
    EPOCHS_A=40, EPOCHS_B=40, LR_A=1e-3, LR_B=1e-4, UNFREEZE_FROM="block_10_expand", MIN_TRAIN_IMAGES_FOR_FINETUNE=30,
    CENTER_RADIUS=1.5, IGNORE_RADIUS=2.5, POS_VISIBLE_FRAC=0.6, CROP_MIN=0.80,
    FOCAL_ALPHA=0.25, FOCAL_GAMMA=2.0, BOX_LOSS_WEIGHT=2.0, CLS_LOSS_WEIGHT=1.0,
    NEG_WEIGHT=0.3,                      # background-loss weight: the dataset leaves uncertain teeth UNLABELLED on purpose, so
                                         # "background" cells may contain real teeth -> they are penalised less (1.0 = standard)
    NMS_IOU=0.45, MAX_DETECTIONS=40, AP_MIN_SCORE=0.01, DEFAULT_SCORE_THRESHOLD=0.30, GEN_WORKERS=2,
    CLASS_CONFIDENCE_THRESHOLD=0.50,     # FDI-number probability below this -> "Unknown / uncertain"
    ASSOC_MIN_OVERLAP=0.30,              # share of the lesion box a tooth box must cover to link the lesion to that tooth
    ASSOC_AMBIGUITY_RATIO=0.80,          # a 2nd tooth covering >= 80 % of the best overlap -> "Uncertain: 16 or 17"
    MIN_ANNOTATED_IMAGES=30, MIN_TRAIN_BOXES_RELIABLE=20,
    INCLUDE_PRIMARY_TEETH=True,          # FDI 51–85 (deciduous) are used if the dataset labels them
    EXCLUDE_DIRS=["only fdi"],           # sub-folders NOT used: 'only fdi' repeats photos of the main set with a different class
                                         # list (0 = 31 instead of 0 = 11) and labels that contradict it (e.g. LFV31: 42 vs 43)
    CLASS_ID_MAPPING=None,               # {yolo_class_id: "16", ...} ONLY if the ZIP has no class-name file and you KNOW the mapping
    ANNOTATION_FRAME="auto",             # box frame: "raw" stored pixels | "exif" EXIF-rotated pixels | "auto" (Label Studio -> exif)
    PATIENT_ID_REGEX=None,               # e.g. r"^(P\d+)_" -> all images of one patient stay in one split
    GROUP_HAMMING=8,                     # near-duplicate photos (dHash) kept in one split
    ALLOW_MODALITY_MISMATCH=False,       # True only if grey-scale FDI images really are photographs (not radiographs)
)
UPLOAD_DIR = "/content/uploads"          # photos uploaded in Section 82 are saved here (created below)

for d in (DATA_ROOT, CACHE_DIR, ZIP_DIR, CKPT_DIR, PREDICTION_DIR, UPLOAD_DIR):
    os.makedirs(d, exist_ok=True)
print("Configuration loaded.")

<br>

---
# PART 3 — GPU SETUP (+ seeds, shared utilities)

## 4. Google Colab / GPU setup
Mounts Google Drive (if enabled), resolves the output folder, detects the GPU and enables mixed precision only on GPUs with compute
capability ≥ 7.0 (T4, L4, A100). Every model output layer is pinned to float32.

In [ ]:
# ============================================================
# SECTION 4 — GOOGLE COLAB / DRIVE / GPU SETUP
# ============================================================
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content/sample_data")
DRIVE_MOUNTED = False
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive not mounted:", e)

OUT_DIR = f"{DRIVE_BASE}/{OUT_SUBDIR}" if DRIVE_MOUNTED else f"/content/{OUT_SUBDIR}"
PACKED_DIR = f"{DRIVE_BASE}/{PACKED_SUBDIR}" if DRIVE_MOUNTED else f"{ZIP_DIR}/{PACKED_SUBDIR}"
DRIVE_CACHE_DIR = f"{DRIVE_BASE}/{DRIVE_CACHE_SUBDIR}" if DRIVE_MOUNTED else None
for d in (OUT_DIR, PACKED_DIR) + ((DRIVE_CACHE_DIR,) if DRIVE_CACHE_DIR else ()):
    os.makedirs(d, exist_ok=True)

MODEL_PATHS = {"stage1": f"{OUT_DIR}/stage1_caries_model.keras",
               "localizer": f"{OUT_DIR}/cavity_localizer_model.keras",
               "gvb": f"{OUT_DIR}/stage3_gvblack_model.keras",
               "icdas": f"{OUT_DIR}/stage4_icdas_model.keras",
               "fdi": f"{OUT_DIR}/stage2_fdi_detector.keras"}           # Stage 2 (optional: trained in Section 24)
MODEL_META_PATHS = {k: v.replace(".keras", "_metadata.json") for k, v in MODEL_PATHS.items()}
METADATA_PATH = f"{OUT_DIR}/pipeline_metadata.json"
print("Outputs ->", OUT_DIR, "(Google Drive)" if DRIVE_MOUNTED else "(local disk — download before the session ends)")

GPUS = tf.config.list_physical_devices("GPU")
GPU_AVAILABLE = len(GPUS) > 0
COMPUTE_CAP = (0, 0)
for g in GPUS:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
if GPU_AVAILABLE:
    try:
        det_ = tf.config.experimental.get_device_details(GPUS[0])
        COMPUTE_CAP = tuple(det_.get("compute_capability", (0, 0)))
        print("GPU:", det_.get("device_name", "unknown"), "| compute capability", COMPUTE_CAP)
    except Exception:
        print("GPU:", GPUS)
else:
    print("NO GPU — Runtime → Change runtime type → T4 GPU. Training will be very slow and "
          "backbone fine-tuning is skipped on CPU (FINE_TUNE_ON_CPU=False).")

MIXED_PRECISION = GPU_AVAILABLE and COMPUTE_CAP >= (7, 0)
keras.mixed_precision.set_global_policy("mixed_float16" if MIXED_PRECISION else "float32")
CAN_FINE_TUNE = GPU_AVAILABLE or FINE_TUNE_ON_CPU
print("Mixed precision:", "ENABLED" if MIXED_PRECISION else "disabled", "| fine-tuning allowed:", CAN_FINE_TUNE)
if not GPU_AVAILABLE:
    S1["BATCH"] = min(S1["BATCH"], 32)

## 5. Random seed setup
Python, NumPy and TensorFlow seeds are fixed. GPU kernels can still be non-deterministic unless `DETERMINISTIC_OPS = True`
(slower), so repeated runs may differ slightly.

In [ ]:
# ============================================================
# SECTION 5 — RANDOM SEEDS
# ============================================================
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
keras.utils.set_random_seed(SEED)                  # python + numpy + tensorflow
if DETERMINISTIC_OPS:
    tf.config.experimental.enable_op_determinism()
print("Seed", SEED, "| deterministic ops:", DETERMINISTIC_OPS)

## 6. Utility functions
Six cells, all definitions (nothing heavy runs):
* **6.1 Preprocessing contract** (+ Stage 2 FDI label / decoding / lesion→tooth association helpers) — **one** function per stage turns an image into model input. The same function builds the training data, the
  evaluation data and the inference input, so training and inference cannot drift apart. Every classifier contains its own normalization
  layer, so the input to every classifier is plain RGB `[0, 255]` → no double normalization.
* **6.2** data staging · **6.3** grouping / splitting / `tf.data` · **6.4** model building and two-phase training ·
  **6.5** evaluation and plots · **6.6** prediction functions + model loading (work after a runtime restart).

In [ ]:
# ============================================================
# SECTION 6.1 — I/O, HASHING, PREPROCESSING CONTRACT, METADATA
# ============================================================
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
DISCLAIMER = "Research proof of concept — not a clinical diagnosis. Please consult a dentist."

TIMINGS = {}
class stopwatch:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time(); return self
    def __exit__(self, *a):
        TIMINGS[self.name] = time.time() - self.t0
        print(f"[timing] {self.name}: {TIMINGS[self.name]:.1f}s")

def load_rgb(source):
    """Path or bytes -> RGB uint8 with EXIF orientation applied. Used by EVERY stage (training + inference)."""
    if isinstance(source, (bytes, bytearray)):
        source = io.BytesIO(source)
    with Image.open(source) as im:
        return np.array(ImageOps.exif_transpose(im).convert("RGB"))

def file_md5(path, block=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block), b""):
            h.update(b)
    return h.hexdigest()

_POPCOUNT = np.unpackbits(np.arange(256, dtype=np.uint8)[:, None], axis=1).sum(1).astype(np.uint8)
def dhash64(rgb_small):
    """64-bit difference hash (packed to 8 bytes) of an RGB image."""
    g = rgb_small.astype(np.float32).mean(axis=2)
    s = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)
    return np.packbits((s[:, 1:] > s[:, :-1]).ravel())

norm_folder = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
strip_uuid = lambda s: re.sub(r"^[0-9a-f]{8}-", "", s)          # Label-Studio exports prefix file names with 8 hex chars
stem = lambda p: os.path.splitext(os.path.basename(p))[0]

# ---------- Stage 1 contract ----------
def s1_prepare(rgb, size):
    """EXIF-corrected RGB -> cv2.resize(size x size, INTER_AREA) -> uint8 [0,255]. Normalisation is inside the model."""
    return cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)

# ---------- Stage 3 contract (square context crop around a lesion box) ----------
def to_px(box, W, H):
    x1 = int(np.floor(box[0] * W)); y1 = int(np.floor(box[1] * H))
    x2 = int(np.ceil(box[2] * W));  y2 = int(np.ceil(box[3] * H))
    return max(x1, 0), max(y1, 0), min(x2, W), min(y2, H)

def square_context_box(box, W, H, pad, min_px):
    """Normalised x1y1x2y2 box -> square pixel window centred on the box, side = longer box side * (1 + 2*pad)."""
    x1, y1, x2, y2 = box[0] * W, box[1] * H, box[2] * W, box[3] * H
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half = max(max(x2 - x1, y2 - y1) * (1 + 2 * pad), min_px) / 2
    return int(round(cx - half)), int(round(cy - half)), int(round(cx + half)), int(round(cy + half))

def crop_with_border(rgb, x1, y1, x2, y2):
    """Crop that may extend past the image edge; outside pixels are black, so the lesion stays centred."""
    H, W = rgb.shape[:2]
    out = np.zeros((max(y2 - y1, 1), max(x2 - x1, 1), 3), np.uint8)
    sx1, sy1, sx2, sy2 = max(x1, 0), max(y1, 0), min(x2, W), min(y2, H)
    if sx2 > sx1 and sy2 > sy1:
        out[sy1 - y1:sy2 - y1, sx1 - x1:sx2 - x1] = rgb[sy1:sy2, sx1:sx2]
    return out

def gvb_prepare_crop(rgb, box, pad, size, min_px):
    """Stage 3 contract: full-resolution RGB + normalised box -> square context crop -> resize -> uint8 [0,255]."""
    H, W = rgb.shape[:2]
    c = crop_with_border(rgb, *square_context_box(box, W, H, pad, min_px))
    interp = cv2.INTER_AREA if c.shape[0] > size else cv2.INTER_LINEAR
    return cv2.resize(c, (size, size), interpolation=interp)

def letterbox(rgb, out_h, out_w):
    """Resize keeping the aspect ratio, centre on a black canvas (no distortion)."""
    H, W = rgb.shape[:2]
    s = min(out_h / H, out_w / W)
    nh, nw = max(1, int(round(H * s))), max(1, int(round(W * s)))
    r = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    out = np.zeros((out_h, out_w, 3), np.uint8)
    t, l = (out_h - nh) // 2, (out_w - nw) // 2
    out[t:t + nh, l:l + nw] = r
    return out

# ---------- Stage 4 contract ----------
def icdas_prepare(rgb, size):
    """Stage 4 contract: tooth / lesion crop -> square letterbox (aspect kept, black padding) -> uint8 [0,255]."""
    return letterbox(rgb, size, size)

def icdas_window_box(box, W, H, mult, min_frac_w, min_frac_h):
    """Lesion box (normalised x1y1x2y2) -> tooth-scale pixel window centred on the lesion and kept inside the image.
    The ICDAS CNN is trained on TOOTH crops; a localizer box marks a LESION, so a window of tooth scale is cut around it."""
    x1, y1, x2, y2 = box[0] * W, box[1] * H, box[2] * W, box[3] * H
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w = float(min(max((x2 - x1) * mult, min_frac_w * W, 8.0), W))
    h = float(min(max((y2 - y1) * mult, min_frac_h * H, 8.0), H))
    left = int(round(np.clip(cx - w / 2, 0, W - w))); top = int(round(np.clip(cy - h / 2, 0, H - h)))
    return left, top, min(int(round(left + w)), W), min(int(round(top + h)), H)

def icdas_prepare_from_box(rgb, box, size, window):
    H, W = rgb.shape[:2]
    x1, y1, x2, y2 = icdas_window_box(box, W, H, window["mult"], window["min_frac_w"], window["min_frac_h"])
    return icdas_prepare(rgb[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1)], size)

# ---------- localizer contract (kept from your notebook) ----------
def loc_prepare(rgb, cache_side, input_size):
    x = cv2.resize(rgb, (cache_side, cache_side), interpolation=cv2.INTER_AREA).astype(np.float32)
    return mobilenet_v2_preprocess(cv2.resize(x, (input_size, input_size), interpolation=cv2.INTER_AREA))

def grid_centres(grid):
    c = (np.arange(grid, dtype=np.float32) + 0.5) / grid
    return np.meshgrid(c, c)

def decode_detections(raw, score_thr, nms_iou, max_det):
    """raw (G,G,5) -> boxes (N,4) normalised x1y1x2y2, scores (N,). NMS = tf.image.non_max_suppression."""
    g = raw.shape[0]; gx, gy = grid_centres(g)
    scores = (1 / (1 + np.exp(-raw[..., 0]))).reshape(-1)
    d = np.logaddexp(0.0, raw[..., 1:5]) / g
    boxes = np.clip(np.stack([gx - d[..., 0], gy - d[..., 1], gx + d[..., 2], gy + d[..., 3]], -1), 0, 1).reshape(-1, 4)
    idx = np.where(scores >= score_thr)[0]
    if idx.size == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)
    keep = tf.image.non_max_suppression(boxes[idx][:, [1, 0, 3, 2]], scores[idx], max_det,
                                        iou_threshold=nms_iou).numpy()
    return boxes[idx[keep]].astype(np.float32), scores[idx[keep]].astype(np.float32)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    inter = iw * ih
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(aa[:, None] + bb[None] - inter, 1e-9)

# ---------- Stage 2 (FDI tooth numbering) helpers: labels, EXIF box frame, decoding, lesion -> tooth association ----------
FDI_PERMANENT = [q * 10 + t for q in (1, 2, 3, 4) for t in range(1, 9)]
FDI_PRIMARY = [q * 10 + t for q in (5, 6, 7, 8) for t in range(1, 6)]
FDI_VALID = set(FDI_PERMANENT) | set(FDI_PRIMARY)
LESION_WORDS = re.compile(r"cari|lesion|cavit|decay", re.I)

def fdi_from_label(name):
    """Annotation label -> FDI code as str ('16') or None. Only ONE explicit, valid two-digit FDI code is accepted
    ('16', 'tooth_16', 'FDI 16'); bare class ids, Universal (1-32) and Palmer labels are never converted."""
    nums = re.findall(r"(?<!\d)(\d{2})(?!\d)", str(name))
    return nums[0] if len(nums) == 1 and int(nums[0]) in FDI_VALID else None

def boxes_raw_to_exif(b, orientation):
    """Normalised x1y1x2y2 boxes in the stored-pixel frame -> the frame after ImageOps.exif_transpose (what load_rgb returns)."""
    b = np.asarray(b, np.float32).reshape(-1, 4); x1, y1, x2, y2 = b.T
    nb = {2: [1 - x2, y1, 1 - x1, y2], 3: [1 - x2, 1 - y2, 1 - x1, 1 - y1], 4: [x1, 1 - y2, x2, 1 - y1],
          5: [y1, x1, y2, x2], 6: [1 - y2, x1, 1 - y1, x2], 7: [1 - y2, 1 - x2, 1 - y1, 1 - x1], 8: [y1, 1 - x2, y2, 1 - x1]}.get(orientation)
    return b if nb is None else np.stack(nb, 1).astype(np.float32)

def decode_fdi_detections(raw, score_thr, nms_iou, max_det):
    """raw (G,G,5+C) -> boxes (N,4) normalised x1y1x2y2, objectness (N,), FDI class probabilities (N,C). Class-agnostic NMS."""
    g = raw.shape[0]; gx, gy = grid_centres(g); n_cls = raw.shape[-1] - 5
    scores = (1 / (1 + np.exp(-raw[..., 0]))).reshape(-1)
    d = np.logaddexp(0.0, raw[..., 1:5]) / g
    boxes = np.clip(np.stack([gx - d[..., 0], gy - d[..., 1], gx + d[..., 2], gy + d[..., 3]], -1), 0, 1).reshape(-1, 4)
    idx = np.where(scores >= score_thr)[0]
    if idx.size == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0, n_cls), np.float32)
    logits = raw[..., 5:].reshape(-1, n_cls)[idx].astype(np.float64)
    probs = np.exp(logits - logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
    keep = tf.image.non_max_suppression(boxes[idx][:, [1, 0, 3, 2]], scores[idx], max_det, iou_threshold=nms_iou).numpy()
    return boxes[idx[keep]].astype(np.float32), scores[idx[keep]].astype(np.float32), probs[keep].astype(np.float32)

def overlap_fraction(box, boxes):
    """Fraction of `box`'s area covered by each of `boxes` (all normalised x1y1x2y2)."""
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4); b = np.asarray(box, np.float32)
    iw = np.clip(np.minimum(b[2], boxes[:, 2]) - np.maximum(b[0], boxes[:, 0]), 0, None)
    ih = np.clip(np.minimum(b[3], boxes[:, 3]) - np.maximum(b[1], boxes[:, 1]), 0, None)
    return iw * ih / max(float((b[2] - b[0]) * (b[3] - b[1])), 1e-9)

def associate_lesions_with_teeth(lesion_boxes, teeth, min_overlap, ambiguity_ratio):
    """Explicit lesion -> tooth link (lesion boxes and tooth boxes come from two different models).
    The tooth covering the largest share of the lesion wins if it covers >= min_overlap. Status:
    ASSIGNED | AMBIGUOUS (a 2nd tooth covers >= ambiguity_ratio x as much) | TOOTH_NUMBER_UNCERTAIN | NO_TOOTH.
    Only ASSIGNED gives an FDI number; everything else is reported as 'Unknown / uncertain' (best guesses kept separately)."""
    tb = np.array([t["box_norm"] for t in teeth], np.float32).reshape(-1, 4); out = []
    for b in lesion_boxes:
        cov = overlap_fraction(b, tb) if len(tb) else np.zeros(0, np.float32)
        if not len(cov) or cov.max() < min_overlap:
            out.append({"fdi_tooth": None, "fdi_display": "Unknown / uncertain", "fdi_status": "NO_TOOTH", "tooth_id": None,
                        "fdi_confidence": None, "fdi_overlap": float(cov.max()) if len(cov) else 0.0, "fdi_also_overlaps": [],
                        "fdi_note": f"no detected tooth covers >= {min_overlap:.0%} of the lesion"})
            continue
        order = np.argsort(-cov); j = int(order[0]); t = teeth[j]
        alts = [teeth[k] for k in order[1:] if cov[k] >= max(min_overlap, ambiguity_ratio * cov[j])]
        rec = {"tooth_id": t.get("tooth_id"), "fdi_confidence": float(t["fdi_confidence"]), "fdi_overlap": float(cov[j]),
               "fdi_best_guess": t["fdi"], "fdi_also_overlaps": [a["fdi"] for a in alts]}
        if t["fdi_uncertain"]:
            rec.update(fdi_tooth=None, fdi_status="TOOTH_NUMBER_UNCERTAIN", fdi_display=f"Unknown / uncertain (best guess {t['fdi']})",
                       fdi_note="the covering tooth was detected but its FDI number is uncertain")
        elif alts:
            rec.update(fdi_tooth=None, fdi_status="AMBIGUOUS",
                       fdi_display="Uncertain: " + " or ".join([t["fdi"]] + [a["fdi"] for a in alts]),
                       fdi_note="the lesion lies across more than one detected tooth")
        else:
            rec.update(fdi_tooth=t["fdi"], fdi_status="ASSIGNED", fdi_display=t["fdi"], fdi_note=None)
        out.append(rec)
    return out

# ---------- metadata ----------
def json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (set, tuple)): return list(o)
    return str(o)

def read_json(path):
    if path and os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}

def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w") as f:
        json.dump(obj, f, indent=2, default=json_default)
    os.replace(path + ".tmp", path)

def read_metadata():
    return read_json(METADATA_PATH)

def update_metadata(updates):
    """Merge top-level keys into pipeline_metadata.json (dict values are merged one level deep)."""
    meta = read_metadata()
    for k, v in updates.items():
        if isinstance(v, dict) and isinstance(meta.get(k), dict):
            meta[k].update(v)
        else:
            meta[k] = v
    write_json(METADATA_PATH, json.loads(json.dumps(meta, default=json_default)))
    return meta

def write_model_meta(stage, meta):
    """Per-model metadata file next to the .keras file + a copy inside pipeline_metadata.json."""
    meta = json.loads(json.dumps(meta, default=json_default))
    write_json(MODEL_META_PATHS[stage], meta)
    update_metadata({stage: meta})
    return meta

def gvb_short(name):
    """'GVB_Class_II' -> 'Class II'."""
    return "Class " + str(name).split("_")[-1]

print("6.1 ready: preprocessing contract (s1_prepare, gvb_prepare_crop, icdas_prepare, loc_prepare), FDI helpers, metadata helpers.")

In [ ]:
# ============================================================
# SECTION 6.2 — DATA STAGING HELPERS (copy / extract once per session)
# ============================================================
def first_existing(paths, kind="any"):
    for p in paths:
        if p and os.path.exists(p) and (kind != "dir" or os.path.isdir(p)):
            return p
    return None

def upload_file(prompt, must_start_with=None):
    from google.colab import files
    print(prompt)
    up = files.upload()
    if not up:
        raise FileNotFoundError("Nothing uploaded.")
    name = list(up.keys())[0]
    if must_start_with and not os.path.basename(name).lower().startswith(must_start_with):
        raise ValueError(f"Expected a file named {must_start_with}*.zip, got {name}")
    dst = os.path.join(ZIP_DIR, os.path.basename(name))
    with open(dst, "wb") as f:
        f.write(up[name])
    return dst

def copy_big_file(src, dst):
    """One sequential copy (fast, even from Drive). Skipped if a same-size copy already exists."""
    if os.path.abspath(src) == os.path.abspath(dst):
        return dst
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    t0 = time.time()
    shutil.copyfile(src, dst + ".part")
    os.replace(dst + ".part", dst)
    print(f"  copied {os.path.basename(src)} ({os.path.getsize(dst)/1e6:.0f} MB) in {time.time()-t0:.0f}s")
    return dst

def stage_zip(zip_path, dest):
    """ZIP -> local copy -> extracted on local disk ONCE. The marker holds name + size, so a new ZIP version re-extracts."""
    marker = os.path.join(dest, ".extracted_ok")
    sig = f"{os.path.basename(zip_path)}|{os.path.getsize(zip_path)}"
    if os.path.exists(marker) and open(marker).read().strip() == sig:
        print(f"  already extracted -> {dest}")
        return dest
    if os.path.exists(dest):
        shutil.rmtree(dest)
    local_zip = zip_path
    if zip_path.startswith("/content/drive"):
        local_zip = copy_big_file(zip_path, os.path.join(ZIP_DIR, os.path.basename(zip_path)))
    os.makedirs(dest, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        bad = zf.testzip()
        if bad:
            raise IOError(f"{zip_path}: corrupted member {bad}")
        zf.extractall(dest)
    with open(marker, "w") as f:
        f.write(sig)
    print(f"  extracted {os.path.basename(zip_path)} in {time.time()-t0:.0f}s -> {dest}")
    return dest

def search_drive_for_folder(names, root="/content/drive/MyDrive", max_depth=3):
    if not os.path.isdir(root):
        return None
    root_depth = root.rstrip("/").count("/")
    for dp, dn, fn in os.walk(root):
        if dp.count("/") - root_depth >= max_depth:
            dn[:] = []
        for d in dn:
            if d in names:
                return os.path.join(dp, d)
    return None

def pack_drive_folder_once(src_dir, local_dest, drive_zip):
    """ONE-TIME: parallel copy of a Drive folder to local disk, then save ONE zip back to Drive."""
    marker = os.path.join(local_dest, ".extracted_ok")
    if os.path.exists(marker):
        return local_dest
    t0 = time.time()
    files_ = [os.path.join(dp, f) for dp, dn, fn in os.walk(src_dir)
              for f in fn if f.lower().endswith(IMG_EXT) and not f.startswith(".")]
    print(f"  one-time packing of {src_dir}: {len(files_):,} images")
    def _cp(p):
        dst = os.path.join(local_dest, os.path.relpath(p, src_dir))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            shutil.copyfile(p, dst)
    with ThreadPoolExecutor(COPY_WORKERS) as ex:
        for k, _ in enumerate(ex.map(_cp, files_), 1):
            if k % 2000 == 0:
                print(f"    copied {k:,}/{len(files_):,}")
    with open(marker, "w") as f:
        f.write("copied from " + src_dir)
    print(f"  local copy finished in {time.time()-t0:.0f}s")
    try:
        tmp = os.path.join(ZIP_DIR, os.path.basename(drive_zip))
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as zf:      # JPEGs are already compressed
            for dp, _, fn in os.walk(local_dest):
                for f in fn:
                    if not f.startswith("."):
                        full = os.path.join(dp, f)
                        zf.write(full, os.path.relpath(full, local_dest))
        copy_big_file(tmp, drive_zip)
        print(f"  saved {drive_zip} -> later sessions copy only this file")
    except Exception as e:
        print("  could not save the packed zip (training still works):", e)
    return local_dest

def scan_images(root):
    out = []
    for dp, dn, fn in os.walk(root):
        dn[:] = sorted(d for d in dn if not d.startswith((".", "__MACOSX")))
        out += [os.path.join(dp, f) for f in fn if not f.startswith(".") and f.lower().endswith(IMG_EXT)]
    return sorted(out)

print("6.2 ready: staging helpers.")

In [ ]:
# ============================================================
# SECTION 6.3 — GROUPING, SPLITTING, tf.data
# ============================================================
def hamming_edges(A, B, t, same, chunk=512):
    """All (i, j) with Hamming(A[i], B[j]) <= t. same=True -> A is B, keep i < j only."""
    rows, cols = [np.zeros(0, np.int64)], [np.zeros(0, np.int64)]
    for s in range(0, len(A), chunk):
        d = _POPCOUNT[A[s:s + chunk, None, :] ^ B[None, :, :]].sum(axis=2, dtype=np.int16)
        i, j = np.nonzero(d <= t); i = i + s
        if same:
            m = i < j; i, j = i[m], j[m]
        rows.append(i.astype(np.int64)); cols.append(j.astype(np.int64))
    return np.concatenate(rows), np.concatenate(cols)

def components(n, edge_lists):
    """Connected components of an undirected graph with n nodes; edge_lists = [(rows, cols), ...]."""
    r = np.concatenate([np.asarray(e[0], np.int64) for e in edge_lists] + [np.arange(n)])
    c = np.concatenate([np.asarray(e[1], np.int64) for e in edge_lists] + [np.arange(n)])
    return connected_components(coo_matrix((np.ones(len(r), np.int8), (r, c)), shape=(n, n)), directed=False)[1]

def stratified_group_split(y, groups, fracs, seed):
    """Stratified GROUP split -> position arrays (train, val, test). A group never straddles splits."""
    y, groups = np.asarray(y), np.asarray(groups); idx = np.arange(len(y))
    n_test = max(2, int(round(1 / fracs[2])))
    n_val = max(2, int(round((1 - fracs[2]) / fracs[1])))
    if len(np.unique(groups)) < max(n_test, n_val) + 1:           # too few groups for k-fold -> plain group shuffle split
        gss = GroupShuffleSplit(n_splits=1, test_size=fracs[2], random_state=seed)
        rest, test = next(gss.split(idx, y, groups))
        gss = GroupShuffleSplit(n_splits=1, test_size=fracs[1] / (1 - fracs[2]), random_state=seed + 1)
        r_tr, r_va = next(gss.split(rest, y[rest], groups[rest]))
        return rest[r_tr], rest[r_va], test
    rest, test = next(StratifiedGroupKFold(n_splits=n_test, shuffle=True, random_state=seed).split(idx, y, groups))
    r_tr, r_va = next(StratifiedGroupKFold(n_splits=n_val, shuffle=True, random_state=seed + 1)
                      .split(rest, y[rest], groups[rest]))
    return rest[r_tr], rest[r_va], test

def fingerprint(items):
    return hashlib.md5("\n".join(sorted(map(str, items))).encode()).hexdigest()

def array_batch_ds(arr, rows, y, batch, shuffle):
    """tf.data pipeline that gathers uint8 images arr[rows] batch-by-batch (arr may be a disk memmap) + prefetch."""
    rows = np.asarray(rows, np.int64); y = np.asarray(y, np.float32)
    shape = tuple(arr.shape[1:]); yshape = tuple(y.shape[1:])
    def _get(pos):
        r = rows[pos]; o = np.argsort(r)
        out = np.empty((len(r),) + shape, np.uint8); out[o] = arr[r[o]]
        return out, y[pos]
    ds = tf.data.Dataset.range(len(rows))
    if shuffle:
        ds = ds.shuffle(len(rows), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch).map(lambda p: tf.numpy_function(_get, [p], [tf.uint8, tf.float32]), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda a, b: (tf.ensure_shape(tf.cast(a, tf.float32), (None,) + shape),
                              tf.ensure_shape(b, (None,) + yshape)), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def subsample_rows(y, n, seed):
    """Stratified random subset of positions (used only to speed up backbone screening)."""
    if len(y) <= n: return np.arange(len(y))
    rng = np.random.default_rng(seed); out = []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        out.append(rng.choice(idx, max(1, int(round(n * len(idx) / len(y)))), replace=False))
    return np.sort(np.concatenate(out))

onehot = lambda y, n: np.eye(n, dtype=np.float32)[np.asarray(y, int)]

def capped_class_weights(y_train, n_classes, cap):
    """'balanced' weights from the TRAINING labels only, capped; classes absent from training get weight 1.0."""
    present = np.unique(y_train)
    w = compute_class_weight("balanced", classes=present, y=y_train)
    out = {c: 1.0 for c in range(n_classes)}
    out.update({int(c): float(min(v, cap)) for c, v in zip(present, w)})
    return out

print("6.3 ready: grouping + split + tf.data helpers.")

In [ ]:
# ============================================================
# SECTION 6.4 — MODEL BUILDING + TWO-PHASE TRAINING (shared by CNN 1, 2, 3)
# ============================================================
BACKBONES = {
    "EfficientNetV2B0": dict(fn=keras.applications.EfficientNetV2B0, prep="builtin", kwargs={"include_preprocessing": True}),
    "EfficientNetB0":   dict(fn=keras.applications.EfficientNetB0,   prep="builtin", kwargs={}),
    "MobileNetV2":      dict(fn=keras.applications.MobileNetV2,      prep="tf",      kwargs={}),
    "DenseNet121":      dict(fn=keras.applications.DenseNet121,      prep="torch",   kwargs={}),
    "ResNet50":         dict(fn=keras.applications.ResNet50,         prep="caffe",   kwargs={}),
}
PREP_DESCRIPTION = {
    "builtin": "raw RGB [0,255]; rescaling/normalisation is built into the Keras EfficientNet backbone",
    "tf": "raw RGB [0,255] -> Rescaling(1/127.5, offset=-1) inside the model (MobileNetV2 convention)",
    "torch": "raw RGB [0,255] -> Rescaling(1/255) -> Normalization(ImageNet mean/std) inside the model (DenseNet convention)",
    "caffe": "raw RGB [0,255] -> fixed 1x1 conv inside the model: RGB->BGR + ImageNet mean subtraction (ResNet50 convention)",
}

def augmentation_block(cfg, name="augmentation"):
    """Mild, dentally plausible augmentation. Active only when training=True; identity at inference.
    It lives INSIDE the model, so validation/test images are never augmented."""
    L = []
    if cfg.get("hflip"):     L.append(layers.RandomFlip("horizontal", seed=SEED, dtype="float32"))
    if cfg.get("rotate"):    L.append(layers.RandomRotation(cfg["rotate"], fill_mode="reflect", seed=SEED, dtype="float32"))
    if cfg.get("translate"): L.append(layers.RandomTranslation(cfg["translate"], cfg["translate"], fill_mode="reflect",
                                                               seed=SEED, dtype="float32"))
    if cfg.get("zoom"):      L.append(layers.RandomZoom(cfg["zoom"], fill_mode="reflect", seed=SEED, dtype="float32"))
    if cfg.get("brightness"):L.append(layers.RandomBrightness(cfg["brightness"], value_range=(0.0, 255.0), seed=SEED, dtype="float32"))
    if cfg.get("contrast"):  L.append(layers.RandomContrast(cfg["contrast"], seed=SEED, dtype="float32"))
    if cfg.get("shear") and hasattr(layers, "RandomShear"):
        # factor passed as a (0, s) range: this form survives .keras save/load (a bare float does not in some Keras 3 versions)
        L.append(layers.RandomShear(x_factor=(0.0, cfg["shear"]), y_factor=(0.0, cfg["shear"]), fill_mode="reflect",
                                    seed=SEED, dtype="float32"))
    return keras.Sequential(L, name=name)

def preprocessing_block(kind, x):
    if kind == "tf":
        return layers.Rescaling(1 / 127.5, offset=-1.0, dtype="float32", name="prep_scale_pm1")(x)
    if kind == "torch":
        x = layers.Rescaling(1 / 255.0, dtype="float32", name="prep_scale_01")(x)
        return layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2],
                                    dtype="float32", name="prep_imagenet_norm")(x)
    if kind == "caffe":                  # RGB->BGR + mean subtraction as a frozen 1x1 conv (serialisable, no Lambda layer)
        conv = layers.Conv2D(3, 1, use_bias=True, trainable=False, dtype="float32", name="prep_caffe_bgr")
        y = conv(x)
        conv.set_weights([np.eye(3, dtype=np.float32)[::-1].T.reshape(1, 1, 3, 3).copy(),
                          -np.array([103.939, 116.779, 123.68], np.float32)])
        return y
    return x

def build_classifier(backbone, input_hw, n_out, cfg, name):
    """RGB [0,255] -> augmentation (train only) -> backbone-specific normalisation -> ImageNet backbone (BatchNorm in
    inference mode) -> GAP -> Dropout -> Dense -> Dropout -> sigmoid/softmax (float32). Returns (model, backbone)."""
    spec = BACKBONES[backbone]
    inp = keras.Input((*input_hw, 3), name="image_rgb_0_255")
    x = augmentation_block(cfg["AUG"])(inp)
    x = preprocessing_block(spec["prep"], x)
    base = spec["fn"](include_top=False, weights=BACKBONE_WEIGHTS, input_shape=(*input_hw, 3), **spec["kwargs"])
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    if cfg.get("HEAD_BN"):
        x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dropout(cfg["DROPOUT"], name="dropout_1")(x)
    x = layers.Dense(cfg["DENSE"], activation="relu", name="dense_head")(x)
    x = layers.Dropout(cfg["DROPOUT"], name="dropout_2")(x)
    out = layers.Dense(n_out, activation="sigmoid" if n_out == 1 else "softmax", dtype="float32", name="probabilities")(x)
    return keras.Model(inp, out, name=name), base

def unfreeze_top(base, frac):
    """Unfreeze the top `frac` of the backbone layers; BatchNormalization layers stay frozen."""
    base.trainable = True
    cut = int(len(base.layers) * (1 - frac))
    for i, l in enumerate(base.layers):
        l.trainable = i >= cut and not isinstance(l, layers.BatchNormalization)
    return sum(int(np.prod(w.shape)) for w in base.trainable_weights)

def fit_phase(model, train_ds, val_ds, epochs, lr, cfg, loss, metrics_fn, monitor, class_weight, ckpt, patience,
              init_best=None, initial_epoch=0, cosine=False):
    """cosine=True: 1 warm-up epoch then cosine decay (used for fine-tuning; ReduceLROnPlateau is then not used,
    because it cannot modify a schedule). Otherwise constant LR + ReduceLROnPlateau."""
    steps = int(train_ds.cardinality().numpy()) if int(train_ds.cardinality().numpy()) > 0 else 100
    if cosine:
        lr_ = keras.optimizers.schedules.CosineDecay(lr / 10, decay_steps=max(1, epochs * steps), alpha=0.02,
                                                     warmup_target=lr, warmup_steps=steps)
    else:
        lr_ = lr
    model.compile(optimizer=keras.optimizers.AdamW(lr_, weight_decay=cfg["WEIGHT_DECAY"], clipnorm=1.0),
                  loss=loss, metrics=metrics_fn())
    cbs = [ModelCheckpoint(ckpt, monitor=monitor, mode="max", save_best_only=True, save_weights_only=True,
                           initial_value_threshold=init_best, verbose=0),
           EarlyStopping(monitor=monitor, mode="max", patience=patience, restore_best_weights=True, verbose=1)]
    if not cosine:
        cbs.append(ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=max(2, patience // 2),
                                     min_lr=1e-7, verbose=1))
    return model.fit(train_ds, validation_data=val_ds, initial_epoch=initial_epoch, epochs=initial_epoch + epochs,
                     class_weight=class_weight, callbacks=cbs, verbose=2)

def screen_backbones(candidates, build_fn, train_ds, val_ds, cfg, loss, metrics_fn, monitor, class_weight, tag):
    """Short frozen-backbone run per candidate; returns (best_name, table). Uses VALIDATION data only — never test."""
    if len(candidates) == 1:
        return candidates[0], pd.DataFrame()
    res = []
    for name in candidates:
        keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
        try:
            model, _ = build_fn(name)
        except Exception as e:
            print(f"  {name}: could not be built ({e}) — skipped"); continue
        t0 = time.time()
        h = fit_phase(model, train_ds, val_ds, cfg["SCREEN_EPOCHS"], cfg["LR_A"], cfg, loss, metrics_fn, monitor,
                      class_weight, f"{CKPT_DIR}/screen_{tag}_{name}.weights.h5", patience=cfg["SCREEN_EPOCHS"])
        res.append({"backbone": name, f"best_{monitor}": float(np.max(h.history[monitor])),
                    "params_M": round(model.count_params() / 1e6, 2), "seconds": round(time.time() - t0, 1)})
        del model
    table = pd.DataFrame(res).sort_values(f"best_{monitor}", ascending=False).reset_index(drop=True)
    print(f"\nBackbone screening ({tag}) — VALIDATION {monitor} (test set not used):\n" + table.to_string(index=False))
    return table.iloc[0]["backbone"], table

def train_phase_a(model, train_ds, val_ds, cfg, loss, metrics_fn, monitor, class_weight, ckpt):
    """PHASE 1: frozen ImageNet backbone, classification head only. Best validation weights restored."""
    h = fit_phase(model, train_ds, val_ds, cfg["EPOCHS_A"], cfg["LR_A"], cfg, loss, metrics_fn, monitor,
                  class_weight, ckpt, cfg["PATIENCE_A"])
    hist = {k: list(map(float, v)) for k, v in h.history.items()}
    model.load_weights(ckpt)
    info = {"epochs_a": len(hist["loss"]), "best_a": float(np.max(hist[monitor])), "epochs_b": 0,
            "fine_tuned": False, "monitor": monitor}
    print(f"Phase 1 done: {info['epochs_a']} epochs, best validation {monitor} {info['best_a']:.4f}")
    return hist, info

def train_phase_b(model, base, train_ds, val_ds, cfg, loss, metrics_fn, monitor, class_weight, ckpt, hist, info):
    """PHASE 2: controlled fine-tuning. The backbone is unfrozen GRADUALLY (cfg["UNFREEZE_STEPS"], e.g. top 25 % then
    top 50 %; default = [UNFREEZE_FRAC]); BatchNorm layers always stay frozen; low LR (cosine decay if PHASE2_COSINE).
    Weights of a step are only kept if they beat everything before on validation (checkpoint threshold)."""
    if not (CAN_FINE_TUNE and cfg["EPOCHS_B"] > 0):
        print("Phase 2 skipped (no GPU or EPOCHS_B = 0)."); return hist, info
    best = info["best_a"]; start = info["epochs_a"]; info["steps_b"] = []
    for step, frac in enumerate(cfg.get("UNFREEZE_STEPS") or [cfg["UNFREEZE_FRAC"]], 1):
        n_tr = unfreeze_top(base, frac)
        lr = cfg["LR_B"] / (2 ** (step - 1))
        print(f"Phase 2.{step}: top {frac:.0%} of the backbone trainable ({n_tr:,} weights, BatchNorm frozen), LR {lr:.1e}"
              + (" with warm-up + cosine decay" if cfg.get("PHASE2_COSINE") else ""))
        h = fit_phase(model, train_ds, val_ds, cfg["EPOCHS_B"], lr, cfg, loss, metrics_fn, monitor, class_weight, ckpt,
                      cfg["PATIENCE_B"], init_best=best, initial_epoch=start, cosine=bool(cfg.get("PHASE2_COSINE")))
        for k, v in h.history.items():
            hist.setdefault(k, []).extend(map(float, v))
        n = len(h.history["loss"]); start += n
        sb = float(np.max(h.history[monitor])); info["steps_b"].append({"unfreeze_frac": frac, "epochs": n, "best": sb, "lr": lr})
        if os.path.exists(ckpt):
            model.load_weights(ckpt)             # best weights so far (this step only overwrote them if it improved)
        best = max(best, sb)
    info.update(epochs_b=start - info["epochs_a"], fine_tuned=True, best_b=max(s["best"] for s in info["steps_b"]))
    info["kept_phase"] = "2 (fine-tuned)" if info["best_b"] > info["best_a"] else "1 (frozen backbone)"
    print(f"Best validation {monitor}: Phase 1 {info['best_a']:.4f} | Phase 2 {info['best_b']:.4f} -> kept Phase {info['kept_phase']}")
    return hist, info

def decide(stage):
    """True = train. 'auto' retrains if no saved model exists or it was trained on a different split (leakage guard)."""
    mode, path = TRAIN_MODE[stage], MODEL_PATHS[stage]
    if mode == "train":
        return True
    saved_fp = read_json(MODEL_META_PATHS[stage]).get("split_fingerprint")
    if mode == "load":
        if not os.path.exists(path):
            raise FileNotFoundError(f"TRAIN_MODE['{stage}']='load' but {path} does not exist")
        if saved_fp != SPLIT_FP[stage]:
            print(f"WARNING: saved {stage} model was trained on a different split — its test metrics may be optimistic.")
        return False
    if not os.path.exists(path):
        return True
    if saved_fp != SPLIT_FP[stage]:
        print(f"Saved {stage} model was trained on a different split -> retraining (prevents test leakage).")
        return True
    return False

def load_keras(path):
    return keras.models.load_model(path, compile=False)

print("6.4 ready: build_classifier, train_phase_a (frozen), train_phase_b (fine-tune), screen_backbones.")

In [ ]:
# ============================================================
# SECTION 6.5 — EVALUATION + PLOTS
# ============================================================
def plot_history(hist, info, keys, title):
    """Training vs validation curves; dotted line = start of fine-tuning."""
    keys = [k for k in keys if k in hist]
    if not keys:
        print("No training history available (model was loaded, not trained, in this or an earlier run)."); return
    fig, ax = plt.subplots(1, len(keys), figsize=(6 * len(keys), 3.8))
    for a, k in zip(np.atleast_1d(ax), keys):
        a.plot(hist.get(k, []), label="train"); a.plot(hist.get("val_" + k, []), label="validation")
        if info.get("fine_tuned"): a.axvline(info["epochs_a"] - 0.5, ls=":", c="grey", label="fine-tuning starts")
        a.set_title(f"{title}: {k}"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

def history_summary(hist, info):
    """Train/val accuracy + loss at the epoch whose weights were kept (best validation monitor)."""
    mon = info.get("monitor", "val_loss")
    if mon not in hist:
        return {}
    e = int(np.argmax(hist[mon])) if mon != "val_loss" else int(np.argmin(hist[mon]))
    out = {"best_epoch": e + 1}
    for k in ("accuracy", "loss"):
        for p in ("", "val_"):
            if p + k in hist: out[("train_" if not p else "val_") + k] = float(hist[p + k][e])
    return out

def clean_eval(model, ds_by_split, loss_fn):
    """Accuracy + loss of the FINAL model on each split, WITHOUT augmentation/dropout (training=False)."""
    rows = {}
    for s, (ds, y) in ds_by_split.items():
        p = model.predict(ds, verbose=0)
        pt = tf.convert_to_tensor(p, tf.float32)
        if p.shape[1] == 1:
            acc = float(((p.ravel() >= 0.5).astype(int) == y).mean())
            loss = float(keras.losses.BinaryCrossentropy()(tf.convert_to_tensor(y[:, None], tf.float32), pt).numpy())
        else:
            acc = float((p.argmax(1) == y).mean())
            loss = float(keras.losses.CategoricalCrossentropy()(tf.convert_to_tensor(onehot(y, p.shape[1])), pt).numpy())
        rows[s] = {"accuracy": acc, "loss": loss, "n": int(len(y))}
    return rows

def evaluate_multiclass(y_true, y_pred, proba, class_names):
    """All multiclass metrics from ACTUAL predictions. Macro averages use the classes present in y_true
    (a class with no test sample has undefined recall and is reported separately); zero_division=0 everywhere."""
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    labels_all = list(range(len(class_names)))
    present = sorted(set(y_true.tolist()))
    R = {"n": int(len(y_true)), "classes_in_test": [class_names[c] for c in present],
         "classes_missing_in_test": [class_names[c] for c in labels_all if c not in present]}
    R["accuracy"] = float(accuracy_score(y_true, y_pred))
    rec_present = [float(((y_pred == c) & (y_true == c)).sum() / max((y_true == c).sum(), 1)) for c in present]
    R["balanced_accuracy"] = float(np.mean(rec_present)) if present else float("nan")
    for avg in ("macro", "weighted"):
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=present, average=avg, zero_division=0)
        R[f"{avg}_precision"], R[f"{avg}_recall"], R[f"{avg}_f1"] = float(p), float(r), float(f)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels_all, zero_division=0)
    R["per_class"] = {class_names[c]: {"precision": float(p[c]), "recall": float(r[c]) if s[c] else None,
                                       "f1": float(f[c]) if s[c] else None, "support": int(s[c]),
                                       "predicted": int((y_pred == c).sum())} for c in labels_all}
    R["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=labels_all).tolist()
    R["classification_report"] = classification_report(y_true, y_pred, labels=present,
                                                       target_names=[class_names[c] for c in present],
                                                       digits=4, zero_division=0)
    # one-vs-rest ROC-AUC / PR-AUC: only for classes with >=1 positive AND >=1 negative test sample
    auc, ap, sup = {}, {}, {}
    if proba is not None:
        for c in present:
            yt = (y_true == c).astype(int)
            if 0 < yt.sum() < len(yt):
                auc[class_names[c]] = float(roc_auc_score(yt, proba[:, c]))
                ap[class_names[c]] = float(average_precision_score(yt, proba[:, c]))
                sup[class_names[c]] = int(yt.sum())
    w = np.array([sup[k] for k in auc], float)
    R["ovr_roc_auc_per_class"], R["ovr_pr_auc_per_class"] = auc, ap
    R["ovr_roc_auc_macro"] = float(np.mean(list(auc.values()))) if auc else None
    R["ovr_roc_auc_weighted"] = float(np.dot(list(auc.values()), w) / w.sum()) if auc else None
    R["ovr_pr_auc_macro"] = float(np.mean(list(ap.values()))) if ap else None
    R["ovr_pr_auc_weighted"] = float(np.dot(list(ap.values()), w) / w.sum()) if ap else None
    return R

def print_results_table(R, title):
    print(title + "\n" + "-" * len(title))
    for label, k in [("Accuracy", "accuracy"), ("Balanced Accuracy", "balanced_accuracy"),
                     ("Macro Precision", "macro_precision"), ("Macro Recall", "macro_recall"), ("Macro F1", "macro_f1"),
                     ("Weighted Precision", "weighted_precision"), ("Weighted Recall", "weighted_recall"),
                     ("Weighted F1", "weighted_f1")]:
        print(f"{label + ':':<20}{R[k]:.4f}")
    if R.get("ovr_roc_auc_macro") is not None:
        print(f"{'OvR ROC-AUC (macro):':<20}{R['ovr_roc_auc_macro']:.4f}   (weighted {R['ovr_roc_auc_weighted']:.4f})")
        print(f"{'OvR PR-AUC (macro):':<20}{R['ovr_pr_auc_macro']:.4f}   (weighted {R['ovr_pr_auc_weighted']:.4f})")
    print(f"Test samples: {R['n']}")
    if R["classes_missing_in_test"]:
        print("Classes with NO test sample (excluded from macro averages):", R["classes_missing_in_test"])

def per_class_table(R, min_support=None):
    df = pd.DataFrame(R["per_class"]).T[["precision", "recall", "f1", "support", "predicted"]]
    if min_support:
        df["reliable?"] = np.where(df["support"] >= min_support, "yes", f"NO (<{min_support} test samples)")
    return df

def plot_confusion(cm, class_names, title):
    cm = np.asarray(cm); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    n = len(class_names); ticks = list(range(n))
    fig, axes = plt.subplots(1, 2, figsize=(max(12, 2.2 * n + 4), max(5, 0.9 * n + 2)))
    for ax, M, t, fmt in [(axes[0], cm, "counts", "{:d}"), (axes[1], cmn, "row-normalised (= recall)", "{:.2f}")]:
        ax.imshow(M, cmap="Blues")
        ax.set_xticks(ticks, class_names, rotation=45, ha="right"); ax.set_yticks(ticks, class_names)
        for i in ticks:
            for j in ticks:
                ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=9,
                        color="white" if M[i, j] > M.max() / 2 else "black")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{title} — {t}")
    plt.tight_layout(); plt.show()

def plot_ovr_curves(y_true, proba, class_names, title):
    y_true = np.asarray(y_true, int)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5)); drawn = 0
    for c, n in enumerate(class_names):
        yt = (y_true == c).astype(int)
        if 0 < yt.sum() < len(yt):
            f_, t_, _ = roc_curve(yt, proba[:, c]); pr_, rc_, _ = precision_recall_curve(yt, proba[:, c])
            ax[0].plot(f_, t_, label=f"{n} (AUC {roc_auc_score(yt, proba[:, c]):.3f}, n={yt.sum()})")
            ax[1].plot(rc_, pr_, label=f"{n} (AP {average_precision_score(yt, proba[:, c]):.3f})"); drawn += 1
    if not drawn:
        plt.close(fig); print("ROC/PR curves not applicable: no class has both positive and negative test samples."); return
    ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")
    ax[0].set_title(f"{title} — one-vs-rest ROC"); ax[0].legend(fontsize=8)
    ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title(f"{title} — one-vs-rest precision-recall")
    ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

def show_examples(images, titles, suptitle, ncols=6):
    n = len(images)
    if n == 0: return
    nrow = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrow, ncols, figsize=(2.6 * ncols, 2.8 * nrow))
    for ax in np.atleast_1d(axes).ravel(): ax.axis("off")
    for ax, im, t in zip(np.atleast_1d(axes).ravel(), images, titles):
        ax.imshow(np.asarray(im).astype(np.uint8)); ax.set_title(t, fontsize=8)
    plt.suptitle(suptitle); plt.tight_layout(); plt.show()

print("6.5 ready: evaluate_multiclass (accuracy, balanced accuracy, macro/weighted P/R/F1, per-class, CM, OvR ROC/PR-AUC).")

In [ ]:
# ============================================================
# SECTION 6.6 — PREDICTION FUNCTIONS + MODEL LOADING (work after a runtime restart)
# ============================================================
PIPELINE = {"models": {}, "meta": {}, "unavailable": {}}   # filled by the training cells or by the load_* functions
MODEL_KEYS = ("stage1", "localizer", "gvb", "icdas")          # required
OPTIONAL_MODEL_KEYS = ("fdi",)                                 # Stage 2: reported as UNAVAILABLE when not trained

def _stage(name):
    if name not in PIPELINE["models"]:
        raise RuntimeError(f"The '{name}' model is not loaded. Train it in its section, or run load_all_models() (Section 75).")
    return PIPELINE["models"][name], PIPELINE["meta"][name]

def _forward(model, x_u8):
    x = tf.convert_to_tensor(np.asarray(x_u8, np.float32)[None])
    return np.asarray(model(x, training=False), np.float64)[0]

# ---------- loading ----------
def load_stage(stage):
    """Load one saved model (.keras) + its metadata JSON into the registry — no retraining."""
    meta = read_json(MODEL_META_PATHS[stage]) or read_metadata().get(stage, {})
    if not meta:
        raise FileNotFoundError(f"No metadata for '{stage}' ({MODEL_META_PATHS[stage]}) — train that stage first.")
    path = MODEL_PATHS[stage] if os.path.exists(MODEL_PATHS[stage]) else meta.get("model_path")
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"Model file for '{stage}' not found ({MODEL_PATHS[stage]}).")
    PIPELINE["models"][stage] = load_keras(path); PIPELINE["meta"][stage] = meta
    print(f"  loaded {stage:9} <- {path}")
    return PIPELINE["models"][stage]

load_stage1_model   = lambda: load_stage("stage1")      # CNN 1
load_localizer      = lambda: load_stage("localizer")   # cavity / lesion localization
load_gvblack_model  = lambda: load_stage("gvb")         # CNN 2
load_icdas_model    = lambda: load_stage("icdas")       # CNN 3
load_fdi_detector   = lambda: load_stage("fdi")         # Stage 2 FDI tooth detector

def load_all_models(force=False):
    for k in MODEL_KEYS:
        if force or k not in PIPELINE["models"]:
            load_stage(k)
    for k in OPTIONAL_MODEL_KEYS:          # the cascade runs without Stage 2 and reports it as UNAVAILABLE (never invents numbers)
        if force or k not in PIPELINE["models"]:
            if os.path.exists(MODEL_PATHS[k]) and os.path.exists(MODEL_META_PATHS[k]):
                load_stage(k); PIPELINE["unavailable"].pop(k, None)
            else:
                PIPELINE["unavailable"][k] = PIPELINE["unavailable"].get(k) or f"no saved model at {MODEL_PATHS[k]} (train it in Section 24)"
                print(f"  {k:9} not available — {PIPELINE['unavailable'][k]}")
    print("All models available:", list(PIPELINE["models"]))

# ---------- CNN 1 ----------
def stage1_probability(rgb):
    m, meta = _stage("stage1")
    return float(_forward(m, s1_prepare(rgb, meta["image_size"])).ravel()[0])

def stage1_decide(p, meta, confidence_threshold=None):
    thr = meta["threshold"]; cthr = meta["confidence_threshold"] if confidence_threshold is None else confidence_threshold
    raw = "CARIES" if p >= thr else "HEALTHY"
    conf = p if raw == "CARIES" else 1 - p
    reasons = []
    if abs(p - thr) < meta["uncertain_margin"]:
        reasons.append(f"P(caries) {p:.3f} is within ±{meta['uncertain_margin']:.2f} of the validation threshold {thr:.3f}")
    if conf < cthr:
        reasons.append(f"confidence {conf * 100:.1f}% < {cthr * 100:.0f}%")
    return {"prediction": "UNCERTAIN" if reasons else raw, "leaning": raw, "confidence": float(conf), "p_caries": float(p),
            "threshold": float(thr), "reason": "; ".join(reasons) or None}

def predict_caries(image_path, confidence_threshold=None, verbose=True):
    """CNN 1 on a full dental image -> HEALTHY / CARIES / UNCERTAIN + confidence."""
    _, meta = _stage("stage1")
    out = stage1_decide(stage1_probability(load_rgb(image_path)), meta, confidence_threshold)
    if verbose:
        print(f"Prediction: {out['prediction']}")
        print(f"Confidence: {out['confidence'] * 100:.2f}%" + (f"  (leaning {out['leaning']})" if out["reason"] else ""))
        if out["reason"]: print("  Reason:", out["reason"])
        print("  " + DISCLAIMER)
    return out

# ---------- crop configuration (from trained model metadata, or config defaults before training) ----------
def _gvb_input_cfg():
    m = PIPELINE["meta"].get("gvb", {})
    return m.get("image_size", GVB["IMG_SIZE"]), m.get("context_pad", GVB["DEFAULT_PAD"]), m.get("min_crop_px", GVB["MIN_CROP_PX"])

def _icdas_input_cfg():
    m = PIPELINE["meta"].get("icdas", {})
    w = m.get("inference_window")
    if w is None:
        mult, scale = ICDAS["DEFAULT_WINDOW"]
        fw, fh = globals().get("TOOTH_FRAC", (0.33, 0.55))
        w = {"mult": mult, "min_scale": scale, "min_frac_w": scale * fw, "min_frac_h": scale * fh, "source": "config default"}
    return m.get("image_size", ICDAS["IMG_SIZE"]), w

# ---------- CNN 2 ----------
def gvb_probabilities(rgb, box=None):
    """box = normalised x1y1x2y2 inside rgb (from the localizer). box=None: rgb IS the lesion crop."""
    m, meta = _stage("gvb")
    size, pad, min_px = _gvb_input_cfg()
    x = letterbox(rgb, size, size) if box is None else gvb_prepare_crop(rgb, np.asarray(box, np.float32), pad, size, min_px)
    return tta_forward(m, x[None], meta.get("tta", "none"))[0]

def predict_gvblack(cavity_crop_path, box=None, confidence_threshold=None, verbose=True):
    """CNN 2: G.V. Black class of one lesion. Pass a lesion-crop file (box=None), or a full image + normalised box."""
    _, meta = _stage("gvb")
    thr = meta["confidence_threshold"] if confidence_threshold is None else confidence_threshold
    p = gvb_probabilities(load_rgb(cavity_crop_path), box)
    k = int(np.argmax(p)); names = meta["class_names"]
    out = {"gvb_class": names[k], "display": gvb_short(names[k]), "confidence": float(p[k]),
           "uncertain": bool(p[k] < thr), "probabilities": {gvb_short(n): float(v) for n, v in zip(names, p)}}
    if verbose:
        if out["uncertain"]:
            print(f"Prediction: UNCERTAIN  (best guess {gvb_short(names[k])}, {p[k] * 100:.1f}% < {thr * 100:.0f}%)")
        else:
            print(f"G.V. Black Class: {gvb_short(names[k])}\nConfidence: {p[k] * 100:.1f}%")
        print("All G.V. Black classes:")
        for n, v in zip(names, p):
            print(f"  {gvb_short(n):<10}: {v * 100:5.1f}%")
        print("  " + DISCLAIMER)
    return out

# ---------- CNN 3 ----------
def icdas_probabilities(rgb, box=None):
    """box=None: rgb IS the tooth/lesion crop. With a box: a tooth-scale window around the lesion is cut first."""
    m, meta = _stage("icdas")
    size, window = _icdas_input_cfg()
    x = icdas_prepare(rgb, size) if box is None else icdas_prepare_from_box(rgb, np.asarray(box, np.float32), size, window)
    return tta_forward(m, x[None], meta.get("tta", "none"))[0]

def predict_icdas(cavity_crop_path, box=None, confidence_threshold=None, verbose=True):
    """CNN 3: ICDAS score of one tooth / lesion crop (or a full image + normalised lesion box)."""
    _, meta = _stage("icdas")
    thr = meta["confidence_threshold"] if confidence_threshold is None else confidence_threshold
    p = icdas_probabilities(load_rgb(cavity_crop_path), box)
    scores = meta["scores"]; k = int(np.argmax(p)); s = int(scores[k])
    out = {"icdas": s, "confidence": float(p[k]), "uncertain": bool(p[k] < thr),
           "probabilities": {f"ICDAS {sc}": float(v) for sc, v in zip(scores, p)},
           "unreliable_class": s in meta.get("unreliable_scores", [])}
    if verbose:
        if out["uncertain"]:
            print(f"Prediction: UNCERTAIN  (best guess ICDAS {s}, {p[k] * 100:.1f}% < {thr * 100:.0f}%)")
        else:
            print(f"Predicted ICDAS: {s}\nConfidence: {p[k] * 100:.1f}%")
        for sc in range(7):
            if sc in scores:
                print(f"  ICDAS {sc}: {p[scores.index(sc)] * 100:5.1f}%")
            else:
                print(f"  ICDAS {sc}:   n/a  (no ICDAS {sc} class in icdascl.zip" +
                      ("; sound teeth are handled by CNN 1" if sc == 0 else "") + ")")
        if out["unreliable_class"]:
            print(f"  WARNING: ICDAS {s} had very few training crops — this class is not reliable.")
        print("  " + DISCLAIMER)
    return out

# ---------- Stage 2: FDI tooth detection (separate model; its boxes are TEETH, never lesions) ----------
def detect_teeth(rgb, score_threshold=None):
    """Stage 2 on a full RGB image -> list of teeth sorted left to right, each with box, detection score, FDI number and its
    probability. Number probability < class_confidence_threshold -> number uncertain; the same number predicted for two
    teeth -> the less confident one is uncertain too (each FDI number exists once per mouth)."""
    m, meta = _stage("fdi")
    thr = meta["score_threshold"] if score_threshold is None else float(score_threshold)
    raw = _forward(m, loc_prepare(rgb, meta["cache_side"], meta["input_size"]))
    boxes, obj, probs = decode_fdi_detections(raw, thr, meta["nms_iou"], meta["max_detections"])
    H, W = rgb.shape[:2]; names = meta["class_names"]; cthr = meta["class_confidence_threshold"]; teeth = []
    for b, s, p in zip(boxes, obj, probs):
        k = int(np.argmax(p)); top = np.argsort(-p)[:3]
        teeth.append({"box_norm": [float(v) for v in b], "box_xyxy_px": list(to_px(b, W, H)), "detection_score": float(s),
                      "fdi": names[k], "fdi_confidence": float(p[k]), "fdi_uncertain": bool(p[k] < cthr),
                      "fdi_unreliable_class": names[k] in meta.get("unreliable_classes", []),
                      "fdi_top3": {names[j]: float(p[j]) for j in top},
                      "fdi_note": f"number probability {p[k]:.0%} < {cthr:.0%}" if p[k] < cthr else None})
    by_num = defaultdict(list)
    for i, t in enumerate(teeth): by_num[t["fdi"]].append(i)
    for num, idx in by_num.items():
        best = max(idx, key=lambda i: teeth[i]["fdi_confidence"])
        for i in idx:
            if i != best:
                teeth[i]["fdi_uncertain"] = True; teeth[i]["fdi_note"] = f"FDI {num} is also predicted for a more confident tooth"
    teeth.sort(key=lambda t: (t["box_norm"][0] + t["box_norm"][2]) / 2)
    for i, t in enumerate(teeth, 1):
        t["tooth_id"] = i
        t["fdi_display"] = t["fdi"] if not t["fdi_uncertain"] else f"Unknown / uncertain (best guess {t['fdi']})"
    return teeth

# ---------- localization + crop generation ----------
def localize(rgb, score_threshold=None):
    m, meta = _stage("localizer")
    thr = meta["score_threshold"] if score_threshold is None else float(score_threshold)
    raw = _forward(m, loc_prepare(rgb, meta["cache_side"], meta["input_size"]))
    return decode_detections(raw, thr, meta["nms_iou"], meta["max_detections"])

def generate_cavity_crops(rgb, score_threshold=None, boxes=None, scores=None):
    """Localize lesions (or use given normalised boxes) -> for EACH lesion: G.V. Black context crop + ICDAS tooth-scale crop."""
    if boxes is None:
        boxes, scores = localize(rgb, score_threshold)
    scores = np.ones(len(boxes), np.float32) if scores is None else scores
    H, W = rgb.shape[:2]
    gsize, pad, min_px = _gvb_input_cfg(); isize, window = _icdas_input_cfg()
    min_box = PIPELINE["meta"].get("localizer", {}).get("min_box_px", LOC["MIN_BOX_PX"])
    out = []
    for b, sc in zip(boxes, scores):
        x1, y1, x2, y2 = to_px(b, W, H)
        if x2 - x1 < min_box or y2 - y1 < min_box:
            continue
        out.append({"box_norm": [float(v) for v in b], "box_xyxy_px": [x1, y1, x2, y2], "localizer_score": float(sc),
                    "gvb_crop": gvb_prepare_crop(rgb, np.asarray(b, np.float32), pad, gsize, min_px),
                    "icdas_window_px": list(icdas_window_box(b, W, H, window["mult"], window["min_frac_w"], window["min_frac_h"])),
                    "icdas_crop": icdas_prepare_from_box(rgb, np.asarray(b, np.float32), isize, window)})
    return out

def save_cavity_crops(image_path, out_dir=f"{PREDICTION_DIR}/cavity_crops", score_threshold=None):
    """Writes <name>_lesionN_gvb.jpg and <name>_lesionN_icdas.jpg — usable with predict_gvblack / predict_icdas."""
    os.makedirs(out_dir, exist_ok=True); base = stem(image_path); paths = []
    for i, c in enumerate(generate_cavity_crops(load_rgb(image_path), score_threshold), 1):
        pg, pi = f"{out_dir}/{base}_lesion{i}_gvb.jpg", f"{out_dir}/{base}_lesion{i}_icdas.jpg"
        cv2.imwrite(pg, cv2.cvtColor(c["gvb_crop"], cv2.COLOR_RGB2BGR)); cv2.imwrite(pi, cv2.cvtColor(c["icdas_crop"], cv2.COLOR_RGB2BGR))
        paths.append((pg, pi))
    return paths

print("6.6 ready: predict_caries, detect_teeth, predict_gvblack, predict_icdas, generate_cavity_crops, load_all_models.")

### 6.7 Stage 3 / Stage 4 optimisation + error-analysis toolkit
* **Feature screening (validation only).** Every candidate backbone × input size (× context padding for G.V. Black) is run **frozen**,
  without augmentation, on the training and validation crops; a small head (Dense-256 + dropout) is trained to convergence on the
  frozen features and scored on **validation**. This ranks architectures far more reliably than a 3-epoch fine-tune, and costs only one
  forward pass per image and candidate. The same features then compare class-balancing strategies and label smoothing.
* **Selection score** = mean of validation accuracy and validation macro-F1 (accuracy alone would favour the majority class,
  macro-F1 alone would ignore how often the model is right). Validation loss is reported too. The **test set is never used** for any choice.
* **Oversampling** repeats minority-class *training* rows (`n_c' = n_max^(1-p) · n_c^p`); every copy is augmented differently inside the
  model = targeted augmentation. Validation / test are never resampled.
* **Test-time augmentation (TTA)** — averaging predictions over a few deterministic views — is kept **only if it helps on validation**;
  the chosen variant is stored in the model metadata and used identically for the test evaluation and for every later prediction.
* **Error analysis** — correct / incorrect examples with confidences, most-confused class pairs, confidence of errors, recall vs training
  support, error rate by crop size and view, high-confidence errors (possible label noise), adjacent-score confusions for ICDAS.

In [ ]:
# ============================================================
# SECTION 6.7 — STAGE 3/4 OPTIMISATION + ERROR-ANALYSIS TOOLKIT
# ============================================================
def selection_score(acc, macro_f1):
    return 0.5 * acc + 0.5 * macro_f1

def rank_candidates(df):
    """Validation ranking: score (½ acc + ½ macro-F1) desc; ties -> higher macro-F1, lower validation loss, fewer parameters."""
    if not len(df): return df
    d = df.copy(); d["_p"] = d["params_M"] if "params_M" in d else 0.0
    d["_s"] = d["score"].round(6); d["_f"] = d["val_macro_f1"].round(6)
    d = d.sort_values(["_s", "_f", "val_loss", "_p"], ascending=[False, False, True, True])
    return d.drop(columns=["_p", "_s", "_f"]).reset_index(drop=True)

def macro_f1_present(y_true, y_pred):
    return float(f1_score(y_true, y_pred, labels=sorted(set(np.asarray(y_true).tolist())), average="macro", zero_division=0))

def oversample_rows(rows, y, power, seed):
    """Training rows with minority classes repeated: n_c' = n_max^(1-power) * n_c^power (power=1 -> no change)."""
    rows, y = np.asarray(rows), np.asarray(y); rng = np.random.default_rng(seed); n_max = np.bincount(y).max(); out = []
    for c in np.unique(y):
        r = rows[y == c]; target = int(round(n_max ** (1 - power) * len(r) ** power))
        out.append(np.concatenate([r, rng.choice(r, max(target - len(r), 0), replace=True)]))
    return rng.permutation(np.concatenate(out))

def backbone_features(backbone, X, rows, size, batch=64):
    """Frozen ImageNet backbone (+ its own normalisation), global-average-pooled features, NO augmentation."""
    keras.backend.clear_session()
    spec = BACKBONES[backbone]
    inp = keras.Input((size, size, 3))
    base = spec["fn"](include_top=False, weights=BACKBONE_WEIGHTS, input_shape=(size, size, 3), pooling="avg", **spec["kwargs"])
    fx = keras.Model(inp, base(preprocessing_block(spec["prep"], inp), training=False))
    rows = np.asarray(rows)
    ds = array_batch_ds(X, rows, np.zeros((len(rows), 1), np.float32), batch, False).map(lambda a, b: a)
    return fx.predict(ds, verbose=0).astype(np.float32)

def fit_feature_head(Ftr, ytr, Fva, yva, n_classes, class_weight=None, oversample_power=None, label_smoothing=0.0,
                     epochs=80, seed=SEED):
    """Small head on frozen features; early stopping on validation loss. Returns validation metrics + probabilities."""
    keras.utils.set_random_seed(seed)
    idx = oversample_rows(np.arange(len(ytr)), ytr, oversample_power, seed) if oversample_power is not None else np.arange(len(ytr))
    inp = keras.Input((Ftr.shape[1],))
    x = layers.Dropout(0.3)(layers.Dense(256, activation="relu")(layers.BatchNormalization()(inp)))
    m = keras.Model(inp, layers.Dense(n_classes, activation="softmax", dtype="float32")(x))
    m.compile(optimizer=keras.optimizers.AdamW(1e-3, weight_decay=1e-4),
              loss=keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing))
    m.fit(Ftr[idx], onehot(ytr[idx], n_classes), validation_data=(Fva, onehot(yva, n_classes)), epochs=epochs, batch_size=64,
          class_weight=class_weight, verbose=0,
          callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)])
    pv = m.predict(Fva, verbose=0); yp = pv.argmax(1)
    acc, mf1 = float((yp == yva).mean()), macro_f1_present(yva, yp)
    vloss = float(keras.losses.CategoricalCrossentropy()(onehot(yva, n_classes), pv).numpy())
    return {"val_accuracy": acc, "val_macro_f1": mf1, "val_loss": vloss, "score": selection_score(acc, mf1)}, pv

def screen_features(candidates, sizes, get_data, n_classes, class_weight, tag):
    """candidates x sizes (x anything get_data encodes). get_data(size) -> (X, train_rows, y_train, val_rows, y_val).
    Returns (table sorted by validation score, cached features)."""
    rows, feats = [], {}
    for size in sizes:
        X, rtr, ytr, rva, yva = get_data(size)
        for bb in candidates:
            t0 = time.time()
            try:
                Ftr, Fva = backbone_features(bb, X, rtr, size), backbone_features(bb, X, rva, size)
            except Exception as e:
                print(f"  {bb}@{size}: skipped ({type(e).__name__}: {e})"); continue
            res, _ = fit_feature_head(Ftr, ytr, Fva, yva, n_classes, class_weight=class_weight)
            feats[(bb, size)] = (Ftr, ytr, Fva, yva)
            rows.append({"backbone": bb, "size": size, **res, "params_M": round(BACKBONE_PARAMS_M.get(bb, float("nan")), 1),
                         "seconds": round(time.time() - t0, 1)})
            print(f"  {tag}: {bb:16} @ {size}px  val acc {res['val_accuracy']:.4f} | val macro-F1 {res['val_macro_f1']:.4f} | "
                  f"val loss {res['val_loss']:.4f}")
    return rank_candidates(pd.DataFrame(rows)), feats

BACKBONE_PARAMS_M = {"EfficientNetV2B0": 5.9, "EfficientNetB0": 4.0, "MobileNetV2": 2.3, "DenseNet121": 7.0, "ResNet50": 23.6}

def screen_training_recipe(Ftr, ytr, Fva, yva, n_classes, balance_candidates, ls_candidates, cap, power):
    """Class balancing x label smoothing, compared on VALIDATION with the frozen features of the chosen backbone."""
    cw = capped_class_weights(ytr, n_classes, cap); rows = []
    for bal in balance_candidates:
        for ls in ls_candidates:
            res, _ = fit_feature_head(Ftr, ytr, Fva, yva, n_classes,
                                      class_weight=cw if "class_weight" in bal else None,
                                      oversample_power=power if "oversample" in bal else None, label_smoothing=ls)
            rows.append({"balancing": bal, "label_smoothing": ls, **res})
    return rank_candidates(pd.DataFrame(rows))

def majority_baseline(y_train, y_eval):
    maj = int(np.bincount(y_train).argmax()); yp = np.full(len(y_eval), maj)
    return {"majority_class": maj, "accuracy": float((yp == y_eval).mean()), "macro_f1": macro_f1_present(y_eval, yp)}

# ---------- test-time augmentation (identical for validation choice, test evaluation and inference) ----------
def tta_views(xb, variant):
    """xb: float32 tensor (B,S,S,3) in [0,255]. Deterministic views; no randomness."""
    views = [xb]
    if "hflip" in variant:
        views.append(tf.image.flip_left_right(xb))
    if "zoom" in variant:
        s = int(xb.shape[1]); views.append(tf.image.resize(tf.image.central_crop(xb, 0.9), (s, s), method="bilinear"))
    return views

def tta_forward(model, X_u8, variant="none"):
    xb = tf.convert_to_tensor(np.asarray(X_u8, np.float32))
    return np.mean([np.asarray(model(v, training=False), np.float64) for v in tta_views(xb, variant)], axis=0)

def predict_rows(model, X, rows, batch=64, variant="none"):
    rows = np.asarray(rows); out = []
    for s in range(0, len(rows), batch):
        out.append(tta_forward(model, np.asarray(X[rows[s:s + batch]]), variant))
    return np.concatenate(out) if out else np.zeros((0, model.output_shape[-1]))

def choose_tta(model, X, rows, y, candidates, batch=64):
    """TTA variant chosen on VALIDATION ('none' kept unless another variant is strictly better)."""
    res = []
    for v in candidates:
        p = predict_rows(model, X, rows, batch, v); yp = p.argmax(1)
        res.append({"tta": v, "val_accuracy": float((yp == y).mean()), "val_macro_f1": macro_f1_present(y, yp),
                    "score": selection_score(float((yp == y).mean()), macro_f1_present(y, yp))})
    t = pd.DataFrame(res)
    best = t.iloc[0]
    for _, r in t.iloc[1:].iterrows():
        if r["score"] > best["score"] + 1e-9: best = r
    return best["tta"], t

# ---------- error analysis ----------
def error_analysis(title, X, idx, y_true, proba, class_names, extra=None, train_counts=None, ordinal_scores=None,
                   n_show=8, high_conf=0.80):
    """Evidence for WHY a model errs. extra: DataFrame aligned with idx (e.g. crop_px, view)."""
    y_true = np.asarray(y_true); y_pred = proba.argmax(1); conf = proba.max(1); ok = y_pred == y_true
    out = {"n": int(len(y_true)), "n_errors": int((~ok).sum())}
    print("=" * 70 + f"\n{title} — ERROR ANALYSIS (test set, no tuning)\n" + "=" * 70)
    print(f"Correct {int(ok.sum())} / {len(ok)} | mean confidence: correct {conf[ok].mean() if ok.any() else float('nan'):.3f}, "
          f"incorrect {conf[~ok].mean() if (~ok).any() else float('nan'):.3f}")
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist([conf[ok], conf[~ok]], bins=20, range=(0, 1), label=["correct", "incorrect"], stacked=False)
    ax.set_xlabel("confidence (max softmax)"); ax.set_ylabel("test samples"); ax.legend(); ax.set_title(f"{title}: confidence")
    plt.tight_layout(); plt.show()
    lab = lambda k: f"true {class_names[y_true[k]]}\npred {class_names[y_pred[k]]} ({conf[k]:.0%})"
    cor = np.where(ok)[0][np.argsort(-conf[ok])][:n_show]; err = np.where(~ok)[0][np.argsort(-conf[~ok])][:n_show]
    show_examples([X[idx[k]] for k in cor], [lab(k) for k in cor], f"{title}: most confident CORRECT predictions", ncols=8)
    show_examples([X[idx[k]] for k in err], [lab(k) for k in err], f"{title}: most confident INCORRECT predictions", ncols=8)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    pairs = [(cm[i, j], i, j) for i in range(len(class_names)) for j in range(len(class_names)) if i != j and cm[i, j] > 0]
    pairs.sort(reverse=True)
    conf_tab = pd.DataFrame([{"true": class_names[i], "predicted as": class_names[j], "count": int(n),
                              "% of true class": f"{n / max(cm[i].sum(), 1):.1%}"} for n, i, j in pairs[:10]])
    print("\nMost confused class pairs:\n" + (conf_tab.to_string(index=False) if len(conf_tab) else "  none"))
    out["top_confusions"] = conf_tab.to_dict("records")
    pc = pd.DataFrame({"test support": cm.sum(1), "recall": np.diag(cm) / np.maximum(cm.sum(1), 1),
                       "mean conf": [conf[y_true == c].mean() if (y_true == c).any() else np.nan for c in range(len(class_names))]},
                      index=class_names)
    if train_counts is not None:
        pc.insert(0, "train samples", train_counts)
    print("\nPer-class recall vs data:\n" + pc.round(3).to_string())
    if train_counts is not None and (pc["test support"] > 0).sum() >= 3:
        from scipy.stats import spearmanr
        m = pc["test support"] > 0
        rho = spearmanr(pc.loc[m, "train samples"], pc.loc[m, "recall"]).correlation
        out["spearman_train_count_vs_recall"] = float(rho)
        print(f"Spearman(train samples, recall) = {rho:.2f}"
              + ("  -> recall rises with training data: accuracy is limited by the amount of data per class" if rho > 0.6 else
                 "  -> no strong link between training data and recall"))
    if extra is not None and len(extra):
        e = extra.reset_index(drop=True).assign(error=~ok)
        for col in e.columns:
            if col == "error": continue
            if pd.api.types.is_numeric_dtype(e[col]) and e[col].nunique() > 4:
                q = pd.qcut(e[col], 4, duplicates="drop")
                t = e.groupby(q, observed=True)["error"].agg(["size", "mean"]).rename(columns={"size": "n", "mean": "error rate"})
                print(f"\nError rate by {col} quartile:\n" + t.round(3).to_string())
                out[f"error_by_{col}"] = {str(k): float(v) for k, v in t["error rate"].items()}
            else:
                t = e.groupby(col)["error"].agg(["size", "mean"]).rename(columns={"size": "n", "mean": "error rate"})
                print(f"\nError rate by {col}:\n" + t.round(3).to_string())
                out[f"error_by_{col}"] = {str(k): float(v) for k, v in t["error rate"].items()}
    hc = np.where(~ok & (conf >= high_conf))[0]
    out["high_confidence_errors"] = int(len(hc))
    print(f"\nHigh-confidence errors (confidence >= {high_conf:.0%}): {len(hc)} of {int((~ok).sum())} errors "
          "-> candidates for label noise or ambiguous images; worth a manual check (labels are NOT changed here).")
    if ordinal_scores is not None:
        st, sp = np.asarray(ordinal_scores)[y_true], np.asarray(ordinal_scores)[y_pred]
        d = np.abs(st - sp)[~ok]
        out["share_errors_adjacent"] = float((d == 1).mean()) if len(d) else None
        print(f"\nOf {len(d)} errors: {int((d == 1).sum())} adjacent (±1), {int((d == 2).sum())} off by 2, {int((d >= 3).sum())} off by >= 3")
        adj = []
        for a, b in zip(ordinal_scores[:-1], ordinal_scores[1:]):
            n_ab = int(((st == a) & (sp == b)).sum()); n_ba = int(((st == b) & (sp == a)).sum())
            adj.append({"pair": f"ICDAS {a} vs {b}", "true lower -> pred higher": n_ab, "true higher -> pred lower": n_ba,
                        "total": n_ab + n_ba, "test crops of the two scores": int(((st == a) | (st == b)).sum())})
        at = pd.DataFrame(adj); out["adjacent_confusions"] = at.to_dict("records")
        print("Adjacent-score confusions:\n" + at.to_string(index=False))
    ind = []
    if train_counts is not None and out.get("spearman_train_count_vs_recall", 0) > 0.6: ind.append("insufficient data for rare classes")
    if train_counts is not None and np.max(train_counts) / max(np.min(train_counts), 1) > 10: ind.append("strong class imbalance")
    if ordinal_scores is not None and (out.get("share_errors_adjacent") or 0) > 0.6: ind.append("visual similarity of adjacent scores")
    if (~ok).sum() and len(hc) / (~ok).sum() > 0.25: ind.append("possible label noise / ambiguous images (many confident errors)")
    for k, v in out.items():
        if k.startswith("error_by_crop") and len(v) >= 2 and list(v.values())[0] > 1.5 * max(list(v.values())[-1], 1e-9):
            ind.append("small / poor crops err more often")
    out["indicators"] = ind
    print("\nINDICATORS (derived from the numbers above; evidence, not proof):", ind or ["no single dominant cause"])
    return out

print("6.7 ready: screen_features, screen_training_recipe, oversample_rows, choose_tta / predict_rows, error_analysis.")

<br>

---
# PART 4 — DATASET INSPECTION (paths, indexing, shared leakage-safe split)

## 7. Dataset path configuration
All data is copied to Colab's **local disk** and extracted **once per session** (an extraction marker stores the ZIP name + size, so a new
ZIP version re-extracts automatically). Every path used later is verified to exist here.

**Stage 4 uses `icdascl.zip` only.** The old `ICD2_ICDAS_0_to_6_Classified.zip` is refused, and the corrected data is extracted to a new
folder (`/content/data/icdascl`) so an old extraction can never be picked up by mistake.

In [ ]:
# ============================================================
# SECTION 7.1 — LOCATE + STAGE ALL DATASETS (once per session)
# ============================================================
def find_icdas_zip():
    hits = []
    for pat in ICDAS_ZIP_PATTERNS:
        hits += sorted(glob.glob(pat))
    hits = [h for h in dict.fromkeys(hits) if os.path.basename(h) not in REFUSED_ICDAS_ZIPS]
    return hits[0] if hits else None

with stopwatch("data staging"):
    GVB_ROOT = os.path.join(DATA_ROOT, "gvblack")
    GVB_ZIP = first_existing(GVB_ZIP_CANDIDATES) or upload_file("Upload the G.V. Black annotation ZIP (lowoccyolo)...")
    stage_zip(GVB_ZIP, GVB_ROOT)

    ICDAS_ZIP = find_icdas_zip() or upload_file("Upload the CORRECTED ICDAS dataset: icdascl.zip", must_start_with="icdascl")
    assert os.path.basename(ICDAS_ZIP) not in REFUSED_ICDAS_ZIPS, "The old ICDAS ZIP must not be used."
    assert os.path.basename(ICDAS_ZIP).lower().startswith("icdascl"), f"Stage 4 must use icdascl.zip, got {ICDAS_ZIP}"
    ICDAS_ROOT = os.path.join(DATA_ROOT, "icdascl")
    stage_zip(ICDAS_ZIP, ICDAS_ROOT)

    CARIES_DIR = os.path.join(DATA_ROOT, "caries")
    cz = first_existing(CARIES_ZIP_CANDIDATES + [f"{PACKED_DIR}/caries.zip"])
    cdir = first_existing(CARIES_DIR_CANDIDATES, "dir")
    if cz:
        stage_zip(cz, CARIES_DIR)
    elif cdir:
        pack_drive_folder_once(cdir, CARIES_DIR, f"{PACKED_DIR}/caries.zip")
    else:
        stage_zip(upload_file("Upload the caries ZIP (Comprehensive visual index of carious lesions)..."), CARIES_DIR)

    HEALTHY_LOCAL = os.path.join(DATA_ROOT, "healthy")
    if not os.path.exists(os.path.join(HEALTHY_LOCAL, ".extracted_ok")):
        packed = first_existing([f"{PACKED_DIR}/healthy.zip"] + HEALTHY_ZIP_CANDIDATES)
        if packed:
            stage_zip(packed, HEALTHY_LOCAL)
        else:
            src = HEALTHY_DIR or first_existing(HEALTHY_DIR_CANDIDATES, "dir") or \
                  search_drive_for_folder({"Teeth or Dental Image Dataset", "healthy", "Healthy"})
            if src is None:
                raise FileNotFoundError(
                    "Healthy dataset not found.\n"
                    f"Open {HEALTHY_DRIVE_LINK}\n"
                    "→ right-click the folder → Organize → Add shortcut → MyDrive/dentalcaries/healthy,\n"
                    "then re-run this cell (or set HEALTHY_DIR in Section 3).")
            pack_drive_folder_once(src, HEALTHY_LOCAL, f"{PACKED_DIR}/healthy.zip")
    else:
        print("  healthy images already on local disk ->", HEALTHY_LOCAL)
    HEALTHY_DIR = HEALTHY_LOCAL

DATA_SOURCES = {"Healthy images (Stage 1)": (HEALTHY_DIR, None), "Caries images (Stage 1)": (CARIES_DIR, cz or cdir),
                "G.V. Black ZIP (localizer + Stage 3)": (GVB_ROOT, GVB_ZIP), "ICDAS ZIP (Stage 4)": (ICDAS_ROOT, ICDAS_ZIP)}
print("\nVERIFIED DATA SOURCES (local disk)")
for k, (v, z) in DATA_SOURCES.items():
    assert os.path.isdir(v), f"missing: {v}"
    print(f"  {k:38}: {v}" + (f"   <- {os.path.basename(z)}" if z else ""))

In [ ]:
# ============================================================
# SECTION 7.2 — DATASET INDEX BUILDERS (parse once, cache, reuse everywhere)
# ============================================================
ICDAS_FOLDER_RE = re.compile(r"^(?:icdas)?[\s_\-]*([0-6])$", re.I)
TOOTH_RE = re.compile(r"^(.*?)__tooth(\d+)$", re.I)

# ---------------- ICDAS: structure straight from the ZIP file ----------------
def inspect_icdas_zip(zip_path):
    """Programmatic structure report of the ZIP itself (no assumptions about its layout)."""
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.filename.startswith("__MACOSX") and "/." not in "/" + i.filename]
    files = [i for i in infos if not i.is_dir()]
    dirs = Counter(os.path.dirname(i.filename) for i in files)
    return {"zip": os.path.basename(zip_path), "zip_mb": os.path.getsize(zip_path) / 1e6, "n_entries": len(infos),
            "n_files": len(files), "uncompressed_mb": sum(i.file_size for i in files) / 1e6,
            "top_level": sorted({i.filename.split("/")[0] for i in infos}),
            "extensions": dict(Counter(os.path.splitext(i.filename)[1].lower() or "<none>" for i in files)),
            "files_per_folder": dict(sorted(dirs.items())), "max_depth": max((i.filename.count("/") for i in files), default=0),
            "non_image_files": [i.filename for i in files if not i.filename.lower().endswith(IMG_EXT)]}

def find_icdas_structure(root):
    """Detect ICDAS class folders (folder name = score), the crop manifest and the no-caries list, wherever they are."""
    class_dirs, manifest, nocaries = defaultdict(list), [], []
    for dp, dn, fn in os.walk(root):
        dn[:] = sorted(d for d in dn if not d.startswith((".", "__MACOSX")))
        m = ICDAS_FOLDER_RE.match(os.path.basename(dp))
        if m and any(f.lower().endswith(IMG_EXT) for f in fn):
            class_dirs[int(m.group(1))].append(dp)
        for f in fn:
            fl = f.lower()
            if fl.endswith(".csv") and "manifest" in fl: manifest.append(os.path.join(dp, f))
            if fl.endswith(".txt") and ("without_caries" in fl or "no_caries" in fl): nocaries.append(os.path.join(dp, f))
    return dict(sorted(class_dirs.items())), (manifest[0] if manifest else None), (nocaries[0] if nocaries else None)

def read_nocaries_list(path):
    """images_without_caries.txt: folder<TAB>filename<TAB>identical copies (';'-separated)."""
    rows = []
    if path:
        for line in open(path, encoding="utf-8", errors="replace"):
            if not line.strip() or line.startswith("#"): continue
            p = line.rstrip("\n").split("\t")
            rows.append({"folder": p[0].strip(), "filename": p[1].strip() if len(p) > 1 else "",
                         "copies": [c.strip() for c in (p[2] if len(p) > 2 else "").split(";") if c.strip()]})
    return pd.DataFrame(rows, columns=["folder", "filename", "copies"])

def build_icdas_index(root, size):
    """One record per crop FILE on disk (+ manifest columns). Decodes every crop ONCE into a uint8 memmap
    (square letterbox, the Stage 4 contract) and computes MD5 + dHash. Cached on local disk."""
    class_dirs, manifest_path, nocaries_path = find_icdas_structure(root)
    assert class_dirs, f"No ICDAS class folders (ICDAS_<k> or <k>) with images found under {root}"
    recs = []
    for score, dirs in class_dirs.items():
        for d in dirs:
            for f in sorted(os.listdir(d)):
                if f.lower().endswith(IMG_EXT) and not f.startswith("."):
                    recs.append({"path": os.path.join(d, f), "folder": os.path.basename(d), "score": score, "file": f})
    df = pd.DataFrame(recs)
    mf = pd.read_csv(manifest_path) if manifest_path else pd.DataFrame()
    base = os.path.dirname(manifest_path) if manifest_path else root
    df["rel"] = [os.path.relpath(p, base).replace(os.sep, "/") for p in df["path"]]
    if len(mf):
        mf["on_disk"] = mf["crop_file"].isin(set(df["rel"]))
        df = df.merge(mf.rename(columns={"icdas": "manifest_icdas"}), left_on="rel", right_on="crop_file", how="left")
    for c in ("source_image", "source_folder", "identical_copies_in_input", "manifest_icdas"):
        if c not in df: df[c] = np.nan
    fallback = [(TOOTH_RE.match(stem(f)).group(1) if TOOTH_RE.match(stem(f)) else stem(f)) for f in df["file"]]
    df["source_id"] = [str(s) if isinstance(s, str) else fb for s, fb in zip(df["source_image"], fallback)]
    df["source_folder"] = df["source_folder"].fillna("")
    df["copies"] = [[c for c in str(v).split(";") if c.strip()] if isinstance(v, str) else [] for v in df["identical_copies_in_input"]]

    cache_npy = os.path.join(CACHE_DIR, f"icdas_crops_{size}.npy"); cache_csv = os.path.join(CACHE_DIR, f"icdas_crops_{size}.csv")
    sig = fingerprint(df["rel"].tolist() + [str(os.path.getsize(ICDAS_ZIP))])
    old = pd.read_csv(cache_csv, dtype={"dhash_hex": str, "md5": str, "error": str, "sig": str}) if os.path.exists(cache_csv) and os.path.exists(cache_npy) else None
    if old is not None and len(old) == len(df) and old["sig"].iloc[0] == sig:
        X = np.load(cache_npy, mmap_mode="r")
        for c in ("ok", "W", "H", "md5", "dhash_hex", "error"): df[c] = old[c].values
        df["error"] = df["error"].fillna("")
        print(f"ICDAS decode cache reused: {X.shape} (zero JPEG decodes)")
    else:
        X = np.lib.format.open_memmap(cache_npy, mode="w+", dtype=np.uint8, shape=(len(df), size, size, 3))
        info = [None] * len(df)
        def _one(i):
            p = df.at[i, "path"]; r = {"ok": False, "W": 0, "H": 0, "md5": file_md5(p), "dhash_hex": "00" * 8, "error": ""}
            try:
                with Image.open(p) as im: im.verify()             # structural check
                rgb = load_rgb(p)                                  # full decode
                X[i] = icdas_prepare(rgb, size)
                r.update(ok=True, W=rgb.shape[1], H=rgb.shape[0], dhash_hex=dhash64(X[i]).tobytes().hex())
            except Exception as e:
                r["error"] = f"{type(e).__name__}: {e}"[:200]
            info[i] = r
        with stopwatch("ICDAS decode + cache"):
            with ThreadPoolExecutor(DECODE_WORKERS) as ex:
                list(ex.map(_one, range(len(df))))
        X.flush()
        for c in ("ok", "W", "H", "md5", "dhash_hex", "error"): df[c] = [r[c] for r in info]
        df[["ok", "W", "H", "md5", "dhash_hex", "error"]].assign(sig=sig).to_csv(cache_csv, index=False)
        X = np.load(cache_npy, mmap_mode="r")
    df["ok"] = df["ok"].astype(bool)
    return df, X, {"class_dirs": class_dirs, "manifest_path": manifest_path, "manifest": mf,
                   "nocaries_path": nocaries_path, "nocaries": read_nocaries_list(nocaries_path)}

# ---------------- G.V. Black: images + YOLO-format text files (read as plain numbers) ----------------
ROMAN = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
def read_gvb_mapping_files(root):
    """Class-id -> name from classes.txt / data.yaml / notes.json if the ZIP has one; else None."""
    found = {}
    for dp, _, fn in os.walk(root):
        for f in fn:
            fl = f.lower(); p = os.path.join(dp, f)
            try:
                if fl == "classes.txt":
                    found[p] = {i: l.strip() for i, l in enumerate(open(p).read().splitlines()) if l.strip()}
                elif fl in ("data.yaml", "data.yml"):
                    import yaml
                    names = (yaml.safe_load(open(p)) or {}).get("names")
                    found[p] = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in (names or {}).items()}
                elif fl == "notes.json":
                    cats = json.load(open(p)).get("categories", [])
                    found[p] = {int(c["id"]): c["name"] for c in cats}
            except Exception as e:
                print("  could not read mapping file", p, e)
    return found

def roman_of(name):
    m = re.search(r"(?:class|cl|gvb|g\.?v\.?\s*black)?[\s_\-]*\b(VI|IV|V|I{1,3})\b", str(name), re.I)
    return m.group(1).upper() if m else None

def build_gvb_index(root, n_classes):
    all_f = [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if not f.startswith(".")]
    imgs = sorted(f for f in all_f if f.lower().endswith(IMG_EXT))
    labs = sorted(f for f in all_f if f.lower().endswith(".txt") and os.path.basename(f).lower() != "classes.txt")
    lab_lookup = {stem(f): f for f in labs}
    ann, malformed, class_ids_seen = {}, [], Counter()
    for s, p in lab_lookup.items():
        boxes, classes = [], []
        for i, line in enumerate(open(p)):
            if not line.strip(): continue
            q = line.strip().split()
            try:
                c = float(q[0]); x, y, w, h = map(float, q[1:5]); ok = len(q) == 5 and c == int(c)
            except (ValueError, IndexError):
                ok = False
            if ok: class_ids_seen[int(c)] += 1
            if not ok or not (0 <= int(c) < n_classes) or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                malformed.append((p, i, line.strip())); continue
            boxes.append([max(x - w / 2, 0.0), max(y - h / 2, 0.0), min(x + w / 2, 1.0), min(y + h / 2, 1.0)]); classes.append(int(c))
        ann[s] = (np.array(boxes, np.float32).reshape(-1, 4), np.array(classes, np.int32))
    pool = []
    for p in imgs:
        s = stem(p); b, c = ann.get(s, (np.zeros((0, 4), np.float32), np.zeros((0,), np.int32)))
        status = ("annotated" if len(b) else "empty_label") if s in ann else "no_label_file"
        pool.append({"id": s, "image_path": p, "gvb_status": status, "boxes": b, "classes": c,
                     "view": (re.match(r"[A-Za-z]+", strip_uuid(s)) or [None])[0], "orig_name": strip_uuid(s)})
    def _inspect(r):
        out = {"md5": file_md5(r["image_path"])}
        try:
            rgb = load_rgb(r["image_path"])
            out.update(readable=True, H=rgb.shape[0], W=rgb.shape[1], dhash=dhash64(s1_prepare(rgb, S1["IMG_PX"])))
        except Exception as e:
            out.update(readable=False, H=0, W=0, dhash=np.zeros(8, np.uint8), error=str(e))
        return out
    with stopwatch("G.V. Black photo inspection"):
        with ThreadPoolExecutor(DECODE_WORKERS) as ex:
            for r, info in zip(pool, ex.map(_inspect, pool)):
                r.update(info)
    orphan_labels = sorted(set(lab_lookup) - {stem(p) for p in imgs})
    return pool, {"n_images": len(imgs), "n_label_files": len(labs), "malformed": malformed, "orphan_labels": orphan_labels,
                  "class_ids_seen": dict(sorted(class_ids_seen.items())), "mapping_files": read_gvb_mapping_files(root)}

print("7.2 ready: inspect_icdas_zip, build_icdas_index, build_gvb_index.")

<br>

---
# PART 5 — STAGE 1: CNN 1 HEALTHY vs CARIES

---
# STAGE 1 — CNN MODEL 1: HEALTHY vs CARIES
Input: full dental image · Output: `0 → Healthy`, `1 → Caries`

## 8. Healthy dataset inspection

In [ ]:
# ============================================================
# SECTION 8 — HEALTHY DATASET INSPECTION
# ============================================================
with stopwatch("healthy scan"):
    healthy_paths = scan_images(HEALTHY_DIR)
assert healthy_paths, f"No healthy images found in {HEALTHY_DIR}"
h_folders = Counter(os.path.relpath(os.path.dirname(p), HEALTHY_DIR) for p in healthy_paths)
print(f"HEALTHY images: {len(healthy_paths):,}")
print("Extensions:", dict(Counter(os.path.splitext(p)[1].lower() for p in healthy_paths)))
print("Images per sub-folder:")
for k, v in sorted(h_folders.items(), key=lambda z: -z[1])[:20]:
    print(f"  {k:50}: {v:,}")
show_examples([load_rgb(p) for p in np.random.default_rng(SEED).choice(healthy_paths, min(6, len(healthy_paths)), replace=False)],
              ["healthy"] * 6, "Healthy examples")

## 9. Caries dataset inspection
The caries archive is organised by photographic view. `icdascl.zip` names the **source photograph** of every ICDAS crop and lists the
photographs in which **no carious tooth** was visible; both are matched to the caries archive **by file name** here (folder name breaks ties).

In [ ]:
# ============================================================
# SECTION 9 — CARIES DATASET INSPECTION (+ links to icdascl.zip)
# ============================================================
with stopwatch("caries scan"):
    caries_paths = scan_images(CARIES_DIR)
assert caries_paths, f"No caries images found in {CARIES_DIR}"
print(f"CARIES images: {len(caries_paths):,}")
print("Extensions:", dict(Counter(os.path.splitext(p)[1].lower() for p in caries_paths)))
print("Images per view folder:")
for k, v in sorted(Counter(os.path.basename(os.path.dirname(p)) for p in caries_paths).items()):
    print(f"  {k:30}: {v:,}")

ICDAS_ZIP_REPORT = inspect_icdas_zip(ICDAS_ZIP)                    # structure only, from the ZIP (details: Section 49)
_, _manifest_path, _nocaries_path = find_icdas_structure(ICDAS_ROOT)
_manifest_df = pd.read_csv(_manifest_path) if _manifest_path else pd.DataFrame(columns=["source_image"])
_nocaries_df = read_nocaries_list(_nocaries_path)
caries_names = Counter(os.path.basename(p).lower() for p in caries_paths)
src_names = set(_manifest_df["source_image"].astype(str).str.lower())
nc_names = set(_nocaries_df["filename"].str.lower())
print(f"\nicdascl.zip source photographs found in the caries archive : {len(src_names & set(caries_names))} / {len(src_names)}")
print(f"'images_without_caries' photographs found in the archive  : {len(nc_names & set(caries_names))} / {len(nc_names)}")
print(f"File names that occur more than once in the archive       : {sum(v > 1 for v in caries_names.values())}")
if src_names and len(src_names & set(caries_names)) < 0.5 * len(src_names):
    print("WARNING: most ICDAS source photographs are not in the caries archive — ICDAS crops will still be grouped by source "
          "photo, but cross-stage leakage control cannot use the archive.")
show_examples([load_rgb(p) for p in np.random.default_rng(SEED).choice(caries_paths, min(6, len(caries_paths)), replace=False)],
              ["caries archive"] * 6, "Caries-archive examples")

## 10. Duplicate / leakage check — ONE shared split for all models
1. **Decode once** (kept from your fast notebook): every Stage 1 image is decoded **once** with EXIF orientation, resized with the Stage 1
   contract and stored in a `uint8` memmap on local disk (copied to Drive, so later sessions skip JPEG decoding). Corrupted files are caught
   per file and excluded. The G.V. Black photos and the ICDAS crops are indexed the same way (MD5 + dHash, decoded once, cached).
2. **Exact / near duplicates** (dHash ≤ `DUP_HAMMING`) inside Stage 1: one copy kept; a photo present in *both* classes is ambiguous → dropped.
3. **Grouping graph.** Nodes: every readable Stage 1 image, every G.V. Black photograph, and every ICDAS source photograph that is not in the
   caries archive. Edges: dHash ≤ `S1_GROUP_HAMMING` (same-mouth bursts; ≤ `POOL_GROUP_HAMMING` among G.V. Black photos), identical MD5,
   ICDAS crop → its source photograph, source photograph → its listed **identical copies**, and ICDAS crops that are near-duplicates of each
   other (dHash ≤ `ICDAS["GROUP_HAMMING"]`). Connected components = **groups**. A group never straddles splits.
4. **Split** (70/15/15, before any augmentation): groups holding G.V. Black boxes or ICDAS crops are split first, **stratified jointly** by
   G.V. Black class and ICDAS score; the remaining Stage 1 groups are split stratified by class × view. Every model uses this split, so a test
   photograph is unseen by **all** models.

No patient IDs exist in any dataset: the split is **group-independent, not guaranteed patient-independent**.

In [ ]:
# ============================================================
# SECTION 10.1 — STAGE 1 DECODE-ONCE CACHE + CORRUPTED FILES
# ============================================================
index = pd.DataFrame([{"src": p, "label": 0} for p in healthy_paths] + [{"src": p, "label": 1} for p in caries_paths])
index["class_name"] = index["label"].map({0: "Healthy", 1: "Caries"})
index["view_folder"] = index["src"].map(lambda p: os.path.basename(os.path.dirname(p)))
N_RAW = len(index)

S1_PX = S1["IMG_PX"]
CACHE_PATH = os.path.join(CACHE_DIR, f"s1_exif_images_{S1_PX}.npy")
INDEX_PATH = os.path.join(CACHE_DIR, f"s1_exif_index_{S1_PX}.csv")
DRIVE_CACHE_PATH = os.path.join(DRIVE_CACHE_DIR, os.path.basename(CACHE_PATH)) if DRIVE_CACHE_DIR else None
DRIVE_INDEX_PATH = os.path.join(DRIVE_CACHE_DIR, os.path.basename(INDEX_PATH)) if DRIVE_CACHE_DIR else None
if DRIVE_CACHE_DIR and not os.path.exists(CACHE_PATH) and os.path.exists(DRIVE_CACHE_PATH) and os.path.exists(DRIVE_INDEX_PATH):
    print("Restoring decoded Stage 1 cache from Drive...")
    copy_big_file(DRIVE_INDEX_PATH, INDEX_PATH); copy_big_file(DRIVE_CACHE_PATH, CACHE_PATH)

cache_hit = False
if os.path.exists(CACHE_PATH) and os.path.exists(INDEX_PATH):
    old = pd.read_csv(INDEX_PATH, dtype={"dhash_hex": str, "md5": str, "error": str})
    cache_hit = len(old) == N_RAW and (old["src"].to_numpy() == index["src"].to_numpy()).all()

if cache_hit:
    index = old
    s1_images = np.load(CACHE_PATH, mmap_mode="r")
    print("Decoded Stage 1 cache reused:", s1_images.shape, "(zero JPEG decodes)")
else:
    with stopwatch("Stage 1 decode + cache"):
        s1_images = np.lib.format.open_memmap(CACHE_PATH, mode="w+", dtype=np.uint8, shape=(N_RAW, S1_PX, S1_PX, 3))
        info = [None] * N_RAW
        def _decode(i):
            p = index.at[i, "src"]
            rec = {"ok": False, "height": 0, "width": 0, "filesize": 0, "md5": "", "error": ""}
            try:
                rec["filesize"] = os.path.getsize(p)
                rec["md5"] = file_md5(p)
                rgb = load_rgb(p)
                s1_images[i] = s1_prepare(rgb, S1_PX)
                rec.update(ok=True, height=rgb.shape[0], width=rgb.shape[1])
            except Exception as e:
                rec["error"] = f"{type(e).__name__}: {e}"[:200]
            info[i] = rec
        with ThreadPoolExecutor(DECODE_WORKERS) as ex:
            for k, _ in enumerate(ex.map(_decode, range(N_RAW)), 1):
                if k % 2000 == 0: print(f"  {k:,}/{N_RAW:,}")
        s1_images.flush()
        index = pd.concat([index, pd.DataFrame(info)], axis=1)
        means = np.zeros((N_RAW, 3), np.float32); hashes = np.zeros((N_RAW, 8), np.uint8)
        for i in range(N_RAW):
            if index.at[i, "ok"]:
                im = np.asarray(s1_images[i]); means[i] = im.reshape(-1, 3).mean(0); hashes[i] = dhash64(im)
        index[["mean_r", "mean_g", "mean_b"]] = means
        index["dhash_hex"] = [h.tobytes().hex() for h in hashes]
        index.to_csv(INDEX_PATH, index=False)
        s1_images = np.load(CACHE_PATH, mmap_mode="r")
    if PERSIST_S1_CACHE_TO_DRIVE and DRIVE_CACHE_DIR:
        try:
            copy_big_file(INDEX_PATH, DRIVE_INDEX_PATH); copy_big_file(CACHE_PATH, DRIVE_CACHE_PATH)
            print("Decoded cache saved to Drive -> the next session skips decoding.")
        except Exception as e:
            print("Could not save the cache to Drive (not required):", e)

index["ok"] = index["ok"].astype(bool)
index["md5"] = index["md5"].fillna("").astype(str)
index["error"] = index["error"].fillna("").astype(str)

# An index cached by an OLDER notebook version stored MD5 only for caries images (healthy rows = ""). Empty fingerprints
# would make all those images look byte-identical. Fill every missing MD5 now (file hashing only, no re-decoding).
_missing = np.where(index["ok"].to_numpy() & (index["md5"].to_numpy() == ""))[0]
if len(_missing):
    with stopwatch(f"MD5 for {len(_missing):,} images missing it in the cached index"):
        with ThreadPoolExecutor(DECODE_WORKERS) as ex:
            index.loc[index.index[_missing], "md5"] = list(ex.map(file_md5, index["src"].to_numpy()[_missing]))
    index.to_csv(INDEX_PATH, index=False)
    if PERSIST_S1_CACHE_TO_DRIVE and DRIVE_CACHE_DIR:
        try: shutil.copyfile(INDEX_PATH, DRIVE_INDEX_PATH)
        except Exception as e: print("Could not update the cached index on Drive (not required):", e)
    print(f"Filled {len(_missing):,} missing MD5 fingerprints (cache from an older notebook version) and updated the cached index.")
assert (index.loc[index["ok"], "md5"] != "").all(), "every readable Stage 1 image must have an MD5"
s1_hash_all = np.stack([np.frombuffer(bytes.fromhex(h), np.uint8) for h in index["dhash_hex"]])
bad = index[~index["ok"]]
print(f"\nCorrupted / unreadable Stage 1 files: {len(bad)}" + (" -> excluded" if len(bad) else ""))
if len(bad): print(bad[["src", "error"]].head(10).to_string(index=False))
print("\nOriginal image sizes (most common per class):")
for nm, d in index[index.ok].groupby("class_name"):
    print(f"  {nm:8}:", ", ".join(f"{w}x{h}:{c}" for (w, h), c in Counter(zip(d.width, d.height)).most_common(4)))

In [ ]:
# ============================================================
# SECTION 10.2 — INDEX THE G.V. BLACK PHOTOS AND THE icdascl.zip CROPS (decode once)
# ============================================================
pool, GVB_INDEX_INFO = build_gvb_index(GVB_ROOT, NUM_GVB)
n_bad_pool = sum(not r["readable"] for r in pool)
pool = [r for r in pool if r["readable"]]
print(f"G.V. Black: {GVB_INDEX_INFO['n_images']} images, {GVB_INDEX_INFO['n_label_files']} label files, "
      f"{n_bad_pool} unreadable -> {len(pool)} photographs kept (details: Sections 28-30)")

icdas_df, X_icdas, ICDAS_INFO = build_icdas_index(ICDAS_ROOT, ICDAS["IMG_SIZE"])
ICDAS_HASH = np.stack([np.frombuffer(bytes.fromhex(h), np.uint8) for h in icdas_df["dhash_hex"]])
print(f"ICDAS: {len(icdas_df)} crop files indexed, {int((~icdas_df.ok).sum())} unreadable (details: Sections 49-54)")

In [ ]:
# ============================================================
# SECTION 10.3 — DUPLICATES, LEAKAGE CONTROL, SHARED GROUP SPLIT
# ============================================================
# ---------- 1) Stage 1 duplicates ----------
ok_rows = np.where(index["ok"].to_numpy())[0]
row2node = {int(r): i for i, r in enumerate(ok_rows)}
H1 = s1_hash_all[ok_rows]; lab1 = index["label"].to_numpy()[ok_rows]
with stopwatch("Stage 1 duplicate search"):
    dup_edges = hamming_edges(H1, H1, DUP_HAMMING, same=True)
    dup_comp = components(len(ok_rows), [dup_edges])
keep_mask = np.zeros(len(ok_rows), bool); n_cross_groups = n_cross_imgs = n_dup_removed = 0
for g, members in pd.Series(np.arange(len(ok_rows))).groupby(dup_comp):
    m = members.to_numpy()
    if len(np.unique(lab1[m])) > 1:
        n_cross_groups += 1; n_cross_imgs += len(m)          # same photo in both classes -> ambiguous -> drop all
    else:
        keep_mask[m[0]] = True; n_dup_removed += len(m) - 1
s1_rows = ok_rows[keep_mask]
print(f"Stage 1: {len(ok_rows):,} readable -> {len(s1_rows):,} after de-duplication "
      f"({n_dup_removed} duplicate copies removed; {n_cross_groups} duplicate groups spanned BOTH classes -> "
      f"{n_cross_imgs} images dropped as ambiguous)")

# ---------- 2) name links: ICDAS source photos / no-caries list -> caries-archive rows ----------
caries_ok = index.iloc[ok_rows]; caries_ok = caries_ok[caries_ok["label"] == 1]
name2rows, stem2rows = defaultdict(list), defaultdict(list)
for r, src, vf in zip(caries_ok.index, caries_ok["src"], caries_ok["view_folder"]):
    name2rows[os.path.basename(src).lower()].append((int(r), norm_folder(vf)))
    stem2rows[stem(src).lower()].append((int(r), norm_folder(vf)))
def rows_for(name, folder=""):
    name = str(name).lower()
    c = name2rows.get(name, []) if os.path.splitext(name)[1] else stem2rows.get(name, [])
    if len(c) > 1 and folder:
        same = [r for r, f in c if f == norm_folder(folder)]
        if same: return same
    return [r for r, _ in c]

n1, n2 = len(ok_rows), len(pool)
extra_nodes = {}
def nodes_of_name(name, folder=""):
    rows = rows_for(name, folder)
    if rows: return [row2node[r] for r in rows]
    key = (norm_folder(folder), str(name).lower())
    if key not in extra_nodes: extra_nodes[key] = n1 + n2 + len(extra_nodes)
    return [extra_nodes[key]]

edges = []
def add_edges(a, b):
    edges.append((np.asarray(a, np.int64), np.asarray(b, np.int64)))

crop_node = np.zeros(len(icdas_df), np.int64); n_src_linked = 0; n_copy_links = 0
for i, (src, fold, copies) in enumerate(zip(icdas_df["source_id"], icdas_df["source_folder"], icdas_df["copies"])):
    ns = nodes_of_name(src, fold)
    n_src_linked += ns[0] < n1
    crop_node[i] = ns[0]
    if len(ns) > 1: add_edges([ns[0]] * (len(ns) - 1), ns[1:])
    for cp in copies:
        nc = nodes_of_name(cp, fold); n_copy_links += 1
        add_edges([ns[0]] * len(nc), nc)
ic_pairs = hamming_edges(ICDAS_HASH, ICDAS_HASH, ICDAS["GROUP_HAMMING"], same=True)
ok_ic = icdas_df["ok"].to_numpy()
m_ = ok_ic[ic_pairs[0]] & ok_ic[ic_pairs[1]]
add_edges(crop_node[ic_pairs[0][m_]], crop_node[ic_pairs[1][m_]])
print(f"ICDAS crops whose source photo is in the caries archive: {n_src_linked}/{len(icdas_df)} | "
      f"identical-copy links: {n_copy_links} | near-duplicate crop pairs (dHash <= {ICDAS['GROUP_HAMMING']}): {int(m_.sum())}")

NOCARIES_ROWS = set()
for fn_, fold, copies in zip(ICDAS_INFO["nocaries"]["filename"], ICDAS_INFO["nocaries"]["folder"], ICDAS_INFO["nocaries"]["copies"]):
    rs = rows_for(fn_, fold)
    NOCARIES_ROWS.update(rs)
    for cp in copies:
        rc = rows_for(cp, fold); NOCARIES_ROWS.update(rc)
        if rs and rc: add_edges([row2node[rs[0]]] * len(rc), [row2node[r] for r in rc])
print(f"Caries-archive photographs listed as 'no carious tooth visible' (incl. identical copies): {len(NOCARIES_ROWS)}")

# ---------- 3) joint near-duplicate grouping: Stage 1 images + G.V. Black photos (+ exact MD5 links) ----------
P_hash = np.stack([r["dhash"] for r in pool]) if pool else np.zeros((0, 8), np.uint8)
ALL_H = np.concatenate([H1, P_hash])
with stopwatch("joint near-duplicate grouping"):
    e_all = hamming_edges(ALL_H, ALL_H, S1_GROUP_HAMMING, same=True)
    pr, pc = hamming_edges(P_hash, P_hash, POOL_GROUP_HAMMING, same=True)
# exact byte-identical files ACROSS ALL DATASETS (Stage 1 images, G.V. Black photos, ICDAS crops) -> same group
md5_nodes = defaultdict(set)
for r in ok_rows:
    if index.at[r, "md5"]: md5_nodes[index.at[r, "md5"]].add(row2node[int(r)])
for i, r in enumerate(pool):
    if r["md5"]: md5_nodes[r["md5"]].add(n1 + i)
for i, (h, ok) in enumerate(zip(icdas_df["md5"], icdas_df["ok"])):
    if ok and h: md5_nodes[h].add(int(crop_node[i]))
n_md5_links = 0
for nodes in md5_nodes.values():
    if len(nodes) > 1:
        nodes = sorted(nodes); add_edges([nodes[0]] * (len(nodes) - 1), nodes[1:]); n_md5_links += len(nodes) - 1
print(f"Byte-identical files linked into one group (all datasets): {n_md5_links}")
N_NODES = n1 + n2 + len(extra_nodes)
gid = components(N_NODES, [dup_edges, e_all, (pr + n1, pc + n1)] + edges)
gsize = Counter(gid)
print(f"Groups: {len(gsize):,} for {N_NODES:,} nodes | largest groups: {sorted(gsize.values(), reverse=True)[:5]}")
if max(gsize.values()) > 0.05 * N_NODES:
    print("WARNING: one group holds >5 % of all nodes; lower S1_GROUP_HAMMING if the split becomes unbalanced.")

# ---------- 4) ICDAS label conflicts among near-identical crops ----------
cf = hamming_edges(ICDAS_HASH, ICDAS_HASH, ICDAS["DUP_HAMMING"], same=True)
cf_m = ok_ic[cf[0]] & ok_ic[cf[1]]
cf_comp = components(len(icdas_df), [(cf[0][cf_m], cf[1][cf_m])])
conf_groups = icdas_df.assign(c=cf_comp).groupby("c")["score"].nunique()
icdas_df["conflict"] = icdas_df.assign(c=cf_comp)["c"].map(conf_groups > 1).values
print(f"ICDAS near-identical crops (dHash <= {ICDAS['DUP_HAMMING']}) with DIFFERENT labels: {int(icdas_df['conflict'].sum())} crops"
      + (" -> excluded from Stage 4" if ICDAS["DROP_CONFLICTING_DUPLICATES"] else " -> kept"))

# ---------- 5) the shared split ----------
SPLIT_NAMES = np.array(["train", "val", "test"])
group_split = {}
ann_y, ann_g = [], []
for s, g_, ok in zip(icdas_df["score"], gid[crop_node], ok_ic):
    if ok: ann_y.append(f"ICDAS{s}"); ann_g.append(g_)
for i, r in enumerate(pool):
    labs = [f"GVB{c}" for c in r["classes"]] or ["GVB_none"]
    ann_y += labs; ann_g += [gid[n1 + i]] * len(labs)
tr, va, te = stratified_group_split(ann_y, ann_g, SPLIT_FRACS, SEED)
for s_name, ids in zip(SPLIT_NAMES, (tr, va, te)):
    for i in ids: group_split[ann_g[i]] = s_name
free = np.array([i for i in range(n1) if gid[i] not in group_split])
if len(free):
    key = (index["class_name"] + " | " + index["view_folder"]).to_numpy()[ok_rows[free]]
    vc = Counter(key); cls_ = index["class_name"].to_numpy()[ok_rows[free]]
    key = np.where([vc[k] >= 10 for k in key], key, cls_ + " | RARE")
    tr, va, te = stratified_group_split(key, gid[free], SPLIT_FRACS, SEED)
    for s_name, ids in zip(SPLIT_NAMES, (tr, va, te)):
        for i in ids: group_split[gid[free[i]]] = s_name
rng_left = np.random.default_rng(SEED)
for g_ in set(gid) - set(group_split):                        # nodes with no image of their own (never used for training)
    group_split[g_] = rng_left.choice(SPLIT_NAMES, p=SPLIT_FRACS)
node_split = np.array([group_split[g_] for g_ in gid])

index["split"] = ""; index["group"] = -1
index.loc[ok_rows, "split"] = node_split[:n1]; index.loc[ok_rows, "group"] = gid[:n1]
for i, r in enumerate(pool):
    r["split"] = node_split[n1 + i]; r["group"] = int(gid[n1 + i])
icdas_df["split"] = node_split[crop_node]; icdas_df["group"] = gid[crop_node]

# ---------- leakage assertions (all stages) ----------
assert pd.Series(node_split).groupby(gid).nunique().max() == 1, "LEAKAGE: a group straddles splits"
md5_all = pd.concat([index.loc[ok_rows, ["md5", "split"]],
                     pd.DataFrame({"md5": [r["md5"] for r in pool], "split": [r["split"] for r in pool]}),
                     icdas_df.loc[icdas_df.ok, ["md5", "split"]]])
md5_all = md5_all[md5_all["md5"].astype(str) != ""]
_bad = md5_all.groupby("md5")["split"].nunique()
if (_bad > 1).any():
    print("Identical files found in more than one split (first 5):", _bad[_bad > 1].index[:5].tolist())
assert (_bad <= 1).all(), "LEAKAGE: identical bytes in two splits"
assert icdas_df.groupby("source_id")["split"].nunique().max() == 1, "LEAKAGE: crops of one source photo in two splits"
assert (icdas_df["split"].values[ic_pairs[0][m_]] == icdas_df["split"].values[ic_pairs[1][m_]]).all(), \
    "LEAKAGE: near-duplicate ICDAS crops in two splits"
print("\nLeakage checks passed: groups, identical bytes (Stage 1 + G.V. Black + ICDAS), ICDAS source photos, near-duplicate crops.")

# tooth-crop size relative to its photograph (TRAINING crops only) -> default ICDAS inference window
_tr = icdas_df[(icdas_df.split == "train") & icdas_df.ok & icdas_df["w"].notna()] if "w" in icdas_df else pd.DataFrame()
TOOTH_FRAC = ((float(np.median(_tr["w"] / _tr["source_width"])), float(np.median(_tr["h"] / _tr["source_height"])))
              if len(_tr) else (0.33, 0.55))
print(f"Median tooth-crop size relative to its photograph (training crops): width {TOOTH_FRAC[0]:.3f}, height {TOOTH_FRAC[1]:.3f}")

SPLIT_FP = {"pool": fingerprint([f"{r['id']}|{r['split']}" for r in pool]),
            "icdas": fingerprint(icdas_df["rel"] + "|" + icdas_df["split"])}
SPLIT_FP.update(localizer=SPLIT_FP["pool"], gvb=SPLIT_FP["pool"])
print("All augmentation happens later, inside the models (training only) — never before this split.")

## 11. Stage 1 dataset preparation
Class mapping (fixed): **0 → Healthy**, **1 → Caries**. Images kept: readable, de-duplicated, not ambiguous (Section 10).
Caries-archive photographs that `icdascl.zip/images_without_caries.txt` lists as "no carious tooth visible" are handled by
`S1["NO_CARIES_LIST_ACTION"]` (default: **excluded** from the Caries class, because using them as "Caries" is label noise).

In [ ]:
# ============================================================
# SECTION 11 — STAGE 1 DATASET PREPARATION
# ============================================================
S1_CLASS_NAMES = ["Healthy", "Caries"]
s1_df = index.iloc[s1_rows].copy()
s1_df["row"] = s1_rows
s1_df["no_caries_listed"] = s1_df["row"].isin(NOCARIES_ROWS) & (s1_df["label"] == 1)
act = S1["NO_CARIES_LIST_ACTION"]
n_listed = int(s1_df["no_caries_listed"].sum())
if act == "exclude":
    s1_df = s1_df[~s1_df["no_caries_listed"]]
elif act == "healthy":
    s1_df.loc[s1_df["no_caries_listed"], "label"] = 0
    s1_df.loc[s1_df["no_caries_listed"], "class_name"] = "Healthy"
elif act != "keep":
    raise ValueError("S1['NO_CARIES_LIST_ACTION'] must be 'exclude', 'healthy' or 'keep'")
s1_df = s1_df.reset_index(drop=True)
print(f"Caries-archive photos listed as 'no carious tooth visible': {n_listed} -> action '{act}'")
print(f"Stage 1 images used: {len(s1_df):,}  (Healthy {int((s1_df.label == 0).sum()):,} | Caries {int((s1_df.label == 1).sum()):,})")
print("Class mapping: 0 -> Healthy | 1 -> Caries")
SPLIT_FP["stage1"] = fingerprint(s1_df["src"] + "|" + s1_df["split"] + "|" + s1_df["label"].astype(str))

## 12. Train / validation / test split
The Stage 1 split is the shared group split from Section 10 (made **before** any augmentation). Augmentation happens only inside the model
during training, so no augmented copy of any image can reach validation or test. The test set is not touched until Section 19.

In [ ]:
# ============================================================
# SECTION 12 — STAGE 1 SPLIT + tf.data PIPELINES
# ============================================================
s1_train_df = s1_df[s1_df.split == "train"].reset_index(drop=True)
s1_val_df   = s1_df[s1_df.split == "val"].reset_index(drop=True)
s1_test_df  = s1_df[s1_df.split == "test"].reset_index(drop=True)
y1_train = s1_train_df["label"].to_numpy().astype(int)
y1_val   = s1_val_df["label"].to_numpy().astype(int)
y1_test  = s1_test_df["label"].to_numpy().astype(int)

assert s1_df.groupby("group")["split"].nunique().max() == 1, "LEAKAGE: a Stage 1 group straddles splits"
_m5 = {s: set(d.md5) - {""} for s, d in [("train", s1_train_df), ("val", s1_val_df), ("test", s1_test_df)]}
assert not (_m5["train"] & _m5["test"]) and not (_m5["val"] & _m5["test"]) and not (_m5["train"] & _m5["val"]), "LEAKAGE: identical bytes"
assert len(np.unique(y1_train)) == 2 and len(np.unique(y1_val)) == 2 and len(np.unique(y1_test)) == 2, "a split lacks one class"

s1_train_ds = array_batch_ds(s1_images, s1_train_df["row"], y1_train[:, None], S1["BATCH"], True)
s1_val_ds   = array_batch_ds(s1_images, s1_val_df["row"],   y1_val[:, None],   S1["BATCH"], False)
s1_test_ds  = array_batch_ds(s1_images, s1_test_df["row"],  y1_test[:, None],  S1["BATCH"], False)
print(f"Train {len(s1_train_df):,} | Validation {len(s1_val_df):,} | Test {len(s1_test_df):,}  (group-level split, leakage checks passed)")

## 13. Class distribution
Class weights are computed from the **training split only**. **Probe A** — a decision tree on file size, image size and mean colour only
(it cannot see a lesion). If it scores close to the CNN, Stage 1 may be separating the two *archives* (camera, lighting, framing).

In [ ]:
# ============================================================
# SECTION 13 — STAGE 1 CLASS DISTRIBUTION + CLASS WEIGHTS + SOURCE PROBE A
# ============================================================
dist1 = pd.crosstab(s1_df["class_name"], s1_df["split"]).reindex(index=S1_CLASS_NAMES, columns=["train", "val", "test"]).fillna(0).astype(int)
dist1["total"] = dist1.sum(1)
print(dist1.to_string())
dist1[["train", "val", "test"]].plot.bar(figsize=(6, 3.2), title="Stage 1 images per class and split"); plt.tight_layout(); plt.show()

counts = np.bincount(y1_train, minlength=2)
IMBALANCE = counts.max() / max(counts.min(), 1)
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y1_train)
use_cw = S1["CLASS_WEIGHTS"] is True or (S1["CLASS_WEIGHTS"] == "auto" and IMBALANCE > 1.5)
S1_CLASS_WEIGHT = {0: float(cw[0]), 1: float(cw[1])} if use_cw else None
print(f"\nTraining imbalance {IMBALANCE:.2f}:1 -> class weights (training split only): {S1_CLASS_WEIGHT}")

def meta_features(d):
    X = d[["width", "height", "filesize", "mean_r", "mean_g", "mean_b"]].astype(float).copy()
    X["aspect"] = X["width"] / X["height"].replace(0, np.nan)
    return X.fillna(0).to_numpy()
probe_a = DecisionTreeClassifier(max_depth=6, random_state=SEED).fit(meta_features(s1_train_df), y1_train)
META_PROBE_ACC = float(accuracy_score(y1_test, probe_a.predict(meta_features(s1_test_df))))
print(f"Probe A (size / file size / mean colour only — no lesion visible) TEST accuracy: {META_PROBE_ACC:.3f}"
      + ("  <-- strong source shortcut; report this next to any Stage 1 result" if META_PROBE_ACC > 0.9 else ""))

## 14. Data augmentation
Mild, dentally plausible augmentation, implemented as Keras layers **inside** the model (active only with `training=True`):
horizontal flip (caries presence is mirror-invariant), small rotation / translation / zoom, brightness and contrast. Below: one training
image and six random augmentations of it.

In [ ]:
# ============================================================
# SECTION 14 — STAGE 1 DATA AUGMENTATION
# ============================================================
print("Stage 1 augmentation:", S1["AUG"])
_aug = augmentation_block(S1["AUG"])
_x = np.asarray(s1_images[s1_train_df["row"].iloc[0]], np.float32)[None]
show_examples([_x[0]] + [np.clip(_aug(_x, training=True).numpy()[0], 0, 255) for _ in range(6)],
              ["original"] + [f"augmented {i}" for i in range(1, 7)], "Stage 1 augmentation (training only)", ncols=7)

## 15. Stage 1 CNN architecture
**Backbone selection:** each candidate (`BACKBONE_CANDIDATES`) trains its head for a few epochs on a stratified training subset; the best
**validation ROC-AUC** wins — the test set is not used for any choice. Architecture: RGB `[0,255]` → augmentation (train only) →
backbone-specific normalization → ImageNet backbone (BatchNorm in inference mode) → GAP → Dropout → Dense(256) → Dropout →
sigmoid = P(caries).

In [ ]:
# ============================================================
# SECTION 15 — STAGE 1 CNN ARCHITECTURE (backbone chosen on VALIDATION)
# ============================================================
S1_METRICS = lambda: [keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.Precision(name="precision"),
                      keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")]
S1_LOSS = keras.losses.BinaryCrossentropy()
build_s1 = lambda bb: build_classifier(bb, (S1_PX, S1_PX), 1, S1, "cnn1_stage1_caries_classifier")
S1_CKPT = f"{CKPT_DIR}/stage1_best.weights.h5"

S1_TRAINED = decide("stage1")
S1_SCREEN = pd.DataFrame(); s1_hist, s1_info = {}, {}
if S1_TRAINED:
    sub = subsample_rows(y1_train, S1["SCREEN_MAX_TRAIN"], SEED)
    screen_ds = array_batch_ds(s1_images, s1_train_df["row"].to_numpy()[sub], y1_train[sub, None], S1["BATCH"], True)
    with stopwatch("Stage 1 backbone screening"):
        S1_BACKBONE, S1_SCREEN = screen_backbones(S1["CANDIDATES"], build_s1, screen_ds, s1_val_ds, S1, S1_LOSS,
                                                  S1_METRICS, "val_auc", S1_CLASS_WEIGHT, "stage1")
    print("Selected Stage 1 backbone (validation):", S1_BACKBONE)
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    stage1_model, s1_base = build_s1(S1_BACKBONE)
else:
    stage1_model = load_keras(MODEL_PATHS["stage1"])
    _m = read_json(MODEL_META_PATHS["stage1"])
    S1_BACKBONE = _m.get("backbone", "unknown"); s1_hist = _m.get("training_history", {}); s1_info = _m.get("training", {})
    print("Loaded the saved Stage 1 model (trained on this split) — no retraining:", MODEL_PATHS["stage1"], "| backbone:", S1_BACKBONE)
print(f"CNN 1: {stage1_model.name} | backbone {S1_BACKBONE} | parameters {stage1_model.count_params():,}")

## 16. Stage 1 training — PHASE 1 (frozen backbone)
ImageNet backbone frozen, classification head only (AdamW, LR 1e-3). EarlyStopping (restore best), ReduceLROnPlateau and a best-weights
checkpoint, all on **validation** ROC-AUC / loss.

In [ ]:
# ============================================================
# SECTION 16 — STAGE 1 PHASE 1: FROZEN BACKBONE
# ============================================================
if S1_TRAINED:
    with stopwatch("Stage 1 phase 1"):
        s1_hist, s1_info = train_phase_a(stage1_model, s1_train_ds, s1_val_ds, S1, S1_LOSS, S1_METRICS, "val_auc",
                                         S1_CLASS_WEIGHT, S1_CKPT)
else:
    print("Skipped — the saved Stage 1 model was loaded in Section 15.")

## 17. Stage 1 fine-tuning — PHASE 2
Top `UNFREEZE_FRAC` of the backbone unfrozen (BatchNorm stays frozen), LR ≈ 50× smaller. The checkpoint is only overwritten if Phase 2 beats
Phase 1 on validation. The model is saved right after training.

In [ ]:
# ============================================================
# SECTION 17 — STAGE 1 PHASE 2: FINE-TUNING + SAVE
# ============================================================
if S1_TRAINED:
    with stopwatch("Stage 1 phase 2"):
        s1_hist, s1_info = train_phase_b(stage1_model, s1_base, s1_train_ds, s1_val_ds, S1, S1_LOSS, S1_METRICS, "val_auc",
                                         S1_CLASS_WEIGHT, S1_CKPT, s1_hist, s1_info)
    stage1_model.save(MODEL_PATHS["stage1"])
    print("Saved", MODEL_PATHS["stage1"])
else:
    print("Skipped — the saved Stage 1 model was loaded in Section 15.")

## 18. Stage 1 training curves
Training vs validation **accuracy**, **loss** and ROC-AUC per epoch (the dotted line marks the start of fine-tuning). Training values are
measured with augmentation + dropout active; the clean re-evaluation below measures the final model on each split without them.

In [ ]:
# ============================================================
# SECTION 18 — STAGE 1 TRAINING CURVES + CLEAN TRAIN/VAL ACCURACY & LOSS
# ============================================================
plot_history(s1_hist, s1_info, ["accuracy", "loss", "auc"], "Stage 1")
S1_HIST_SUMMARY = history_summary(s1_hist, s1_info)
print("At the kept epoch:", S1_HIST_SUMMARY)
S1_CLEAN = clean_eval(stage1_model, {"train": (array_batch_ds(s1_images, s1_train_df["row"], y1_train[:, None], S1["BATCH"], False), y1_train),
                                     "val": (s1_val_ds, y1_val)}, S1_LOSS)
for s, v in S1_CLEAN.items():
    print(f"Clean {s:5} accuracy {v['accuracy']:.4f} | loss {v['loss']:.4f}  (n={v['n']:,}, threshold 0.5, no augmentation)")

## 19. Stage 1 evaluation (untouched test set)
1. **Decision threshold** (Youden's J) and **uncertainty band** are chosen on **validation only**. The band is the smallest margin around the
   threshold such that validation predictions outside it reach `UNCERTAIN_TARGET_ACC`.
2. The test set is predicted **once**, after both are frozen: accuracy, precision, recall / sensitivity, specificity, F1, ROC-AUC,
   average precision.

In [ ]:
# ============================================================
# SECTION 19 — STAGE 1 EVALUATION (threshold from validation, test predicted once)
# ============================================================
p_val = stage1_model.predict(s1_val_ds, verbose=0).ravel().astype(np.float64)
fpr_v, tpr_v, thr_v = roc_curve(y1_val, p_val)
S1_THRESHOLD = float(thr_v[int(np.argmax(tpr_v - fpr_v))])
if not np.isfinite(S1_THRESHOLD) or not (0.0 < S1_THRESHOLD < 1.0):
    print("WARNING: degenerate validation ROC -> threshold falls back to 0.5"); S1_THRESHOLD = 0.5
S1_MARGIN = S1["UNCERTAIN_MAX_MARGIN"]
for m in np.round(np.arange(0.0, S1["UNCERTAIN_MAX_MARGIN"] + 1e-9, 0.01), 2):
    conf = np.abs(p_val - S1_THRESHOLD) >= m
    if conf.mean() >= 0.5 and accuracy_score(y1_val[conf], p_val[conf] >= S1_THRESHOLD) >= S1["UNCERTAIN_TARGET_ACC"]:
        S1_MARGIN = float(m); break
else:
    print(f"WARNING: validation accuracy {S1['UNCERTAIN_TARGET_ACC']} is not reached with >=50 % coverage; "
          f"using the maximum band ±{S1_MARGIN:.2f}.")
print(f"Validation ROC-AUC {roc_auc_score(y1_val, p_val):.4f} | threshold (Youden J) {S1_THRESHOLD:.4f} | "
      f"uncertainty band ±{S1_MARGIN:.2f} | confidence threshold {S1['CONFIDENCE_THRESHOLD']:.2f}")

p_test = stage1_model.predict(s1_test_ds, verbose=0).ravel().astype(np.float64)       # the ONE test prediction
yhat = (p_test >= S1_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y1_test, yhat, labels=[0, 1]).ravel()
S1_TEST = {"n_test": int(len(y1_test)), "accuracy": accuracy_score(y1_test, yhat),
           "precision": precision_score(y1_test, yhat, zero_division=0),
           "recall_sensitivity": recall_score(y1_test, yhat, zero_division=0),
           "specificity": tn / max(tn + fp, 1), "f1": f1_score(y1_test, yhat, zero_division=0),
           "roc_auc": roc_auc_score(y1_test, p_test), "average_precision": average_precision_score(y1_test, p_test),
           "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
_dec = [stage1_decide(p, {"threshold": S1_THRESHOLD, "uncertain_margin": S1_MARGIN,
                          "confidence_threshold": S1["CONFIDENCE_THRESHOLD"]}) for p in p_test]
conf_t = np.array([d["prediction"] != "UNCERTAIN" for d in _dec])
S1_TEST["coverage_not_uncertain"] = float(conf_t.mean())
S1_TEST["accuracy_not_uncertain"] = float(accuracy_score(y1_test[conf_t], yhat[conf_t])) if conf_t.any() else None
S1_TEST = {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in S1_TEST.items()}

print("\n" + "=" * 60 + f"\nSTAGE 1 — HEALTHY vs CARIES — TEST SET ({S1_BACKBONE})\n" + "=" * 60)
for label, k in [("Accuracy", "accuracy"), ("Precision", "precision"), ("Recall / Sensitivity", "recall_sensitivity"),
                 ("Specificity", "specificity"), ("F1-score", "f1"), ("ROC-AUC", "roc_auc"), ("Average precision", "average_precision")]:
    print(f"  {label:22}: {S1_TEST[k]:.4f}")
print(f"  {'Test images':22}: {S1_TEST['n_test']:,}")
print(f"  Not UNCERTAIN: {S1_TEST['coverage_not_uncertain']:.1%} of test images, accuracy on them "
      + (f"{S1_TEST['accuracy_not_uncertain']:.4f}" if S1_TEST['accuracy_not_uncertain'] is not None else "n/a"))

## 20. Stage 1 confusion matrix

In [ ]:
# ============================================================
# SECTION 20 — STAGE 1 CONFUSION MATRIX
# ============================================================
plot_confusion(confusion_matrix(y1_test, yhat, labels=[0, 1]), S1_CLASS_NAMES, f"Stage 1 test (threshold {S1_THRESHOLD:.3f})")
print(f"TN {tn:,} | FP {fp:,} | FN {fn:,} | TP {tp:,}")

## 21. Stage 1 classification report

In [ ]:
# ============================================================
# SECTION 21 — STAGE 1 CLASSIFICATION REPORT
# ============================================================
print(classification_report(y1_test, yhat, labels=[0, 1], target_names=S1_CLASS_NAMES, digits=4, zero_division=0))

## 22. Stage 1 ROC curve
Plus the precision–recall curve and **Probe B** (colour histogram + logistic regression), which repeats the source-shortcut check.

In [ ]:
# ============================================================
# SECTION 22 — STAGE 1 ROC + PR CURVES + SOURCE PROBE B
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
f_, t_, _ = roc_curve(y1_test, p_test)
ax[0].plot(f_, t_, lw=2, label=f"AUC {S1_TEST['roc_auc']:.4f}"); ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set_xlabel("False positive rate (1 − specificity)"); ax[0].set_ylabel("Sensitivity"); ax[0].set_title("Stage 1 ROC — test"); ax[0].legend()
pr_, rc_, _ = precision_recall_curve(y1_test, p_test)
ax[1].plot(rc_, pr_, lw=2, label=f"AP {S1_TEST['average_precision']:.4f}"); ax[1].axhline(y1_test.mean(), ls="--", c="k", label="chance")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Stage 1 precision–recall — test"); ax[1].legend()
plt.tight_layout(); plt.show()

def hists(rows, bins=32):
    out = np.zeros((len(rows), bins * 3), np.float32); off = np.arange(3) * bins
    for i, r in enumerate(rows):
        X = np.asarray(s1_images[r])
        out[i] = np.bincount(((X >> 3).astype(np.int32) + off).ravel(), minlength=bins * 3)[:bins * 3]
    return out / (S1_PX * S1_PX)
sub = np.random.RandomState(SEED).permutation(len(s1_train_df))[:5000]
hp = LogisticRegression(max_iter=2000).fit(hists(s1_train_df["row"].to_numpy()[sub]), y1_train[sub]).predict(hists(s1_test_df["row"].to_numpy()))
PROBE_B_ACC = float(accuracy_score(y1_test, hp))
print(f"Probe A (metadata only) accuracy    : {META_PROBE_ACC:.4f}")
print(f"Probe B (colour histogram) accuracy : {PROBE_B_ACC:.4f}")
print(f"CNN 1 test accuracy                 : {S1_TEST['accuracy']:.4f}")
print("If a probe is close to the CNN, Stage 1 may be recognising the ARCHIVE (camera/lighting/framing) rather than lesions.")

PIPELINE["models"]["stage1"] = stage1_model
PIPELINE["meta"]["stage1"] = write_model_meta("stage1", {
    "model": "CNN 1 — Stage 1 healthy vs caries", "architecture": f"{S1_BACKBONE} (ImageNet) + GAP + Dense({S1['DENSE']}) + sigmoid",
    "backbone": S1_BACKBONE, "image_size": S1_PX, "class_names": S1_CLASS_NAMES, "class_mapping": {"0": "Healthy", "1": "Caries"},
    "output": "sigmoid = P(caries)", "threshold": S1_THRESHOLD, "threshold_rule": "Youden J on validation",
    "uncertain_margin": S1_MARGIN, "uncertain_rule": f"smallest band with validation accuracy >= {S1['UNCERTAIN_TARGET_ACC']} outside it",
    "confidence_threshold": S1["CONFIDENCE_THRESHOLD"],
    "preprocessing": "load_rgb (EXIF) -> cv2.resize(image_size, INTER_AREA) -> uint8 [0,255]; " +
                     PREP_DESCRIPTION.get(BACKBONES.get(S1_BACKBONE, {}).get("prep"), "see model"),
    "class_weight": S1_CLASS_WEIGHT, "no_caries_list_action": S1["NO_CARIES_LIST_ACTION"],
    "training_config": {k: v for k, v in S1.items()}, "backbone_screening": S1_SCREEN.to_dict("records"),
    "training": s1_info, "training_history": s1_hist, "history_at_kept_epoch": S1_HIST_SUMMARY, "clean_train_val": S1_CLEAN,
    "test_metrics": S1_TEST, "probe_metadata_acc": META_PROBE_ACC, "probe_colour_hist_acc": PROBE_B_ACC,
    "split_counts": {s: int((s1_df.split == s).sum()) for s in SPLIT_NAMES}, "split_fingerprint": SPLIT_FP["stage1"],
    "model_path": MODEL_PATHS["stage1"],
})
print("\nCNN 1 registered + metadata saved:", MODEL_META_PATHS["stage1"])

## 23. Stage 1 — unseen image prediction
`predict_caries(image_path)` loads the image (EXIF orientation), applies exactly the training preprocessing, and returns
**HEALTHY / CARIES / UNCERTAIN** with a confidence. UNCERTAIN = inside the validation-derived band **or** confidence below
`S1["CONFIDENCE_THRESHOLD"]` (override per call: `predict_caries(path, confidence_threshold=0.8)`). Confidences are sigmoid outputs,
not calibrated clinical probabilities. Demonstrated on held-out **test** images; try your own with `predict_caries("/content/my_photo.jpg")`.

In [ ]:
# ============================================================
# SECTION 23 — STAGE 1 PREDICTION ON UNSEEN (TEST) IMAGES
# ============================================================
for lab in (0, 1):
    cand = s1_test_df[s1_test_df.label == lab]
    if len(cand):
        p = cand.sample(1, random_state=SEED)["src"].iloc[0]
        print(f"--- {os.path.basename(p)}  (true: {S1_CLASS_NAMES[lab]})")
        predict_caries(p); print()

<br>

---
# PART 6 — STAGE 2: FDI TOOTH DETECTION + NUMBERING

---
# STAGE 2 — FDI TOOTH NUMBERING
## 24. Stage 2 — FDI tooth detection + numbering (`caries_FDI_dataset_final.zip`)
Stage 2 answers **"which tooth is where, and what is its FDI number?"** It is a **separate detector** with its own data, weights and file
(`stage2_fdi_detector.keras`). It never sees lesion boxes, and the lesion localizer never sees tooth boxes. The two outputs are joined only
afterwards by an explicit **lesion → tooth association**: a lesion is linked to the detected tooth box that covers the largest part of it.

| Cell | What it does |
|---|---|
| **24.1** | finds the ZIP **or the extracted folder** on Drive (`dentalcaries/caries_FDI_dataset_final/`), copies it to local disk once (skipping `FDI["EXCLUDE_DIRS"]`), or asks for an upload, and prints its structure (folders, file types, annotation-like files). These are dataset facts, not results. |
| **24.2** | parses the annotations (YOLO `.txt` + `classes.txt` / `data.yaml` / `notes.json`, COCO JSON incl. DENTEX-style quadrant + tooth categories, Pascal-VOC XML, LabelMe JSON, Label Studio JSON, CSV), maps the labels to FDI codes and decides whether the data can train a detector. |
| **24.3** | decodes and caches the images, then makes a **leakage-safe group split**: identical files, near-duplicates (dHash), Roboflow augmented copies and an optional patient-ID pattern always stay in the same split. The dataset's own train/valid/test folders are **not** trusted. |
| **24.4** | detector: the **same TensorFlow/Keras anchor-free design as the cavity localizer** (MobileNetV2 C4 + C5, stride 16) plus an FDI class head. Loss = focal objectness + GIoU + softmax cross-entropy. |
| **24.5** | **test** evaluation, run once: mAP@0.5, COCO-style mAP@[0.5:0.95], class-agnostic tooth AP, FDI-number accuracy on matched teeth, and precision / recall / F1 at the score threshold chosen on validation. Then the model and metadata are saved. |
| **24.6** | a self-check of the association rule, a test-image demo and, if the ZIP also contains caries boxes, a measured lesion → tooth association accuracy. |

**Rules (no invented numbers).**
* A label counts only if it **is an explicit, valid two-digit FDI code**. Examples: `16`, `tooth_16`, `FDI 16`, or a DENTEX quadrant + tooth pair.
* A YOLO class id without a name file is **not** assumed to be an FDI number. Set `FDI["CLASS_ID_MAPPING"]` only if you know the mapping.
* Universal (1–32) and Palmer label sets are refused.
* No horizontal flip during training: flipping a photo moves tooth 16 to where tooth 26 normally appears.
* At inference, a tooth whose number probability is below `FDI["CLASS_CONFIDENCE_THRESHOLD"]`, or whose number is also predicted for a
  more confident tooth, is reported as **Unknown / uncertain**.
* If a lesion is not covered by any detected tooth, it is also reported as Unknown / uncertain.

If the ZIP is missing, lacks FDI box annotations, or contains radiographs instead of colour photographs, every Stage 2 cell prints the reason.
The pipeline then reports Stage 2 as **UNAVAILABLE**, and the rest of the cascade is unchanged.

In [ ]:
# ============================================================
# SECTION 24.1 — STAGE 2: LOCATE + INSPECT caries_FDI_dataset_final.zip
# ============================================================
FDI_ROOT = os.path.join(DATA_ROOT, "fdi")
FDI_ZIP, FDI_DATA_OK, FDI_STOP_REASON, FDI_ZIP_REPORT = None, False, None, {}

def fdi_zip_report(zip_path):
    """Structure of the ZIP itself (no assumption about its layout)."""
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.filename.startswith("__MACOSX") and "/." not in "/" + i.filename]
    files_ = [i for i in infos if not i.is_dir()]
    ext = Counter(os.path.splitext(i.filename)[1].lower() or "<none>" for i in files_)
    folders = Counter(os.path.dirname(i.filename) or "<root>" for i in files_)
    ann_like = [i.filename for i in files_ if os.path.splitext(i.filename)[1].lower() in (".json", ".xml", ".csv", ".yaml", ".yml", ".names")
                or os.path.basename(i.filename).lower() == "classes.txt"]
    return {"zip": os.path.basename(zip_path), "zip_mb": os.path.getsize(zip_path) / 1e6, "n_files": len(files_),
            "uncompressed_mb": sum(i.file_size for i in files_) / 1e6,
            "top_level": sorted({i.filename.split("/")[0] for i in infos}), "extensions": dict(ext.most_common()),
            "max_depth": max((i.filename.count("/") for i in files_), default=0),
            "n_images": sum(v for k, v in ext.items() if k in IMG_EXT), "n_txt": ext.get(".txt", 0),
            "files_per_folder": dict(sorted(folders.items())), "annotation_like_files": ann_like}

def fdi_find_dataset_dir(candidates, max_depth=2):
    """First folder (candidate or a sub-folder <= max_depth) holding classes.txt / data.yaml or images/ + labels/."""
    excl = {d.lower() for d in FDI["EXCLUDE_DIRS"]}
    for c in candidates:
        if not (c and os.path.isdir(c)): continue
        base_depth = c.rstrip("/").count("/")
        for dp, dn, fn in os.walk(c):
            dn[:] = sorted(d for d in dn if not d.startswith(".") and d.lower() not in excl)
            if dp.count("/") - base_depth >= max_depth: dn[:] = []
            if {"classes.txt", "data.yaml"} & set(fn) or {"images", "labels"} <= set(dn):
                return dp
    return None

def fdi_copy_dir_once(src, dst):
    """Drive folder -> local disk (parallel, once). EXCLUDE_DIRS are not copied. Marker = source + file count."""
    excl = {d.lower() for d in FDI["EXCLUDE_DIRS"]}
    files_ = []
    for dp, dn, fn in os.walk(src):
        dn[:] = [d for d in dn if not d.startswith(".") and d.lower() not in excl]
        files_ += [os.path.join(dp, f) for f in fn if not f.startswith(".")]
    sig = f"{src}|{len(files_)}"; marker = os.path.join(dst, ".copied_ok")
    if os.path.exists(marker) and open(marker).read().strip() == sig:
        print(f"  already copied -> {dst}"); return dst
    if os.path.exists(dst): shutil.rmtree(dst)
    def _cp(p):
        d = os.path.join(dst, os.path.relpath(p, src)); os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(p, d)
    t0 = time.time()
    with ThreadPoolExecutor(COPY_WORKERS) as ex:
        list(ex.map(_cp, files_))
    open(marker, "w").write(sig)
    print(f"  copied {len(files_)} files from Drive in {time.time() - t0:.0f}s -> {dst}")
    return dst

def fdi_dir_report(root):
    files_ = [os.path.join(dp, f) for dp, dn, fn in os.walk(root) for f in fn if not f.startswith(".")]
    ext = Counter(os.path.splitext(f)[1].lower() or "<none>" for f in files_)
    folders = Counter(os.path.relpath(os.path.dirname(f), root) for f in files_)
    return {"zip": os.path.basename(root.rstrip("/")) + "/ (folder)", "zip_mb": sum(os.path.getsize(f) for f in files_) / 1e6,
            "n_files": len(files_), "uncompressed_mb": sum(os.path.getsize(f) for f in files_) / 1e6,
            "top_level": sorted(os.listdir(root)), "extensions": dict(ext.most_common()),
            "max_depth": max((os.path.relpath(f, root).count(os.sep) for f in files_), default=0),
            "n_images": sum(v for k, v in ext.items() if k in IMG_EXT), "n_txt": ext.get(".txt", 0),
            "files_per_folder": dict(sorted(folders.items())),
            "annotation_like_files": [os.path.relpath(f, root) for f in files_ if os.path.splitext(f)[1].lower() in
                                      (".json", ".xml", ".csv", ".yaml", ".yml", ".names") or os.path.basename(f).lower() == "classes.txt"]}

FDI_SRC_DIR = None
for pat in FDI_ZIP_PATTERNS:
    hits = sorted(glob.glob(pat))
    if hits:
        FDI_ZIP = hits[0]; break
if FDI_ZIP is None:                       # extracted folder on Drive (e.g. dentalcaries/caries_FDI_dataset_final/caries_FDI_dataset)
    FDI_SRC_DIR = fdi_find_dataset_dir(FDI_DIR_CANDIDATES)
if FDI_ZIP is None and FDI_SRC_DIR is None and FDI["ASK_UPLOAD"] and IN_COLAB:
    try:
        FDI_ZIP = upload_file("Upload the FDI dataset: caries_FDI_dataset_final.zip  (Cancel = Stage 2 is not trained)")
    except FileNotFoundError as e:
        print("No FDI ZIP uploaded:", e)

if FDI_SRC_DIR is not None:
    print("FDI dataset folder:", FDI_SRC_DIR, "| excluded sub-folders:", FDI["EXCLUDE_DIRS"] or "none")
    fdi_copy_dir_once(FDI_SRC_DIR, FDI_ROOT)
    FDI_ZIP = FDI_SRC_DIR                  # name used in reports / metadata
    FDI_ZIP_REPORT = fdi_dir_report(FDI_ROOT)
elif FDI_ZIP is None:
    FDI_STOP_REASON = ("FDI dataset not found (no ZIP: " + ", ".join(FDI_ZIP_PATTERNS) + "; no folder: " + ", ".join(FDI_DIR_CANDIDATES) + ")")
elif not zipfile.is_zipfile(FDI_ZIP):
    FDI_STOP_REASON = f"{FDI_ZIP} is not a valid ZIP file"
else:
    FDI_ZIP_REPORT = fdi_zip_report(FDI_ZIP)
    stage_zip(FDI_ZIP, FDI_ROOT)
if FDI_ZIP_REPORT:
    R_ = FDI_ZIP_REPORT
    print("FDI DATASET STRUCTURE\n" + "-" * 21)
    print(f"ZIP file          : {R_['zip']}  ({R_['zip_mb']:.1f} MB, {R_['uncompressed_mb']:.1f} MB uncompressed)")
    print(f"Files             : {R_['n_files']}  | images {R_['n_images']} | .txt {R_['n_txt']} | max depth {R_['max_depth']}")
    print(f"Top-level entries : {R_['top_level']}")
    print(f"File extensions   : {R_['extensions']}")
    print("Folders (files directly inside, first 40):")
    for k, v in list(R_["files_per_folder"].items())[:40]:
        print(f"  {k:60}: {v}")
    if len(R_["files_per_folder"]) > 40:
        print(f"  ... {len(R_['files_per_folder']) - 40} more folders")
    print("Annotation-like files (first 20):", R_["annotation_like_files"][:20] or "none")
if FDI_STOP_REASON:
    print("STAGE 2 cannot be trained:", FDI_STOP_REASON)
    print("-> Stage 2 will be reported as UNAVAILABLE. Put the ZIP or the extracted folder in", DRIVE_BASE, "and re-run Section 24.")

In [ ]:
# ============================================================
# SECTION 24.2 — STAGE 2: PARSE ANNOTATIONS -> FDI MAPPING -> SUITABILITY DECISION
# ============================================================
import xml.etree.ElementTree as ET

def fdi_walk(root):
    """os.walk without hidden folders, __MACOSX and FDI['EXCLUDE_DIRS'] (e.g. a duplicate subset with conflicting labels)."""
    excl = {d.lower() for d in FDI["EXCLUDE_DIRS"]}
    for dp, dn, fn in os.walk(root):
        dn[:] = sorted(d for d in dn if not d.startswith((".", "__MACOSX")) and d.lower() not in excl)
        yield dp, dn, [f for f in fn if not f.startswith(".")]

def fdi_read_class_names(root):
    """YOLO class-id -> name files: classes.txt / *.names / data.yaml / notes.json (Label Studio). {path: {id: name}}."""
    found = {}
    for dp, dn, fn in fdi_walk(root):
        for f in fn:
            fl, p = f.lower(), os.path.join(dp, f)
            try:
                if fl in ("classes.txt", "_darknet.labels") or fl.endswith(".names"):
                    found[p] = {i: l.strip() for i, l in enumerate(open(p, encoding="utf-8", errors="replace").read().splitlines()) if l.strip()}
                elif fl in ("data.yaml", "data.yml", "dataset.yaml", "dataset.yml"):
                    import yaml
                    names = (yaml.safe_load(open(p)) or {}).get("names")
                    if names:
                        found[p] = dict(enumerate(map(str, names))) if isinstance(names, list) else {int(k): str(v) for k, v in names.items()}
                elif fl == "notes.json":
                    cats = json.load(open(p)).get("categories", [])
                    if cats: found[p] = {int(c["id"]): str(c["name"]) for c in cats}
            except Exception as e:
                print(f"  could not read class-name file {p}: {type(e).__name__}: {e}")
    return found

def _px_box(x1, y1, x2, y2):
    return [float(min(x1, x2)), float(min(y1, y2)), float(max(x1, x2)), float(max(y1, y2))]

def fdi_parse_annotations(root):
    """All annotation files under root -> entries {format, ann_file, image_ref, size (W,H) or None, objects:[(label, box, kind)]}
    kind: 'norm' = normalised x1y1x2y2, 'px' = pixel x1y1x2y2 in the frame given by size (or the raw image if size is None)."""
    files_ = [os.path.join(dp, f) for dp, dn, fn in fdi_walk(root) for f in fn]
    names = fdi_read_class_names(root)
    problems = []
    by_dir = defaultdict(dict)                         # folder -> {name file: mapping}
    for p_, m_ in names.items(): by_dir[os.path.dirname(p_)][p_] = m_
    for d_, ms in by_dir.items():
        if len({json.dumps(m, sort_keys=True) for m in ms.values()}) > 1:
            problems.append(f"class-name files in {os.path.relpath(d_, root) or '.'} DISAGREE: {[os.path.basename(k) for k in ms]} "
                            f"— using {os.path.basename(next(iter(ms)))}")
    if len({json.dumps(m, sort_keys=True) for m in names.values()}) > 1:
        problems.append("different folders have DIFFERENT class-name files -> each label file uses the one in its nearest parent folder")
    user_map = {int(k): str(v) for k, v in FDI["CLASS_ID_MAPPING"].items()} if FDI["CLASS_ID_MAPPING"] else {}
    if user_map: problems.append("class names for folders without a name file taken from FDI['CLASS_ID_MAPPING'] (user supplied)")
    def id2name_for(path):
        """Mapping from the class-name file in the nearest parent folder of `path` (YOLO ids are only meaningful per export)."""
        d = os.path.dirname(path)
        while True:
            if d in by_dir: return next(iter(by_dir[d].values()))
            if os.path.abspath(d) == os.path.abspath(root) or os.path.dirname(d) == d: return user_map
            d = os.path.dirname(d)
    id2name = id2name_for(os.path.join(root, "x"))
    name_files = set(names)
    entries = []
    for p in sorted(files_):
        fl = p.lower(); base = os.path.basename(fl)
        if p in name_files:
            continue
        try:
            if fl.endswith(".json"):
                J = json.load(open(p, encoding="utf-8"))
                if isinstance(J, dict) and "annotations" in J and "images" in J:            # COCO (+ DENTEX hierarchical variant)
                    imgs = {im["id"]: im for im in J["images"]}
                    cats = {c["id"]: str(c["name"]) for c in J.get("categories", [])}
                    q_cats = {c["id"]: str(c["name"]) for c in J.get("categories_1", [])}
                    t_cats = {c["id"]: str(c["name"]) for c in J.get("categories_2", [])}
                    d_cats = {c["id"]: str(c["name"]) for c in J.get("categories_3", [])}
                    per = defaultdict(list)
                    for a in J["annotations"]:
                        if "bbox" not in a or a.get("image_id") not in imgs: continue
                        x, y, w, h = map(float, a["bbox"][:4])
                        if "category_id_1" in a and "category_id_2" in a and q_cats and t_cats:
                            lab = f"FDI {q_cats.get(a['category_id_1'], '?')}{t_cats.get(a['category_id_2'], '?')}"
                            if "category_id_3" in a and d_cats: lab += f" [{d_cats.get(a['category_id_3'], '')}]"
                        else:
                            lab = cats.get(a.get("category_id"), f"<category id {a.get('category_id')}>")
                        per[a["image_id"]].append((lab, _px_box(x, y, x + w, y + h), "px"))
                    fmt = "COCO (DENTEX quadrant+tooth)" if q_cats and t_cats else "COCO JSON"
                    for iid, objs in per.items():
                        im = imgs[iid]
                        entries.append({"format": fmt, "ann_file": p, "image_ref": im.get("file_name", ""),
                                        "size": (im["width"], im["height"]) if im.get("width") and im.get("height") else None,
                                        "objects": objs})
                elif isinstance(J, dict) and "shapes" in J:                                  # LabelMe
                    objs = []
                    for s in J["shapes"]:
                        pts = np.asarray(s.get("points", []), float).reshape(-1, 2)
                        if len(pts) >= 2:
                            objs.append((str(s.get("label", "")), _px_box(pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()), "px"))
                    entries.append({"format": "LabelMe JSON", "ann_file": p, "image_ref": J.get("imagePath") or stem(p),
                                    "size": (J["imageWidth"], J["imageHeight"]) if J.get("imageWidth") else None, "objects": objs})
                elif isinstance(J, list) and J and isinstance(J[0], dict) and "data" in J[0]:     # Label Studio JSON export
                    for task in J:
                        ref = next((v for v in (task.get("data") or {}).values() if isinstance(v, str)), "")
                        objs = []
                        for ann in task.get("annotations", []) or task.get("completions", []):
                            for r in ann.get("result", []):
                                v = r.get("value", {})
                                labs = v.get("rectanglelabels") or v.get("labels") or []
                                if {"x", "y", "width", "height"} <= set(v) and labs:
                                    objs.append((str(labs[0]), [v["x"] / 100, v["y"] / 100, (v["x"] + v["width"]) / 100,
                                                                (v["y"] + v["height"]) / 100], "norm"))
                        if objs:
                            entries.append({"format": "Label Studio JSON", "ann_file": p, "image_ref": ref, "size": None, "objects": objs})
            elif fl.endswith(".xml"):                                                          # Pascal VOC
                r = ET.parse(p).getroot()
                if r.tag == "annotation":
                    sz = r.find("size")
                    W_, H_ = (int(float(sz.findtext("width", "0"))), int(float(sz.findtext("height", "0")))) if sz is not None else (0, 0)
                    objs = []
                    for o in r.findall("object"):
                        bb = o.find("bndbox")
                        if bb is not None:
                            objs.append((o.findtext("name", "").strip(), _px_box(*(float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))), "px"))
                    entries.append({"format": "Pascal VOC XML", "ann_file": p, "image_ref": r.findtext("filename") or stem(p),
                                    "size": (W_, H_) if W_ and H_ else None, "objects": objs})
            elif fl.endswith(".txt") and base not in ("readme.txt", "license.txt", "images_without_caries.txt"):  # YOLO
                objs, bad, id2name_p = [], 0, id2name_for(p)
                for line in open(p, encoding="utf-8", errors="replace"):
                    q = line.split()
                    if not q: continue
                    try:
                        c = float(q[0]); v = list(map(float, q[1:]))
                    except ValueError:
                        bad += 1; continue
                    if c != int(c) or len(v) < 4 or (len(v) > 4 and len(v) % 2):
                        bad += 1; continue
                    if len(v) == 4:                                    # cx cy w h
                        x, y, w, h = v; box = [x - w / 2, y - h / 2, x + w / 2, y + h / 2]
                    else:                                              # YOLO polygon -> enclosing box
                        xs, ys = v[0::2], v[1::2]; box = [min(xs), min(ys), max(xs), max(ys)]
                    lab = id2name_p.get(int(c), f"<class id {int(c)}>")
                    objs.append((lab, box, "norm"))
                if objs or bad == 0:
                    entries.append({"format": "YOLO txt", "ann_file": p, "image_ref": stem(p), "size": None, "objects": objs,
                                    "malformed_lines": bad})
            elif fl.endswith(".csv"):
                df = pd.read_csv(p)
                cols = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in df.columns}
                pick = lambda opts: next((cols[o] for o in opts if o in cols), None)
                fcol = pick(["filename", "file", "image", "imagename", "imagepath", "path", "sourceimage", "imagefile"])
                lcol = pick(["fdi", "fdinumber", "toothfdi", "fdicode", "toothnumber", "tooth", "toothid",
                             "class", "classname", "label", "name", "category"])
                xyxy = [pick(o) for o in (["xmin", "x1", "left"], ["ymin", "y1", "top"], ["xmax", "x2", "right"], ["ymax", "y2", "bottom"])]
                xywh = [pick(o) for o in (["x", "bboxx"], ["y", "bboxy"], ["w", "width", "bboxw", "bboxwidth"], ["h", "height", "bboxh", "bboxheight"])]
                wcol, hcol = pick(["imagewidth", "sourcewidth", "imgwidth"]), pick(["imageheight", "sourceheight", "imgheight"])
                if fcol and lcol and (all(xyxy) or all(xywh)):
                    per = defaultdict(list); sizes = {}
                    for _, r in df.iterrows():
                        if all(xyxy):
                            b = _px_box(*(float(r[c]) for c in xyxy))
                        else:
                            x, y, w, h = (float(r[c]) for c in xywh); b = _px_box(x, y, x + w, y + h)
                        per[str(r[fcol])].append((str(r[lcol]), b, "px"))
                        if wcol and hcol: sizes[str(r[fcol])] = (int(r[wcol]), int(r[hcol]))
                    for ref, objs in per.items():
                        entries.append({"format": "CSV", "ann_file": p, "image_ref": ref, "size": sizes.get(ref), "objects": objs})
                else:
                    problems.append(f"CSV without recognisable file / label / box columns (ignored): {os.path.relpath(p, root)} "
                                    f"columns={list(df.columns)[:12]}")
        except Exception as e:
            problems.append(f"could not parse {os.path.relpath(p, root)}: {type(e).__name__}: {e}")
    return entries, names, id2name, problems

def fdi_match_images(entries, root):
    """image_ref -> image file on disk (same stem; Label-Studio 8-hex prefixes tolerated; same / sibling 'images' folder preferred)."""
    imgs = sorted(os.path.join(dp, f) for dp, dn, fn in fdi_walk(root) for f in fn if f.lower().endswith(IMG_EXT))
    by_stem = defaultdict(list)
    for p in imgs:
        by_stem[stem(p).lower()].append(p); by_stem[strip_uuid(stem(p)).lower()].append(p)
    out, orphans = [], []
    for e in entries:
        s = stem(str(e["image_ref"])).lower()
        cands = list(dict.fromkeys(by_stem.get(s, []) or by_stem.get(strip_uuid(s), [])))
        if not cands:
            orphans.append(e["ann_file"]); continue
        if len(cands) > 1:
            d = os.path.dirname(e["ann_file"])
            sib = [d, d.replace(os.sep + "labels", os.sep + "images"), d.replace(os.sep + "annotations", os.sep + "images")]
            same = [c for c in cands if os.path.dirname(c) in sib]
            cands = same or sorted(cands, key=lambda c: -len(os.path.commonpath([c, e["ann_file"]])))
        out.append({**e, "image_path": cands[0]})
    return out, orphans, imgs

def fdi_kind(label):
    """'tooth' (label carries ONE valid FDI code) | 'lesion' (caries word, no FDI code) | 'other'."""
    code = fdi_from_label(label)
    if code and (FDI["INCLUDE_PRIMARY_TEETH"] or int(code) in FDI_PERMANENT):
        return "tooth", code
    return ("lesion" if LESION_WORDS.search(str(label)) else "other"), None

def _raw_size_orientation(path):
    with Image.open(path) as im:
        return im.size, int(im.getexif().get(0x0112, 1) or 1)

FDI_RECORDS, FDI_CLASSES, FDI_PARSE = [], [], {}
if FDI_STOP_REASON is None:
    entries, name_files, id2name, problems = fdi_parse_annotations(FDI_ROOT)
    matched, orphans, all_imgs = fdi_match_images(entries, FDI_ROOT)
    fmt_imgs = defaultdict(set)
    for e in matched: fmt_imgs[e["format"]].add(e["image_path"])
    fmt_rank = sorted(fmt_imgs, key=lambda f: -len(fmt_imgs[f]))
    is_label_studio = any(os.path.basename(p).lower() == "notes.json" for p in name_files) or "Label Studio JSON" in fmt_imgs
    frame = FDI["ANNOTATION_FRAME"] if FDI["ANNOTATION_FRAME"] != "auto" else ("exif" if is_label_studio else "raw")

    by_img = {}
    for e in sorted(matched, key=lambda e: fmt_rank.index(e["format"])):     # one format per image (the most widely used one)
        cur = by_img.get(e["image_path"])
        if cur is None:
            by_img[e["image_path"]] = {"format": e["format"], "objects": list(e["objects"]), "size": e["size"]}
        elif cur["format"] == e["format"]:
            cur["objects"] += [o for o in e["objects"] if o not in cur["objects"]]

    label_counts, kind_of, numeric_non_fdi = Counter(), {}, Counter()
    size_mismatch = rotated = 0
    unreadable = []
    for ip, a in by_img.items():
        try:
            (rw, rh), ori = _raw_size_orientation(ip)
        except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as e:
            unreadable.append(f"{os.path.relpath(ip, FDI_ROOT)}: {type(e).__name__}: {e}"); continue
        ew, eh = (rh, rw) if ori in (5, 6, 7, 8) else (rw, rh)          # size after EXIF orientation (= what load_rgb returns)
        rotated += ori != 1
        boxes, fdis, les, les_fdi, other = [], [], [], [], 0
        for lab, b, kind in a["objects"]:
            label_counts[lab] += 1
            k, code = fdi_kind(lab); kind_of[lab] = k if k != "tooth" else f"tooth FDI {code}"
            if k == "other" and re.fullmatch(r"\D*\d{1,2}\D*", str(lab)): numeric_non_fdi[lab] += 1
            if kind == "px":
                fw, fh = a["size"] or (rw, rh)
                in_exif_frame = a["size"] is not None and (fw, fh) == (ew, eh) and (ew, eh) != (rw, rh)
                if a["size"] and (fw, fh) not in ((rw, rh), (ew, eh)): size_mismatch += 1
                nb = np.array([b[0] / fw, b[1] / fh, b[2] / fw, b[3] / fh], np.float32)
            else:
                in_exif_frame = False; nb = np.asarray(b, np.float32)
            if ori != 1 and frame == "raw" and not in_exif_frame:
                nb = boxes_raw_to_exif(nb, ori)[0]
            nb = np.clip(nb, 0, 1)
            if nb[2] - nb[0] <= 1e-4 or nb[3] - nb[1] <= 1e-4: continue
            if k == "tooth": boxes.append(nb); fdis.append(code)
            elif k == "lesion": les.append(nb); les_fdi.append(None)
            else: other += 1
        FDI_RECORDS.append({"image_path": ip, "rel": os.path.relpath(ip, FDI_ROOT), "W": ew, "H": eh, "orientation": ori,
                            "format": a["format"], "boxes": np.array(boxes, np.float32).reshape(-1, 4), "fdi": fdis,
                            "lesion_boxes": np.array(les, np.float32).reshape(-1, 4), "n_other": other})

    # folders named by an FDI code that contain images = classification-style data (no boxes)
    fdi_folders = Counter(fdi_from_label(os.path.basename(os.path.dirname(p))) for p in all_imgs)
    fdi_folders.pop(None, None)
    tooth_recs = [r for r in FDI_RECORDS if len(r["boxes"])]
    box_per_class = Counter(c for r in tooth_recs for c in r["fdi"])
    img_per_class = Counter(c for r in tooth_recs for c in set(r["fdi"]))
    FDI_CLASSES = sorted(box_per_class, key=int)
    n_numeric = sum(box_per_class.values()) + sum(numeric_non_fdi.values())

    print("=" * 70 + "\nFDI DATASET — ANNOTATION REPORT\n" + "=" * 70)
    print(f"Images in ZIP                      : {len(all_imgs)}")
    print(f"Annotation formats (images)        : { {f: len(v) for f, v in fmt_imgs.items()} or 'none found'}")
    print(f"Class-name files                   : {[os.path.relpath(p, FDI_ROOT) for p in name_files] or 'none'}")
    print(f"Folders excluded (FDI['EXCLUDE_DIRS']): {FDI['EXCLUDE_DIRS'] or 'none'}")
    print(f"Annotated images                   : {len(by_img)}  | annotation entries without an image: {len(orphans)}")
    print(f"Images with >= 1 FDI tooth box     : {len(tooth_recs)}")
    print(f"Tooth boxes / lesion boxes / other : {sum(box_per_class.values())} / {sum(len(r['lesion_boxes']) for r in FDI_RECORDS)} / "
          f"{sum(r['n_other'] for r in FDI_RECORDS)}")
    print(f"Box frame                          : '{frame}' ({'Label Studio export detected' if is_label_studio else 'config'}) | "
          f"images with EXIF rotation: {rotated} | declared size != image size: {size_mismatch}")
    for pr in problems: print("  NOTE:", pr)
    if unreadable: print(f"  UNREADABLE images excluded ({len(unreadable)}):", unreadable[:5])
    print("\nLabel -> interpretation (first 60 labels):")
    for lab, n in sorted(label_counts.items(), key=lambda z: (-z[1], z[0]))[:60]:
        print(f"  {str(lab)[:40]:40} {n:6}  -> {kind_of.get(lab)}")
    if FDI_CLASSES:
        print("\nFDI classes represented (boxes / images):")
        print("  " + "  ".join(f"{c}:{box_per_class[c]}/{img_per_class[c]}" for c in FDI_CLASSES))
        missing = [str(c) for c in FDI_PERMANENT if str(c) not in box_per_class]
        print(f"  Permanent FDI codes with NO box ({len(missing)}): {missing or 'none'}")
        print(f"  Primary (deciduous) codes present: {[c for c in FDI_CLASSES if int(c) in FDI_PRIMARY] or 'none'}")
        bpi = np.array([len(r["boxes"]) for r in tooth_recs])
        print(f"  Tooth boxes per image: median {np.median(bpi):.0f}, min {bpi.min()}, max {bpi.max()}"
              + ("  <-- few boxes per photo: probably only SOME teeth (e.g. carious ones) are annotated; unannotated teeth are background"
                 if np.median(bpi) <= 3 else ""))
    if fdi_folders:
        print("\nImages inside folders named by an FDI code (classification-style data):", dict(fdi_folders))

    # ---- suitability decision ----
    if not fmt_imgs:
        FDI_STOP_REASON = ("no bounding-box annotation file found (no YOLO txt / COCO / VOC / LabelMe / Label Studio / CSV)"
                           + (" — images are only sorted into FDI-named folders: classification data, a tooth DETECTOR cannot be "
                              "trained from it" if fdi_folders else ""))
    elif not FDI_CLASSES and any(str(l).startswith("<class id") for l in label_counts):
        FDI_STOP_REASON = ("YOLO class ids have no names (no classes.txt / data.yaml / notes.json). The ids are NOT assumed to be FDI "
                           "numbers — add the name file to the ZIP or set FDI['CLASS_ID_MAPPING'] if you know the mapping")
    elif n_numeric and sum(numeric_non_fdi.values()) / n_numeric > 0.10:
        FDI_STOP_REASON = (f"{sum(numeric_non_fdi.values())} numeric labels are not valid FDI codes (e.g. {list(numeric_non_fdi)[:6]}) — the "
                           "label set looks like Universal (1–32) or Palmer numbering; it is NOT converted automatically")
    elif not FDI_CLASSES:
        FDI_STOP_REASON = "no label is a valid two-digit FDI code — the ZIP contains no FDI tooth-number annotations"
    elif len(tooth_recs) < FDI["MIN_ANNOTATED_IMAGES"]:
        FDI_STOP_REASON = f"only {len(tooth_recs)} images with FDI tooth boxes (< FDI['MIN_ANNOTATED_IMAGES'] = {FDI['MIN_ANNOTATED_IMAGES']})"
    elif len(FDI_CLASSES) < 2:
        FDI_STOP_REASON = f"only one FDI class ({FDI_CLASSES}) — tooth numbering cannot be learned"
    FDI_DATA_OK = FDI_STOP_REASON is None
    FDI_PARSE = {"formats": {f: len(v) for f, v in fmt_imgs.items()}, "box_frame": frame, "n_images_in_zip": len(all_imgs),
                 "n_annotated_images": len(by_img), "n_images_with_fdi_boxes": len(tooth_recs),
                 "boxes_per_class": dict(box_per_class), "images_per_class": dict(img_per_class),
                 "n_lesion_boxes": int(sum(len(r["lesion_boxes"]) for r in FDI_RECORDS)), "labels": dict(label_counts),
                 "orphan_annotations": len(orphans), "unreadable_images": unreadable, "problems": problems,
                 "suitability": "object detection (bounding boxes with FDI codes)" if FDI_DATA_OK else FDI_STOP_REASON}
print("\nSUITABILITY:", "OK — object detection with FDI tooth classes" if FDI_DATA_OK else f"NOT SUITABLE — {FDI_STOP_REASON}")

In [ ]:
# ============================================================
# SECTION 24.3 — STAGE 2: IMAGE CACHE (disk memmap) + LEAKAGE-SAFE GROUP SPLIT
# ============================================================
FDI_S_IN = FDI["INPUT_SIZE"]; FDI_GRID = FDI_S_IN // FDI["STRIDE"]; FDI_CACHE_SIDE = int(FDI_S_IN * 1.25)
ROBOFLOW_RE = re.compile(r"^(.*?)_(?:jpe?g|png|bmp)\.rf\.[0-9a-f]+$", re.I)   # Roboflow augmented / re-exported copies

if FDI_DATA_OK:
    fdi_recs = [r for r in FDI_RECORDS if len(r["boxes"])]
    FDI_CLASS_INDEX = {c: i for i, c in enumerate(FDI_CLASSES)}
    for r in fdi_recs:
        r["cls"] = np.array([FDI_CLASS_INDEX[c] for c in r["fdi"]], np.int32)
    cache_npy = os.path.join(CACHE_DIR, f"fdi_images_{FDI_CACHE_SIDE}.npy")
    FDI_X = np.lib.format.open_memmap(cache_npy, mode="w+", dtype=np.uint8, shape=(len(fdi_recs), FDI_CACHE_SIDE, FDI_CACHE_SIDE, 3))
    def _fdi_cache_one(i):
        r = fdi_recs[i]; out = {"md5": file_md5(r["image_path"])}
        try:
            rgb = load_rgb(r["image_path"])                      # EXIF-corrected RGB, the same loader as inference
            FDI_X[i] = cv2.resize(rgb, (FDI_CACHE_SIDE, FDI_CACHE_SIDE), interpolation=cv2.INTER_AREA)
            sm = cv2.resize(rgb, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)
            out.update(readable=True, dhash=dhash64(sm.astype(np.uint8)),
                       gray=float(np.abs(sm - sm.mean(2, keepdims=True)).mean()) < 2.0)   # R≈G≈B -> radiograph-like
        except Exception as e:
            out.update(readable=False, dhash=np.zeros(8, np.uint8), gray=False, error=f"{type(e).__name__}: {e}")
        return out
    with stopwatch("FDI image cache"):
        with ThreadPoolExecutor(DECODE_WORKERS) as ex:
            for r, info in zip(fdi_recs, ex.map(_fdi_cache_one, range(len(fdi_recs)))):
                r.update(info)
    FDI_X.flush()
    bad = [r["rel"] for r in fdi_recs if not r["readable"]]
    if bad: print(f"Unreadable images excluded ({len(bad)}):", bad[:5])
    keep = np.array([r["readable"] for r in fdi_recs])
    gray_frac = float(np.mean([r["gray"] for r in fdi_recs if r["readable"]]))
    print(f"Grey-scale (radiograph-like) images: {gray_frac:.1%}")
    if gray_frac > 0.5 and not FDI.get("ALLOW_MODALITY_MISMATCH", False):
        FDI_DATA_OK = False
        FDI_STOP_REASON = (f"{gray_frac:.0%} of the FDI images are grey-scale (radiographs?), but the pipeline analyses COLOUR intraoral "
                           "photographs — a detector trained on them would not transfer. Set FDI['ALLOW_MODALITY_MISMATCH']=True "
                           "only if these really are photographs.")
        print("STAGE 2 STOPPED:", FDI_STOP_REASON)

if FDI_DATA_OK:
    n = len(fdi_recs)
    # ---- groups: identical files + near-duplicates (dHash) + Roboflow copies + optional patient-ID pattern ----
    D = np.stack([r["dhash"] for r in fdi_recs])
    edges = [hamming_edges(D, D, FDI["GROUP_HAMMING"], same=True)]
    keyed = defaultdict(list)
    for i, r in enumerate(fdi_recs):
        keyed["md5:" + r["md5"]].append(i)
        m = ROBOFLOW_RE.match(stem(r["image_path"]))
        if m: keyed["rf:" + m.group(1).lower()].append(i)
        if FDI["PATIENT_ID_REGEX"]:
            pm = re.search(FDI["PATIENT_ID_REGEX"], os.path.basename(r["image_path"]))
            if pm: keyed["pid:" + pm.group(1)].append(i)
    rr, cc = [], []
    for v in keyed.values():
        rr += [v[0]] * (len(v) - 1); cc += v[1:]
    edges.append((np.array(rr, np.int64), np.array(cc, np.int64)))
    groups = components(n, edges)
    by_md5 = defaultdict(list)
    for r in fdi_recs: by_md5[r["md5"]].append(r)
    conflicts = [(v[0]["rel"], v[1]["rel"]) for v in by_md5.values() if len(v) > 1 and
                 len({tuple(sorted(x["fdi"], key=int)) for x in v}) > 1]
    print(f"Identical image files: {sum(len(v) - 1 for v in by_md5.values() if len(v) > 1)} extra copies | copies whose FDI labels "
          f"DISAGREE: {len(conflicts)}" + (f"  e.g. {conflicts[:3]}" if conflicts else "") + "  (copies always share one split)")
    groups = np.where(keep, groups, -1 - np.arange(n))            # unreadable rows: own group, removed below
    n_rf = sum(len(v) for k, v in keyed.items() if k.startswith("rf:"))
    print(f"Groups: {len(np.unique(groups[keep]))} for {int(keep.sum())} readable images | Roboflow copies linked: {n_rf}"
          + (f" | patient regex {FDI['PATIENT_ID_REGEX']!r} used" if FDI["PATIENT_ID_REGEX"] else " | no patient IDs (FDI['PATIENT_ID_REGEX'] = None)"))
    # stratify on arch: 0 = only upper teeth, 1 = only lower, 2 = both
    arch = np.array([2 if len({int(c) // 10 in (1, 2, 5, 6) for c in r["fdi"]}) > 1 else (0 if int(r["fdi"][0]) // 10 in (1, 2, 5, 6) else 1)
                     for r in fdi_recs])
    idx_ok = np.where(keep)[0]
    tr, va, te = stratified_group_split(arch[idx_ok], groups[idx_ok], SPLIT_FRACS, SEED)
    for name_, pos in (("train", tr), ("val", va), ("test", te)):
        for p_ in pos: fdi_recs[idx_ok[p_]]["split"] = name_
    fdi_split = {s: [r for r in fdi_recs if r.get("split") == s] for s in ("train", "val", "test")}
    assert not ({r["md5"] for r in fdi_split["train"]} & {r["md5"] for r in fdi_split["test"]}), "identical file in train and test"
    assert not ({groups[i] for i, r in enumerate(fdi_recs) if r.get("split") == "train"} &
                {groups[i] for i, r in enumerate(fdi_recs) if r.get("split") == "test"}), "group straddles train/test"
    if min(len(v) for v in fdi_split.values()) == 0:
        FDI_DATA_OK = False; FDI_STOP_REASON = "a split is empty — too few independent image groups"
        print("STAGE 2 STOPPED:", FDI_STOP_REASON)

if FDI_DATA_OK:
    FDI_TRAIN_COUNTS = Counter(FDI_CLASSES[c] for r in fdi_split["train"] for c in r["cls"])
    tab = pd.DataFrame({s: Counter(FDI_CLASSES[c] for r in v for c in r["cls"]) for s, v in fdi_split.items()}).reindex(FDI_CLASSES).fillna(0).astype(int)
    print("Images: " + " | ".join(f"{s} {len(v)}" for s, v in fdi_split.items()))
    print("Tooth boxes per FDI class and split:\n" + tab.T.to_string())
    FDI_UNRELIABLE = [c for c in FDI_CLASSES if FDI_TRAIN_COUNTS.get(c, 0) < FDI["MIN_TRAIN_BOXES_RELIABLE"]]
    print(f"FDI classes with < {FDI['MIN_TRAIN_BOXES_RELIABLE']} training boxes (flagged unreliable):", FDI_UNRELIABLE or "none")
    FDI_SPLIT_FP = fingerprint([f"{r['rel']}|{r['split']}" for r in fdi_recs if r.get("split")] + ["classes:" + ",".join(FDI_CLASSES)])
    SPLIT_FP = globals().get("SPLIT_FP") or {}
    SPLIT_FP["fdi"] = FDI_SPLIT_FP
    for i, r in enumerate(fdi_recs): r["row"] = i                  # row in the FDI_X memmap

In [ ]:
# ============================================================
# SECTION 24.4 — STAGE 2: FDI TOOTH DETECTOR (localizer design + FDI class head) — TRAIN OR LOAD
# ============================================================
_FGX, _FGY = grid_centres(FDI_GRID)

def fdi_encode_targets(boxes, cls, ignore_boxes=()):
    """(G,G,6): state (1 pos | 0 neg | -1 ignore), x1 y1 x2 y2, class index. Positives = cells near a box centre (smallest box
    wins); cells inside a box within IGNORE_RADIUS of its centre, or inside a partly cropped box, are ignored."""
    y = np.zeros((FDI_GRID, FDI_GRID, 6), np.float32)
    best = np.full((FDI_GRID, FDI_GRID), np.inf, np.float32)
    R = FDI["CENTER_RADIUS"]; ri = int(np.ceil(R))
    for (x1, y1, x2, y2), c in zip(boxes, cls):
        area = (x2 - x1) * (y2 - y1); bcx, bcy = (x1 + x2) / 2, (y1 + y2) / 2
        ci, cj = min(int(bcy * FDI_GRID), FDI_GRID - 1), min(int(bcx * FDI_GRID), FDI_GRID - 1)
        for i in range(max(ci - ri, 0), min(ci + ri + 1, FDI_GRID)):
            for j in range(max(cj - ri, 0), min(cj + ri + 1, FDI_GRID)):
                cx, cy = _FGX[i, j], _FGY[i, j]
                inside = x1 < cx < x2 and y1 < cy < y2
                near = abs(cx - bcx) * FDI_GRID <= R and abs(cy - bcy) * FDI_GRID <= R
                if ((i == ci and j == cj) or (inside and near)) and area < best[i, j]:
                    best[i, j] = area; y[i, j] = (1.0, x1, y1, x2, y2, c)
    IR = FDI["IGNORE_RADIUS"]
    for (x1, y1, x2, y2) in boxes:
        bcx, bcy = (x1 + x2) / 2, (y1 + y2) / 2
        m = ((_FGX > x1) & (_FGX < x2) & (_FGY > y1) & (_FGY < y2) & (y[..., 0] == 0) &
             (np.abs(_FGX - bcx) * FDI_GRID <= IR) & (np.abs(_FGY - bcy) * FDI_GRID <= IR))
        y[m, 0] = -1.0
    for (x1, y1, x2, y2) in ignore_boxes:
        m = (_FGX > x1) & (_FGX < x2) & (_FGY > y1) & (_FGY < y2) & (y[..., 0] == 0)
        y[m, 0] = -1.0
    return y

def fdi_augment(img, boxes, cls, rng):
    """Train-only: zoom-crop (boxes cut below POS_VISIBLE_FRAC become 'ignore'), colour jitter. NO flip (FDI side would swap)."""
    ign = np.zeros((0, 4), np.float32)
    if rng.random() < 0.7:
        sw = rng.uniform(FDI["CROP_MIN"], 1.0); sh = float(np.clip(sw * rng.uniform(0.9, 1.1), FDI["CROP_MIN"], 1.0))
        ox, oy = rng.uniform(0, 1 - sw), rng.uniform(0, 1 - sh)
        nb = (boxes - [ox, oy, ox, oy]) / [sw, sh, sw, sh]; cb = np.clip(nb, 0, 1)
        full = (nb[:, 2] - nb[:, 0]) * (nb[:, 3] - nb[:, 1])
        seen = np.clip(cb[:, 2] - cb[:, 0], 0, None) * np.clip(cb[:, 3] - cb[:, 1], 0, None)
        frac = seen / np.maximum(full, 1e-9); ok = frac >= FDI["POS_VISIBLE_FRAC"]
        if ok.any():
            H, W = img.shape[:2]
            img = img[int(round(oy * H)):int(round((oy + sh) * H)), int(round(ox * W)):int(round((ox + sw) * W))]
            ign = cb[(~ok) & (frac > 0)].astype(np.float32)
            boxes, cls = cb[ok].astype(np.float32), cls[ok]
    img = img.astype(np.float32) * rng.uniform(0.85, 1.15)
    m = img.mean(); img = (img - m) * rng.uniform(0.85, 1.15) + m
    g = img.mean(axis=2, keepdims=True); img = g + (img - g) * rng.uniform(0.85, 1.15)
    return np.clip(img, 0, 255), boxes, cls, ign

def fdi_input_from_cache(img):
    """Same contract as loc_prepare: load_rgb -> resize(cache side) -> resize(input) -> MobileNetV2 preprocess_input."""
    return mobilenet_v2_preprocess(cv2.resize(img.astype(np.float32), (FDI_S_IN, FDI_S_IN), interpolation=cv2.INTER_AREA))

def fdi_prepare(rec, augment, rng=None):
    img, boxes, cls, ign = np.asarray(FDI_X[rec["row"]]), rec["boxes"], rec["cls"], ()
    if augment:
        img, boxes, cls, ign = fdi_augment(img, boxes, cls, rng)
    return fdi_input_from_cache(img), fdi_encode_targets(boxes, cls, ign)

def fdi_conv_bn(x, f, k=1):
    x = layers.Conv2D(f, k, padding="same", use_bias=False)(x)
    return layers.ReLU(6.0)(layers.BatchNormalization()(x))

def fdi_sep_bn(x, f):
    x = layers.SeparableConv2D(f, 3, padding="same", use_bias=False)(x)
    return layers.ReLU(6.0)(layers.BatchNormalization()(x))

def build_fdi_detector(n_cls):
    base = keras.applications.MobileNetV2(input_shape=(FDI_S_IN, FDI_S_IN, 3), include_top=False, weights=BACKBONE_WEIGHTS)
    backbone = models.Model(base.input, [base.get_layer("block_13_expand_relu").output, base.get_layer("out_relu").output],
                            name="fdi_mobilenetv2_c4_c5")
    backbone.trainable = False
    inp = layers.Input((FDI_S_IN, FDI_S_IN, 3), name="image")
    c4, c5 = backbone(inp, training=False)
    p = layers.Add()([fdi_conv_bn(c4, 128), layers.UpSampling2D(2)(fdi_conv_bn(c5, 128))])
    p = fdi_sep_bn(p, 128); p = layers.SpatialDropout2D(0.2)(p); p = fdi_sep_bn(p, 128)
    obj = layers.Conv2D(1, 1, bias_initializer=keras.initializers.Constant(-4.6), name="tooth_objectness")(fdi_sep_bn(p, 96))
    box = layers.Conv2D(4, 1, name="tooth_box_ltrb")(fdi_sep_bn(p, 96))
    cls = layers.Conv2D(n_cls, 1, name="fdi_class_logits")(fdi_sep_bn(fdi_sep_bn(p, 128), 128))
    out = layers.Activation("linear", dtype="float32", name="fdi_output")(layers.Concatenate(name="fdi_concat")([obj, box, cls]))
    return models.Model(inp, out, name="fdi_tooth_detector_mobilenetv2"), backbone

_FTGX, _FTGY = tf.constant(_FGX[None]), tf.constant(_FGY[None])
def fdi_decode_boxes_tf(raw):
    d = tf.nn.softplus(raw) / FDI_GRID
    return tf.stack([_FTGX - d[..., 0], _FTGY - d[..., 1], _FTGX + d[..., 2], _FTGY + d[..., 3]], -1)

def fdi_giou_tf(a, b):
    iw = tf.nn.relu(tf.minimum(a[..., 2], b[..., 2]) - tf.maximum(a[..., 0], b[..., 0]))
    ih = tf.nn.relu(tf.minimum(a[..., 3], b[..., 3]) - tf.maximum(a[..., 1], b[..., 1]))
    inter = iw * ih
    union = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1]) + (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1]) - inter + 1e-9
    enc = (tf.maximum(a[..., 2], b[..., 2]) - tf.minimum(a[..., 0], b[..., 0])) * \
          (tf.maximum(a[..., 3], b[..., 3]) - tf.minimum(a[..., 1], b[..., 1])) + 1e-9
    return inter / union - (enc - union) / enc

def _fdi_loss_parts(y_true, y_pred):
    y_pred = tf.cast(y_pred, tf.float32)
    state = y_true[..., 0]
    pos = tf.cast(tf.equal(state, 1.0), tf.float32); valid = tf.cast(state >= 0.0, tf.float32)
    npos = tf.maximum(tf.reduce_sum(pos), 1.0)
    logit = y_pred[..., 0]; p = tf.sigmoid(logit)
    ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=pos, logits=logit)
    p_t = pos * p + (1 - pos) * (1 - p)
    a_t = pos * FDI["FOCAL_ALPHA"] + (1 - pos) * (1 - FDI["FOCAL_ALPHA"]) * FDI["NEG_WEIGHT"]   # partial labels -> softer background
    obj = tf.reduce_sum(a_t * tf.pow(1 - p_t, FDI["FOCAL_GAMMA"]) * ce * valid) / npos
    box = tf.reduce_sum((1.0 - fdi_giou_tf(fdi_decode_boxes_tf(y_pred[..., 1:5]), y_true[..., 1:5])) * pos) / npos
    cls = tf.reduce_sum(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=tf.cast(y_true[..., 5], tf.int32),
                                                                       logits=y_pred[..., 5:]) * pos) / npos
    return obj, box, cls

def fdi_loss(y_true, y_pred):
    o, b, c = _fdi_loss_parts(y_true, y_pred)
    return o + FDI["BOX_LOSS_WEIGHT"] * b + FDI["CLS_LOSS_WEIGHT"] * c
def fdi_obj_loss(y_true, y_pred): return _fdi_loss_parts(y_true, y_pred)[0]
def fdi_box_loss(y_true, y_pred): return _fdi_loss_parts(y_true, y_pred)[1]
def fdi_cls_loss(y_true, y_pred): return _fdi_loss_parts(y_true, y_pred)[2]

def fdi_dataset(recs, augment, seed=0):
    sig = (tf.TensorSpec((FDI_S_IN, FDI_S_IN, 3), tf.float32), tf.TensorSpec((FDI_GRID, FDI_GRID, 6), tf.float32))
    if augment:
        def gen(worker_seed):
            rng = np.random.default_rng(SEED + int(worker_seed))
            while True:
                yield fdi_prepare(recs[rng.integers(len(recs))], True, rng)
        return (tf.data.Dataset.range(FDI["GEN_WORKERS"])
                .interleave(lambda s: tf.data.Dataset.from_generator(gen, args=(s,), output_signature=sig),
                            cycle_length=FDI["GEN_WORKERS"], num_parallel_calls=FDI["GEN_WORKERS"])
                .batch(FDI["BATCH"]).prefetch(AUTOTUNE))
    def gen_fixed():
        for r in recs:
            yield fdi_prepare(r, False)
    return tf.data.Dataset.from_generator(gen_fixed, output_signature=sig).batch(FDI["BATCH"]).prefetch(AUTOTUNE)

def fdi_predict_raw(model, recs):
    """Raw (G,G,5+C) outputs for records (no augmentation), batch by batch — nothing large is held in memory."""
    out = []
    for s in range(0, len(recs), FDI["BATCH"]):
        xb = np.stack([fdi_input_from_cache(np.asarray(FDI_X[r["row"]])) for r in recs[s:s + FDI["BATCH"]]])
        out.append(np.asarray(model(xb, training=False), np.float32))
    return np.concatenate(out) if out else np.zeros((0, FDI_GRID, FDI_GRID, 5 + len(FDI_CLASSES)), np.float32)

fdi_hist = {}
if FDI_DATA_OK:
    FDI_CKPT = f"{CKPT_DIR}/fdi_detector_best.weights.h5"
    FDI_TRAINED = decide("fdi")
    if not FDI_TRAINED:
        fdi_model = load_keras(MODEL_PATHS["fdi"])
        saved_classes = read_json(MODEL_META_PATHS["fdi"]).get("class_names")
        if saved_classes != FDI_CLASSES or fdi_model.output_shape[-1] != 5 + len(FDI_CLASSES):
            print("Saved FDI detector has a different class list -> retraining.")
            FDI_TRAINED = True
        else:
            fdi_hist = read_json(MODEL_META_PATHS["fdi"]).get("training_history", {})
            print("Loaded the saved FDI detector (trained on this split) — no retraining:", MODEL_PATHS["fdi"])
    if FDI_TRAINED:
        keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
        fdi_model, fdi_backbone = build_fdi_detector(len(FDI_CLASSES))
        assert tuple(fdi_model.output_shape[1:]) == (FDI_GRID, FDI_GRID, 5 + len(FDI_CLASSES))
        print(f"FDI detector: {fdi_model.count_params():,} parameters | grid {FDI_GRID}x{FDI_GRID} | {len(FDI_CLASSES)} FDI classes")
        tr_ds, va_ds = fdi_dataset(fdi_split["train"], True), fdi_dataset(fdi_split["val"], False)
        # >= MIN_STEPS_PER_EPOCH: on small datasets one pass is only a few steps, and the head's BatchNorm moving statistics
        # (used at validation / inference) would not converge within EPOCHS_A epochs
        steps = FDI["STEPS_PER_EPOCH"] or max(FDI["MIN_STEPS_PER_EPOCH"], math.ceil(len(fdi_split["train"]) / FDI["BATCH"]))
        def _compile(lr):
            fdi_model.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=5.0), loss=fdi_loss,
                              metrics=[fdi_obj_loss, fdi_box_loss, fdi_cls_loss])
        _compile(FDI["LR_A"])
        with stopwatch("FDI detector training A (frozen backbone)"):
            h = fdi_model.fit(tr_ds, steps_per_epoch=steps, epochs=FDI["EPOCHS_A"], validation_data=va_ds, verbose=2,
                              callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
                                         ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
                                         ModelCheckpoint(FDI_CKPT, monitor="val_loss", save_best_only=True, save_weights_only=True)])
        fdi_hist = {k: list(map(float, v)) for k, v in h.history.items()}
        fdi_model.load_weights(FDI_CKPT)
        best_a, n_a = float(np.min(fdi_hist["val_loss"])), len(fdi_hist["loss"])
        if CAN_FINE_TUNE and len(fdi_split["train"]) >= FDI["MIN_TRAIN_IMAGES_FOR_FINETUNE"] and FDI["EPOCHS_B"] > 0:
            fdi_backbone.trainable = True
            names_ = [l.name for l in fdi_backbone.layers]; cut = names_.index(FDI["UNFREEZE_FROM"])
            for i, l in enumerate(fdi_backbone.layers):
                l.trainable = i >= cut and not isinstance(l, layers.BatchNormalization)
            _compile(FDI["LR_B"])
            with stopwatch("FDI detector training B (fine-tuning)"):
                h = fdi_model.fit(tr_ds, steps_per_epoch=steps, initial_epoch=n_a, epochs=n_a + FDI["EPOCHS_B"],
                                  validation_data=va_ds, verbose=2,
                                  callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                                             ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
                                             ModelCheckpoint(FDI_CKPT, monitor="val_loss", save_best_only=True,
                                                             save_weights_only=True, initial_value_threshold=best_a)])
            for k, v in h.history.items(): fdi_hist.setdefault(k, []).extend(map(float, v))
            fdi_model.load_weights(FDI_CKPT)
            print("Fine-tuning best val_loss %.4f vs frozen-backbone %.4f" % (np.min(h.history["val_loss"]), best_a))
        else:
            print("FDI fine-tuning skipped (no GPU or too few training images).")
        fdi_model.save(MODEL_PATHS["fdi"])
        print("Saved", MODEL_PATHS["fdi"])
    if fdi_hist:
        fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
        for ax, k in zip(axes, ["loss", "fdi_obj_loss", "fdi_box_loss", "fdi_cls_loss"]):
            ax.plot(fdi_hist.get(k, []), label="train"); ax.plot(fdi_hist.get("val_" + k, []), label="validation")
            ax.set_title(f"FDI detector: {k}"); ax.legend(); ax.grid(alpha=.3)
        plt.tight_layout(); plt.show()
else:
    print("Stage 2 detector not trained:", FDI_STOP_REASON)

In [ ]:
# ============================================================
# SECTION 24.5 — STAGE 2: EVALUATION (threshold on VALIDATION, test ONCE) + SAVE METADATA
# ============================================================
def fdi_match(pb, ps, gb, iou_thr, pl=None, gl=None):
    """Greedy score-ordered matching; with labels (pl, gl) a match also needs the same FDI class.
    -> list of (pred_idx, score, is_tp, gt_idx, iou)."""
    ious = iou_matrix(pb, gb); used, out = set(), []
    for k in np.argsort(-ps):
        cand = [(ious[k, g], g) for g in range(len(gb)) if g not in used and ious[k, g] >= iou_thr
                and (pl is None or pl[k] == gl[g])]
        if cand:
            iou, g = max(cand); used.add(g); out.append((k, float(ps[k]), True, g, float(iou)))
        else:
            out.append((k, float(ps[k]), False, -1, 0.0))
    return out

def fdi_ap(scores, tps, n_gt):
    """Area under the interpolated precision-recall curve (all points, VOC2010+/COCO style)."""
    if n_gt == 0 or not scores: return 0.0
    o = np.argsort(-np.array(scores)); t = np.array(tps)[o]
    tp, fp = np.cumsum(t), np.cumsum(~t)
    rec = tp / n_gt; prec = tp / np.maximum(tp + fp, 1e-9)
    mrec = np.r_[0, rec, 1]; mpre = np.r_[0, prec, 0]
    for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1]))

def fdi_decode_all(raws, thr):
    """-> per image (boxes, ranking score = objectness × number probability, predicted class index, objectness)."""
    out = []
    for raw in raws:
        b, s, p = decode_fdi_detections(raw, thr, FDI["NMS_IOU"], FDI["MAX_DETECTIONS"])
        k = p.argmax(1) if len(p) else np.zeros(0, int)
        out.append((b, s * (p.max(1) if len(p) else 0), k, s))
    return out

def fdi_map(recs, dets, iou_thr):
    """Per-class AP (classes with >= 1 GT box in recs) -> (mAP, {class: AP}); + class-agnostic tooth AP."""
    ap = {}
    for c in sorted({int(x) for r in recs for x in r["cls"]}):
        sc, tps, n = [], [], 0
        for r, (b, s, k, _) in zip(recs, dets):
            gb = r["boxes"][r["cls"] == c]; m = k == c; n += len(gb)
            for _, score, ok, _, _ in fdi_match(b[m], s[m], gb, iou_thr):
                sc.append(score); tps.append(ok)
        ap[FDI_CLASSES[c]] = fdi_ap(sc, tps, n)
    sc, tps, n = [], [], 0
    for r, (b, s, k, obj) in zip(recs, dets):
        n += len(r["boxes"])
        for _, score, ok, _, _ in fdi_match(b, obj, r["boxes"], iou_thr):
            sc.append(score); tps.append(ok)
    return (float(np.mean(list(ap.values()))) if ap else 0.0), ap, fdi_ap(sc, tps, n)

def fdi_eval_at(recs, raws, thr, iou_thr=0.5, cthr=None):
    """At one score threshold: tooth P/R/F1 (box only), FDI P/R/F1 (box + correct number), number accuracy on matched teeth,
    and — with cthr — accuracy / coverage of the numbers that are NOT flagged uncertain."""
    tp_b = fp_b = fn_b = tp_c = fp_c = fn_c = 0; n_match = n_right = n_conf = n_conf_right = 0
    for r, raw in zip(recs, raws):
        b, s, p = decode_fdi_detections(raw, thr, FDI["NMS_IOU"], FDI["MAX_DETECTIONS"])
        k = p.argmax(1) if len(p) else np.zeros(0, int)
        mb = fdi_match(b, s, r["boxes"], iou_thr)
        found = [(kk, g) for kk, _, ok, g, _ in mb if ok]
        tp_b += len(found); fp_b += len(b) - len(found); fn_b += len(r["boxes"]) - len(found)
        mc = fdi_match(b, s, r["boxes"], iou_thr, k, r["cls"]); fc = sum(ok for _, _, ok, _, _ in mc)
        tp_c += fc; fp_c += len(b) - fc; fn_c += len(r["boxes"]) - fc
        for kk, g in found:
            n_match += 1; right = k[kk] == r["cls"][g]; n_right += right
            if cthr is not None and p[kk].max() >= cthr:
                n_conf += 1; n_conf_right += right
    f1 = lambda tp, fp, fn: 2 * tp / max(2 * tp + fp + fn, 1)
    out = {"threshold": float(thr), "tooth_precision": tp_b / max(tp_b + fp_b, 1), "tooth_recall": tp_b / max(tp_b + fn_b, 1),
           "tooth_f1": f1(tp_b, fp_b, fn_b), "fdi_precision": tp_c / max(tp_c + fp_c, 1), "fdi_recall": tp_c / max(tp_c + fn_c, 1),
           "fdi_f1": f1(tp_c, fp_c, fn_c), "number_accuracy_on_matched": n_right / max(n_match, 1), "n_matched": n_match,
           "tp_fdi": tp_c, "fp_fdi": fp_c, "fn_fdi": fn_c}
    if cthr is not None:
        out.update(confident_coverage=n_conf / max(n_match, 1), confident_number_accuracy=n_conf_right / max(n_conf, 1))
    return out

if FDI_DATA_OK:
    raw_val = fdi_predict_raw(fdi_model, fdi_split["val"])
    sweep = [fdi_eval_at(fdi_split["val"], raw_val, t) for t in np.round(np.arange(0.05, 0.96, 0.05), 2)]
    best = max(sweep, key=lambda e: (e["fdi_f1"], e["threshold"]))
    FDI_THRESHOLD = best["threshold"] if best["fdi_f1"] > 0 else FDI["DEFAULT_SCORE_THRESHOLD"]
    if best["fdi_f1"] == 0:
        print("WARNING: no correct validation tooth+number at IoU 0.5 — the FDI detector is NOT reliable.")
    print("Validation sweep (score thr: FDI-F1 / tooth-F1 / number acc. on matched):")
    for e in sweep[::2]:
        print(f"  {e['threshold']:.2f}: {e['fdi_f1']:.3f} / {e['tooth_f1']:.3f} / {e['number_accuracy_on_matched']:.3f}")
    val_at = fdi_eval_at(fdi_split["val"], raw_val, FDI_THRESHOLD, cthr=FDI["CLASS_CONFIDENCE_THRESHOLD"])
    v_map50, _, _ = fdi_map(fdi_split["val"], fdi_decode_all(raw_val, FDI["AP_MIN_SCORE"]), 0.5)
    print(f"Score threshold (best FDI-F1 on validation): {FDI_THRESHOLD:.2f} | val mAP@0.5 {v_map50:.3f} | "
          f"val numbers with probability >= {FDI['CLASS_CONFIDENCE_THRESHOLD']:.2f}: coverage {val_at['confident_coverage']:.1%}, "
          f"accuracy {val_at['confident_number_accuracy']:.1%} (vs {val_at['number_accuracy_on_matched']:.1%} for all)")

    # ---------- TEST: predicted once ----------
    raw_test = fdi_predict_raw(fdi_model, fdi_split["test"])
    dets_test = fdi_decode_all(raw_test, FDI["AP_MIN_SCORE"])
    map50, ap50, tooth_ap50 = fdi_map(fdi_split["test"], dets_test, 0.5)
    maps = [fdi_map(fdi_split["test"], dets_test, t)[0] for t in np.round(np.arange(0.5, 0.96, 0.05), 2)]
    test_at = fdi_eval_at(fdi_split["test"], raw_test, FDI_THRESHOLD, cthr=FDI["CLASS_CONFIDENCE_THRESHOLD"])
    n_tb = sum(len(r["boxes"]) for r in fdi_split["test"])
    FDI_TEST = {"mAP50": map50, "mAP50_95": float(np.mean(maps)), "tooth_AP50_class_agnostic": tooth_ap50, "AP50_per_class": ap50,
                "n_test_images": len(fdi_split["test"]), "n_test_teeth": n_tb, **{f"at_threshold_{k}": v for k, v in test_at.items()}}
    print("\n" + "=" * 64 + f"\nSTAGE 2 FDI DETECTOR — TEST ({len(fdi_split['test'])} images, {n_tb} tooth boxes)\n" + "=" * 64)
    print(f"mAP@0.5 {map50:.3f} | mAP@[0.5:0.95] {np.mean(maps):.3f} | class-agnostic tooth AP@0.5 {tooth_ap50:.3f}")
    print(f"@score {FDI_THRESHOLD:.2f}, IoU >= 0.5: tooth P {test_at['tooth_precision']:.3f} R {test_at['tooth_recall']:.3f} "
          f"F1 {test_at['tooth_f1']:.3f} | tooth+number P {test_at['fdi_precision']:.3f} R {test_at['fdi_recall']:.3f} F1 {test_at['fdi_f1']:.3f}")
    print(f"FDI-number accuracy on matched teeth: {test_at['number_accuracy_on_matched']:.3f} (n={test_at['n_matched']}) | "
          f"numbers NOT flagged uncertain: coverage {test_at['confident_coverage']:.1%}, accuracy {test_at['confident_number_accuracy']:.1%}")
    ap_tab = pd.DataFrame({"AP@0.5": ap50, "train boxes": {c: FDI_TRAIN_COUNTS.get(c, 0) for c in ap50},
                           "test boxes": Counter(FDI_CLASSES[c] for r in fdi_split["test"] for c in r["cls"])}).fillna(0)
    ap_tab["reliable?"] = np.where(ap_tab.index.isin(FDI_UNRELIABLE), f"NO (<{FDI['MIN_TRAIN_BOXES_RELIABLE']} train boxes)", "yes")
    print("\nPer-class AP@0.5 (test):\n" + ap_tab.round(3).to_string())
    print("NOTE: the dataset labels only teeth whose FDI number was certain. A correct detection of an UNLABELLED tooth counts as a "
          "false positive here, so precision and mAP are lower bounds; number accuracy on matched teeth is not affected.")

    PIPELINE["models"]["fdi"] = fdi_model
    PIPELINE["unavailable"].pop("fdi", None)
    PIPELINE["meta"]["fdi"] = write_model_meta("fdi", {
        "model": "Stage 2 FDI tooth detector (tooth boxes + FDI two-digit number)",
        "architecture": "anchor-free grid detector, MobileNetV2 C4+C5 backbone (cavity-localizer design) + FDI softmax class head",
        "dataset": os.path.basename(FDI_ZIP), "class_names": FDI_CLASSES, "num_classes": len(FDI_CLASSES),
        "input_size": FDI_S_IN, "cache_side": FDI_CACHE_SIDE, "stride": FDI["STRIDE"], "grid": FDI_GRID,
        "preprocessing": "load_rgb -> cv2.resize(cache_side, INTER_AREA) -> cv2.resize(input_size, INTER_AREA) -> mobilenet_v2 preprocess_input (= loc_prepare)",
        "output": "ch0 objectness logit | ch1-4 softplus(l,t,r,b) in grid-cell units | ch5.. FDI class logits (order = class_names)",
        "score_threshold": FDI_THRESHOLD, "score_threshold_rule": "best tooth+number F1 @IoU0.5 on validation",
        "class_confidence_threshold": FDI["CLASS_CONFIDENCE_THRESHOLD"], "nms_iou": FDI["NMS_IOU"],
        "max_detections": FDI["MAX_DETECTIONS"], "assoc_min_overlap": FDI["ASSOC_MIN_OVERLAP"],
        "assoc_ambiguity_ratio": FDI["ASSOC_AMBIGUITY_RATIO"], "unreliable_classes": FDI_UNRELIABLE,
        "train_boxes_per_class": dict(FDI_TRAIN_COUNTS), "box_frame": FDI_PARSE.get("box_frame"),
        "training_config": FDI, "training_history": fdi_hist, "val_metrics": {"mAP50": v_map50, **val_at}, "test_metrics": FDI_TEST,
        "dataset_report": FDI_PARSE, "split_images": {s: len(v) for s, v in fdi_split.items()},
        "split_fingerprint": FDI_SPLIT_FP, "model_path": MODEL_PATHS["fdi"],
    })
elif os.path.exists(MODEL_PATHS["fdi"]) and os.path.exists(MODEL_META_PATHS["fdi"]):
    print("Stage 2 evaluation skipped in this session:", FDI_STOP_REASON)
    print("A previously saved FDI detector exists and is loaded by load_all_models() (its stored test metrics apply):",
          MODEL_PATHS["fdi"])
else:
    PIPELINE["unavailable"]["fdi"] = f"Stage 2 not trained: {FDI_STOP_REASON}"
    print("Stage 2 evaluation skipped:", FDI_STOP_REASON)

In [ ]:
# ============================================================
# SECTION 24.6 — LESION -> TOOTH ASSOCIATION: SELF-CHECK, TEST DEMO, MEASURED ACCURACY (if the ZIP has caries boxes)
# ============================================================
# (a) Rule self-check on hand-made boxes (no model involved): verifies the association logic itself.
_t = [{"tooth_id": 1, "box_norm": [0.10, 0.20, 0.30, 0.80], "fdi": "16", "fdi_confidence": 0.9, "fdi_uncertain": False},
      {"tooth_id": 2, "box_norm": [0.30, 0.20, 0.50, 0.80], "fdi": "15", "fdi_confidence": 0.8, "fdi_uncertain": False},
      {"tooth_id": 3, "box_norm": [0.60, 0.20, 0.80, 0.80], "fdi": "14", "fdi_confidence": 0.3, "fdi_uncertain": True}]
_l = [[0.15, 0.40, 0.20, 0.50],     # inside tooth 16                       -> 16
      [0.27, 0.40, 0.33, 0.50],     # half on 16, half on 15                -> ambiguous 16 / 15
      [0.65, 0.40, 0.70, 0.50],     # inside a tooth whose number is uncertain -> Unknown / uncertain
      [0.90, 0.05, 0.95, 0.10]]     # outside every tooth                   -> Unknown / uncertain
_a = associate_lesions_with_teeth(_l, _t, FDI["ASSOC_MIN_OVERLAP"], FDI["ASSOC_AMBIGUITY_RATIO"])
assert [x["fdi_status"] for x in _a] == ["ASSIGNED", "AMBIGUOUS", "TOOTH_NUMBER_UNCERTAIN", "NO_TOOTH"], _a
assert _a[0]["fdi_tooth"] == "16" and _a[1]["fdi_tooth"] is None and _a[2]["fdi_tooth"] is None and _a[3]["fdi_tooth"] is None
assert associate_lesions_with_teeth(_l, [], 0.3, 0.8)[0]["fdi_status"] == "NO_TOOTH"
assert associate_lesions_with_teeth([], _t, 0.3, 0.8) == []
print("Association rule self-check passed:", [x["fdi_display"] for x in _a])

FDI_ASSOC_TEST = None
if FDI_DATA_OK:
    # (b) demo on held-out TEST photographs: ground-truth teeth (white) vs detected teeth (orange)
    fig, axes = plt.subplots(1, min(3, len(fdi_split["test"])), figsize=(18, 6), squeeze=False)
    for ax, r in zip(axes.ravel(), fdi_split["test"][:3]):
        rgb = load_rgb(r["image_path"]); H, W = rgb.shape[:2]; t = max(2, int(max(H, W) / 500)); vis = rgb.copy()
        for gb, c in zip(r["boxes"], r["fdi"]):
            p = to_px(gb, W, H); cv2.rectangle(vis, p[:2], p[2:], (255, 255, 255), t)
        for tt in detect_teeth(rgb):
            x1, y1, x2, y2 = tt["box_xyxy_px"]; cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 140, 0), t)
            cv2.putText(vis, tt["fdi"] if not tt["fdi_uncertain"] else "?", (x1 + 4, y1 + 12 * t),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45 * t, (255, 140, 0), max(1, t // 2), cv2.LINE_AA)
        ax.imshow(vis); ax.axis("off"); ax.set_title(f"{os.path.basename(r['image_path'])}\nGT FDI: {sorted(r['fdi'], key=int)}", fontsize=8)
    plt.suptitle("TEST photographs: white = ground-truth teeth, orange = Stage 2 prediction ('?' = number uncertain)")
    plt.tight_layout(); plt.show()

    # (c) measured association accuracy: caries boxes in the FDI ZIP -> ground-truth tooth (by the same overlap rule on the
    #     GROUND-TRUTH tooth boxes) vs the tooth found with the PREDICTED teeth. Only lesions with an unambiguous GT tooth count.
    raw_ids = {id(r) for r in fdi_split["test"]}
    test_les = [r for r in FDI_RECORDS if len(r["lesion_boxes"]) and id(r) in raw_ids]
    n = right = unknown = 0
    for r in test_les:
        gt_teeth = [{"tooth_id": i, "box_norm": b.tolist(), "fdi": c, "fdi_confidence": 1.0, "fdi_uncertain": False}
                    for i, (b, c) in enumerate(zip(r["boxes"], r["fdi"]))]
        gt = associate_lesions_with_teeth(r["lesion_boxes"], gt_teeth, FDI["ASSOC_MIN_OVERLAP"], FDI["ASSOC_AMBIGUITY_RATIO"])
        pr = associate_lesions_with_teeth(r["lesion_boxes"], detect_teeth(load_rgb(r["image_path"])),
                                          FDI["ASSOC_MIN_OVERLAP"], FDI["ASSOC_AMBIGUITY_RATIO"])
        for g, p_ in zip(gt, pr):
            if g["fdi_status"] != "ASSIGNED": continue
            n += 1; right += p_["fdi_tooth"] == g["fdi_tooth"]; unknown += p_["fdi_tooth"] is None
    if n:
        FDI_ASSOC_TEST = {"n_lesions": n, "accuracy": right / n, "unknown_rate": unknown / n,
                          "accuracy_when_assigned": right / max(n - unknown, 1),
                          "definition": "GT caries box -> GT tooth (overlap rule on GT tooth boxes) vs tooth from predicted teeth"}
        print(f"Lesion -> tooth association on TEST caries boxes (n={n}): accuracy {FDI_ASSOC_TEST['accuracy']:.3f} | "
              f"reported Unknown/uncertain {FDI_ASSOC_TEST['unknown_rate']:.1%} | accuracy when a number was given "
              f"{FDI_ASSOC_TEST['accuracy_when_assigned']:.3f}")
    else:
        print("The FDI ZIP has no caries boxes on test images that fall on an annotated tooth -> the association accuracy cannot be "
              "measured on this data (the rule itself is verified by the self-check above).")
    PIPELINE["meta"]["fdi"] = write_model_meta("fdi", {**PIPELINE["meta"]["fdi"], "association_test": FDI_ASSOC_TEST})

<br>

---
# PART 7 — CAVITY / LESION LOCALIZATION

---
# CAVITY / LESION LOCALIZATION
## 25. Existing cavity localization (TensorFlow/Keras, kept from your notebook)
The localizer answers only **"WHERE is a lesion?"** (class-agnostic). It never assigns a G.V. Black class or an ICDAS score — that is the
job of CNN 2 and CNN 3.

* Architecture unchanged: MobileNetV2 C4 (`block_13_expand_relu`, stride 16) + upsampled C5 → separable-conv neck → objectness logit +
  4 softplus box distances per grid cell → `(26, 26, 5)`. Focal loss + GIoU loss, TF NMS at inference.
* Trained on the G.V. Black box annotations (all classes merged into "lesion"), on the photographs of the **shared split**.
* Train-only augmentation: lesion-preserving zoom-crop, horizontal flip (safe: no class or laterality is predicted), colour jitter.

In [ ]:
# ============================================================
# SECTION 25.1 — LOCALIZER DATA
# ============================================================
gvb_records = [r for r in pool if r["gvb_status"] == "annotated"]
gvb_train = [r for r in gvb_records if r["split"] == "train"]
gvb_val   = [r for r in gvb_records if r["split"] == "val"]
gvb_test  = [r for r in gvb_records if r["split"] == "test"]
print(f"Annotated photographs: train {len(gvb_train)} | val {len(gvb_val)} | test {len(gvb_test)}")
assert gvb_train and gvb_val and gvb_test, "each split needs annotated G.V. Black photographs"

S_IN = LOC["INPUT_SIZE"]; GRID = S_IN // LOC["STRIDE"]; CACHE_SIDE = int(S_IN * 1.25)
STEPS_PER_EPOCH = LOC["STEPS_PER_EPOCH"] or max(1, math.ceil(len(gvb_train) / LOC["BATCH"]))

with stopwatch("localizer image cache"):
    def _cache_one(r):
        return r["image_path"], cv2.resize(load_rgb(r["image_path"]), (CACHE_SIDE, CACHE_SIDE), interpolation=cv2.INTER_AREA)
    with ThreadPoolExecutor(DECODE_WORKERS) as ex:
        loc_cache = dict(ex.map(_cache_one, gvb_records))

_GX, _GY = grid_centres(GRID)
def encode_targets(boxes):
    y = np.zeros((GRID, GRID, 5), np.float32)
    best = np.full((GRID, GRID), np.inf, np.float32)
    R = LOC["CENTER_RADIUS"]; ri = int(np.ceil(R))
    for (x1, y1, x2, y2) in boxes:
        area = (x2 - x1) * (y2 - y1); bcx, bcy = (x1 + x2) / 2, (y1 + y2) / 2
        ci, cj = min(int(bcy * GRID), GRID - 1), min(int(bcx * GRID), GRID - 1)
        for i in range(max(ci - ri, 0), min(ci + ri + 1, GRID)):
            for j in range(max(cj - ri, 0), min(cj + ri + 1, GRID)):
                cx, cy = _GX[i, j], _GY[i, j]
                inside = x1 < cx < x2 and y1 < cy < y2
                near = abs(cx - bcx) * GRID <= R and abs(cy - bcy) * GRID <= R
                if ((i == ci and j == cj) or (inside and near)) and area < best[i, j]:
                    best[i, j] = area; y[i, j] = (1.0, x1, y1, x2, y2)
    for (x1, y1, x2, y2) in boxes:
        m = (_GX > x1) & (_GX < x2) & (_GY > y1) & (_GY < y2) & (y[..., 0] == 0)
        y[m, 0] = -1.0
    return y

def augment_sample(img, boxes, rng):
    vis = LOC["POS_VISIBLE_FRAC"]
    for _ in range(10):
        sw = rng.uniform(0.70, 1.0); sh = float(np.clip(sw * rng.uniform(0.9, 1.1), 0.6, 1.0))
        ox, oy = rng.uniform(0, 1 - sw), rng.uniform(0, 1 - sh)
        nb = (boxes - [ox, oy, ox, oy]) / [sw, sh, sw, sh]; cb = np.clip(nb, 0, 1)
        full = (nb[:, 2] - nb[:, 0]) * (nb[:, 3] - nb[:, 1])
        seen = np.clip(cb[:, 2] - cb[:, 0], 0, None) * np.clip(cb[:, 3] - cb[:, 1], 0, None)
        frac = seen / np.maximum(full, 1e-9)
        if np.all((frac >= vis) | (frac <= 0)) and np.any(frac >= vis):
            H, W = img.shape[:2]
            img = img[int(round(oy * H)):int(round((oy + sh) * H)), int(round(ox * W)):int(round((ox + sw) * W))]
            boxes = cb[frac >= vis].astype(np.float32)
            break
    if rng.random() < LOC["HFLIP_PROB"]:
        img = img[:, ::-1]
        boxes = np.stack([1 - boxes[:, 2], boxes[:, 1], 1 - boxes[:, 0], boxes[:, 3]], 1).astype(np.float32)
    img = img.astype(np.float32) * rng.uniform(0.85, 1.15)
    m = img.mean(); img = (img - m) * rng.uniform(0.85, 1.15) + m
    g = img.mean(axis=2, keepdims=True); img = g + (img - g) * rng.uniform(0.85, 1.15)
    return np.clip(img, 0, 255), boxes

def loc_input_from_cache(img):
    return mobilenet_v2_preprocess(cv2.resize(img.astype(np.float32), (S_IN, S_IN), interpolation=cv2.INTER_AREA))

def prepare_loc(rec, augment, rng=None):
    img, boxes = loc_cache[rec["image_path"]], rec["boxes"]
    if augment:
        img, boxes = augment_sample(img, boxes, rng)
    return loc_input_from_cache(img), encode_targets(boxes), boxes

loc_sig = (tf.TensorSpec((S_IN, S_IN, 3), tf.float32), tf.TensorSpec((GRID, GRID, 5), tf.float32))
def loc_generator(worker_seed):
    rng = np.random.default_rng(SEED + int(worker_seed))
    while True:
        x, y, _ = prepare_loc(gvb_train[rng.integers(len(gvb_train))], True, rng)
        yield x, y

loc_train_ds = (tf.data.Dataset.range(LOC["GEN_WORKERS"])
                .interleave(lambda s: tf.data.Dataset.from_generator(loc_generator, args=(s,), output_signature=loc_sig),
                            cycle_length=LOC["GEN_WORKERS"], num_parallel_calls=LOC["GEN_WORKERS"])
                .batch(LOC["BATCH"]).prefetch(AUTOTUNE))

def stack_split(recs):
    xs, ys = zip(*[prepare_loc(r, False)[:2] for r in recs])
    return np.stack(xs), np.stack(ys)
X_loc_val, Y_loc_val = stack_split(gvb_val)
X_loc_test, Y_loc_test = stack_split(gvb_test)
loc_val_ds = tf.data.Dataset.from_tensor_slices((X_loc_val, Y_loc_val)).batch(LOC["BATCH"]).prefetch(AUTOTUNE)
print("Grid", GRID, "x", GRID, "| steps/epoch", STEPS_PER_EPOCH, "| val", X_loc_val.shape, "| test", X_loc_test.shape)

rng_d = np.random.default_rng(1)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax in axes:
    x, y, b = prepare_loc(gvb_train[rng_d.integers(len(gvb_train))], True, rng_d)
    vis = ((x + 1) * 127.5).astype(np.uint8).copy()
    for bb in b:
        p = (bb * S_IN).astype(int); cv2.rectangle(vis, tuple(map(int, p[:2])), tuple(map(int, p[2:])), (0, 255, 0), 2)
    ii, jj = np.where(y[..., 0] == 1)
    ax.imshow(vis); ax.scatter((jj + .5) * LOC["STRIDE"], (ii + .5) * LOC["STRIDE"], s=10, c="red"); ax.axis("off")
plt.suptitle("Localizer training view: augmented ground-truth boxes + positive cells"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# SECTION 25.2 — LOCALIZER MODEL + LOSS + TRAINING (unchanged architecture)
# ============================================================
def conv_bn(x, f, k=1):
    x = layers.Conv2D(f, k, padding="same", use_bias=False)(x)
    return layers.ReLU(6.0)(layers.BatchNormalization()(x))

def sep_bn(x, f):
    x = layers.SeparableConv2D(f, 3, padding="same", use_bias=False)(x)
    return layers.ReLU(6.0)(layers.BatchNormalization()(x))

def build_localizer():
    base = keras.applications.MobileNetV2(input_shape=(S_IN, S_IN, 3), include_top=False, weights=BACKBONE_WEIGHTS)
    backbone = models.Model(base.input, [base.get_layer("block_13_expand_relu").output,
                                         base.get_layer("out_relu").output], name="mobilenetv2_c4_c5")
    backbone.trainable = False
    inp = layers.Input((S_IN, S_IN, 3), name="image")
    c4, c5 = backbone(inp, training=False)
    p = layers.Add()([conv_bn(c4, 128), layers.UpSampling2D(2)(conv_bn(c5, 128))])
    p = sep_bn(p, 128); p = layers.SpatialDropout2D(0.2)(p); p = sep_bn(p, 128)
    obj = layers.Conv2D(1, 1, bias_initializer=keras.initializers.Constant(-4.6), name="objectness")(sep_bn(p, 96))
    box = layers.Conv2D(4, 1, name="box_ltrb")(sep_bn(p, 96))
    out = layers.Concatenate(name="concat")([obj, box])
    out = layers.Activation("linear", dtype="float32", name="loc_output")(out)
    return models.Model(inp, out, name="cavity_localizer_mobilenetv2"), backbone

_TGX, _TGY = tf.constant(_GX[None]), tf.constant(_GY[None])
def decode_boxes_tf(raw):
    d = tf.nn.softplus(raw) / GRID
    return tf.stack([_TGX - d[..., 0], _TGY - d[..., 1], _TGX + d[..., 2], _TGY + d[..., 3]], -1)

def giou_tf(a, b):
    iw = tf.nn.relu(tf.minimum(a[..., 2], b[..., 2]) - tf.maximum(a[..., 0], b[..., 0]))
    ih = tf.nn.relu(tf.minimum(a[..., 3], b[..., 3]) - tf.maximum(a[..., 1], b[..., 1]))
    inter = iw * ih
    union = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1]) + (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1]) - inter + 1e-9
    enc = (tf.maximum(a[..., 2], b[..., 2]) - tf.minimum(a[..., 0], b[..., 0])) * \
          (tf.maximum(a[..., 3], b[..., 3]) - tf.minimum(a[..., 1], b[..., 1])) + 1e-9
    return inter / union - (enc - union) / enc

def _loss_parts(y_true, y_pred):
    y_pred = tf.cast(y_pred, tf.float32)
    state = y_true[..., 0]
    pos = tf.cast(tf.equal(state, 1.0), tf.float32)
    valid = tf.cast(state >= 0.0, tf.float32)
    npos = tf.maximum(tf.reduce_sum(pos), 1.0)
    logit = y_pred[..., 0]; p = tf.sigmoid(logit)
    ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=pos, logits=logit)
    p_t = pos * p + (1 - pos) * (1 - p)
    a_t = pos * LOC["FOCAL_ALPHA"] + (1 - pos) * (1 - LOC["FOCAL_ALPHA"])
    obj = tf.reduce_sum(a_t * tf.pow(1 - p_t, LOC["FOCAL_GAMMA"]) * ce * valid) / npos
    box = tf.reduce_sum((1.0 - giou_tf(decode_boxes_tf(y_pred[..., 1:5]), y_true[..., 1:5])) * pos) / npos
    return obj, box

def localization_loss(y_true, y_pred):
    o, b = _loss_parts(y_true, y_pred); return o + LOC["BOX_LOSS_WEIGHT"] * b
def obj_loss(y_true, y_pred): return _loss_parts(y_true, y_pred)[0]
def box_loss(y_true, y_pred): return _loss_parts(y_true, y_pred)[1]

def compile_localizer(lr):
    localizer.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=5.0), loss=localization_loss, metrics=[obj_loss, box_loss])

LOC_TRAINED = decide("localizer")
LOC_CKPT = f"{CKPT_DIR}/localizer_best.weights.h5"
loc_hist = {}
if LOC_TRAINED:
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    localizer, loc_backbone = build_localizer()
    assert tuple(localizer.output_shape[1:]) == (GRID, GRID, 5)
    print("Localizer parameters:", f"{localizer.count_params():,}")
    compile_localizer(LOC["LR_A"])
    with stopwatch("localizer training A"):
        h = localizer.fit(loc_train_ds, steps_per_epoch=STEPS_PER_EPOCH, epochs=LOC["EPOCHS_A"],
                          validation_data=loc_val_ds, verbose=2,
                          callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
                                     ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
                                     ModelCheckpoint(LOC_CKPT, monitor="val_loss", save_best_only=True, save_weights_only=True)])
    loc_hist = {k: list(map(float, v)) for k, v in h.history.items()}
    localizer.load_weights(LOC_CKPT)
    best_a = float(np.min(loc_hist["val_loss"])); n_a = len(loc_hist["loss"])
    if CAN_FINE_TUNE and len(gvb_train) >= LOC["MIN_TRAIN_IMAGES_FOR_FINETUNE"] and LOC["EPOCHS_B"] > 0:
        loc_backbone.trainable = True
        names = [l.name for l in loc_backbone.layers]; cut = names.index(LOC["UNFREEZE_FROM"])
        for i, l in enumerate(loc_backbone.layers):
            l.trainable = i >= cut and not isinstance(l, layers.BatchNormalization)
        compile_localizer(LOC["LR_B"])
        with stopwatch("localizer training B"):
            h = localizer.fit(loc_train_ds, steps_per_epoch=STEPS_PER_EPOCH, initial_epoch=n_a,
                              epochs=n_a + LOC["EPOCHS_B"], validation_data=loc_val_ds, verbose=2,
                              callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                                         ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
                                         ModelCheckpoint(LOC_CKPT, monitor="val_loss", save_best_only=True,
                                                         save_weights_only=True, initial_value_threshold=best_a)])
        for k, v in h.history.items(): loc_hist.setdefault(k, []).extend(map(float, v))
        localizer.load_weights(LOC_CKPT)
        print("Stage B best val_loss %.4f vs Stage A %.4f" % (np.min(h.history["val_loss"]), best_a))
    else:
        print("Localizer Stage B skipped (no GPU or too few training images).")
    localizer.save(MODEL_PATHS["localizer"])
    print("Saved", MODEL_PATHS["localizer"])
else:
    localizer = load_keras(MODEL_PATHS["localizer"])
    loc_hist = read_json(MODEL_META_PATHS["localizer"]).get("training_history", {})
    print("Loaded the saved localizer (trained on this split) — no retraining:", MODEL_PATHS["localizer"])
if loc_hist:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, k in zip(axes, ["loss", "obj_loss", "box_loss"]):
        ax.plot(loc_hist.get(k, []), label="train"); ax.plot(loc_hist.get("val_" + k, []), label="validation")
        ax.set_title(f"Localizer: {k}"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 26. Localization testing
* Score threshold chosen on **validation** (best F1 at IoU ≥ 0.5). The test set is never used for tuning.
* **Test:** class-agnostic AP@0.5 and AP@0.3, precision / recall / F1 at the tuned threshold, mean IoU of matched boxes, recall per
  G.V. Black class, image-level hit rate. IoU 0.3 is also shown because small, loosely drawn lesion boxes make 0.5 strict.

In [ ]:
# ============================================================
# SECTION 26 — LOCALIZATION TESTING
# ============================================================
def match_image(pb, ps, gb, iou_thr):
    """Greedy score-ordered matching -> list of (pred_idx, score, is_tp, gt_idx, iou)."""
    ious = iou_matrix(pb, gb); used, out = set(), []
    for k in np.argsort(-ps):
        cand = [(ious[k, g], g) for g in range(len(gb)) if g not in used and ious[k, g] >= iou_thr]
        if cand:
            iou, g = max(cand); used.add(g); out.append((k, float(ps[k]), True, g, float(iou)))
        else:
            out.append((k, float(ps[k]), False, -1, 0.0))
    return out

def average_precision(scores, tps, n_gt):
    if n_gt == 0 or not scores: return 0.0
    o = np.argsort(-np.array(scores)); t = np.array(tps)[o]
    tp, fp = np.cumsum(t), np.cumsum(~t)
    rec = tp / n_gt; prec = tp / np.maximum(tp + fp, 1e-9)
    mrec = np.r_[0, rec, 1]; mpre = np.r_[0, prec, 0]
    for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1]))

def eval_loc(recs, raws, thr, iou_thr):
    tp = fp = fn = hit = 0; ious = []
    found_c = np.zeros(NUM_GVB, int); total_c = np.zeros(NUM_GVB, int)
    for r, raw in zip(recs, raws):
        b, s = decode_detections(raw, thr, LOC["NMS_IOU"], LOC["MAX_DETECTIONS"])
        m = match_image(b, s, r["boxes"], iou_thr)
        found = {g for _, _, ok, g, _ in m if ok}
        tp += len(found); fp += sum(not ok for _, _, ok, _, _ in m); fn += len(r["boxes"]) - len(found)
        ious += [iou for *_, ok, _, iou in m if ok]; hit += bool(found)
        for g, c in enumerate(r["classes"]):
            total_c[c] += 1; found_c[c] += g in found
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=2 * P * R / max(P + R, 1e-9),
                mean_iou=float(np.mean(ious)) if ious else 0.0, image_hit_rate=hit / max(len(recs), 1),
                found_c=found_c, total_c=total_c)

def ap_at(recs, raws, iou_thr):
    sc, tps, n = [], [], 0
    for r, raw in zip(recs, raws):
        b, s = decode_detections(raw, LOC["AP_MIN_SCORE"], LOC["NMS_IOU"], LOC["MAX_DETECTIONS"])
        for _, score, ok, _, _ in match_image(b, s, r["boxes"], iou_thr):
            sc.append(score); tps.append(ok)
        n += len(r["boxes"])
    return average_precision(sc, tps, n)

raw_val = localizer.predict(X_loc_val, batch_size=LOC["BATCH"], verbose=0)
sweep = [(t, eval_loc(gvb_val, raw_val, t, 0.5)) for t in np.round(np.arange(0.05, 0.96, 0.05), 2)]
best_t, best_e = max(sweep, key=lambda z: (z[1]["f1"], z[0]))
LOC_THRESHOLD = float(best_t) if best_e["f1"] > 0 else LOC["DEFAULT_SCORE_THRESHOLD"]
if best_e["f1"] == 0:
    print("WARNING: no correct validation detection at IoU 0.5 — the localizer is NOT reliable.")
print("Validation sweep (thr: F1 / P / R):")
for t, e in sweep[::2]:
    print(f"  {t:.2f}: {e['f1']:.3f} / {e['precision']:.3f} / {e['recall']:.3f}")
print(f"Localizer score threshold (validation): {LOC_THRESHOLD:.2f}   | val AP@0.5 {ap_at(gvb_val, raw_val, 0.5):.3f}")

raw_test = localizer.predict(X_loc_test, batch_size=LOC["BATCH"], verbose=0)
n_test_boxes = sum(len(r["boxes"]) for r in gvb_test)
LOC_TEST = {"AP50": ap_at(gvb_test, raw_test, 0.5), "AP30": ap_at(gvb_test, raw_test, 0.3), "n_test_images": len(gvb_test),
            "n_test_lesions": n_test_boxes}
print("\n" + "=" * 64 + f"\nCAVITY LOCALIZATION — TEST ({len(gvb_test)} images, {n_test_boxes} lesions)\n" + "=" * 64)
print(f"AP@0.5 {LOC_TEST['AP50']:.3f} | AP@0.3 {LOC_TEST['AP30']:.3f}")
loc_test_eval = {}
for it in (0.5, 0.3):
    e = eval_loc(gvb_test, raw_test, LOC_THRESHOLD, it); loc_test_eval[it] = e
    LOC_TEST[f"iou{it}"] = {k: float(e[k]) for k in ("precision", "recall", "f1", "mean_iou", "image_hit_rate")}
    print(f"\n@score {LOC_THRESHOLD:.2f}, IoU >= {it}: TP {e['tp']} FP {e['fp']} FN {e['fn']}")
    print(f"  Localization precision {e['precision']:.3f} | recall {e['recall']:.3f} | F1 {e['f1']:.3f}")
    print(f"  Mean IoU (matched) {e['mean_iou']:.3f} | images with >=1 correct box {e['image_hit_rate']*100:.1f}%")
e = loc_test_eval[0.5]
print("\nRecall per G.V. Black class (IoU >= 0.5):")
for c in range(NUM_GVB):
    print(f"  {gvb_short(GVB_CLASS_NAMES[c]):<10}: " + (f"{e['found_c'][c]}/{e['total_c'][c]}" if e["total_c"][c] else "no test lesions"))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, r, raw in zip(axes.ravel(), gvb_test[:6], raw_test[:6]):
    full = load_rgb(r["image_path"]); H, W = full.shape[:2]; t = max(2, int(max(H, W) / 400))
    for gb in r["boxes"]:
        p = to_px(gb, W, H); cv2.rectangle(full, p[:2], p[2:], (255, 255, 255), t)
    for pb in decode_detections(raw, LOC_THRESHOLD, LOC["NMS_IOU"], LOC["MAX_DETECTIONS"])[0]:
        p = to_px(pb, W, H); cv2.rectangle(full, p[:2], p[2:], (0, 255, 0), t)
    ax.imshow(full); ax.axis("off"); ax.set_title(os.path.basename(r["image_path"]), fontsize=9)
for ax in axes.ravel()[len(gvb_test[:6]):]: ax.axis("off")
plt.suptitle("TEST photographs: white = ground truth, green = localizer"); plt.tight_layout(); plt.show()

PIPELINE["models"]["localizer"] = localizer
PIPELINE["meta"]["localizer"] = write_model_meta("localizer", {
    "model": "cavity / lesion localizer (class-agnostic)", "architecture": "anchor-free grid detector, MobileNetV2 C4+C5 backbone",
    "input_size": S_IN, "cache_side": CACHE_SIDE, "stride": LOC["STRIDE"], "grid": GRID,
    "preprocessing": "load_rgb -> cv2.resize(cache_side, INTER_AREA) -> cv2.resize(input_size, INTER_AREA) -> mobilenet_v2 preprocess_input",
    "output": "ch0 objectness logit | ch1-4 softplus(l,t,r,b) in grid-cell units",
    "score_threshold": LOC_THRESHOLD, "score_threshold_rule": "best F1 @IoU0.5 on validation",
    "nms_iou": LOC["NMS_IOU"], "max_detections": LOC["MAX_DETECTIONS"], "min_box_px": LOC["MIN_BOX_PX"],
    "training_config": LOC, "training_history": loc_hist, "test_metrics": LOC_TEST,
    "split_images": {"train": len(gvb_train), "val": len(gvb_val), "test": len(gvb_test)},
    "split_fingerprint": SPLIT_FP["localizer"], "model_path": MODEL_PATHS["localizer"],
})

## 27. Cavity crop generation
`generate_cavity_crops(rgb)` runs the localizer and, for **each** lesion box, cuts two inputs — both internally, the user never crops:
* **CNN 2 input** — square context crop centred on the lesion (side = longer box side × (1 + 2·pad)); `pad` is chosen on validation in
  Section 35 (until then `GVB["DEFAULT_PAD"]`).
* **CNN 3 input** — a **tooth-scale window** around the lesion, because the ICDAS CNN is trained on tooth crops
  (window = max(lesion side × mult, min_scale × median tooth-crop fraction of the photo)); `(mult, min_scale)` is chosen on validation
  lesions in Section 74 (until then `ICDAS["DEFAULT_WINDOW"]`).

`save_cavity_crops(image_path)` writes the crops as files, usable with `predict_gvblack(...)` and `predict_icdas(...)`.

In [ ]:
# ============================================================
# SECTION 27 — CAVITY CROP GENERATION (demo on a held-out TEST photograph)
# ============================================================
demo_r = gvb_test[0]
demo_rgb = load_rgb(demo_r["image_path"])
demo_crops = generate_cavity_crops(demo_rgb)
print(f"{demo_r['id']}: {len(demo_crops)} lesion(s) localized (score >= {LOC_THRESHOLD:.2f}); ground truth has {len(demo_r['boxes'])}")
if not demo_crops:
    print("No lesion above the threshold — showing crops from the ground-truth boxes instead.")
    demo_crops = generate_cavity_crops(demo_rgb, boxes=demo_r["boxes"])
ims, tts = [], []
for i, c in enumerate(demo_crops[:4], 1):
    ims += [c["gvb_crop"], c["icdas_crop"]]; tts += [f"lesion {i}: CNN 2 input", f"lesion {i}: CNN 3 input"]
show_examples(ims, tts, "Cavity crops generated from localizer boxes", ncols=4)
saved = save_cavity_crops(demo_r["image_path"])
print("Crop files written:", saved[:3], "..." if len(saved) > 3 else "")

<br>

---
# PART 8 — STAGE 3: CNN 2 G.V. BLACK

---
# STAGE 3 — CNN MODEL 2: G.V. BLACK CLASSIFICATION
Input: lesion / cavity crop · Output: G.V. Black class. The YOLO-format `.txt` files are **annotations** (class + box), used to cut
lesion crops — they are not the CNN.

## 28. G.V. Black dataset inspection

In [ ]:
# ============================================================
# SECTION 28 — G.V. BLACK DATASET INSPECTION
# ============================================================
pool_df = pd.DataFrame([{k: r[k] for k in ("id", "gvb_status", "view", "W", "H", "split")} | {"n_boxes": len(r["boxes"])} for r in pool])
print("=" * 64 + "\nG.V. BLACK DATASET\n" + "=" * 64)
print(f"Image files                 : {GVB_INDEX_INFO['n_images']}  (unreadable, excluded: {n_bad_pool})")
print(f"YOLO-format label files     : {GVB_INDEX_INFO['n_label_files']}  (label files without an image: {len(GVB_INDEX_INFO['orphan_labels'])})")
print("Annotation status per photo :", pool_df.gvb_status.value_counts().to_dict())
print("Photos per view (file-name prefix):", pool_df.view.value_counts().to_dict())
print("Image sizes (W x H, most common):", Counter(zip(pool_df.W, pool_df.H)).most_common(5))

## 29. Annotation inspection
Each line of a `.txt` file is `class x_center y_center width height` (normalised). Invalid lines (wrong field count, non-integer or
out-of-range class, coordinates outside [0, 1]) are reported and ignored.

In [ ]:
# ============================================================
# SECTION 29 — ANNOTATION INSPECTION
# ============================================================
ann = [r for r in pool if r["gvb_status"] == "annotated"]
box_counts = Counter(np.concatenate([r["classes"] for r in ann]).tolist()) if ann else Counter()
print(f"Photos with >= 1 box : {len(ann)} | total boxes: {sum(box_counts.values())}")
print(f"Malformed lines      : {len(GVB_INDEX_INFO['malformed'])}")
for p, i, l in GVB_INDEX_INFO["malformed"][:5]:
    print(f"   {os.path.basename(p)} line {i + 1}: {l!r}")
print("Class ids found in the label files (all lines):", GVB_INDEX_INFO["class_ids_seen"])
print("Boxes per photo:", dict(sorted(Counter(len(r["boxes"]) for r in ann).items())))
if ann:
    bw = np.concatenate([r["boxes"][:, 2] - r["boxes"][:, 0] for r in ann])
    bh = np.concatenate([r["boxes"][:, 3] - r["boxes"][:, 1] for r in ann])
    print("Box width / height as a fraction of the photo (5 / 50 / 95 %):",
          np.round(np.percentile(bw, [5, 50, 95]), 3), "/", np.round(np.percentile(bh, [5, 50, 95]), 3))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, r in zip(axes, ann[:3]):
    im = load_rgb(r["image_path"]); H, W = im.shape[:2]; t = max(2, int(max(H, W) / 400))
    for b, c in zip(r["boxes"], r["classes"]):
        p = to_px(b, W, H); cv2.rectangle(im, p[:2], p[2:], (0, 255, 0), t)
        cv2.putText(im, str(int(c)), (p[0], max(p[1] - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, max(0.6, W / 1500), (255, 255, 0), 2)
    ax.imshow(im); ax.axis("off"); ax.set_title(r["id"], fontsize=9)
plt.suptitle("Annotation check: box + class id as written in the .txt file"); plt.tight_layout(); plt.show()

## 30. Verify the G.V. Black class mapping
1. If the ZIP contains `classes.txt`, `data.yaml` or `notes.json`, the mapping is **read from it** (the Roman numeral in each class name is
   parsed) and compared with the confirmed mapping; the file wins if they differ.
2. Otherwise the mapping confirmed by the project owner (`0 → Class I … 5 → Class VI`) is used, and it is checked for **consistency** with
   the photographic view in the file name (Class I / VI lesions should come mostly from occlusal views, Class III / IV from front views).
   Consistency supports the mapping but cannot prove it — that is stated in the output.

In [ ]:
# ============================================================
# SECTION 30 — VERIFY G.V. BLACK CLASS MAPPING (printed before training)
# ============================================================
GVB_MAPPING_SOURCE = "project owner's confirmed mapping (no mapping file in the ZIP) — NOT provable from the data alone"
mf = GVB_INDEX_INFO["mapping_files"]
if mf:
    for path, mp in mf.items():
        print(f"Mapping file found: {path}\n  contents: {mp}")
        romans = {int(k): roman_of(v) for k, v in mp.items()}
        if set(romans) >= set(range(NUM_GVB)) and all(romans[i] for i in range(NUM_GVB)) and \
           len({romans[i] for i in range(NUM_GVB)}) == NUM_GVB:
            file_names = [f"GVB_Class_{romans[i]}" for i in range(NUM_GVB)]
            print("  -> agrees with the confirmed mapping" if file_names == GVB_CLASS_NAMES else
                  "  -> DIFFERS from the confirmed mapping — the mapping FILE is used")
            GVB_CLASS_NAMES = file_names; GVB_MAPPING_SOURCE = f"mapping file {os.path.basename(path)}"
            break
        print("  -> could not interpret this file as G.V. Black Class I..VI; ignored")
unexpected = [c for c in GVB_INDEX_INFO["class_ids_seen"] if not 0 <= c < NUM_GVB]
if unexpected:
    print("WARNING: class ids outside 0..5 in the label files (ignored as malformed):", unexpected)

VIEW_NAMES = {"LOV": "lower occlusal", "UOV": "upper occlusal", "LFV": "lower front", "UFV": "upper front",
              "LLV": "lower left", "ULV": "upper left", "LRV": "lower right", "URV": "upper right"}
vt = defaultdict(Counter)
for r in ann:
    for c in r["classes"]:
        vt[GVB_CLASS_NAMES[c]][r["view"]] += 1
view_tab = pd.DataFrame(vt).T.reindex(GVB_CLASS_NAMES).fillna(0).astype(int)
print("\nG.V. Black boxes by photographic view (file-name prefix):\n" + view_tab.to_string())
def _frac(cls, views):
    if cls not in view_tab.index: return float("nan")
    return view_tab.loc[cls, [v for v in views if v in view_tab.columns]].sum() / max(view_tab.loc[cls].sum(), 1)
print(f"\nClass I   in occlusal views: {_frac('GVB_Class_I', ['LOV', 'UOV']):.0%}   (pits & fissures -> expected mostly occlusal)")
print(f"Class VI  in occlusal views: {_frac('GVB_Class_VI', ['LOV', 'UOV']):.0%}   (cusp tips -> expected occlusal)")
print(f"Class III in front views   : {_frac('GVB_Class_III', ['LFV', 'UFV']):.0%}   (anterior proximal -> expected front)")
print(f"Class IV  in front views   : {_frac('GVB_Class_IV', ['LFV', 'UFV']):.0%}   (anterior incisal angle -> expected front)")

print("\n" + "=" * 50 + "\nFINAL VERIFIED G.V. BLACK MAPPING (used for training)\n" + "=" * 50)
for i, n in enumerate(GVB_CLASS_NAMES):
    print(f"  {i} -> {gvb_short(n)}")
print("Source:", GVB_MAPPING_SOURCE)

## 31. Generate lesion crops
G.V. Black classes are defined by **where** the lesion sits on the tooth, so each crop is a **square window centred on the box**
(side = longer box side × (1 + 2 × `pad`)), cut from the full-resolution photo (black border past the edge), resized to the model size — no
aspect distortion, one resize only (lesion detail preserved). `pad` and the input size are chosen on validation (Section 35).
The same function (`gvb_prepare_crop`) is applied to localizer boxes at inference.
`JITTER_COPIES` extra **training** crops per box with ±10 % box jitter prepare CNN 2 for imperfect localizer boxes; validation and test use
exact ground-truth boxes only.

In [ ]:
# ============================================================
# SECTION 31 — LESION CROP GENERATION (from the YOLO-format boxes)
# ============================================================
def jitter_box(box, rng, frac):
    w, h = box[2] - box[0], box[3] - box[1]
    b = np.clip(box + rng.uniform(-frac, frac, 4) * np.array([w, h, w, h]), 0, 1)
    return b if (b[2] - b[0]) > 0 and (b[3] - b[1]) > 0 else box

def build_gvb_crops(recs, pad, jitter_copies, seed, size):
    """Crops ONLY from the photographs passed in (one split at a time) -> X uint8, y, source photo id, is_exact."""
    rng = np.random.default_rng(seed)
    jobs = [(r, int(rng.integers(1 << 31))) for r in recs]
    def one(job):
        r, s = job; rr = np.random.default_rng(s)
        full = load_rgb(r["image_path"]); out = []
        for b, c in zip(r["boxes"], r["classes"]):
            for k in range(1 + jitter_copies):
                bb = b if k == 0 else jitter_box(b, rr, GVB["JITTER_FRAC"])
                out.append((gvb_prepare_crop(full, bb, pad, size, GVB["MIN_CROP_PX"]), int(c), r["id"], k == 0))
        return out
    X, y, ids, exact = [], [], [], []
    with ThreadPoolExecutor(DECODE_WORKERS) as ex:
        for res in ex.map(one, jobs):
            for img, c, i, e in res:
                X.append(img); y.append(c); ids.append(i); exact.append(e)
    return np.stack(X).astype(np.uint8), np.array(y, np.int32), np.array(ids), np.array(exact)

def gvb_crop_meta(recs):
    """One row per EXACT crop, in the same order as build_gvb_crops(jitter_copies=0): source photo, view, lesion size in pixels."""
    return pd.DataFrame([{"id": r["id"], "view": r["view"], "crop_px": float(max((b[2] - b[0]) * r["W"], (b[3] - b[1]) * r["H"]))}
                         for r in recs for b in r["boxes"]])

demo = gvb_train[0]; _s = GVB["IMG_SIZE"]
fig, axes = plt.subplots(1, len(GVB["PAD_CANDIDATES"]) + 1, figsize=(4 * (len(GVB["PAD_CANDIDATES"]) + 1), 4))
b0 = demo["boxes"][0]; full = load_rgb(demo["image_path"]); H, W = full.shape[:2]
x1, y1, x2, y2 = to_px(b0, W, H)
axes[0].imshow(cv2.resize(full[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1)], (_s, _s))); axes[0].set_title("tight box, squashed (not used)")
for ax, pad in zip(axes[1:], GVB["PAD_CANDIDATES"]):
    ax.imshow(gvb_prepare_crop(full, b0, pad, _s, GVB["MIN_CROP_PX"])); ax.set_title(f"square context, pad {pad}")
for ax in axes: ax.axis("off")
plt.suptitle(f"{demo['id']} — {gvb_short(GVB_CLASS_NAMES[demo['classes'][0]])}"); plt.tight_layout(); plt.show()

## 32. Original image-level train / validation / test split
**ORIGINAL IMAGE → split → crops.** The split is made on original photographs (the shared group split of Section 10: exact duplicates,
near-duplicates and burst photos of the same mouth are one group), and crops are generated **per split** afterwards, so crops from one
photograph — or from a duplicate of it — can never appear in two splits. Identical crops are checked again after crop generation (Section 35).

In [ ]:
# ============================================================
# SECTION 32 — ORIGINAL-IMAGE-LEVEL SPLIT (verified)
# ============================================================
ids_split = {s: {r["id"] for r in recs} for s, recs in [("train", gvb_train), ("val", gvb_val), ("test", gvb_test)]}
assert not (ids_split["train"] & ids_split["val"]) and not (ids_split["train"] & ids_split["test"]) and \
       not (ids_split["val"] & ids_split["test"]), "LEAKAGE: a G.V. Black photograph is in two splits"
grp_split = {s: {r["group"] for r in recs} for s, recs in [("train", gvb_train), ("val", gvb_val), ("test", gvb_test)]}
assert not (grp_split["train"] & grp_split["test"]) and not (grp_split["val"] & grp_split["test"]) and \
       not (grp_split["train"] & grp_split["val"]), "LEAKAGE: a group in two splits"
md5c = Counter(r["md5"] for r in gvb_records)
gsz = Counter(r["group"] for r in gvb_records)
print(f"Byte-identical G.V. Black photographs: {sum(v - 1 for v in md5c.values() if v > 1)} extra copies "
      f"| near-duplicate groups with > 1 photo: {sum(v > 1 for v in gsz.values())} (each kept inside ONE split)")
print(pd.DataFrame({s: {"photographs": len(recs), "boxes": sum(len(r["boxes"]) for r in recs)}
                    for s, recs in [("train", gvb_train), ("val", gvb_val), ("test", gvb_test)]}).T.to_string())
print("Photograph-level and group-level leakage checks passed.")

## 33. Class distribution

In [ ]:
# ============================================================
# SECTION 33 — G.V. BLACK CLASS DISTRIBUTION (actual counts)
# ============================================================
yg_count = pd.DataFrame({s: Counter(np.concatenate([r["classes"] for r in recs]).tolist())
                         for s, recs in [("train", gvb_train), ("val", gvb_val), ("test", gvb_test)]}) \
             .reindex(range(NUM_GVB)).fillna(0).astype(int)
yg_count.index = [gvb_short(n) for n in GVB_CLASS_NAMES]
yg_count["total"] = yg_count.sum(1)
print("G.V. Black lesions (boxes = exact crops) — all photographs:")
for n, v in yg_count["total"].items():
    print(f"  {n:<10}: {v}")
print("\nPer split:\n" + yg_count.to_string())
GVB_THIN = [n for n in yg_count.index if yg_count.loc[n, "test"] < GVB["MIN_RELIABLE_TEST"]]
if GVB_THIN:
    print(f"\nDATASET LIMITATION: {GVB_THIN} have < {GVB['MIN_RELIABLE_TEST']} TEST crops. Their per-class metrics are very "
          "uncertain — a high score on a handful of crops does NOT show that the class is reliable.")
print(f"Imbalance (largest / smallest class, train): {yg_count['train'].max() / max(yg_count['train'].min(), 1):.1f} : 1")
yg_count[["train", "val", "test"]].plot.bar(stacked=True, figsize=(8, 3.5), title="G.V. Black crops per class and split")
plt.tight_layout(); plt.show()

## 34. Class weights
Balanced weights from the **training split only**, capped at `MAX_CLASS_WEIGHT` so a very rare class cannot dominate the loss.
Whether class weights, training-set oversampling, both, or neither work best is decided on **validation** in Section 35.

In [ ]:
# ============================================================
# SECTION 34 — G.V. BLACK CLASS WEIGHTS (training split only)
# ============================================================
_ytr_boxes = np.concatenate([r["classes"] for r in gvb_train])
GVB_CLASS_WEIGHT_FULL = capped_class_weights(_ytr_boxes, NUM_GVB, GVB["MAX_CLASS_WEIGHT"])
for c, v in GVB_CLASS_WEIGHT_FULL.items():
    print(f"  {gvb_short(GVB_CLASS_NAMES[c]):<10}: {v:.3f}" + ("   (absent from training)" if c not in set(_ytr_boxes) else ""))

## 34.1 One-time RAM cleanup before Section 35
Deletes only objects that no cell after Section 34 uses (localizer training/validation caches, Stage 1 tf.data pipelines and the memory-mapped Stage 1 cache). `X_loc_test` is kept for Section 78. Run once, right before Section 35.

In [ ]:
# ============================================================
# ONE-TIME RAM CLEANUP — run ONCE, right before Section 35.1
# (only deletes objects that NO cell after Section 34 uses)
# ============================================================
import gc, ctypes

def _ram():
    try:
        import psutil
        vm = psutil.virtual_memory(); rss = psutil.Process().memory_info().rss
        return f"RAM used {(vm.total - vm.available) / 1e9:.2f} / {vm.total / 1e9:.2f} GB | this Python process {rss / 1e9:.2f} GB"
    except Exception:
        return "(psutil not available)"

print("Before cleanup:", _ram())

# Localizer training/validation data (Sections 25.1 / 26). The trained localizer is already saved and loaded; Section 78
# only needs X_loc_test, which is KEPT.  loc_cache ≈ 917 photos × 520×520×3 ≈ 0.74 GB, X_loc_val float32 ≈ 0.28 GB and
# loc_val_ds holds a SECOND copy of X_loc_val inside TensorFlow (from_tensor_slices) ≈ 0.28 GB.
# Stage 1 tf.data pipelines + the 2.4 GB memory-mapped Stage 1 cache (Sections 10.1 / 12) — not used after Section 23;
# Stage 1 is already trained, evaluated and saved, and later cells reload images from the original files.
_TO_DELETE = ["loc_cache", "X_loc_val", "Y_loc_val", "loc_val_ds", "loc_train_ds", "raw_val", "raw_test",
              "s1_train_ds", "s1_val_ds", "s1_test_ds", "s1_images"]
_deleted = [n for n in _TO_DELETE if n in globals()]
for n in _deleted:
    del globals()[n]
print("Deleted:", _deleted)

keras.backend.clear_session()           # drops leaked tf.data / tf.function graphs of the deleted pipelines
gc.collect()

_LIBC = None
try:
    _LIBC = ctypes.CDLL("libc.so.6")
    # Fixed mmap threshold (disables glibc's dynamic threshold): every allocation >= 128 KB (each 224/288 px crop, every
    # decoded photo, every image batch) is mmap-ed and handed back to the OS the moment it is freed, instead of piling up
    # as fragmented heap in the 8 decoder threads' malloc arenas.
    _LIBC.mallopt(-3, 128 * 1024)       # -3 = M_MMAP_THRESHOLD
    _LIBC.malloc_trim(0)                # return free heap pages to the OS now
except Exception as e:
    print("malloc tuning not available (not required):", e)

print("After cleanup: ", _ram())


## 35. Stage 3 CNN architecture — validation search
**35.1 Architecture, input size and context padding.** For every `pad` × input size × backbone (EfficientNetV2B0, EfficientNetB0,
DenseNet121, ResNet50, MobileNetV2), the frozen ImageNet backbone embeds the exact training and validation crops and a small head is
trained to convergence (Section 6.7). Ranked by validation score = ½ accuracy + ½ macro-F1; validation loss and parameter count are shown.
**Baselines** on validation: majority class, and logistic regression on the best frozen features.

**35.2 Training recipe.** With the chosen features: class balancing (none / class weights / oversampling / both) × label smoothing
(0 / 0.1), again on validation.

**35.3 Model.** Chosen backbone + size + pad + recipe → RGB `[0,255]` → augmentation (train only) → backbone normalisation → backbone →
GAP → BatchNorm → Dropout → Dense(256) → Dropout → softmax(6).

**Memory-safe version.** Each frozen backbone is built once per (pad, size) and embeds the crops without `model.predict()` / XLA; crops are built straight into one uint8 array; RAM is printed after every step; finished candidates are cached in `/content/cache/gvb_arch_screen`, so a re-run resumes instead of restarting.

In [ ]:
# ============================================================
# SECTION 35.1 — CNN 2 ARCHITECTURE / SIZE / PADDING SEARCH (VALIDATION ONLY) — MEMORY-SAFE VERSION
# Same candidates (5 backbones × 2 sizes × 3 pads), same crops, same frozen features, same head, same ranking,
# same selected backbone / size / pad, same baselines, same printed table. Only HOW memory is used changed:
#   * each frozen backbone is built ONCE per (pad, size) and embeds train + val with a single non-XLA tf.function
#     (the original built it twice and ran model.predict(), which XLA-compiles every backbone on the GPU and keeps the
#     compiled programs in host RAM — the RAM that kept growing until the crash);
#   * no tf.data / tf.numpy_function pipeline holding the whole crop array; uint8 batches are cast to float32 on the GPU;
#   * crops are written straight into one pre-allocated uint8 array (no list -> np.stack -> astype copies, no
#     np.concatenate of train + val);
#   * explicit cleanup after every backbone and every (pad, size);
#   * finished candidates are cached on local disk, so a re-run after any interruption resumes instead of restarting.
# ============================================================
import gc, ctypes

try:
    _LIBC = ctypes.CDLL("libc.so.6")
except OSError:
    _LIBC = None

def _free_ram():
    gc.collect()
    if _LIBC is not None:
        try: _LIBC.malloc_trim(0)
        except Exception: pass

def _ram():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f"RAM used {(vm.total - vm.available) / 1e9:.2f} / {vm.total / 1e9:.2f} GB"
    except Exception:
        return ""

# ---------- G.V. Black crops: SAME signature, SAME crops, order, RNG stream and dtypes as Section 31 ----------
# (redefined here so Section 35.2 and every later cell also use the low-memory version)
def build_gvb_crops(recs, pad, jitter_copies, seed, size):
    """Crops ONLY from the photographs passed in (one split at a time) -> X uint8, y, source photo id, is_exact."""
    rng = np.random.default_rng(seed)
    jobs = [(r, int(rng.integers(1 << 31))) for r in recs]                       # identical per-photo seeds
    counts = [min(len(r["boxes"]), len(r["classes"])) * (1 + jitter_copies) for r in recs]
    offsets = np.concatenate([[0], np.cumsum(counts, dtype=np.int64)]).astype(np.int64)
    n = int(offsets[-1])
    X = np.empty((n, size, size, 3), np.uint8)                                   # the ONLY image buffer
    y = np.empty(n, np.int32); exact = np.empty(n, bool)
    def one(i):
        r, s = jobs[i]; rr = np.random.default_rng(s)
        full = load_rgb(r["image_path"]); j = int(offsets[i])
        for b, c in zip(r["boxes"], r["classes"]):
            for k in range(1 + jitter_copies):
                bb = b if k == 0 else jitter_box(b, rr, GVB["JITTER_FRAC"])
                X[j] = gvb_prepare_crop(full, bb, pad, size, GVB["MIN_CROP_PX"])
                y[j] = int(c); exact[j] = (k == 0); j += 1
        del full
    with ThreadPoolExecutor(DECODE_WORKERS) as ex:
        list(ex.map(one, range(len(recs))))
    ids = np.array([r["id"] for r, cnt in zip(recs, counts) for _ in range(cnt)])
    return X, y, ids, exact

# ---------- frozen-backbone features: same model as Section 6.7 backbone_features(), no XLA, no tf.data ----------
GVB_FEATURE_BATCH = 64                    # = backbone_features() default -> identical batch composition

def frozen_features_lowmem(backbone, size, sources, batch=GVB_FEATURE_BATCH):
    """Builds the frozen ImageNet backbone (+ its own normalisation, GAP) ONCE and embeds every (X, rows) in `sources`.
    Same batches as backbone_features(): rows taken in order, `batch` at a time (X may be a disk memmap)."""
    keras.backend.clear_session()
    spec = BACKBONES[backbone]
    inp = keras.Input((size, size, 3))
    base = spec["fn"](include_top=False, weights=BACKBONE_WEIGHTS, input_shape=(size, size, 3), pooling="avg", **spec["kwargs"])
    fx = keras.Model(inp, base(preprocessing_block(spec["prep"], inp), training=False))
    run = tf.function(lambda xb: tf.cast(fx(tf.cast(xb, tf.float32), training=False), tf.float32),
                      input_signature=[tf.TensorSpec((None, size, size, 3), tf.uint8)], jit_compile=False, autograph=False)
    try:
        outs = []
        for X, rows in sources:
            rows = np.asarray(rows, np.int64); parts = []
            for s in range(0, len(rows), batch):
                r = rows[s:s + batch]; o = np.argsort(r)
                xb = np.empty((len(r),) + tuple(X.shape[1:]), np.uint8); xb[o] = X[r[o]]   # sorted reads, original order
                parts.append(run(xb).numpy())
            outs.append(np.concatenate(parts).astype(np.float32, copy=False) if parts
                        else np.zeros((0, int(fx.output_shape[-1])), np.float32))
        return outs
    finally:
        del run, fx, base, inp
        keras.backend.clear_session(); _free_ram()

def _gvb_backbone_features(backbone, size, Xtr, Xva):
    return frozen_features_lowmem(backbone, size, [(Xtr, np.arange(len(Xtr))), (Xva, np.arange(len(Xva)))])

def screen_features_lowmem(candidates, sizes, get_data, n_classes, class_weight, tag):
    """Drop-in replacement for screen_features() (Section 6.7): same arguments, same table, same feature dict.
    Used by Section 59.1 (ICDAS). One backbone build per candidate, no model.predict / XLA, cleanup after each."""
    rows_, feats = [], {}
    for size in sizes:
        X, rtr, ytr, rva, yva = get_data(size)
        for bb in candidates:
            t0 = time.time()
            try:
                Ftr, Fva = frozen_features_lowmem(bb, size, [(X, rtr), (X, rva)])
            except Exception as e:
                print(f"  {bb}@{size}: skipped ({type(e).__name__}: {e})"); _free_ram(); continue
            res, _ = fit_feature_head(Ftr, ytr, Fva, yva, n_classes, class_weight=class_weight)
            feats[(bb, size)] = (Ftr, ytr, Fva, yva)
            rows_.append({"backbone": bb, "size": size, **res, "params_M": round(BACKBONE_PARAMS_M.get(bb, float("nan")), 1),
                          "seconds": round(time.time() - t0, 1)})
            print(f"  {tag}: {bb:16} @ {size}px  val acc {res['val_accuracy']:.4f} | val macro-F1 {res['val_macro_f1']:.4f} | "
                  f"val loss {res['val_loss']:.4f}")
            keras.backend.clear_session(); _free_ram()
        del X; _free_ram()
        print(f"    [{tag} @ {size}px done] {_ram()}")
    return rank_candidates(pd.DataFrame(rows_)), feats

# ---------- resume cache (local disk, keyed by the exact split + boxes + settings) ----------
GVB_SCREEN_RESUME = True
GVB_SCREEN_DIR = f"{CACHE_DIR}/gvb_arch_screen"
_GVB_SPLIT_SIG = fingerprint([f"{s}|{r['id']}|{np.asarray(r['boxes'], np.float32).round(6).tolist()}|"
                              f"{np.asarray(r['classes']).astype(int).tolist()}"
                              for s, recs in [("train", gvb_train), ("val", gvb_val)] for r in recs])
def _gvb_screen_path(pad, size, bb):
    key = hashlib.md5(f"{_GVB_SPLIT_SIG}|{pad}|{size}|{bb}|{BACKBONE_WEIGHTS}|{GVB['MIN_CROP_PX']}|"
                      f"{GVB_FEATURE_BATCH}|{tf.__version__}|{keras.mixed_precision.global_policy().name}".encode()).hexdigest()
    return f"{GVB_SCREEN_DIR}/{bb}_{size}_pad{pad}_{key[:12]}.npz"

# ============================================================
MACRO_F1 = lambda: keras.metrics.F1Score(average="macro", threshold=None, name="macro_f1")
GVB_METRICS = lambda: [keras.metrics.CategoricalAccuracy(name="accuracy"), MACRO_F1()]
GVB_CKPT = f"{CKPT_DIR}/gvb_best.weights.h5"
GVB_TRAINED = decide("gvb")
GVB_ARCH_SCREEN = GVB_RECIPE_SCREEN = GVB_TTA_SCREEN = pd.DataFrame(); gvb_hist, gvb_info = {}, {}; GVB_BASELINES = {}

if GVB_TRAINED:
    os.makedirs(GVB_SCREEN_DIR, exist_ok=True)
    _free_ram(); print("Before screening:", _ram())
    tabs, GVB_FEATS = [], {}
    for pad in GVB["PAD_CANDIDATES"]:
        tag, rows_pad = f"pad {pad}", []
        with stopwatch(f"G.V. Black feature screening pad {pad}"):
            for size in GVB["SIZE_CANDIDATES"]:
                paths = {bb: _gvb_screen_path(pad, size, bb) for bb in GVB["CANDIDATES"]}
                todo = [bb for bb in GVB["CANDIDATES"] if not (GVB_SCREEN_RESUME and os.path.exists(paths[bb]))]
                Xtr = Xva = ytr = yva = None
                if todo:                                   # crops are only built if some backbone still needs them
                    Xtr, ytr, _, _ = build_gvb_crops(gvb_train, pad, 0, SEED, size)
                    Xva, yva, _, _ = build_gvb_crops(gvb_val, pad, 0, SEED, size)
                for bb in GVB["CANDIDATES"]:
                    t0 = time.time()
                    try:
                        if bb in todo:
                            Ftr, Fva = _gvb_backbone_features(bb, size, Xtr, Xva)
                            yt, yv = ytr, yva
                            if GVB_SCREEN_RESUME:
                                np.savez(paths[bb] + ".tmp.npz", Ftr=Ftr, Fva=Fva, ytr=yt, yva=yv)
                                os.replace(paths[bb] + ".tmp.npz", paths[bb])
                        else:
                            with np.load(paths[bb]) as z:
                                Ftr, Fva, yt, yv = z["Ftr"], z["Fva"], z["ytr"], z["yva"]
                    except Exception as e:
                        print(f"  {bb}@{size}: skipped ({type(e).__name__}: {e})"); _free_ram(); continue
                    res, _ = fit_feature_head(Ftr, yt, Fva, yv, NUM_GVB, class_weight=GVB_CLASS_WEIGHT_FULL)
                    GVB_FEATS[(pad, bb, size)] = (Ftr, yt, Fva, yv)
                    rows_pad.append({"backbone": bb, "size": size, **res,
                                     "params_M": round(BACKBONE_PARAMS_M.get(bb, float("nan")), 1),
                                     "seconds": round(time.time() - t0, 1)})
                    print(f"  {tag}: {bb:16} @ {size}px  val acc {res['val_accuracy']:.4f} | val macro-F1 "
                          f"{res['val_macro_f1']:.4f} | val loss {res['val_loss']:.4f}" + ("" if bb in todo else "  (resumed)"))
                    keras.backend.clear_session(); _free_ram()
                del Xtr, Xva; _free_ram()
                print(f"    [{tag} @ {size}px done] {_ram()}")
        tabs.append(rank_candidates(pd.DataFrame(rows_pad)).assign(pad=pad))
    GVB_ARCH_SCREEN = rank_candidates(pd.concat(tabs))
    print("\nG.V. Black architecture search — VALIDATION (test set not used):")
    print(GVB_ARCH_SCREEN[["backbone", "size", "pad", "val_accuracy", "val_macro_f1", "val_loss", "score", "params_M", "seconds"]]
          .round(4).to_string(index=False))
    best = GVB_ARCH_SCREEN.iloc[0]
    GVB_BACKBONE, GVB_SIZE, GVB_PAD = str(best["backbone"]), int(best["size"]), float(best["pad"])
    Ftr, ytr_f, Fva, yva_f = GVB_FEATS[(GVB_PAD, GVB_BACKBONE, GVB_SIZE)]
    lr_ = LogisticRegression(max_iter=3000, class_weight="balanced").fit(Ftr, ytr_f)
    yp_ = lr_.predict(Fva)
    GVB_BASELINES = {"majority_val": majority_baseline(ytr_f, yva_f),
                     "logreg_frozen_features_val": {"accuracy": float((yp_ == yva_f).mean()), "macro_f1": macro_f1_present(yva_f, yp_)}}
    print(f"\nSelected: {GVB_BACKBONE} @ {GVB_SIZE}px, pad {GVB_PAD}")
    print("Validation baselines:", {k: {kk: round(vv, 4) if isinstance(vv, float) else vv for kk, vv in v.items()}
                                   for k, v in GVB_BASELINES.items()})
    # only the winner's features are needed from here on (Section 35.2) — drop the other 29 feature sets
    GVB_FEATS = {(GVB_PAD, GVB_BACKBONE, GVB_SIZE): (Ftr, ytr_f, Fva, yva_f)}
    del lr_; _free_ram(); print("After screening:", _ram())
else:
    gvb_model = load_keras(MODEL_PATHS["gvb"])
    _m = read_json(MODEL_META_PATHS["gvb"])
    GVB_BACKBONE, GVB_SIZE = _m.get("backbone", "unknown"), int(_m.get("image_size", GVB["IMG_SIZE"]))
    GVB_PAD = float(_m.get("context_pad", GVB["DEFAULT_PAD"]))
    GVB_BAL, GVB_LS, GVB_TTA = _m.get("balancing", "class_weight"), float(_m.get("label_smoothing", 0.0)), _m.get("tta", "none")
    gvb_hist, gvb_info = _m.get("training_history", {}), _m.get("training", {})
    GVB_BASELINES = _m.get("baselines", {})
    GVB_ARCH_SCREEN = pd.DataFrame(_m.get("architecture_screening", []))
    GVB_RECIPE_SCREEN = pd.DataFrame(_m.get("recipe_screening", []))
    GVB_TTA_SCREEN = pd.DataFrame(_m.get("tta_screening", []))
    print("Loaded the saved G.V. Black model (trained on this split) — no retraining:", MODEL_PATHS["gvb"])
    print(f"  {GVB_BACKBONE} @ {GVB_SIZE}px, pad {GVB_PAD}, balancing {GVB_BAL}, label smoothing {GVB_LS}, TTA {GVB_TTA}")


In [ ]:
# ============================================================
# SECTION 35.2 — TRAINING RECIPE (balancing x label smoothing, VALIDATION) + FINAL CROPS + MODEL — MEMORY-SAFE VERSION
# Identical logic and outputs. build_gvb_crops is the low-memory version from 35.1 (same crops), so the jittered
# training set (3 crops per box) is built without the old list -> np.stack -> astype triple copy; cleanup added.
# ============================================================
if GVB_TRAINED:
    GVB_RECIPE_SCREEN = screen_training_recipe(Ftr, ytr_f, Fva, yva_f, NUM_GVB, GVB["BALANCE_CANDIDATES"], GVB["LS_CANDIDATES"],
                                               GVB["MAX_CLASS_WEIGHT"], GVB["OVERSAMPLE_POWER"])
    print("Training-recipe search — VALIDATION:\n" + GVB_RECIPE_SCREEN.round(4).to_string(index=False))
    GVB_BAL, GVB_LS = str(GVB_RECIPE_SCREEN.iloc[0]["balancing"]), float(GVB_RECIPE_SCREEN.iloc[0]["label_smoothing"])
    print(f"Selected recipe: balancing = {GVB_BAL}, label smoothing = {GVB_LS}")
    del GVB_FEATS
    keras.backend.clear_session(); _free_ram()        # drop the 8 small recipe heads before building the crops

with stopwatch("G.V. Black final crops"):
    Xg_train, yg_train, ids_tr, exact_tr = build_gvb_crops(gvb_train, GVB_PAD, GVB["JITTER_COPIES"] if GVB_TRAINED else 0,
                                                           SEED, GVB_SIZE)
    Xg_val, yg_val, ids_va, _ = build_gvb_crops(gvb_val, GVB_PAD, 0, SEED, GVB_SIZE)
    Xg_test, yg_test, ids_te, _ = build_gvb_crops(gvb_test, GVB_PAD, 0, SEED, GVB_SIZE)
_free_ram()
assert not (set(ids_tr) & set(ids_te)) and not (set(ids_tr) & set(ids_va)) and not (set(ids_va) & set(ids_te)), "crop leakage"
_h = {s: {hashlib.md5(x.tobytes()).hexdigest() for x in X} for s, X in [("train", Xg_train), ("val", Xg_val), ("test", Xg_test)]}
assert not (_h["train"] & _h["test"]) and not (_h["val"] & _h["test"]) and not (_h["train"] & _h["val"]), "identical crop in two splits"
del _h
print(f"Crops: train {len(Xg_train)}" + (" (incl. jitter copies)" if GVB_TRAINED else " (exact)") +
      f" | val {len(Xg_val)} | test {len(Xg_test)} — no photograph and no identical crop shared between splits")
print(f"Crop arrays in RAM: train {Xg_train.nbytes / 1e9:.2f} GB | val {Xg_val.nbytes / 1e9:.2f} GB | "
      f"test {Xg_test.nbytes / 1e9:.2f} GB | {_ram()}")

GVB_CLASS_WEIGHT = GVB_CLASS_WEIGHT_FULL if "class_weight" in GVB_BAL else None
gvb_train_rows = (oversample_rows(np.arange(len(Xg_train)), yg_train, GVB["OVERSAMPLE_POWER"], SEED)
                  if "oversample" in GVB_BAL else np.arange(len(Xg_train)))
if "oversample" in GVB_BAL:
    print("Training rows per class after oversampling (training set only):",
          {gvb_short(GVB_CLASS_NAMES[c]): int(n) for c, n in enumerate(np.bincount(yg_train[gvb_train_rows], minlength=NUM_GVB))})
GVB_LOSS = keras.losses.CategoricalCrossentropy(label_smoothing=GVB_LS)
gvb_train_ds = array_batch_ds(Xg_train, gvb_train_rows, onehot(yg_train[gvb_train_rows], NUM_GVB), GVB["BATCH"], True)
gvb_val_ds = array_batch_ds(Xg_val, np.arange(len(Xg_val)), onehot(yg_val, NUM_GVB), GVB["BATCH"], False)
if GVB_TRAINED:
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    gvb_model, gvb_base = build_classifier(GVB_BACKBONE, (GVB_SIZE, GVB_SIZE), NUM_GVB, GVB, "cnn2_stage3_gvblack_classifier")
print(f"CNN 2: {gvb_model.name} | {GVB_BACKBONE} @ {GVB_SIZE}px | pad {GVB_PAD} | parameters {gvb_model.count_params():,}")


## 36. Stage 3 training — PHASE 1 (frozen backbone)
ImageNet backbone frozen; new head trained with AdamW (LR 1e-3), dropout, head BatchNorm, EarlyStopping + ModelCheckpoint +
ReduceLROnPlateau on validation.

In [ ]:
# ============================================================
# SECTION 36 — CNN 2 PHASE 1: FROZEN BACKBONE
# ============================================================
if GVB_TRAINED:
    with stopwatch("G.V. Black phase 1"):
        gvb_hist, gvb_info = train_phase_a(gvb_model, gvb_train_ds, gvb_val_ds, GVB, GVB_LOSS, GVB_METRICS, GVB["MONITOR"],
                                           GVB_CLASS_WEIGHT, GVB_CKPT)
else:
    print("Skipped — the saved G.V. Black model was loaded in Section 35.")

## 37. Stage 3 fine-tuning — PHASE 2
**Gradual** unfreezing (`UNFREEZE_STEPS`, e.g. top 25 % then top 50 % of the backbone — never the whole network at once), BatchNorm layers
kept frozen (avoids destroying ImageNet statistics with small batches), LR 1e-4 → 5e-5 with warm-up + cosine decay, checkpointing on
validation macro-F1. A step's weights are only kept if they beat everything before. Then **test-time augmentation** is chosen on validation.

In [ ]:
# ============================================================
# SECTION 37 — CNN 2 PHASE 2: FINE-TUNING + TTA CHOICE (VALIDATION) + SAVE
# ============================================================
if GVB_TRAINED:
    with stopwatch("G.V. Black phase 2"):
        gvb_hist, gvb_info = train_phase_b(gvb_model, gvb_base, gvb_train_ds, gvb_val_ds, GVB, GVB_LOSS, GVB_METRICS,
                                           GVB["MONITOR"], GVB_CLASS_WEIGHT, GVB_CKPT, gvb_hist, gvb_info)
    GVB_TTA, GVB_TTA_SCREEN = choose_tta(gvb_model, Xg_val, np.arange(len(Xg_val)), yg_val, GVB["TTA_CANDIDATES"])
    print("Test-time augmentation — VALIDATION:\n" + GVB_TTA_SCREEN.round(4).to_string(index=False) + f"\n-> using '{GVB_TTA}'")
    gvb_model.save(MODEL_PATHS["gvb"])
    print("Saved", MODEL_PATHS["gvb"])
else:
    print("Skipped — the saved G.V. Black model was loaded in Section 35.")
_pv = predict_rows(gvb_model, Xg_val, np.arange(len(Xg_val)), variant=GVB_TTA)
GVB_VAL_FINAL = {"accuracy": float((_pv.argmax(1) == yg_val).mean()), "macro_f1": macro_f1_present(yg_val, _pv.argmax(1))}
print(f"Final model on VALIDATION (the split used for all choices): accuracy {GVB_VAL_FINAL['accuracy'] * 100:.2f}% | "
      f"macro-F1 {GVB_VAL_FINAL['macro_f1'] * 100:.2f}%")

## 38. Stage 3 training curves
Training vs validation **accuracy**, **loss** and macro-F1 per epoch (dotted line = start of fine-tuning). Then the **final** model is
re-evaluated without augmentation on the exact training crops and on validation (training / validation accuracy and loss).

In [ ]:
# ============================================================
# SECTION 38 — CNN 2 TRAINING CURVES + TRAINING/VALIDATION ACCURACY & LOSS
# ============================================================
plot_history(gvb_hist, gvb_info, ["accuracy", "loss", "macro_f1"], "G.V. Black (CNN 2)")
GVB_HIST_SUMMARY = history_summary(gvb_hist, gvb_info)
ex_tr = np.where(exact_tr)[0]
GVB_CLEAN = clean_eval(gvb_model, {
    "train": (array_batch_ds(Xg_train, ex_tr, onehot(yg_train[ex_tr], NUM_GVB), GVB["BATCH"], False), yg_train[ex_tr]),
    "val": (gvb_val_ds, yg_val)}, keras.losses.CategoricalCrossentropy())
print("At the kept epoch (augmentation + dropout active during training):")
for k, v in GVB_HIST_SUMMARY.items(): print(f"  {k:16}: {v:.4f}" if isinstance(v, float) else f"  {k:16}: {v}")
print("Final model, clean re-evaluation (no augmentation, no TTA):")
for s, v in GVB_CLEAN.items():
    print(f"  {'Training' if s == 'train' else 'Validation'} accuracy: {v['accuracy']:.4f} | loss: {v['loss']:.4f}  (n={v['n']})")
gap = GVB_CLEAN["train"]["accuracy"] - GVB_CLEAN["val"]["accuracy"]
print(f"Train - validation accuracy gap: {gap:+.3f}" + ("  <-- overfitting: more data / stronger regularisation needed" if gap > 0.15 else ""))

<br>

---
# PART 9 — STAGE 3 COMPLETE EVALUATION (untouched test set)

## 39. Stage 3 accuracy (untouched test set)
The test crops (exact ground-truth boxes of **test photographs only**) are predicted **once**, with the TTA variant fixed on validation.
All metrics in Sections 39–46 and the error analysis come from these predictions. The majority-class baseline is shown for reference only.

In [ ]:
# ============================================================
# SECTION 39 — CNN 2 TEST PREDICTION (ONCE) + ACCURACY
# ============================================================
PIPELINE["models"]["gvb"] = gvb_model
PIPELINE["meta"]["gvb"] = {"class_names": GVB_CLASS_NAMES, "image_size": GVB_SIZE, "context_pad": GVB_PAD, "tta": GVB_TTA,
                           "min_crop_px": GVB["MIN_CROP_PX"], "confidence_threshold": GVB["CONFIDENCE_THRESHOLD"]}
pg_test = predict_rows(gvb_model, Xg_test, np.arange(len(Xg_test)), variant=GVB_TTA)       # the ONE test prediction
yg_pred = pg_test.argmax(1)
GVB_TEST = evaluate_multiclass(yg_test, yg_pred, pg_test, [gvb_short(n) for n in GVB_CLASS_NAMES])
GVB_BASELINES["majority_test"] = majority_baseline(yg_train[ex_tr], yg_test)
print(f"G.V. Black TEST accuracy: {GVB_TEST['accuracy'] * 100:.2f}%  ({int((yg_test == yg_pred).sum())}/{len(yg_test)} crops)")
print(f"(reference) majority-class baseline on the same test crops: {GVB_BASELINES['majority_test']['accuracy'] * 100:.2f}%")

## 40. Stage 3 balanced accuracy
Mean of the per-class recalls over the classes present in the test set — not inflated by the large classes.

In [ ]:
# ============================================================
# SECTION 40 — CNN 2 BALANCED ACCURACY
# ============================================================
print(f"Balanced accuracy: {GVB_TEST['balanced_accuracy'] * 100:.2f}%  (over {len(GVB_TEST['classes_in_test'])} classes present in test)")

## 41. Stage 3 precision

In [ ]:
# ============================================================
# SECTION 41 — CNN 2 PRECISION (macro / weighted / per class)
# ============================================================
print(f"Macro precision   : {GVB_TEST['macro_precision'] * 100:.2f}%\nWeighted precision: {GVB_TEST['weighted_precision'] * 100:.2f}%")
print(per_class_table(GVB_TEST)[["precision", "support", "predicted"]].to_string())

## 42. Stage 3 recall

In [ ]:
# ============================================================
# SECTION 42 — CNN 2 RECALL (macro / weighted / per class)
# ============================================================
print(f"Macro recall   : {GVB_TEST['macro_recall'] * 100:.2f}%\nWeighted recall: {GVB_TEST['weighted_recall'] * 100:.2f}%")
print(per_class_table(GVB_TEST)[["recall", "support"]].to_string())

## 43. Stage 3 F1 score

In [ ]:
# ============================================================
# SECTION 43 — CNN 2 F1 (macro / weighted / per class)
# ============================================================
print(f"Macro F1   : {GVB_TEST['macro_f1'] * 100:.2f}%\nWeighted F1: {GVB_TEST['weighted_f1'] * 100:.2f}%")
print(per_class_table(GVB_TEST)[["f1", "support"]].to_string())

## 44. Stage 3 classification report
Final results table, then per-class precision / recall / F1 / support. Classes with fewer than `MIN_RELIABLE_TEST` test crops are marked.

In [ ]:
# ============================================================
# SECTION 44 — CNN 2 FINAL TABLE + CLASSIFICATION REPORT
# ============================================================
print_results_table(GVB_TEST, f"G.V. BLACK FINAL TEST RESULTS  (CNN 2: {GVB_BACKBONE} @ {GVB_SIZE}px, pad {GVB_PAD}, TTA {GVB_TTA})")
print("\nPer-class results:")
print(per_class_table(GVB_TEST, GVB["MIN_RELIABLE_TEST"]).to_string())
print("\n" + GVB_TEST["classification_report"])
low = pg_test.max(1) < GVB["CONFIDENCE_THRESHOLD"]
print(f"Test crops below the confidence threshold ({GVB['CONFIDENCE_THRESHOLD']:.0%}) -> UNCERTAIN: {low.mean():.1%}; "
      f"accuracy on the rest: " + (f"{accuracy_score(yg_test[~low], yg_pred[~low]) * 100:.2f}%" if (~low).any() else "n/a"))

## 45. Stage 3 confusion matrix

In [ ]:
# ============================================================
# SECTION 45 — CNN 2 CONFUSION MATRIX
# ============================================================
plot_confusion(GVB_TEST["confusion_matrix"], [gvb_short(n) for n in GVB_CLASS_NAMES], "G.V. Black test")

## 46. Stage 3 ROC / PR curves
One-vs-rest ROC and precision–recall curves for every class that has both positive and negative test crops (a class missing from the test
set is skipped, not scored).

In [ ]:
# ============================================================
# SECTION 46 — CNN 2 ONE-VS-REST ROC / PR CURVES
# ============================================================
plot_ovr_curves(yg_test, pg_test, [gvb_short(n) for n in GVB_CLASS_NAMES], "G.V. Black")
print("OvR ROC-AUC per class:", {k: round(v, 4) for k, v in GVB_TEST["ovr_roc_auc_per_class"].items()})
print("OvR PR-AUC  per class:", {k: round(v, 4) for k, v in GVB_TEST["ovr_pr_auc_per_class"].items()})

<br>

---
# PART 10 — STAGE 3 ERROR ANALYSIS (+ unseen-image prediction)

## 47. Stage 3 error analysis
Which samples fail and why: most confident correct / incorrect test crops, confidence of errors, most-confused G.V. Black classes, recall
versus the number of training crops per class (insufficient data), error rate by lesion size (poor / tiny crops) and by photographic view,
and high-confidence errors (candidates for label noise). Nothing here changes the model or the test set.

In [ ]:
# ============================================================
# SECTION 47 — CNN 2 ERROR ANALYSIS + SAVE METADATA
# ============================================================
_cm_test = gvb_crop_meta(gvb_test)
GVB_ERRORS = error_analysis("G.V. Black", Xg_test, np.arange(len(Xg_test)), yg_test, pg_test,
                            [gvb_short(n) for n in GVB_CLASS_NAMES], extra=_cm_test[["crop_px", "view"]],
                            train_counts=np.bincount(yg_train[ex_tr], minlength=NUM_GVB))

PIPELINE["meta"]["gvb"] = write_model_meta("gvb", {
    "model": "CNN 2 — Stage 3 G.V. Black classifier",
    "architecture": f"{GVB_BACKBONE} (ImageNet) + GAP + BN + Dense({GVB['DENSE']}) + softmax({NUM_GVB})",
    "backbone": GVB_BACKBONE, "class_names": GVB_CLASS_NAMES,
    "class_mapping": {str(i): gvb_short(n) for i, n in enumerate(GVB_CLASS_NAMES)}, "mapping_source": GVB_MAPPING_SOURCE,
    "image_size": GVB_SIZE, "context_pad": GVB_PAD, "min_crop_px": GVB["MIN_CROP_PX"], "tta": GVB_TTA,
    "balancing": GVB_BAL, "label_smoothing": GVB_LS,
    "preprocessing": "load_rgb (EXIF) -> square context crop centred on the box (side = longer box side*(1+2*pad), "
                     "black border outside the image) -> resize to image_size -> uint8 [0,255]; " +
                     PREP_DESCRIPTION.get(BACKBONES.get(GVB_BACKBONE, {}).get("prep"), "") + f"; TTA '{GVB_TTA}'",
    "confidence_threshold": GVB["CONFIDENCE_THRESHOLD"], "displayed_confidence": "softmax probability (not calibrated)",
    "class_weight": ({gvb_short(GVB_CLASS_NAMES[int(c)]): v for c, v in GVB_CLASS_WEIGHT.items()} if GVB_CLASS_WEIGHT else None),
    "training_config": GVB, "architecture_screening": GVB_ARCH_SCREEN.to_dict("records"),
    "recipe_screening": GVB_RECIPE_SCREEN.to_dict("records"), "tta_screening": GVB_TTA_SCREEN.to_dict("records"),
    "baselines": GVB_BASELINES, "validation_final": GVB_VAL_FINAL,
    "training": gvb_info, "training_history": gvb_hist, "history_at_kept_epoch": GVB_HIST_SUMMARY, "clean_train_val": GVB_CLEAN,
    "class_distribution": yg_count.to_dict(), "thin_classes": GVB_THIN,
    "test_metrics_gt_crops": {k: v for k, v in GVB_TEST.items() if k != "classification_report"}, "error_analysis": GVB_ERRORS,
    "split_fingerprint": SPLIT_FP["gvb"], "model_path": MODEL_PATHS["gvb"],
})
print("\nCNN 2 metadata saved:", MODEL_META_PATHS["gvb"])

## 48. Stage 3 — unseen image prediction
`predict_gvblack(image_path)` takes a lesion-crop file (e.g. one written by `save_cavity_crops`), or a full image plus a normalised
box (`predict_gvblack(path, box=[x1, y1, x2, y2])`, the same context crop as in training). It applies exactly the training preprocessing
and the validation-chosen TTA, and returns prediction, confidence and the probability of every class; **UNCERTAIN** below
`GVB["CONFIDENCE_THRESHOLD"]` (override per call with `confidence_threshold=`).

In [ ]:
# ============================================================
# SECTION 48 — CNN 2 PREDICTION ON UNSEEN (TEST) LESIONS
# ============================================================
r = gvb_test[0]
crop_path = f"{PREDICTION_DIR}/gvb_demo_crop.jpg"
cv2.imwrite(crop_path, cv2.cvtColor(gvb_prepare_crop(load_rgb(r["image_path"]), r["boxes"][0], GVB_PAD, GVB_SIZE,
                                                     GVB["MIN_CROP_PX"]), cv2.COLOR_RGB2BGR))
print(f"{r['id']} box 0 — true: {gvb_short(GVB_CLASS_NAMES[r['classes'][0]])}  (crop file: {crop_path})\n")
_ = predict_gvblack(crop_path)

<br>

---
# PART 11 — STAGE 4: CNN 3 ICDAS (icdascl.zip)

---
# STAGE 4 — CNN MODEL 3: ICDAS CLASSIFICATION (corrected dataset `icdascl.zip`)
Input: tooth / lesion crop · Output: ICDAS score. **Only `icdascl.zip` is used**; nothing from the old ICDAS ZIP, its folder layout or
its class counts is reused. Every structure, count and mapping below is read from the uploaded ZIP.

## 49. NEW ICDAS ZIP inspection
**IMPORTANT: `icdascl.zip` ONLY.** Programmatic report of the ZIP file itself: top-level entries, folders, file counts, extensions,
nesting depth, non-image files.

In [ ]:
# ============================================================
# SECTION 49 — NEW ICDAS ZIP INSPECTION (icdascl.zip ONLY)
# ============================================================
assert os.path.basename(ICDAS_ZIP) not in REFUSED_ICDAS_ZIPS and os.path.basename(ICDAS_ZIP).lower().startswith("icdascl")
R_ = ICDAS_ZIP_REPORT
print("ICDAS DATASET STRUCTURE\n-----------------------")
print(f"ZIP file            : {R_['zip']}  ({R_['zip_mb']:.1f} MB, {R_['uncompressed_mb']:.1f} MB uncompressed)")
print(f"Entries / files     : {R_['n_entries']} / {R_['n_files']}")
print(f"Top-level entries   : {R_['top_level']}")
print(f"Max nesting depth   : {R_['max_depth']} folder level(s)")
print(f"File extensions     : {R_['extensions']}")
print("Folders (files directly inside):")
for d, n in R_["files_per_folder"].items():
    print(f"  {d or '<zip root>':60}: {n}")
print("Non-image files     :", R_["non_image_files"])

## 50. Extract the corrected ICDAS dataset
Extracted once to `/content/data/icdascl` in Section 7 (a new folder — an extraction of the old ZIP can never be reused). Calling
`stage_zip` again is a no-op unless the ZIP changed.

In [ ]:
# ============================================================
# SECTION 50 — EXTRACT CORRECTED ICDAS DATASET (idempotent)
# ============================================================
stage_zip(ICDAS_ZIP, ICDAS_ROOT)
for dp, dn, fn in os.walk(ICDAS_ROOT):
    dn[:] = sorted(d for d in dn if not d.startswith((".", "__MACOSX")))
    depth = 0 if dp == ICDAS_ROOT else os.path.relpath(dp, ICDAS_ROOT).count(os.sep) + 1
    if depth <= 3:
        print("  " * depth + os.path.basename(dp) + f"/   ({len([f for f in fn if not f.startswith('.')])} files)")

## 51. Detect the actual folder / class structure
Class folders are found by name (`ICDAS_<k>` or `<k>`, anywhere in the tree); **the folder name is the ICDAS score** and is used unchanged.
The model's output index is mapped to the ICDAS score explicitly, so metrics and the ordinal error are computed on true ICDAS scores.

In [ ]:
# ============================================================
# SECTION 51 — DETECT CLASS STRUCTURE + MAPPING
# ============================================================
class_dirs = ICDAS_INFO["class_dirs"]
ICDAS_SCORES = sorted(class_dirs)                   # e.g. [1, 2, 3, 4, 5, 6] — from the folders, not assumed
NUM_ICDAS = len(ICDAS_SCORES)
ICDAS_CLASS_NAMES = [f"ICDAS {s}" for s in ICDAS_SCORES]
print("Class/Folders:")
for s in ICDAS_SCORES:
    print(f"  ICDAS {s}: " + ", ".join(os.path.relpath(d, ICDAS_ROOT) for d in class_dirs[s]))
missing_scores = [s for s in range(7) if s not in ICDAS_SCORES]
if missing_scores:
    print(f"\nICDAS scores WITHOUT a folder in icdascl.zip: {missing_scores}")
    if 0 in missing_scores:
        nc = ICDAS_INFO["nocaries_path"]
        print("  ICDAS 0 is absent by design" + (f" — {os.path.basename(nc)} lists the sound photographs instead." if nc else "."))
        print("  -> CNN 3 is a", NUM_ICDAS, "-class problem; 'no caries' is decided by CNN 1, not by CNN 3.")
print("\nModel output index -> ICDAS score (used everywhere below):")
for i, s in enumerate(ICDAS_SCORES):
    print(f"  {i} -> ICDAS {s}")

ICDAS_MANIFEST = ICDAS_INFO["manifest"]
if len(ICDAS_MANIFEST):
    mis = icdas_df[icdas_df["manifest_icdas"].notna() & (icdas_df["manifest_icdas"] != icdas_df["score"])]
    print(f"\nManifest '{os.path.basename(ICDAS_INFO['manifest_path'])}': {len(ICDAS_MANIFEST)} rows | "
          f"crop files matched to a manifest row: {int(icdas_df['manifest_icdas'].notna().sum())}/{len(icdas_df)} | "
          f"folder vs manifest label disagreements: {len(mis)}" + (" (folder is used)" if len(mis) else ""))
else:
    print("\nNo crop manifest in the ZIP -> source photograph taken from the file name pattern '<photo>__toothNN'.")
fm = icdas_df["file"].map(lambda f: bool(TOOTH_RE.match(stem(f))))
print(f"File names matching '<source photo>__tooth<NN>': {fm.mean():.1%}  -> file names DO identify the source photograph "
      "and the tooth index" if fm.mean() > 0.9 else f"File-name pattern matched for {fm.mean():.1%} of crops")
print("View codes in file names:", dict(Counter(icdas_df["file"].str.extract(r"^([A-Za-z]+)")[0])))

## 52. Count images per ICDAS class

In [ ]:
# ============================================================
# SECTION 52 — IMAGES PER ICDAS CLASS (actual files on disk)
# ============================================================
on_disk = icdas_df["score"].value_counts()
print("ICDAS images per class (files in icdascl.zip):")
for s in range(7):
    print(f"  ICDAS {s}: {int(on_disk.get(s, 0)):>5}" + ("   (no folder)" if s not in ICDAS_SCORES else
          ("   <-- VERY RARE: metrics for this class will not be reliable" if on_disk.get(s, 0) < 100 else "")))
print(f"  Total  : {len(icdas_df):>5}")
if len(ICDAS_MANIFEST):
    mc = ICDAS_MANIFEST.groupby("icdas")["on_disk"].agg(["size", "sum"]).rename(columns={"size": "in manifest", "sum": "on disk"})
    mc["missing from ZIP"] = mc["in manifest"] - mc["on disk"]
    print("\nManifest vs files actually in the ZIP:\n" + mc.to_string())
    if mc["missing from ZIP"].sum():
        print(f"\nNOTE: {int(mc['missing from ZIP'].sum())} crops listed in the manifest are NOT in the ZIP. Training uses only the "
              "files that exist; re-create the ZIP with all crops to use them (nothing else needs to change).")
ICDAS_ONDISK_COUNTS = {f"ICDAS {s}": int(on_disk.get(s, 0)) for s in ICDAS_SCORES}

## 53. Check for corrupt images
Every crop was opened, verified (`PIL.verify`) and fully decoded once in Section 10 (the decoded result is cached). Unreadable files are
excluded.

In [ ]:
# ============================================================
# SECTION 53 — CORRUPT IMAGES + IMAGE SIZES
# ============================================================
bad_ic = icdas_df[~icdas_df["ok"]]
print(f"Corrupted / unreadable crops: {len(bad_ic)}" + (" -> excluded" if len(bad_ic) else ""))
if len(bad_ic): print(bad_ic[["rel", "error"]].head(10).to_string(index=False))
okd = icdas_df[icdas_df.ok]
print("Crop width  (px) 1/5/50/95/99 %:", np.percentile(okd["W"], [1, 5, 50, 95, 99]).round(0))
print("Crop height (px) 1/5/50/95/99 %:", np.percentile(okd["H"], [1, 5, 50, 95, 99]).round(0))
print(f"Model input: square letterbox {ICDAS['IMG_SIZE']}x{ICDAS['IMG_SIZE']} (aspect ratio kept)")
_ex = []
for s in ICDAS_SCORES:
    _ex += list(okd[okd.score == s].index[:3])
show_examples([X_icdas[i] for i in _ex], [f"ICDAS {icdas_df.at[i, 'score']}" for i in _ex], "Crops per class (model input)", ncols=9)

## 54. Duplicate / near-duplicate check

In [ ]:
# ============================================================
# SECTION 54 — EXACT + NEAR DUPLICATES, SOURCE PHOTOGRAPHS, LABEL CONFLICTS
# ============================================================
md5_dup = icdas_df[icdas_df.ok].groupby("md5").filter(lambda d: len(d) > 1)
print(f"Byte-identical crops: {len(md5_dup)} files in {md5_dup['md5'].nunique()} groups"
      + (f", {int((md5_dup.groupby('md5')['score'].nunique() > 1).sum())} with conflicting labels" if len(md5_dup) else ""))
src_arr = icdas_df["source_id"].to_numpy(); sc_arr = icdas_df["score"].to_numpy()
for t in sorted({0, ICDAS["DUP_HAMMING"], 4, ICDAS["GROUP_HAMMING"]}):
    i, j = hamming_edges(ICDAS_HASH, ICDAS_HASH, t, same=True)
    k = ok_ic[i] & ok_ic[j]; i, j = i[k], j[k]
    print(f"dHash <= {t}: {len(i):>4} near-duplicate pairs | from different source photos: {int((src_arr[i] != src_arr[j]).sum()):>4}"
          f" | with different ICDAS labels: {int((sc_arr[i] != sc_arr[j]).sum()):>4}")
print(f"-> pairs with dHash <= {ICDAS['GROUP_HAMMING']} are kept in the SAME split (Section 10); "
      f"near-identical (<= {ICDAS['DUP_HAMMING']}) with conflicting labels: {int(icdas_df.conflict.sum())} crops "
      + ("excluded" if ICDAS["DROP_CONFLICTING_DUPLICATES"] else "kept"))
per_src = icdas_df.groupby("source_id")
print(f"\nSource photographs: {per_src.ngroups} | crops per photo: {dict(sorted(Counter(per_src.size()).items()))}")
print(f"Source photos containing teeth with DIFFERENT ICDAS scores: {int((per_src['score'].nunique() > 1).sum())}")
print(f"Crops whose source photo has identical copies listed in the manifest: {int(icdas_df['copies'].map(len).gt(0).sum())}")
print("-> all crops of a source photograph (and of its identical copies) share one split — verified in Section 55.")
i, j = hamming_edges(ICDAS_HASH, ICDAS_HASH, ICDAS["DUP_HAMMING"], same=True)
k = ok_ic[i] & ok_ic[j] & (src_arr[i] != src_arr[j])
pairs = list(zip(i[k], j[k]))[:4]
if pairs:
    show_examples([X_icdas[a] for p in pairs for a in p],
                  [f"{icdas_df.at[a, 'source_id']} (ICDAS {icdas_df.at[a, 'score']})" for p in pairs for a in p],
                  "Near-identical crops from different source photos (kept in one split)", ncols=4)

## 55. Train / validation / test split
The ICDAS split is the shared **group** split of Section 10 (original source photographs, their identical copies and near-duplicate crops
are grouped; the split was made before any augmentation, stratified by ICDAS score). Validation and test crops are never augmented.

In [ ]:
# ============================================================
# SECTION 55 — ICDAS SPLIT (group level, verified)
# ============================================================
use_ic = icdas_df["ok"] & ~(ICDAS["DROP_CONFLICTING_DUPLICATES"] & icdas_df["conflict"])
IC_ROWS = np.where(use_ic.to_numpy())[0]                    # rows of icdas_df / X_icdas used by Stage 4
y_icdas_all = np.array([ICDAS_SCORES.index(s) for s in icdas_df["score"]], np.int32)     # model index
split_ic = icdas_df["split"].to_numpy()
ic_idx = {s: IC_ROWS[split_ic[IC_ROWS] == s] for s in SPLIT_NAMES}
sub_ = icdas_df.iloc[IC_ROWS]
assert sub_.groupby("source_id")["split"].nunique().max() == 1, "LEAKAGE: one source photo in two splits"
assert sub_.groupby("group")["split"].nunique().max() == 1, "LEAKAGE: one group in two splits"
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not (set(icdas_df["source_id"].iloc[ic_idx[a]]) & set(icdas_df["source_id"].iloc[ic_idx[b]])), f"LEAKAGE {a}/{b}"
    assert not (set(icdas_df["md5"].iloc[ic_idx[a]]) & set(icdas_df["md5"].iloc[ic_idx[b]])), f"identical crop in {a}/{b}"
print(f"Crops used: {len(IC_ROWS)} of {len(icdas_df)} (excluded: {int((~icdas_df.ok).sum())} unreadable, "
      f"{int((icdas_df.ok & ~use_ic).sum())} conflicting near-identical)")
for s, v in ic_idx.items():
    print(f"  {s:5}: {len(v):>5} crops from {icdas_df['source_id'].iloc[v].nunique():>5} source photographs")
print("Leakage checks passed: no source photograph, group or identical crop is shared between splits.")

## 56. ICDAS class distribution

In [ ]:
# ============================================================
# SECTION 56 — ICDAS CLASS DISTRIBUTION PER SPLIT
# ============================================================
dist4 = pd.DataFrame({s: np.bincount(y_icdas_all[ic_idx[s]], minlength=NUM_ICDAS) for s in SPLIT_NAMES}, index=ICDAS_CLASS_NAMES)
dist4["total"] = dist4.sum(1); dist4["share"] = (dist4["total"] / dist4["total"].sum()).round(3)
print("ICDAS crops used for Stage 4:")
for n, v in dist4["total"].items():
    print(f"  {n}: {v}")
print("\n" + dist4.to_string())
ICDAS_UNRELIABLE = [ICDAS_SCORES[k] for k in range(NUM_ICDAS) if dist4.iloc[k]["train"] < ICDAS["MIN_RELIABLE_TRAIN"]]
print(f"\nImbalance (largest / smallest class, train): {dist4['train'].max() / max(dist4['train'].min(), 1):.1f} : 1")
if ICDAS_UNRELIABLE:
    print(f"DATASET LIMITATION: {[f'ICDAS {s}' for s in ICDAS_UNRELIABLE]} have < {ICDAS['MIN_RELIABLE_TRAIN']} TRAINING crops — "
          "metrics for them are NOT reliable, whatever their value.")
for s in SPLIT_NAMES:
    miss = [ICDAS_CLASS_NAMES[k] for k in range(NUM_ICDAS) if dist4.iloc[k][s] == 0]
    if miss: print(f"Classes with no {s} crop: {miss}")
dist4[["train", "val", "test"]].plot.bar(stacked=True, figsize=(8, 3.5), title="ICDAS crops per class and split")
plt.tight_layout(); plt.show()

## 57. Data augmentation
Realistic, mild augmentation inside the model (training only): horizontal flip (severity is mirror-invariant), small rotation /
translation / zoom, brightness, contrast. No colour-hue changes (discolouration is part of the ICDAS criteria).

In [ ]:
# ============================================================
# SECTION 57 — ICDAS DATA AUGMENTATION
# ============================================================
print("Stage 4 augmentation:", ICDAS["AUG"])
_aug = augmentation_block(ICDAS["AUG"])
_x = np.asarray(X_icdas[ic_idx["train"][0]], np.float32)[None]
show_examples([_x[0]] + [np.clip(_aug(_x, training=True).numpy()[0], 0, 255) for _ in range(6)],
              ["original"] + [f"augmented {i}" for i in range(1, 7)], "Stage 4 augmentation (training only)", ncols=7)

## 58. Class weights
Balanced weights from the **training split only**, capped at `MAX_CLASS_WEIGHT` (ICDAS 4 would otherwise get an extreme weight).
Whether class weights, training-set oversampling, both or neither work best is decided on **validation** in Section 59.

In [ ]:
# ============================================================
# SECTION 58 — ICDAS CLASS WEIGHTS (training split only)
# ============================================================
ICDAS_CLASS_WEIGHT_FULL = capped_class_weights(y_icdas_all[ic_idx["train"]], NUM_ICDAS, ICDAS["MAX_CLASS_WEIGHT"])
for c, v in ICDAS_CLASS_WEIGHT_FULL.items():
    print(f"  {ICDAS_CLASS_NAMES[c]}: {v:.3f}")

## 59. Stage 4 CNN architecture — validation search
Completely separate CNN 3 (own weights, own file — it does not reuse CNN 1 or CNN 2).
**59.1** Every backbone (EfficientNetV2B0, EfficientNetB0, DenseNet121, ResNet50, MobileNetV2) × input size is compared with frozen
ImageNet features + a small head trained to convergence, scored on **validation** (½ accuracy + ½ macro-F1; loss and size shown).
Crops for other sizes are decoded once into their own cache. Baselines: majority class and logistic regression on frozen features.
**59.2** Class balancing × label smoothing compared on validation with the chosen features.
**59.3** Model: RGB `[0,255]` → augmentation (train only) → backbone normalisation → backbone → GAP → BatchNorm → Dropout → Dense(256) →
Dropout → softmax over the ICDAS classes present in `icdascl.zip`.

In [ ]:
# ============================================================
# SECTION 59.1 — CNN 3 ARCHITECTURE / INPUT-SIZE SEARCH (VALIDATION ONLY)
# (uses screen_features_lowmem from Section 35.1: same candidates, table and selection, without the RAM build-up)
# ============================================================
ICDAS_METRICS = lambda: [keras.metrics.CategoricalAccuracy(name="accuracy"), MACRO_F1()]
ICDAS_CKPT = f"{CKPT_DIR}/icdas_best.weights.h5"
X_ICDAS_BY_SIZE = {ICDAS["IMG_SIZE"]: X_icdas}
def icdas_cache(size):
    """Decoded crop cache at `size` (row order identical to icdas_df — verified)."""
    if size not in X_ICDAS_BY_SIZE:
        df_s, X_s, _ = build_icdas_index(ICDAS_ROOT, size)
        assert (df_s["rel"].values == icdas_df["rel"].values).all(), "crop order differs between caches"
        X_ICDAS_BY_SIZE[size] = X_s
    return X_ICDAS_BY_SIZE[size]

ICDAS_TRAINED = decide("icdas")
ICDAS_ARCH_SCREEN = ICDAS_RECIPE_SCREEN = ICDAS_TTA_SCREEN = pd.DataFrame(); icdas_hist, icdas_info = {}, {}; ICDAS_BASELINES = {}
ytr_i, yva_i = y_icdas_all[ic_idx["train"]], y_icdas_all[ic_idx["val"]]
if ICDAS_TRAINED:
    with stopwatch("ICDAS feature screening"):
        ICDAS_ARCH_SCREEN, IC_FEATS = screen_features_lowmem(   # memory-safe drop-in, defined in Section 35.1
            ICDAS["CANDIDATES"], ICDAS["SIZE_CANDIDATES"],
            lambda s: (icdas_cache(s), ic_idx["train"], ytr_i, ic_idx["val"], yva_i), NUM_ICDAS, ICDAS_CLASS_WEIGHT_FULL, "ICDAS")
    print("\nICDAS architecture search — VALIDATION (test set not used):")
    print(ICDAS_ARCH_SCREEN[["backbone", "size", "val_accuracy", "val_macro_f1", "val_loss", "score", "params_M", "seconds"]]
          .round(4).to_string(index=False))
    ICDAS_BACKBONE, IC_SIZE = str(ICDAS_ARCH_SCREEN.iloc[0]["backbone"]), int(ICDAS_ARCH_SCREEN.iloc[0]["size"])
    Ftr_i, _, Fva_i, _ = IC_FEATS[(ICDAS_BACKBONE, IC_SIZE)]
    _lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(Ftr_i, ytr_i); _yp = _lr.predict(Fva_i)
    ICDAS_BASELINES = {"majority_val": majority_baseline(ytr_i, yva_i),
                       "logreg_frozen_features_val": {"accuracy": float((_yp == yva_i).mean()), "macro_f1": macro_f1_present(yva_i, _yp)}}
    print(f"\nSelected: {ICDAS_BACKBONE} @ {IC_SIZE}px")
    print("Validation baselines:", {k: {kk: round(vv, 4) if isinstance(vv, float) else vv for kk, vv in v.items()}
                                   for k, v in ICDAS_BASELINES.items()})
else:
    icdas_model = load_keras(MODEL_PATHS["icdas"])
    _m = read_json(MODEL_META_PATHS["icdas"])
    assert _m.get("scores") == ICDAS_SCORES, "saved ICDAS model has different classes — set TRAIN_MODE['icdas']='train'"
    ICDAS_BACKBONE, IC_SIZE = _m.get("backbone", "unknown"), int(_m.get("image_size", ICDAS["IMG_SIZE"]))
    ICDAS_BAL, ICDAS_LS, ICDAS_TTA = _m.get("balancing", "class_weight"), float(_m.get("label_smoothing", 0.0)), _m.get("tta", "none")
    icdas_hist, icdas_info = _m.get("training_history", {}), _m.get("training", {})
    ICDAS_BASELINES = _m.get("baselines", {})
    ICDAS_ARCH_SCREEN = pd.DataFrame(_m.get("architecture_screening", []))
    ICDAS_RECIPE_SCREEN = pd.DataFrame(_m.get("recipe_screening", [])); ICDAS_TTA_SCREEN = pd.DataFrame(_m.get("tta_screening", []))
    print("Loaded the saved ICDAS model (trained on this split) — no retraining:", MODEL_PATHS["icdas"])
    print(f"  {ICDAS_BACKBONE} @ {IC_SIZE}px, balancing {ICDAS_BAL}, label smoothing {ICDAS_LS}, TTA {ICDAS_TTA}")
X_ic = icdas_cache(IC_SIZE)                  # crops at the chosen input size (same rows as X_icdas)

In [ ]:
# ============================================================
# SECTION 59.2 — TRAINING RECIPE (VALIDATION) + DATASETS + MODEL
# ============================================================
if ICDAS_TRAINED:
    ICDAS_RECIPE_SCREEN = screen_training_recipe(Ftr_i, ytr_i, Fva_i, yva_i, NUM_ICDAS, ICDAS["BALANCE_CANDIDATES"],
                                                 ICDAS["LS_CANDIDATES"], ICDAS["MAX_CLASS_WEIGHT"], ICDAS["OVERSAMPLE_POWER"])
    print("Training-recipe search — VALIDATION:\n" + ICDAS_RECIPE_SCREEN.round(4).to_string(index=False))
    ICDAS_BAL, ICDAS_LS = str(ICDAS_RECIPE_SCREEN.iloc[0]["balancing"]), float(ICDAS_RECIPE_SCREEN.iloc[0]["label_smoothing"])
    print(f"Selected recipe: balancing = {ICDAS_BAL}, label smoothing = {ICDAS_LS}")
    del IC_FEATS
ICDAS_CLASS_WEIGHT = ICDAS_CLASS_WEIGHT_FULL if "class_weight" in ICDAS_BAL else None
ic_train_rows = (oversample_rows(ic_idx["train"], y_icdas_all[ic_idx["train"]], ICDAS["OVERSAMPLE_POWER"], SEED)
                 if "oversample" in ICDAS_BAL else ic_idx["train"])
if "oversample" in ICDAS_BAL:
    print("Training rows per class after oversampling (training set only):",
          dict(zip(ICDAS_CLASS_NAMES, np.bincount(y_icdas_all[ic_train_rows], minlength=NUM_ICDAS).tolist())))
ICDAS_LOSS = keras.losses.CategoricalCrossentropy(label_smoothing=ICDAS_LS)
yi_1h = onehot(y_icdas_all, NUM_ICDAS)
icdas_train_ds = array_batch_ds(X_ic, ic_train_rows, yi_1h[ic_train_rows], ICDAS["BATCH"], True)
icdas_val_ds   = array_batch_ds(X_ic, ic_idx["val"], yi_1h[ic_idx["val"]], ICDAS["BATCH"], False)
if ICDAS_TRAINED:
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    icdas_model, icdas_base = build_classifier(ICDAS_BACKBONE, (IC_SIZE, IC_SIZE), NUM_ICDAS, ICDAS, "cnn3_stage4_icdas_classifier")
print(f"CNN 3: {icdas_model.name} | {ICDAS_BACKBONE} @ {IC_SIZE}px | {NUM_ICDAS} classes {ICDAS_SCORES} | "
      f"parameters {icdas_model.count_params():,}")

## 60. Stage 4 training — PHASE 1 (frozen backbone)
Frozen ImageNet backbone + ICDAS classification head (BatchNorm, dropout); AdamW; EarlyStopping, ModelCheckpoint and ReduceLROnPlateau
on validation.

In [ ]:
# ============================================================
# SECTION 60 — CNN 3 PHASE 1: FROZEN BACKBONE
# ============================================================
if ICDAS_TRAINED:
    with stopwatch("ICDAS phase 1"):
        icdas_hist, icdas_info = train_phase_a(icdas_model, icdas_train_ds, icdas_val_ds, ICDAS, ICDAS_LOSS, ICDAS_METRICS,
                                               ICDAS["MONITOR"], ICDAS_CLASS_WEIGHT, ICDAS_CKPT)
else:
    print("Skipped — the saved ICDAS model was loaded in Section 59.")

## 61. Stage 4 fine-tuning — PHASE 2
Gradual unfreezing (`UNFREEZE_STEPS`), BatchNorm frozen, low LR with warm-up + cosine decay, validation checkpointing; then the
test-time-augmentation variant is chosen on **validation**.

In [ ]:
# ============================================================
# SECTION 61 — CNN 3 PHASE 2: FINE-TUNING + TTA CHOICE (VALIDATION) + SAVE
# ============================================================
if ICDAS_TRAINED:
    with stopwatch("ICDAS phase 2"):
        icdas_hist, icdas_info = train_phase_b(icdas_model, icdas_base, icdas_train_ds, icdas_val_ds, ICDAS, ICDAS_LOSS,
                                               ICDAS_METRICS, ICDAS["MONITOR"], ICDAS_CLASS_WEIGHT, ICDAS_CKPT, icdas_hist, icdas_info)
    ICDAS_TTA, ICDAS_TTA_SCREEN = choose_tta(icdas_model, X_ic, ic_idx["val"], yva_i, ICDAS["TTA_CANDIDATES"])
    print("Test-time augmentation — VALIDATION:\n" + ICDAS_TTA_SCREEN.round(4).to_string(index=False) + f"\n-> using '{ICDAS_TTA}'")
    icdas_model.save(MODEL_PATHS["icdas"])
    print("Saved", MODEL_PATHS["icdas"])
else:
    print("Skipped — the saved ICDAS model was loaded in Section 59.")
_pv = predict_rows(icdas_model, X_ic, ic_idx["val"], variant=ICDAS_TTA)
ICDAS_VAL_FINAL = {"accuracy": float((_pv.argmax(1) == yva_i).mean()), "macro_f1": macro_f1_present(yva_i, _pv.argmax(1))}
print(f"Final model on VALIDATION (the split used for all choices): accuracy {ICDAS_VAL_FINAL['accuracy'] * 100:.2f}% | "
      f"macro-F1 {ICDAS_VAL_FINAL['macro_f1'] * 100:.2f}%")

## 62. Stage 4 training curves
Training / validation accuracy and loss per epoch, and a clean re-evaluation of the final model (no augmentation).

In [ ]:
# ============================================================
# SECTION 62 — CNN 3 TRAINING CURVES + TRAINING/VALIDATION ACCURACY & LOSS
# ============================================================
plot_history(icdas_hist, icdas_info, ["accuracy", "loss", "macro_f1"], "ICDAS (CNN 3)")
ICDAS_HIST_SUMMARY = history_summary(icdas_hist, icdas_info)
ICDAS_CLEAN = clean_eval(icdas_model, {
    "train": (array_batch_ds(X_ic, ic_idx["train"], yi_1h[ic_idx["train"]], ICDAS["BATCH"], False), y_icdas_all[ic_idx["train"]]),
    "val": (icdas_val_ds, y_icdas_all[ic_idx["val"]])}, keras.losses.CategoricalCrossentropy())
print("At the kept epoch (augmentation + dropout active during training):")
for k, v in ICDAS_HIST_SUMMARY.items(): print(f"  {k:16}: {v:.4f}" if isinstance(v, float) else f"  {k:16}: {v}")
print("Final model, clean re-evaluation (no augmentation, no TTA):")
for s, v in ICDAS_CLEAN.items():
    print(f"  {'Training' if s == 'train' else 'Validation'} accuracy: {v['accuracy']:.4f} | loss: {v['loss']:.4f}  (n={v['n']})")
gap = ICDAS_CLEAN["train"]["accuracy"] - ICDAS_CLEAN["val"]["accuracy"]
print(f"Train - validation accuracy gap: {gap:+.3f}" + ("  <-- overfitting" if gap > 0.15 else ""))

<br>

---
# PART 12 — STAGE 4 COMPLETE EVALUATION (untouched test set)

## 63. Stage 4 accuracy (untouched ICDAS test set)
The test crops are predicted **once**; Sections 63–72 are all computed from these predictions.

In [ ]:
# ============================================================
# SECTION 63 — CNN 3 TEST PREDICTION (ONCE) + ACCURACY
# ============================================================
PIPELINE["models"]["icdas"] = icdas_model
PIPELINE["meta"]["icdas"] = {"scores": ICDAS_SCORES, "class_names": ICDAS_CLASS_NAMES, "image_size": IC_SIZE, "tta": ICDAS_TTA,
                             "confidence_threshold": ICDAS["CONFIDENCE_THRESHOLD"], "unreliable_scores": ICDAS_UNRELIABLE}
pi_test = predict_rows(icdas_model, X_ic, ic_idx["test"], variant=ICDAS_TTA)        # the ONE test prediction
yi_test = y_icdas_all[ic_idx["test"]]; yi_pred = pi_test.argmax(1)
ICDAS_TEST = evaluate_multiclass(yi_test, yi_pred, pi_test, ICDAS_CLASS_NAMES)
ICDAS_BASELINES["majority_test"] = majority_baseline(ytr_i, yi_test)
print(f"ICDAS TEST accuracy: {ICDAS_TEST['accuracy'] * 100:.2f}%  ({int((yi_test == yi_pred).sum())}/{len(yi_test)} crops)")
print(f"(reference) majority-class baseline on the same test crops "
      f"(always '{ICDAS_CLASS_NAMES[ICDAS_BASELINES['majority_test']['majority_class']]}'): "
      f"{ICDAS_BASELINES['majority_test']['accuracy'] * 100:.2f}%")

## 64. Stage 4 balanced accuracy

In [ ]:
# ============================================================
# SECTION 64 — CNN 3 BALANCED ACCURACY
# ============================================================
print(f"Balanced accuracy: {ICDAS_TEST['balanced_accuracy'] * 100:.2f}%  (over {len(ICDAS_TEST['classes_in_test'])} classes present in test)")

## 65. Stage 4 precision

In [ ]:
# ============================================================
# SECTION 65 — CNN 3 PRECISION
# ============================================================
print(f"Macro precision   : {ICDAS_TEST['macro_precision'] * 100:.2f}%\nWeighted precision: {ICDAS_TEST['weighted_precision'] * 100:.2f}%")
print(per_class_table(ICDAS_TEST)[["precision", "support", "predicted"]].to_string())

## 66. Stage 4 recall

In [ ]:
# ============================================================
# SECTION 66 — CNN 3 RECALL
# ============================================================
print(f"Macro recall   : {ICDAS_TEST['macro_recall'] * 100:.2f}%\nWeighted recall: {ICDAS_TEST['weighted_recall'] * 100:.2f}%")
print(per_class_table(ICDAS_TEST)[["recall", "support"]].to_string())

## 67. Stage 4 F1 score

In [ ]:
# ============================================================
# SECTION 67 — CNN 3 F1
# ============================================================
print(f"Macro F1   : {ICDAS_TEST['macro_f1'] * 100:.2f}%\nWeighted F1: {ICDAS_TEST['weighted_f1'] * 100:.2f}%")
print(per_class_table(ICDAS_TEST)[["f1", "support"]].to_string())

## 68. Stage 4 classification report

In [ ]:
# ============================================================
# SECTION 68 — CNN 3 FINAL TABLE + CLASSIFICATION REPORT
# ============================================================
print_results_table(ICDAS_TEST, f"ICDAS FINAL TEST RESULTS  (CNN 3: {ICDAS_BACKBONE} @ {IC_SIZE}px, {NUM_ICDAS} classes, TTA {ICDAS_TTA})")
_pc = per_class_table(ICDAS_TEST)
_pc["train crops"] = dist4["train"].values
_pc["reliable?"] = np.where(_pc["train crops"] >= ICDAS["MIN_RELIABLE_TRAIN"], "yes", f"NO (<{ICDAS['MIN_RELIABLE_TRAIN']} train)")
print("\nPer-class results:\n" + _pc.to_string())
print("\n" + ICDAS_TEST["classification_report"])

## 69. Stage 4 confusion matrix

In [ ]:
# ============================================================
# SECTION 69 — CNN 3 CONFUSION MATRIX
# ============================================================
plot_confusion(ICDAS_TEST["confusion_matrix"], ICDAS_CLASS_NAMES, "ICDAS test")

## 70. Stage 4 ROC / PR curves
One-vs-rest per class (classes missing from the test set are skipped); macro and support-weighted averages in Section 68.

In [ ]:
# ============================================================
# SECTION 70 — CNN 3 ONE-VS-REST ROC / PR CURVES
# ============================================================
plot_ovr_curves(yi_test, pi_test, ICDAS_CLASS_NAMES, "ICDAS")
print("OvR ROC-AUC per class:", {k: round(v, 4) for k, v in ICDAS_TEST["ovr_roc_auc_per_class"].items()})
print("OvR PR-AUC  per class:", {k: round(v, 4) for k, v in ICDAS_TEST["ovr_pr_auc_per_class"].items()})

## 71. ICDAS mean absolute error
ICDAS is an ordered severity scale, so the size of an error matters: `MAE = mean(|true ICDAS − predicted ICDAS|)`, computed on the
**ICDAS scores** (not on the model's output indices). These are **additional** metrics — they do not replace accuracy / F1.

In [ ]:
# ============================================================
# SECTION 71 — ICDAS MAE (ordinal)
# ============================================================
s_true = np.array(ICDAS_SCORES)[yi_test]; s_pred = np.array(ICDAS_SCORES)[yi_pred]
err = s_pred - s_true
ICDAS_ORDINAL = {"mae": float(np.abs(err).mean()), "exact_agreement": float((err == 0).mean()),
                 "within_1": float((np.abs(err) <= 1).mean()), "mean_signed_error": float(err.mean()),
                 "quadratic_weighted_kappa": float(cohen_kappa_score(s_true, s_pred, weights="quadratic")),
                 "mae_per_true_class": {f"ICDAS {s}": float(np.abs(err[s_true == s]).mean()) for s in ICDAS_SCORES if (s_true == s).any()}}
print(f"ICDAS MAE:\n{ICDAS_ORDINAL['mae']:.2f}")
print(f"\nMean signed error (bias): {ICDAS_ORDINAL['mean_signed_error']:+.3f}  (+ = over-scoring)")
print("MAE per true class:", {k: round(v, 2) for k, v in ICDAS_ORDINAL["mae_per_true_class"].items()})

## 72. ICDAS within ±1 accuracy
Percentage of test crops whose predicted score is within one ICDAS point of the true score (true 3 / predicted 4 counts; true 3 /
predicted 6 does not). Also: exact agreement and quadratic-weighted Cohen's κ.

In [ ]:
# ============================================================
# SECTION 72 — WITHIN ±1 + ERROR DISTRIBUTION + SAVE METADATA
# ============================================================
print(f"Within ±1:\n{ICDAS_ORDINAL['within_1'] * 100:.2f}%")
print(f"\nExact agreement           : {ICDAS_ORDINAL['exact_agreement'] * 100:.2f}%")
print(f"Quadratic-weighted kappa  : {ICDAS_ORDINAL['quadratic_weighted_kappa']:.3f}")
fig, ax = plt.subplots(figsize=(7, 3.2))
vals, cnts = np.unique(err, return_counts=True)
ax.bar(vals, cnts); ax.set_xlabel("predicted − true ICDAS"); ax.set_ylabel("test crops"); ax.set_title("ICDAS ordinal error distribution")
plt.tight_layout(); plt.show()
if ICDAS_UNRELIABLE:
    print("Not reliable (too few training crops):", [f"ICDAS {s}" for s in ICDAS_UNRELIABLE])

def icdas_meta_payload(extra=None):
    d = {"model": "CNN 3 — Stage 4 ICDAS classifier (icdascl.zip)",
         "architecture": f"{ICDAS_BACKBONE} (ImageNet) + GAP + BN + Dense({ICDAS['DENSE']}) + softmax({NUM_ICDAS})",
         "backbone": ICDAS_BACKBONE, "dataset": os.path.basename(ICDAS_ZIP), "level": "tooth / lesion crop",
         "scores": ICDAS_SCORES, "class_names": ICDAS_CLASS_NAMES,
         "class_mapping": {str(i): f"ICDAS {s}" for i, s in enumerate(ICDAS_SCORES)},
         "mapping_source": "icdascl.zip class folder names (folder name = ICDAS score)", "absent_scores": missing_scores,
         "image_size": IC_SIZE, "preprocessing": f"load_rgb (EXIF) -> square letterbox {IC_SIZE}x{IC_SIZE} (aspect kept, black padding) "
                                                 "-> uint8 [0,255]; " + PREP_DESCRIPTION.get(BACKBONES.get(ICDAS_BACKBONE, {}).get("prep"), "")
                                                 + f"; TTA '{ICDAS_TTA}'",
         "tta": ICDAS_TTA, "balancing": ICDAS_BAL, "label_smoothing": ICDAS_LS, "baselines": ICDAS_BASELINES,
         "validation_final": ICDAS_VAL_FINAL, "architecture_screening": ICDAS_ARCH_SCREEN.to_dict("records"),
         "recipe_screening": ICDAS_RECIPE_SCREEN.to_dict("records"), "tta_screening": ICDAS_TTA_SCREEN.to_dict("records"),
         "error_analysis": globals().get("ICDAS_ERRORS"),
         "confidence_threshold": ICDAS["CONFIDENCE_THRESHOLD"], "unreliable_scores": ICDAS_UNRELIABLE,
         "inference_window": PIPELINE["meta"].get("icdas", {}).get("inference_window"),
         "class_weight": ({ICDAS_CLASS_NAMES[c]: v for c, v in ICDAS_CLASS_WEIGHT.items()} if ICDAS_CLASS_WEIGHT else None),
         "class_distribution": dist4[["train", "val", "test"]].to_dict(), "files_on_disk_per_class": ICDAS_ONDISK_COUNTS,
         "excluded_conflicting_near_duplicates": int((icdas_df.ok & ~use_ic).sum()),
         "training_config": ICDAS, "training": icdas_info,
         "training_history": icdas_hist, "history_at_kept_epoch": ICDAS_HIST_SUMMARY, "clean_train_val": ICDAS_CLEAN,
         "test_metrics": {k: v for k, v in ICDAS_TEST.items() if k != "classification_report"}, "test_ordinal": ICDAS_ORDINAL,
         "split_fingerprint": SPLIT_FP["icdas"], "model_path": MODEL_PATHS["icdas"]}
    d.update(extra or {})
    return d
PIPELINE["models"]["icdas"] = icdas_model
PIPELINE["meta"]["icdas"] = write_model_meta("icdas", icdas_meta_payload())
print("CNN 3 metadata saved:", MODEL_META_PATHS["icdas"])

<br>

---
# PART 13 — STAGE 4 ERROR ANALYSIS (+ unseen-image prediction)

## 73. Stage 4 error analysis
Which test crops fail and why: most confident correct / incorrect predictions with their confidence, confidence of errors, most-confused
scores, **adjacent-score confusions** (1 vs 2, 2 vs 3, 3 vs 4, 4 vs 5, 5 vs 6 — ICDAS is ordinal, so ±1 confusions are the clinically
expected kind), recall versus training crops per class (insufficient data), error rate by crop size and photographic view, and
high-confidence errors (label-noise candidates). Nothing here changes the model or the test set.

In [ ]:
# ============================================================
# SECTION 73 — CNN 3 ERROR ANALYSIS (test predictions from Section 63) + SAVE METADATA
# ============================================================
_te = icdas_df.iloc[ic_idx["test"]]
_extra = pd.DataFrame({"crop_px": np.maximum(_te["W"].values, _te["H"].values).astype(float),
                       "view": _te["file"].str.extract(r"^([A-Za-z]+)")[0].values})
ICDAS_ERRORS = error_analysis("ICDAS", X_ic, ic_idx["test"], yi_test, pi_test, ICDAS_CLASS_NAMES, extra=_extra,
                              train_counts=dist4["train"].values, ordinal_scores=ICDAS_SCORES)
PIPELINE["meta"]["icdas"] = write_model_meta("icdas", icdas_meta_payload())
print("\nCNN 3 metadata updated with the error analysis.")

## 74. Stage 4 — unseen image prediction
**74.1** `predict_icdas(cavity_crop_path)` on a held-out test crop: predicted score, confidence, the probability of every ICDAS score
(ICDAS 0 is shown as *n/a* because `icdascl.zip` has no ICDAS 0 class), **UNCERTAIN** below `ICDAS["CONFIDENCE_THRESHOLD"]`.

**72.2 ICDAS on localizer-style lesion crops.** In the pipeline CNN 3 receives a window around a *lesion box*, not a manifest tooth crop.
Where a G.V. Black photograph is also an ICDAS source photograph (matched by MD5 through the caries archive, or by file name) with the same
pixel size, each ground-truth lesion box is linked to the manifest tooth box that covers it (≥ 50 % of the lesion area) and inherits that
tooth's ICDAS score. The window `(mult, min_scale)` is chosen on **validation** lesions (macro-F1); **test** lesions are evaluated once
with the chosen window. If no links exist, `ICDAS["DEFAULT_WINDOW"]` is used and this is stated.

In [ ]:
# ============================================================
# SECTION 74.1 — CNN 3 PREDICTION ON AN UNSEEN (TEST) CROP
# ============================================================
k = ic_idx["test"][0]
print(f"{icdas_df.at[k, 'rel']} — true ICDAS {icdas_df.at[k, 'score']}\n")
_ = predict_icdas(icdas_df.at[k, "path"])

In [ ]:
# ============================================================
# SECTION 74.2 — LESION -> TOOTH LINKS, WINDOW CHOSEN ON VALIDATION, TEST ONCE
# ============================================================
def window_from(mult, scale):
    return {"mult": float(mult), "min_scale": float(scale), "min_frac_w": float(scale * TOOTH_FRAC[0]),
            "min_frac_h": float(scale * TOOTH_FRAC[1])}

def link_lesions_to_teeth(recs):
    """[(record, lesion_box, icdas_score)] for G.V. Black lesions covered by a manifest tooth box of the same photograph."""
    if not len(ICDAS_MANIFEST) or not {"x", "y", "w", "h", "source_width", "source_height"} <= set(ICDAS_MANIFEST.columns):
        return []
    by_stem = defaultdict(list)
    for row in ICDAS_MANIFEST.itertuples(index=False):
        by_stem[stem(str(row.source_image)).lower()].append(row)
    md5_names = defaultdict(set)
    for r_ in ok_rows:
        if index.at[r_, "label"] == 1: md5_names[index.at[r_, "md5"]].add(stem(index.at[r_, "src"]).lower())
    out = []
    for r in recs:
        names = md5_names.get(r["md5"], set()) | {str(r["orig_name"]).lower()}
        teeth = [t for n in names for t in by_stem.get(n, []) if (t.source_width, t.source_height) == (r["W"], r["H"])]
        if not teeth: continue
        tb = np.array([[t.x / t.source_width, t.y / t.source_height, (t.x + t.w) / t.source_width,
                        (t.y + t.h) / t.source_height] for t in teeth], np.float32)
        ts = np.array([t.icdas for t in teeth])
        for b in r["boxes"]:
            iw = np.clip(np.minimum(b[2], tb[:, 2]) - np.maximum(b[0], tb[:, 0]), 0, None)
            ih = np.clip(np.minimum(b[3], tb[:, 3]) - np.maximum(b[1], tb[:, 1]), 0, None)
            cov = iw * ih / max((b[2] - b[0]) * (b[3] - b[1]), 1e-9)
            j = int(np.argmax(cov))
            if cov[j] >= 0.5 and int(ts[j]) in ICDAS_SCORES:
                out.append((r, b, int(ts[j])))
    return out

def predict_lesion_windows(links, window):
    by_img = defaultdict(list)
    for i, (r, b, s) in enumerate(links): by_img[r["image_path"]].append((i, b))
    X = np.zeros((len(links), IC_SIZE, IC_SIZE, 3), np.uint8)
    for p, items in by_img.items():
        rgb = load_rgb(p)
        for i, b in items: X[i] = icdas_prepare_from_box(rgb, b, IC_SIZE, window)
    return predict_rows(icdas_model, X, np.arange(len(X)), variant=ICDAS_TTA), X

LESION_LINKS = {s: link_lesions_to_teeth(recs) for s, recs in [("val", gvb_val), ("test", gvb_test)]}
print(f"Linked lesions (G.V. Black box -> manifest tooth box): val {len(LESION_LINKS['val'])} | test {len(LESION_LINKS['test'])}")
ICDAS_WINDOW_SCREEN = pd.DataFrame(); ICDAS_LESION_TEST = None
if len(LESION_LINKS["val"]) >= 10:
    rows = []
    yv = np.array([ICDAS_SCORES.index(s) for *_, s in LESION_LINKS["val"]])
    for mult, scale in ICDAS["WINDOW_CANDIDATES"]:
        pv, _ = predict_lesion_windows(LESION_LINKS["val"], window_from(mult, scale))
        ev = evaluate_multiclass(yv, pv.argmax(1), pv, ICDAS_CLASS_NAMES)
        rows.append({"mult": mult, "min_scale": scale, "val_macro_f1": ev["macro_f1"], "val_accuracy": ev["accuracy"]})
    ICDAS_WINDOW_SCREEN = pd.DataFrame(rows).sort_values(["val_macro_f1", "val_accuracy"], ascending=False).reset_index(drop=True)
    print("Window screening on VALIDATION lesions:\n" + ICDAS_WINDOW_SCREEN.to_string(index=False))
    best = ICDAS_WINDOW_SCREEN.iloc[0]
    ICDAS_WINDOW = window_from(best["mult"], best["min_scale"]); ICDAS_WINDOW["source"] = "chosen on validation lesions (macro-F1)"
else:
    ICDAS_WINDOW = window_from(*ICDAS["DEFAULT_WINDOW"])
    ICDAS_WINDOW["source"] = "config default (fewer than 10 linked validation lesions — window could not be tuned)"
print("ICDAS inference window:", ICDAS_WINDOW)

if LESION_LINKS["test"]:
    yt = np.array([ICDAS_SCORES.index(s) for *_, s in LESION_LINKS["test"]])
    pt, Xlt = predict_lesion_windows(LESION_LINKS["test"], ICDAS_WINDOW)
    ICDAS_LESION_TEST = evaluate_multiclass(yt, pt.argmax(1), pt, ICDAS_CLASS_NAMES)
    e_ = np.array(ICDAS_SCORES)[pt.argmax(1)] - np.array(ICDAS_SCORES)[yt]
    ICDAS_LESION_TEST.update(mae=float(np.abs(e_).mean()), within_1=float((np.abs(e_) <= 1).mean()))
    print_results_table(ICDAS_LESION_TEST, "ICDAS ON LESION WINDOWS (test photographs, ground-truth lesion boxes)")
    print(f"MAE {ICDAS_LESION_TEST['mae']:.2f} | within ±1 {ICDAS_LESION_TEST['within_1'] * 100:.2f}%")
    show_examples(list(Xlt[:6]), [f"true ICDAS {s}" for *_, s in LESION_LINKS["test"][:6]], "CNN 3 input cut around test lesions")
else:
    print("No linked test lesions — lesion-window ICDAS accuracy cannot be measured on this data.")

PIPELINE["meta"]["icdas"] = write_model_meta("icdas", icdas_meta_payload({
    "inference_window": ICDAS_WINDOW, "window_screening": ICDAS_WINDOW_SCREEN.to_dict("records"),
    "tooth_frac_median": TOOTH_FRAC,
    "lesion_window_test_metrics": ({k: v for k, v in ICDAS_LESION_TEST.items() if k != "classification_report"}
                                   if ICDAS_LESION_TEST else None),
    "lesion_links": {s: len(v) for s, v in LESION_LINKS.items()}}))
print("CNN 3 metadata updated with the inference window.")

<br>

---
# PART 14 — FINAL MULTI-CNN INFERENCE

---
# FINAL PIPELINE — MULTI-CNN CASCADE
## 75. Load all models (three CNNs + localizer + optional Stage 2 FDI detector)
Uses the models in memory if this session trained them; otherwise loads the saved `.keras` files + metadata (no retraining).
**After a runtime restart:** run Sections 2–6, then start here.

In [ ]:
# ============================================================
# SECTION 75 — LOAD ALL THREE CNNs (+ localizer)
# ============================================================
load_all_models()                       # loads only what is not already in memory
for k, title in [("stage1", "CNN 1  Stage 1 healthy vs caries"), ("localizer", "Localizer (where)"),
                 ("gvb", "CNN 2  Stage 3 G.V. Black"), ("icdas", "CNN 3  Stage 4 ICDAS")]:
    m = PIPELINE["meta"][k]
    extra = {"stage1": lambda: f"threshold {m['threshold']:.3f} ± {m['uncertain_margin']:.2f}, confidence ≥ {m['confidence_threshold']:.2f}",
             "localizer": lambda: f"score ≥ {m['score_threshold']:.2f}, input {m['input_size']}",
             "gvb": lambda: f"classes {[gvb_short(n) for n in m['class_names']]}, pad {m['context_pad']}",
             "icdas": lambda: f"scores {m['scores']}, window {({kk: m['inference_window'][kk] for kk in ('mult', 'min_scale')} if m.get('inference_window') else 'default')}"}[k]()
    print(f"{title:36}: {m.get('backbone', 'MobileNetV2 C4+C5')} | {extra}")
if "fdi" in PIPELINE["models"]:
    m = PIPELINE["meta"]["fdi"]
    print(f"{'Stage 2  FDI tooth detector':36}: MobileNetV2 C4+C5 | {len(m['class_names'])} FDI classes, "
          f"score ≥ {m['score_threshold']:.2f}, number confidence ≥ {m['class_confidence_threshold']:.2f}")
else:
    print(f"{'Stage 2  FDI tooth detector':36}: UNAVAILABLE — {PIPELINE['unavailable'].get('fdi', 'not loaded')}")
assert PIPELINE["models"]["stage1"] is not PIPELINE["models"]["gvb"] is not PIPELINE["models"]["icdas"]
print("\nThree independent classifier CNNs + one localizer" + (" + the Stage 2 FDI detector" if "fdi" in PIPELINE["models"] else "")
      + " are loaded (separate weights, separate files).")

## 76. Final multi-CNN prediction function
`predict_dental_condition(image_path)` / `run_pipeline(rgb)`:
```
FULL DENTAL IMAGE → CNN 1 (healthy vs caries) → HEALTHY: stop | UNCERTAIN: stop (unless continue_if_uncertain=True) | CARIES ↓
→ Stage 2 FDI detector: tooth boxes + FDI numbers (UNAVAILABLE if not trained)
→ cavity localization → one or more lesion boxes
→ lesion → tooth association (largest share of the lesion box covered by a tooth box; 'Unknown / uncertain' if none / ambiguous)
→ for EACH lesion independently: CNN 2 (G.V. Black) + CNN 3 (ICDAS) → result dict
```
Result keys: `stage1`, `stage2`, `localization`, `stage3`, `stage4` (each with a `status` + reason when not run), `teeth` (Stage 2
detections) and `lesions` (one dict per lesion: id, box, localizer score, `fdi_tooth` / `fdi_display` / `fdi_status`, G.V. Black and ICDAS
with their own confidence and UNCERTAIN flag). Caries but no lesion above the localizer threshold → "no lesion localized" (no tooth,
G.V. Black or ICDAS is invented). A lower `score_threshold` adds less certain lesions, flagged `below_validated_threshold`.

In [ ]:
# ============================================================
# SECTION 76 — FINAL MULTI-CNN PREDICTION FUNCTION
#   Stage 1 -> Stage 2 FDI teeth -> lesion localization -> lesion->tooth link -> Stage 3 + Stage 4 per lesion
# ============================================================
def _batch_forward(model, X):
    return np.asarray(model(tf.convert_to_tensor(np.asarray(X, np.float32)), training=False), np.float64)

def run_pipeline(rgb, score_threshold=None, s1_confidence_threshold=None, continue_if_uncertain=False, keep_crops=False):
    """Full cascade on one RGB image -> JSON-serialisable result dict.
    Every value comes from a model output. A stage that does not run keeps status NOT RUN / UNAVAILABLE + a reason; nothing
    is filled in. keep_crops=True adds the CNN inputs under '_crops' (not JSON-serialisable; stripped before saving)."""
    m1, ml, mg, mi = (PIPELINE["meta"][k] for k in ("stage1", "localizer", "gvb", "icdas"))
    d = stage1_decide(stage1_probability(rgb), m1, s1_confidence_threshold)
    res = {"stage1": d, "stage2": {"status": "NOT RUN"}, "localization": {"status": "NOT RUN"},
           "stage3": {"status": "NOT RUN"}, "stage4": {"status": "NOT RUN"}, "teeth": [], "lesions": []}
    if d["prediction"] == "HEALTHY" or (d["prediction"] == "UNCERTAIN" and not continue_if_uncertain):
        res["overall"] = ("HEALTHY — NO CARIES DETECTED" if d["prediction"] == "HEALTHY" else
                          "UNCERTAIN — Stage 1 is not confident; retake a clear, well-lit photo or see a dentist")
        for k in ("stage2", "localization", "stage3", "stage4"):
            res[k]["reason"] = f"Stage 1 = {d['prediction']} (caries-specific cascade stopped)"
        return res

    # ---- Stage 2: teeth + FDI numbers ----
    teeth = None
    if "fdi" in PIPELINE["models"]:
        mf = PIPELINE["meta"]["fdi"]
        teeth = detect_teeth(rgb)
        res["teeth"] = teeth
        res["stage2"] = {"status": "RAN", "n_teeth": len(teeth), "n_number_uncertain": int(sum(t["fdi_uncertain"] for t in teeth)),
                         "score_threshold": float(mf["score_threshold"]),
                         "class_confidence_threshold": float(mf["class_confidence_threshold"])}
    else:
        res["stage2"] = {"status": "UNAVAILABLE", "reason": PIPELINE["unavailable"].get(
            "fdi", "Stage 2 FDI detector is not loaded (train it in Section 24, then run load_all_models())")}

    # ---- lesion localization (validated threshold unless the caller lowers it) ----
    validated = float(ml["score_threshold"]); used = validated if score_threshold is None else float(score_threshold)
    boxes, scores = localize(rgb, used)
    crops = generate_cavity_crops(rgb, boxes=boxes, scores=scores)
    res["localization"] = {"status": "RAN", "n_regions": int(len(boxes)), "n_valid_crops": len(crops),
                           "score_threshold_used": used, "validated_threshold": validated}

    # ---- explicit lesion -> tooth association ----
    if teeth is not None:
        links = associate_lesions_with_teeth([c["box_norm"] for c in crops], teeth, mf["assoc_min_overlap"], mf["assoc_ambiguity_ratio"])
    else:
        links = [{"fdi_tooth": None, "fdi_display": "Unavailable", "fdi_status": "STAGE2_UNAVAILABLE", "tooth_id": None,
                  "fdi_confidence": None, "fdi_overlap": None, "fdi_also_overlaps": [], "fdi_note": "Stage 2 unavailable"}
                 for _ in crops]

    # ---- Stage 3 + Stage 4 on EACH lesion crop (same preprocessing and validation-chosen TTA as the test evaluation) ----
    if crops:
        pg = tta_forward(PIPELINE["models"]["gvb"], np.stack([c["gvb_crop"] for c in crops]), mg.get("tta", "none"))
        pi = tta_forward(PIPELINE["models"]["icdas"], np.stack([c["icdas_crop"] for c in crops]), mi.get("tta", "none"))
    for i, (c, lk) in enumerate(zip(crops, links)):
        kg, ki = int(np.argmax(pg[i])), int(np.argmax(pi[i]))
        res["lesions"].append({
            "lesion": i + 1, "box_xyxy_px": c["box_xyxy_px"], "box_norm": c["box_norm"], "localizer_score": c["localizer_score"],
            "below_validated_threshold": bool(c["localizer_score"] < validated), "icdas_window_px": c["icdas_window_px"],
            **lk,
            "gvb_class": mg["class_names"][kg], "gvb_display": gvb_short(mg["class_names"][kg]),
            "gvb_confidence": float(pg[i][kg]), "gvb_uncertain": bool(pg[i][kg] < mg["confidence_threshold"]),
            "gvb_probabilities": {gvb_short(n): float(v) for n, v in zip(mg["class_names"], pg[i])},
            "icdas": int(mi["scores"][ki]), "icdas_confidence": float(pi[i][ki]),
            "icdas_uncertain": bool(pi[i][ki] < mi["confidence_threshold"]),
            "icdas_unreliable_class": int(mi["scores"][ki]) in mi.get("unreliable_scores", []),
            "icdas_probabilities": {f"ICDAS {s}": float(v) for s, v in zip(mi["scores"], pi[i])},
            "icdas_basis": "model prediction from a photograph — not clinically confirmed"})
    for k in ("stage3", "stage4"):
        res[k] = ({"status": "RAN", "n_lesions": len(crops)} if crops else
                  {"status": "NOT RUN", "reason": "no valid lesion crop (no lesion localized above the score threshold)"})
    res["overall"] = "CARIES DETECTED" if res["lesions"] else "CARIES DETECTED — but no lesion localized above the threshold"
    if d["prediction"] == "UNCERTAIN":
        res["overall"] = "UNCERTAIN (Stage 1) — cascade continued on request: " + res["overall"].lower()
    if keep_crops:
        res["_crops"] = [{"gvb_crop": c["gvb_crop"], "icdas_crop": c["icdas_crop"]} for c in crops]
    return res

def result_lines(res, name=None):
    L = []
    if name: L += ["INPUT:", name, ""]
    s1 = res["stage1"]
    L += ["Overall Result:", res["overall"], "", "Caries Confidence:" if s1["prediction"] == "CARIES" else "Stage 1 Confidence:",
          f"{s1['confidence'] * 100:.1f}%" + (f"  (leaning {s1['leaning']}; {s1['reason']})" if s1["prediction"] == "UNCERTAIN" else "")]
    if not res["lesions"] and res["stage2"]["status"] == "NOT RUN":
        return L
    st2 = res["stage2"]
    L += ["", "FDI (Stage 2):", f"{st2['n_teeth']} teeth detected: " + (", ".join(t["fdi_display"] for t in res["teeth"]) or "none")
          if st2["status"] == "RAN" else f"{st2['status']} — {st2.get('reason')}"]
    if not res["lesions"]:
        L += ["", "No lesion localized — G.V. Black and ICDAS not assigned (try a lower score_threshold)."]
    for les in res["lesions"]:
        g = f"G.V. Black: {les['gvb_display']}" if not les["gvb_uncertain"] else f"G.V. Black: UNCERTAIN (best guess {les['gvb_display']})"
        ic = f"ICDAS: {les['icdas']}" if not les["icdas_uncertain"] else f"ICDAS: UNCERTAIN (best guess {les['icdas']})"
        L += ["", f"Lesion {les['lesion']}:  tooth {les['fdi_display']}", g, f"Confidence: {les['gvb_confidence'] * 100:.1f}%", "",
              ic, f"Confidence: {les['icdas_confidence'] * 100:.1f}%"
              + ("  [class rarely seen in training]" if les["icdas_unreliable_class"] else "")]
    return L

TOOTH_COLOR, LESION_COLOR = (255, 140, 0), (0, 255, 0)
def _put_label(img, text, x, y, color, fs, ft, above=True):
    font = cv2.FONT_HERSHEY_SIMPLEX; H, W = img.shape[:2]
    (tw, th), _ = cv2.getTextSize(text, font, fs, ft)
    x = int(max(0, min(x, W - tw - 6)))                     # keep the label inside the image
    ty = y - 6 if above and y - th - 10 > 0 else min(y + th + 6, H - 2)
    cv2.rectangle(img, (x, ty - th - 4), (min(x + tw + 6, W), ty + 4), color, -1)
    cv2.putText(img, text, (x + 3, ty), font, fs, (0, 0, 0), ft, cv2.LINE_AA)

def draw_result(rgb, res, color=LESION_COLOR, show_teeth=True):
    """Tooth boxes (orange, FDI number or '?') and lesion boxes (green, lesion id -> tooth | G.V. Black | ICDAS)."""
    out = rgb.copy(); H, W = out.shape[:2]; s = max(H, W) / 1000.0
    th, fs, ft = max(2, round(3 * s)), max(0.5, 0.8 * s), max(1, round(2 * s))
    if show_teeth:
        for t in res.get("teeth", []):
            x1, y1, x2, y2 = t["box_xyxy_px"]; cv2.rectangle(out, (x1, y1), (x2, y2), TOOTH_COLOR, max(1, th // 2))
            _put_label(out, t["fdi"] if not t["fdi_uncertain"] else "?", x1, y1 + 2 * th + int(22 * s), TOOTH_COLOR,
                       0.8 * fs, max(1, ft - 1), above=True)
    for les in res["lesions"]:
        x1, y1, x2, y2 = les["box_xyxy_px"]; cv2.rectangle(out, (x1, y1), (x2, y2), color, th)
        tooth = les["fdi_tooth"] or "?"
        _put_label(out, f"L{les['lesion']} T{tooth}: {les['gvb_display']} | ICDAS {les['icdas']}", x1, y1 if y1 > 0 else y2, color, fs, ft,
                   above=y1 > 0)
    return out

def predict_dental_condition(image_path, score_threshold=None, show=True, save_dir=PREDICTION_DIR, verbose=True,
                             continue_if_uncertain=False):
    """MAIN ENTRY POINT: full dental image -> CNN 1 -> Stage 2 FDI -> localization -> per lesion: tooth + CNN 2 + CNN 3."""
    rgb = load_rgb(image_path); res = run_pipeline(rgb, score_threshold, continue_if_uncertain=continue_if_uncertain)
    res["input"] = os.path.basename(str(image_path))
    if verbose:
        print("\n".join(result_lines(res, res["input"]))); print("\n" + DISCLAIMER)
    if show:
        plt.figure(figsize=(12, 12 * rgb.shape[0] / rgb.shape[1])); plt.imshow(draw_result(rgb, res)); plt.axis("off")
        plt.title(res["overall"]); plt.show()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True); b = os.path.join(save_dir, stem(image_path))
        cv2.imwrite(b + "_result.jpg", cv2.cvtColor(draw_result(rgb, res), cv2.COLOR_RGB2BGR))
        write_json(b + "_result.json", {k: v for k, v in res.items() if not k.startswith("_")})
    return res

print("Ready: predict_dental_condition(image_path)  [run_pipeline(rgb) returns the raw result dict]")

## 77. End-to-end example
**77.1** Held-out **test** photographs (never used for training, model selection or threshold tuning) through the full cascade.
**77.2** End-to-end evaluation on all annotated test photographs: Stage 1 gate outcome → lesion-level localization (IoU ≥ 0.3) →
G.V. Black accuracy on matched lesions → ICDAS on matched lesions that are linked to a manifest tooth score; and how often the full pipeline
wrongly reports a lesion on **healthy** test images.

In [ ]:
# ============================================================
# SECTION 77.1 — END-TO-END EXAMPLE (held-out test photographs)
# ============================================================
EXAMPLE_PHOTOS = [r for r in gvb_test if len(r["boxes"])][:2] if "gvb_test" in globals() else []
if "gvb_test" in globals(): print("Examples are TEST photographs (unseen by every model).\n")
for r in EXAMPLE_PHOTOS:
    print("#" * 70 + f"\nTRUE: G.V. Black {[gvb_short(GVB_CLASS_NAMES[c]) for c in r['classes']]}\n")
    predict_dental_condition(r["image_path"])
if "s1_test_df" in globals() and (s1_test_df.label == 0).any():
    hp_ = s1_test_df[s1_test_df.label == 0]["src"].iloc[0]
    print("#" * 70 + "\nTRUE: Healthy\n"); predict_dental_condition(hp_)
if not EXAMPLE_PHOTOS:
    print("Datasets are not loaded in this session. Run the cascade on any image with:\n"
          "    predict_dental_condition('/content/your_photo.jpg')")

In [ ]:
# ============================================================
# SECTION 77.2 — END-TO-END EVALUATION (test split only)
# ============================================================
E2E = {}
if "gvb_test" in globals() and "match_image" in globals():
    link_score = {(r["image_path"], tuple(np.round(b, 6))): s for r, b, s in LESION_LINKS.get("test", [])}
    gate = Counter(); tp = fp = fn = 0; g_t, g_p, i_t, i_p = [], [], [], []
    with stopwatch("end-to-end evaluation"):
        for r in gvb_test:
            rgb = load_rgb(r["image_path"]); H, W = rgb.shape[:2]
            res = run_pipeline(rgb); gate[res["stage1"]["prediction"]] += 1
            pb = np.array([[x1 / W, y1 / H, x2 / W, y2 / H] for x1, y1, x2, y2 in (l["box_xyxy_px"] for l in res["lesions"])],
                          np.float32).reshape(-1, 4)
            ps = np.array([l["localizer_score"] for l in res["lesions"]], np.float32)
            m = match_image(pb, ps, r["boxes"], 0.3); found = {g for _, _, ok, g, _ in m if ok}
            tp += len(found); fp += sum(not ok for _, _, ok, _, _ in m); fn += len(r["boxes"]) - len(found)
            for k_, _, ok, g, _ in m:
                if not ok: continue
                g_t.append(int(r["classes"][g])); g_p.append(GVB_CLASS_NAMES.index(res["lesions"][k_]["gvb_class"]))
                s_ = link_score.get((r["image_path"], tuple(np.round(r["boxes"][g], 6))))
                if s_ is not None:
                    i_t.append(s_); i_p.append(res["lesions"][k_]["icdas"])
    P, R_ = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    E2E = {"n_test_photos": len(gvb_test), "stage1_outcomes": dict(gate), "lesion_precision_iou03": P, "lesion_recall_iou03": R_,
           "tp": tp, "fp": fp, "fn": fn}
    print("=" * 64 + f"\nEND-TO-END on {len(gvb_test)} annotated TEST photographs\n" + "=" * 64)
    print("Stage 1 outcome:", dict(gate))
    print(f"Lesion level (IoU >= 0.3, after the Stage 1 gate): precision {P:.3f} | recall {R_:.3f} | TP {tp} FP {fp} FN {fn}")
    if g_t:
        E2E.update(gvb_n=len(g_t), gvb_acc=float(accuracy_score(g_t, g_p)),
                   gvb_balanced_acc=evaluate_multiclass(g_t, g_p, None, GVB_CLASS_NAMES)["balanced_accuracy"],
                   gvb_macro_f1=float(f1_score(g_t, g_p, labels=sorted(set(g_t)), average="macro", zero_division=0)))
        print(f"G.V. Black on matched lesions: accuracy {E2E['gvb_acc']:.3f} | balanced {E2E['gvb_balanced_acc']:.3f} | "
              f"macro-F1 {E2E['gvb_macro_f1']:.3f} (n={len(g_t)})")
    else:
        print("No matched lesions — end-to-end G.V. Black accuracy cannot be computed.")
    if i_t:
        it_, ip_ = np.array(i_t), np.array(i_p)
        E2E.update(icdas_n=len(i_t), icdas_acc=float((it_ == ip_).mean()), icdas_mae=float(np.abs(it_ - ip_).mean()),
                   icdas_within1=float((np.abs(it_ - ip_) <= 1).mean()))
        print(f"ICDAS on matched + tooth-linked lesions: accuracy {E2E['icdas_acc']:.3f} | MAE {E2E['icdas_mae']:.2f} | "
              f"within ±1 {E2E['icdas_within1']:.1%} (n={len(i_t)})")
    else:
        print("No matched lesion with a manifest ICDAS score — end-to-end ICDAS accuracy cannot be computed.")
    healthy_test = s1_test_df[s1_test_df["label"] == 0]["src"].tolist()[:300]
    fa = Counter()
    for pth in healthy_test:
        res = run_pipeline(load_rgb(pth))
        fa["lesion reported (false alarm)" if res["lesions"] else res["stage1"]["prediction"]] += 1
    E2E["healthy_test_outcomes"] = dict(fa)
    print(f"Healthy TEST images through the full pipeline (n={len(healthy_test)}):", dict(fa))
    update_metadata({"end_to_end": E2E})
else:
    print("Datasets not loaded in this session — end-to-end evaluation skipped (prediction still works).")

<br>

---
# PART 15 — MODEL SAVING

## 78. Model saving
Each model was saved as `.keras` right after training; each evaluation cell wrote the model's own metadata JSON (class names, class
mapping, image size, preprocessing, confidence threshold, architecture, training configuration, history, test metrics). This cell checks
the files, writes the run-level metadata, and **reloads every model from disk to confirm it gives the same output** as the model in memory.

| File | Content |
|---|---|
| `stage1_caries_model.keras` + `_metadata.json` | CNN 1 |
| `stage3_gvblack_model.keras` + `_metadata.json` | CNN 2 |
| `stage4_icdas_model.keras` + `_metadata.json` | CNN 3 |
| `cavity_localizer_model.keras` + `_metadata.json` | localizer |
| `pipeline_metadata.json` | everything above + run info + end-to-end results |

In [ ]:
# ============================================================
# SECTION 78 — MODEL SAVING (verify + run metadata + reload check)
# ============================================================
for k in MODEL_KEYS:
    assert os.path.exists(MODEL_PATHS[k]) and os.path.exists(MODEL_META_PATHS[k]), f"missing saved files for {k}"
    print(f"  {k:9} {os.path.basename(MODEL_PATHS[k]):34} {os.path.getsize(MODEL_PATHS[k]) / 1e6:6.1f} MB  + {os.path.basename(MODEL_META_PATHS[k])}")
DATA_IN_MEMORY = all(v in globals() for v in ("s1_test_df", "X_loc_test", "Xg_test", "X_ic", "ic_idx", "pool", "index"))
if DATA_IN_MEMORY:
    run_info = {"run": {
        "created": time.strftime("%Y-%m-%d %H:%M:%S"), "tensorflow": tf.__version__, "keras": keras.__version__, "seed": SEED,
        "status": "research proof of concept — not clinically validated",
        "architecture": "multi-CNN cascade: CNN1 (stage 1) -> Stage 2 FDI tooth detector -> localizer -> lesion->tooth association "
                        "-> CNN2 (G.V. Black) + CNN3 (ICDAS) per lesion",
        "stage2_fdi": ("trained" if os.path.exists(MODEL_PATHS["fdi"]) else "UNAVAILABLE: " + PIPELINE["unavailable"].get("fdi", "not trained")),
        "icdas_dataset": os.path.basename(ICDAS_ZIP), "refused_datasets": list(REFUSED_ICDAS_ZIPS),
        "model_paths": MODEL_PATHS, "model_metadata_paths": MODEL_META_PATHS, "split_fracs": SPLIT_FRACS,
        "leakage_control": {"dup_hamming": DUP_HAMMING, "s1_group_hamming": S1_GROUP_HAMMING, "pool_group_hamming": POOL_GROUP_HAMMING,
                            "icdas_group_hamming": ICDAS["GROUP_HAMMING"], "split": "one stratified GROUP split shared by all models"},
        "gpu": bool(GPU_AVAILABLE), "mixed_precision": bool(MIXED_PRECISION), "split_fingerprints": SPLIT_FP,
        "dataset_audit": {"healthy_images": len(healthy_paths), "caries_images": len(caries_paths),
                          "stage1_unreadable": int((~index["ok"]).sum()), "stage1_cross_class_duplicates_dropped": int(n_cross_imgs),
                          "no_caries_listed_rows": len(NOCARIES_ROWS), "gvb_photographs": len(pool),
                          "gvb_boxes_per_class": {gvb_short(GVB_CLASS_NAMES[c]): int(box_counts.get(c, 0)) for c in range(NUM_GVB)},
                          "icdas_files_per_class": ICDAS_ONDISK_COUNTS, "icdas_manifest_rows": int(len(ICDAS_MANIFEST)),
                          "icdas_manifest_rows_missing_from_zip": int((~ICDAS_MANIFEST["on_disk"]).sum()) if len(ICDAS_MANIFEST) else 0}}}
    update_metadata(run_info)
    chk = {"stage1": s1_prepare(load_rgb(s1_test_df["src"].iloc[0]), S1_PX), "localizer": X_loc_test[0],
           "gvb": Xg_test[0], "icdas": np.asarray(X_ic[ic_idx["test"][0]])}
    print("\nReload check (saved file vs model in memory):")
    for k, x in chk.items():
        reloaded = load_keras(MODEL_PATHS[k])
        a = np.asarray(PIPELINE["models"][k](x[None].astype(np.float32), training=False), np.float32)
        b = np.asarray(reloaded(x[None].astype(np.float32), training=False), np.float32)
        print(f"  {k:9} max |Δ| after reload = {np.abs(a - b).max():.2e}")
        assert np.abs(a - b).max() < 1e-3, f"{k}: reloaded model differs"
else:
    print("Datasets not in memory (prediction-only session): files verified, run metadata and reload check skipped.")
print(f"  metadata  {METADATA_PATH}")
if os.path.exists(MODEL_PATHS["fdi"]):
    print(f"  fdi       {os.path.basename(MODEL_PATHS['fdi']):34} {os.path.getsize(MODEL_PATHS['fdi']) / 1e6:6.1f} MB  (Stage 2, optional)")
else:
    print("  fdi       Stage 2 FDI detector not saved —", PIPELINE["unavailable"].get("fdi", "not trained (Section 24)"))
print("\nTo download:  from google.colab import files; [files.download(p) for p in "
      "list(MODEL_PATHS.values()) + list(MODEL_META_PATHS.values()) + [METADATA_PATH] if os.path.exists(p)]")

<br>

---
# PART 16 — MODEL LOADING

## 79. Model loading
Proof that the pipeline works from the saved files alone: the registry is emptied, all models are loaded with `load_all_models()`
(`load_stage1_model()`, `load_localizer()`, `load_gvblack_model()`, `load_icdas_model()` load them one by one), and the same test photo gives
the same result.

In [ ]:
# ============================================================
# SECTION 79 — MODEL LOADING (from disk, no retraining)
# ============================================================
PROBE_IMAGE = None                      # set to any image path to test with your own photo
if PROBE_IMAGE is None and EXAMPLE_PHOTOS:
    PROBE_IMAGE = EXAMPLE_PHOTOS[0]["image_path"]
elif PROBE_IMAGE is None and "s1_test_df" in globals():
    PROBE_IMAGE = s1_test_df["src"].iloc[0]
before = run_pipeline(load_rgb(PROBE_IMAGE)) if PROBE_IMAGE else None
PIPELINE["models"].clear(); PIPELINE["meta"].clear()
load_all_models()                       # = load_stage1_model() + load_localizer() + load_gvblack_model() + load_icdas_model()
if PROBE_IMAGE:
    after = run_pipeline(load_rgb(PROBE_IMAGE))
    same = before["stage1"]["prediction"] == after["stage1"]["prediction"] and \
           [(l["gvb_class"], l["icdas"]) for l in before["lesions"]] == [(l["gvb_class"], l["icdas"]) for l in after["lesions"]]
    print(f"Same result after reloading from disk ({os.path.basename(PROBE_IMAGE)}): {same}")
    assert same
else:
    print("Models loaded from disk. Set PROBE_IMAGE to compare a prediction before/after reloading.")

<br>

---
# PART 17 — LIMITATIONS (+ final summary)

## 80. Final limitations (read before reporting results)
* **Proof of concept, not a medical device.** No output of this notebook is a diagnosis.
* **Stage 1 source confound.** Healthy and caries images come from different archives, so "class" and "source" are the same variable.
  Probes A and B measure how much of the Stage 1 score is reachable without looking at a lesion; report them next to the CNN metrics.
  External validation on photos from another clinic / camera is needed before any claim of generalisation.
* **No patient IDs.** Splits are group-level (source photos, identical copies, near-duplicate / burst photos kept together); this reduces
  but cannot eliminate patient-level leakage.
* **ICDAS has no score 0 in `icdascl.zip`.** CNN 3 only distinguishes ICDAS 1–6; a sound tooth must be caught by CNN 1. CNN 3 will always
  output a score ≥ 1 for whatever crop it is given.
* **`icdascl.zip` is incomplete relative to its own manifest** (ICDAS 5 and 6 crops missing — counts in Section 52); ICDAS 5 in particular
  is under-represented compared to the full export.
* **ICDAS 4** has only a few dozen crops; its test support is a handful of crops, so its metrics are not meaningful. Adjacent-score
  confusions (1 vs 2, 5 vs 6) are clinically expected from photographs (ICDAS 1 vs 2 depends on dried vs wet enamel).
* **Lesion vs tooth crops.** CNN 3 is trained on tooth crops; in the pipeline it receives a tooth-scale window around each lesion box
  (Section 74). Section 74.2 / 75.2 measure this only where lesion ↔ tooth links exist.
* **G.V. Black Class IV and Class VI** are thin; their per-class metrics have wide uncertainty. The class mapping comes from a mapping file
  if present, otherwise from the project owner (checked only for consistency with the photo views).
* **Small validation / test sets** → wide confidence intervals; backbone / padding / window choices made on validation can be noisy.
* **Displayed confidences** are softmax / sigmoid outputs, not calibrated clinical probabilities.
* **End-to-end quality** depends on the localizer: a missed lesion gets no G.V. Black / ICDAS, and loose boxes change the crops.
* **Stage 2 FDI numbering** is only as good as `caries_FDI_dataset_final.zip`: its measured test mAP / number accuracy are printed in
  Section 24.5. Photos show only some teeth, occlusal views are mirror images, and rare FDI classes have few boxes. The lesion → tooth link
  is a geometric overlap rule; its accuracy is measured only if the FDI ZIP also contains caries boxes (Section 24.6). Without the ZIP,
  Stage 2 is reported as UNAVAILABLE and no tooth number is shown.

The cell below prints the **data-driven** limitations of this particular run.

In [ ]:
# ============================================================
# SECTION 80 — DATA-DRIVEN LIMITATIONS OF THIS RUN (read from the saved metadata)
# ============================================================
M = {k: PIPELINE["meta"].get(k) or read_json(MODEL_META_PATHS[k]) for k in MODEL_KEYS}
t1 = M["stage1"]["test_metrics"]
pa, pb_ = M["stage1"].get("probe_metadata_acc"), M["stage1"].get("probe_colour_hist_acc")
print(f"Stage 1: CNN test accuracy {t1['accuracy']:.4f} vs probe A {pa:.4f} / probe B {pb_:.4f}"
      + ("  <-- source shortcut likely" if max(pa, pb_) > t1["accuracy"] - 0.05 else ""))
print(f"G.V. Black classes with < {GVB['MIN_RELIABLE_TEST']} test crops:", M["gvb"].get("thin_classes") or "none")
print(f"ICDAS classes with < {ICDAS['MIN_RELIABLE_TRAIN']} training crops:", [f"ICDAS {s}" for s in M["icdas"].get("unreliable_scores", [])] or "none")
print("ICDAS scores absent from icdascl.zip:", M["icdas"].get("absent_scores") or "none")
aud = read_metadata().get("run", {}).get("dataset_audit", {})
if aud: print("Manifest crops missing from icdascl.zip:", aud.get("icdas_manifest_rows_missing_from_zip"))
print("Lesion<->tooth links (val / test):", M["icdas"].get("lesion_links"), "| window:", (M["icdas"].get("inference_window") or {}).get("source"))
print("Test support per ICDAS class:", {k: v["support"] for k, v in M["icdas"]["test_metrics"]["per_class"].items()})
print("Test support per G.V. Black class:", {k: v["support"] for k, v in M["gvb"]["test_metrics_gt_crops"]["per_class"].items()})
fm = PIPELINE["meta"].get("fdi") or read_json(MODEL_META_PATHS["fdi"])
if fm.get("test_metrics"):
    print(f"Stage 2 FDI: {len(fm['class_names'])} classes | test mAP@0.5 {fm['test_metrics']['mAP50']:.3f} | classes with few training "
          f"boxes: {fm.get('unreliable_classes') or 'none'} | association test: {fm.get('association_test') or 'not measurable on this data'}")
else:
    print("Stage 2 FDI: UNAVAILABLE —", PIPELINE["unavailable"].get("fdi", "no saved FDI detector"))

## 81. Final summary
Every number below is read from the evaluations run above (nothing is typed in by hand).

In [ ]:
# ============================================================
# SECTION 81 — FINAL SUMMARY (computed values only, read from the saved metadata)
# ============================================================
M = {k: PIPELINE["meta"].get(k) or read_json(MODEL_META_PATHS[k]) for k in MODEL_KEYS}
f4 = lambda v: "—" if v is None else f"{v:.4f}"
t1, tl, tg, ti, to = (M["stage1"]["test_metrics"], M["localizer"]["test_metrics"], M["gvb"]["test_metrics_gt_crops"],
                      M["icdas"]["test_metrics"], M["icdas"]["test_ordinal"])
summary = pd.DataFrame([
    {"model": "CNN 1  Healthy vs Caries", "backbone": M["stage1"]["backbone"], "test n": t1["n_test"], "accuracy": f4(t1["accuracy"]),
     "balanced acc.": f4((t1["recall_sensitivity"] + t1["specificity"]) / 2), "macro F1": "—", "weighted F1": "—",
     "other": f"sens {t1['recall_sensitivity']:.3f} | spec {t1['specificity']:.3f} | AUC {t1['roc_auc']:.3f}"},
    {"model": "Localizer (where)", "backbone": "MobileNetV2 C4+C5", "test n": tl.get("n_test_lesions"), "accuracy": "—",
     "balanced acc.": "—", "macro F1": "—", "weighted F1": "—",
     "other": f"AP@0.5 {tl['AP50']:.3f} | AP@0.3 {tl['AP30']:.3f} | R@0.3 {tl['iou0.3']['recall']:.3f}"},
    {"model": "CNN 2  G.V. Black", "backbone": M["gvb"]["backbone"], "test n": tg["n"], "accuracy": f4(tg["accuracy"]),
     "balanced acc.": f4(tg["balanced_accuracy"]), "macro F1": f4(tg["macro_f1"]), "weighted F1": f4(tg["weighted_f1"]),
     "other": f"macro P {tg['macro_precision']:.3f} | macro R {tg['macro_recall']:.3f}"},
    {"model": "CNN 3  ICDAS", "backbone": M["icdas"]["backbone"], "test n": ti["n"], "accuracy": f4(ti["accuracy"]),
     "balanced acc.": f4(ti["balanced_accuracy"]), "macro F1": f4(ti["macro_f1"]), "weighted F1": f4(ti["weighted_f1"]),
     "other": f"MAE {to['mae']:.2f} | within ±1 {to['within_1'] * 100:.1f}% | QWK {to['quadratic_weighted_kappa']:.3f}"},
])
fm = PIPELINE["meta"].get("fdi") or read_json(MODEL_META_PATHS["fdi"])
if fm.get("test_metrics"):
    tf_ = fm["test_metrics"]
    summary = pd.concat([summary.iloc[:1], pd.DataFrame([{
        "model": "Stage 2 FDI tooth detector", "backbone": "MobileNetV2 C4+C5", "test n": tf_["n_test_teeth"], "accuracy": "—",
        "balanced acc.": "—", "macro F1": "—", "weighted F1": "—",
        "other": f"mAP@0.5 {tf_['mAP50']:.3f} | mAP@.5:.95 {tf_['mAP50_95']:.3f} | number acc. {tf_['at_threshold_number_accuracy_on_matched']:.3f}"}]),
        summary.iloc[1:]], ignore_index=True)
print(summary.to_string(index=False))
e2e = read_metadata().get("end_to_end")
if e2e:
    print("\nEnd-to-end (test photographs):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in e2e.items()})
update_metadata({"summary": summary.to_dict("records")})
print("\nPipeline: CNN 1 → Stage 2 FDI (" + ("trained" if fm.get("test_metrics") else "UNAVAILABLE") +
      ") → localization → lesion→tooth link → CNN 2 (G.V. Black) + CNN 3 (ICDAS) per lesion.")
print("Models:", ", ".join(os.path.basename(p) for p in MODEL_PATHS.values() if os.path.exists(p)), "->", OUT_DIR)
print(DISCLAIMER)

## 82. Complete assessment — patient information → questionnaire → image upload → Stage 1 → Stage 2 → localization → Stage 3 → Stage 4 → report
Run **82** (definitions) and then **82.1**. Steps:
1. **Patient information** — optional name or ID, plus age.
2. **Oral-habits questionnaire** — 7 questions; type the option number or the answer text. It gives a habit-based risk score
   (LOW ≤ 3, MODERATE ≤ 7, HIGH, out of 13) and personal advice. **This score is separate from the image analysis. It never changes a
   CNN prediction and it does not diagnose caries.**
3. **Image upload** — JPG, JPEG or PNG. The file is checked: real image format, full decode, EXIF orientation corrected, converted to
   RGB. It is saved under `UPLOAD_DIR` (`/content/uploads`) with a unique name and shown before analysis. A cancelled upload, an invalid
   or corrupted file, or an unsupported format is reported, and the cascade is not run on it.
4. **Stage 1 → Stage 2 → localization → lesion → tooth link → Stage 3 → Stage 4**, using `run_pipeline` (Section 76).
5. **Final report** — sections 1–10, a per-lesion table, the annotated image (teeth in orange, lesions in green), and saved files
   (`*_result.jpg`, `*_result.json`, `*_report.txt`, `*_lesions.csv` in `/content/predictions`).

Scripted use without `input()`:
`predict_complete_dental_image(image_path="/content/photo.jpg", patient={"age": 34}, answers={"Q1": 1, "Q2": 2, ...})`.
`score_threshold=None` uses the localizer's validated threshold. A lower value, e.g. `0.2`, adds low-certainty regions, flagged as such.

In [ ]:
# ============================================================
# SECTION 82 — PATIENT INFO + ORAL-HABITS QUESTIONNAIRE -> IMAGE UPLOAD -> STAGE 1 -> STAGE 2 (FDI) -> LESION LOCALIZATION
#              -> STAGE 3 (G.V. Black) -> STAGE 4 (ICDAS) -> FINAL DENTAL REPORT
# ============================================================
import textwrap, uuid
from PIL import UnidentifiedImageError
try:
    from IPython.display import display
except ImportError:                       # plain Python: tables are printed instead
    display = None

LINE = "=" * 70
UPLOAD_DIR = UPLOAD_DIR if "UPLOAD_DIR" in globals() else "/content/uploads"   # set in Section 3 (re-stated for stand-alone use)
os.makedirs(UPLOAD_DIR, exist_ok=True)
ALLOWED_UPLOAD_EXT = (".jpg", ".jpeg", ".png")
ALLOWED_PIL_FORMATS = {"JPEG", "MPO", "PNG"}          # MPO = multi-picture JPEG written by many phone cameras
MIN_UPLOAD_PX = 64

# ---------- oral-habits questionnaire (questions + scoring unchanged) ----------
RISK_QUESTIONS = [
    {"key": "brush_twice", "text": "Do you brush your teeth twice a day?", "options": {"yes": 0, "no": 2},
     "advice": {"no": "Brush twice a day (morning and before bed) for 2 minutes."}},
    {"key": "junk_sugary", "text": "How often do you eat junk food or sugary foods / drinks?",
     "options": {"rarely": 0, "once a day": 1, "several times a day": 3},
     "advice": {"once a day": "Reduce junk and sugary food; prefer it with meals.",
                "several times a day": "Cut down junk / sugary food and drinks — frequent sugar intake is a major caries risk."}},
    {"key": "snack_night", "text": "Do you eat or drink something sugary between meals or after brushing at night?",
     "options": {"no": 0, "sometimes": 1, "often": 2},
     "advice": {"sometimes": "Avoid sugary snacks between meals and after night brushing.",
                "often": "Avoid sugary snacks between meals and after night brushing."}},
    {"key": "fluoride", "text": "Do you use a fluoride toothpaste?", "options": {"yes": 0, "not sure": 1, "no": 1},
     "advice": {"not sure": "Use a fluoride toothpaste.", "no": "Use a fluoride toothpaste."}},
    {"key": "dental_visit", "text": "When did you last visit a dentist?",
     "options": {"within 6 months": 0, "6-12 months ago": 1, "more than 1 year / never": 2},
     "advice": {"6-12 months ago": "Plan a routine dental check-up.",
                "more than 1 year / never": "Visit a dentist for a check-up (recommended every 6–12 months)."}},
    {"key": "pain", "text": "Do you have tooth pain, or sensitivity to sweet / hot / cold?", "options": {"no": 0, "yes": 2},
     "advice": {"yes": "Pain or sensitivity should be checked by a dentist soon."}},
    {"key": "tobacco", "text": "Do you smoke or chew tobacco?", "options": {"no": 0, "yes": 1},
     "advice": {"yes": "Stopping tobacco use improves oral health."}},
]
RISK_MAX = sum(max(q["options"].values()) for q in RISK_QUESTIONS)
RISK_LEVELS = [(3, "LOW"), (7, "MODERATE"), (RISK_MAX, "HIGH")]
assert RISK_MAX == 13

def _center(t): return t.center(70).rstrip()

def _match_option(q, reply):
    """Option number ('1') or answer text ('yes'), case-insensitive; a unique prefix ('sev') is accepted too."""
    opts = list(q["options"]); r = re.sub(r"\s+", " ", str(reply).strip().lower())
    if r.isdigit() and 1 <= int(r) <= len(opts): return opts[int(r) - 1]
    if r in opts: return r
    starts = [o for o in opts if o.startswith(r)] if r else []
    return starts[0] if len(starts) == 1 else None

def _print_question(n, q):
    lines = textwrap.wrap(q["text"], 62)
    print(f"\nQ{n}. " + lines[0])
    for l in lines[1:]: print("    " + l)
    opts = [f"[{k}] {o}" for k, o in enumerate(q["options"], 1)]
    if len("   ".join(opts)) <= 60:
        print("    " + "   ".join(opts))
    else:
        for o in opts: print("    " + o)

def ask_age(age=None):
    print("Q0. What is your age?")
    def _parse(v):
        m = re.fullmatch(r"\s*(\d{1,3})\s*(years?|yrs?|y)?\s*", str(v), re.I)
        return int(m.group(1)) if m and 1 <= int(m.group(1)) <= 120 else None
    if age is not None:
        a = _parse(age)
        if a is None: raise ValueError(f"Invalid age {age!r}: enter a whole number between 1 and 120.")
        print(f"    Enter your age: {age}"); return a
    while True:
        a = _parse(input("    Enter your age: "))
        if a is not None: return a
        print("    Please enter your age as a whole number between 1 and 120.")

def ask_patient_info(patient=None, age=None):
    """Optional name / ID + age (required, 1–120). patient={'patient_id': ..., 'age': ...} answers it without input()."""
    print("\nPATIENT INFORMATION\n")
    if patient is not None or age is not None:
        patient = dict(patient or {})
        pid = str(patient.get("patient_id") or patient.get("name") or "").strip() or None
        a = patient.get("age", age)
        if a is None: raise ValueError("patient age is required")
        print(f"Patient name / ID (optional): {pid or '-'}")
        return {"patient_id": pid, "age": ask_age(a)}
    pid = input("Patient name / ID (optional — press Enter to skip): ").strip() or None
    return {"patient_id": pid, "age": ask_age()}

def ask_oral_habits(answers=None):
    """Interactive (input()) or scripted (answers={key or 'Q1': reply}). Invalid interactive answers repeat the question."""
    chosen = {}
    for n, q in enumerate(RISK_QUESTIONS, 1):
        _print_question(n, q)
        if answers is not None:
            reply = answers.get(q["key"], answers.get(f"Q{n}"))
            a = _match_option(q, reply) if reply is not None else None
            if a is None:
                raise ValueError(f"Q{n}: invalid or missing answer {reply!r}; options: {list(q['options'])}")
            print(f"    Your answer: {reply}")
        else:
            while True:
                a = _match_option(q, input("    Your answer: "))
                if a is not None: break
                print(f"    Please type an option number (1–{len(q['options'])}) or the answer text.")
        chosen[q["key"]] = a
    score = int(sum(q["options"][chosen[q["key"]]] for q in RISK_QUESTIONS))
    level = next(name for limit, name in RISK_LEVELS if score <= limit)
    advice = list(dict.fromkeys(q["advice"][chosen[q["key"]]] for q in RISK_QUESTIONS if chosen[q["key"]] in q.get("advice", {})))
    points = {q["key"]: q["options"][chosen[q["key"]]] for q in RISK_QUESTIONS}
    return {"answers": chosen, "points": points, "score": score, "max": RISK_MAX, "level": level, "advice": advice}

def print_risk_result(risk):
    print("\n" + LINE + "\n" + _center("ORAL-HABITS RISK RESULT") + "\n" + LINE + "\n")
    print("Habit-based caries risk assessment (questionnaire only — not a diagnosis)")
    print(f"Habit-based caries risk: {risk['level']}")
    print(f"Risk score: {risk['score']}/{risk['max']}")
    print("Points per question:", ", ".join(f"Q{i} {p}" for i, p in enumerate(risk["points"].values(), 1)))
    print("Note: this score comes from your answers only. The image analysis below is separate and image-based;\n"
          "      the questionnaire never changes a CNN prediction.")

# ---------- image upload + validation ----------
def _safe_upload_name(name):
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", os.path.basename(str(name))).strip("._") or "image"
    return f"{time.strftime('%Y%m%d-%H%M%S')}_{uuid.uuid4().hex[:6]}_{base}"     # unique: no collisions between uploads

def upload_dental_images():
    """Colab file picker -> each file saved under UPLOAD_DIR with a unique name. Returns the saved paths ([] if cancelled).
    Outside Colab: asks for a file path instead."""
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    if not IN_COLAB:
        p = input("\nPath of the intraoral image to analyse (Enter = skip): ").strip()
        return [p] if p else []
    from google.colab import files
    print(f"\nSelect an intraoral photograph (JPG, JPEG or PNG). It will be saved in {UPLOAD_DIR}")
    try:
        up = files.upload()
    except Exception as e:                # the Colab widget raises when the upload is cancelled or interrupted
        print(f"Upload did not complete ({type(e).__name__}: {e}).")
        return []
    if not up:
        print("No file was uploaded (upload cancelled).")
        return []
    paths = []
    for name, data in up.items():
        p = os.path.join(UPLOAD_DIR, _safe_upload_name(name))
        with open(p, "wb") as f:
            f.write(data)
        print(f"  saved '{name}' -> {p}  ({len(data) / 1024:.0f} KB)")
        paths.append(p)
    return paths

def validate_input_image(path):
    """-> (EXIF-corrected RGB uint8, None) or (None, reason). Checks existence, extension, real file format, full decode, size."""
    if not path or not os.path.isfile(path):
        return None, f"file not found: {path}"
    ext = os.path.splitext(path)[1].lower()
    if ext not in ALLOWED_UPLOAD_EXT:
        return None, f"unsupported file type '{ext or '(none)'}' — please upload a JPG, JPEG or PNG image"
    if os.path.getsize(path) == 0:
        return None, "the file is empty"
    try:
        with Image.open(path) as im:
            fmt = im.format
            im.verify()
    except (UnidentifiedImageError, OSError, SyntaxError, ValueError, Image.DecompressionBombError) as e:
        return None, f"not a readable image ({type(e).__name__}: {e})"
    if fmt not in ALLOWED_PIL_FORMATS:
        return None, f"the file content is {fmt}, not JPEG / PNG"
    try:
        rgb = load_rgb(path)                       # EXIF orientation applied, converted to RGB (same loader as training)
    except (OSError, SyntaxError, ValueError) as e:
        return None, f"the image is corrupted and could not be decoded ({type(e).__name__}: {e})"
    if min(rgb.shape[:2]) < MIN_UPLOAD_PX:
        return None, f"the image is too small ({rgb.shape[1]}x{rgb.shape[0]} px; at least {MIN_UPLOAD_PX} px per side)"
    return rgb, None

def show_uploaded_image(rgb, name, show=True):
    print(f"Image uploaded successfully: {name}  ({rgb.shape[1]}x{rgb.shape[0]} px, EXIF orientation corrected, RGB)")
    if show:
        plt.figure(figsize=(10, 10 * rgb.shape[0] / rgb.shape[1])); plt.imshow(rgb); plt.axis("off")
        plt.title(f"Uploaded image — {name}", fontsize=10); plt.show()

# ---------- report ----------
_pct = lambda v: "Unavailable" if v is None else f"{v * 100:.1f}%"

def lesion_table(res):
    """One row per localized lesion; every value is a model output (or Unavailable / Unknown / Uncertain)."""
    rows = []
    for l in res.get("lesions", []):
        rows.append({
            "Lesion": l["lesion"],
            "FDI tooth number": l["fdi_display"],
            "Tooth-number confidence": _pct(l.get("fdi_confidence")),
            "Localization confidence": _pct(l["localizer_score"]) + (" (low-certainty region)" if l["below_validated_threshold"] else ""),
            "G.V. Black class": l["gvb_display"] if not l["gvb_uncertain"] else f"Uncertain (best guess {l['gvb_display']})",
            "G.V. Black confidence": _pct(l["gvb_confidence"]),
            "ICDAS score (model-predicted)": (f"ICDAS {l['icdas']}" if not l["icdas_uncertain"] else f"Uncertain (best guess ICDAS {l['icdas']})")
                                             + (" [rare class]" if l["icdas_unreliable_class"] else ""),
            "ICDAS confidence": _pct(l["icdas_confidence"])})
    return pd.DataFrame(rows, columns=["Lesion", "FDI tooth number", "Tooth-number confidence", "Localization confidence",
                                       "G.V. Black class", "G.V. Black confidence", "ICDAS score (model-predicted)", "ICDAS confidence"])

def teeth_table(res):
    return pd.DataFrame([{"Tooth": t["tooth_id"], "FDI number": t["fdi_display"], "Number confidence": _pct(t["fdi_confidence"]),
                          "Detection score": _pct(t["detection_score"]), "Note": t.get("fdi_note") or ""} for t in res.get("teeth", [])])

def lesion_label(les):
    g = les["gvb_display"] if not les["gvb_uncertain"] else f"uncertain ({les['gvb_display']}?)"
    i = f"ICDAS {les['icdas']}" if not les["icdas_uncertain"] else f"ICDAS uncertain ({les['icdas']}?)"
    return (f"Lesion {les['lesion']}: tooth {les['fdi_display']} | G.V. Black {g} ({les['gvb_confidence'] * 100:.0f}%) | "
            f"{i} ({les['icdas_confidence'] * 100:.0f}%)" + ("  [low-certainty region]" if les["below_validated_threshold"] else ""))

def general_advice(res, risk):
    adv = list(risk["advice"]) if risk else []
    s1 = (res.get("stage1") or {}).get("prediction")
    if s1 in ("CARIES", "UNCERTAIN") or res.get("overall") == "INVALID INPUT":
        adv.insert(0, "Book a dental examination to confirm or rule out the image findings.")
    adv += ["Brush twice daily with a fluoride toothpaste and limit sugary snacks and drinks.",
            "Regular dental check-ups (every 6–12 months) catch caries early."]
    return list(dict.fromkeys(adv))

STAGE_TITLES = {"stage2": "Stage 2 FDI tooth numbering", "localization": "Lesion localization",
                "stage3": "Stage 3 G.V. Black", "stage4": "Stage 4 ICDAS"}

def report_limitations(res):
    L = ["RESEARCH PROTOTYPE — not a substitute for professional dental diagnosis."]
    off = [f"{STAGE_TITLES[k]}: {res[k]['status']} ({res[k].get('reason', '')})" for k in STAGE_TITLES
           if k in res and res[k].get("status") != "RAN"]
    if off: L += ["Stages not run / unavailable for this image: " + "; ".join(off)]
    fm = PIPELINE["meta"].get("fdi") or read_json(MODEL_META_PATHS["fdi"])
    if fm.get("test_metrics"):
        t = fm["test_metrics"]
        L += [f"Stage 2 detector (measured on its held-out test images): mAP@0.5 {t['mAP50']:.3f}, FDI-number accuracy on matched "
              f"teeth {t['at_threshold_number_accuracy_on_matched']:.3f}." + (f" Lesion->tooth association accuracy "
              f"{fm['association_test']['accuracy']:.3f} (n={fm['association_test']['n_lesions']})." if fm.get("association_test") else
              " Lesion->tooth association accuracy could not be measured (no caries boxes in the FDI test data).")]
    L += ["Stage 1 healthy and caries images come from different archives (source confound); no external validation yet.",
          "FDI numbers come from one photograph; mirror (occlusal) views, missing or partially visible teeth reduce reliability. "
          "'Unknown / uncertain' is shown when the number is not confident or the lesion lies across two teeth.",
          "ICDAS scores are MODEL PREDICTIONS from a photograph, not clinically confirmed scores: ICDAS requires a cleaned, air-dried "
          "tooth, visual-tactile examination and often radiographs. ICDAS 0 (sound) is not modelled — Stage 4 outputs 1–6 only.",
          "G.V. Black classes are predicted from the lesion's appearance in a photograph; proximal lesions are often not visible.",
          "Confidences are softmax / sigmoid outputs, not calibrated clinical probabilities. Missed lesions receive no classification.",
          "Please consult a dentist for diagnosis and treatment."]
    return L

def final_report_lines(res, patient, risk):
    L = [LINE, _center("FINAL DENTAL REPORT"), _center("research prototype — not a clinical diagnosis"), LINE, ""]
    L += ["1. PATIENT INFORMATION"]
    if patient:
        L += [f"   Name / ID: {patient.get('patient_id') or '-'}", f"   Age: {patient['age']} years" if patient.get("age") else "   Age: -"]
    else:
        L += ["   not collected"]
    L += ["", "2. ORAL-HABITS QUESTIONNAIRE (habit-based risk — separate from the image analysis, not a diagnosis)"]
    if risk:
        for n, q in enumerate(RISK_QUESTIONS, 1):
            a = risk["answers"][q["key"]]
            L += [f"   Q{n}. {q['text']}", f"       -> {a}  ({q['options'][a]} point{'s' if q['options'][a] != 1 else ''})"]
        L += [f"   Habit-based caries risk: {risk['level']}  (score {risk['score']}/{risk['max']})"]
    else:
        L += ["   questionnaire skipped"]
    L += ["", "IMAGE-BASED CNN ANALYSIS" + (f"  —  {res.get('input')}" if res.get("input") else ""), "-" * 70]
    s1 = res.get("stage1")
    L += ["3. Stage 1 — Healthy vs Caries:"]
    if s1:
        L += ["   " + {"CARIES": "CARIES", "HEALTHY": "HEALTHY — the model did not detect caries",
                       "UNCERTAIN": f"UNCERTAIN (leaning {s1['leaning']}) — {s1['reason']}"}[s1["prediction"]],
              f"   Confidence: {s1['confidence'] * 100:.1f}%  (P(caries) {s1['p_caries']:.3f}, threshold {s1['threshold']:.3f})"]
    else:
        L += ["   not run — " + str(res.get("error") or res.get("overall"))]
    st2 = res.get("stage2", {"status": "NOT RUN"})
    L += ["", "4. Stage 2 — FDI tooth detection:"]
    if st2["status"] == "RAN":
        L += [f"   {st2['n_teeth']} teeth detected ({st2['n_number_uncertain']} with an uncertain number)"]
        L += [f"   Tooth {t['tooth_id']}: FDI {t['fdi_display']}   number confidence {t['fdi_confidence'] * 100:.1f}%   "
              f"detection {t['detection_score'] * 100:.1f}%" for t in res.get("teeth", [])]
    else:
        L += [f"   {st2['status']} — {st2.get('reason', '')}"]
    loc = res.get("localization", {"status": "NOT RUN"})
    L += ["", "5. Lesion localization:"]
    L += ([f"   {loc['n_regions']} region(s) at score >= {loc['score_threshold_used']:.2f} (validated threshold "
           f"{loc['validated_threshold']:.2f}); {loc['n_valid_crops']} valid lesion crop(s)"] if loc["status"] == "RAN" else
          [f"   {loc['status']} — {loc.get('reason', '')}"])
    L += ["", "6–8. Per-lesion results (tooth, G.V. Black, ICDAS — model predictions, each with its own confidence):"]
    tab = lesion_table(res)
    if len(tab):
        L += ["   " + l for l in tab.to_string(index=False).splitlines()]
        for les in res["lesions"]:
            if les.get("fdi_note"): L += [f"   Lesion {les['lesion']} tooth: {les['fdi_note']}"]
    else:
        reason = res.get("stage3", {}).get("reason") or res.get("overall")
        L += [f"   No lesion classified — G.V. Black: Unavailable, ICDAS: Unavailable ({reason})"]
    L += ["", LINE, "FINAL RESULT", LINE]
    if s1 is None:
        L += ["Image not analysed: " + str(res.get("error") or res.get("overall"))]
    elif s1["prediction"] == "HEALTHY":
        L += ["The image model did not detect caries (Stage 1 = HEALTHY). Stages 2–4 were not required."]
    elif s1["prediction"] == "UNCERTAIN" and not res.get("lesions"):
        L += ["Image result uncertain — please retake a clear, well-lit photo or have a dentist examine the teeth."]
    else:
        L += [res["overall"]]
        for les in res.get("lesions", []):
            L += ["  " + lesion_label(les)]
    L += ["", "9. PREVENTIVE ADVICE (from the questionnaire + general advice)"] + [f"   • {a}" for a in general_advice(res, risk)]
    L += ["", "10. LIMITATIONS"] + ["   " + t for t in report_limitations(res)]
    L += ["", DISCLAIMER, LINE]
    return L

def show_annotated_report(rgb, res, title):
    """Original image with tooth boxes + FDI labels (orange) and lesion boxes + results (green)."""
    plt.figure(figsize=(12, 12 * rgb.shape[0] / rgb.shape[1]))
    plt.imshow(draw_result(rgb, res) if res.get("stage1") else rgb); plt.axis("off")
    plt.title(title + "\norange = Stage 2 teeth (FDI, '?' = uncertain) | green = lesions (L# T<tooth>: G.V. Black | ICDAS)", fontsize=9)
    plt.show()

def save_assessment(res, rgb, lines, base):
    cv2.imwrite(base + "_result.jpg", cv2.cvtColor(draw_result(rgb, res), cv2.COLOR_RGB2BGR))
    write_json(base + "_result.json", {k: v for k, v in res.items() if not k.startswith("_")})
    with open(base + "_report.txt", "w") as f:
        f.write("\n".join(lines))
    lesion_table(res).to_csv(base + "_lesions.csv", index=False)
    return [base + s for s in ("_result.jpg", "_result.json", "_report.txt", "_lesions.csv")]

def _show_table(df, title):
    if not len(df): return
    print(title)
    if display is not None and IN_COLAB: display(df)
    else: print(df.to_string(index=False))

# ---------- main entry point ----------
def predict_complete_dental_image(score_threshold=None, ask_questions=True, *, image_path=None, answers=None, patient=None,
                                  age=None, show=True, continue_if_uncertain=False):
    """Patient info + questionnaire -> upload -> Stage 1 -> Stage 2 -> localization -> Stage 3 -> Stage 4 -> final report.
    score_threshold=None uses the localizer's validated threshold (lower values add low-certainty regions, flagged as such).
    The questionnaire never influences any CNN prediction."""
    if any(k not in PIPELINE["models"] for k in MODEL_KEYS + OPTIONAL_MODEL_KEYS):
        load_all_models()                  # loads only what is missing; Stage 2 is optional and reported if unavailable
    print(LINE + "\n" + _center("COMPLETE DENTAL ASSESSMENT") + "\n" + _center("research prototype — not a clinical diagnosis") + "\n" + LINE)
    patient_info, risk = None, None
    if ask_questions:
        patient_info = ask_patient_info(patient, age)
        print("\n\nORAL-HABITS QUESTIONNAIRE\n" + LINE)
        risk = ask_oral_habits(answers)
        print_risk_result(risk)
    paths = ([image_path] if isinstance(image_path, str) else list(image_path)) if image_path else upload_dental_images()
    out = {"patient": patient_info, "habit_risk": risk, "images": {}}
    if not paths:
        print("\nNo image selected — image analysis skipped. Run the cell again to upload an image.")
        return out
    for p in paths:
        name = os.path.basename(str(p))
        print("\n" + LINE + "\n" + _center("IMAGE ANALYSIS") + "\n" + LINE + "\n")
        rgb, err = validate_input_image(p)
        if err:
            print(f"Image could not be used ({name}): {err}")
            res = {"input": name, "overall": "INVALID INPUT", "error": err, "stage1": None, "lesions": [], "teeth": [],
                   **{k: {"status": "NOT RUN", "reason": "invalid input image"} for k in STAGE_TITLES}}
        else:
            show_uploaded_image(rgb, name, show)
            res = run_pipeline(rgb, score_threshold, continue_if_uncertain=continue_if_uncertain, keep_crops=True)
            res["input"] = name; res["image_path"] = p
            s1, st2, loc = res["stage1"], res["stage2"], res["localization"]
            print("Running Stage 1 (healthy vs caries)...")
            print(f"   -> {s1['prediction']}  (confidence {s1['confidence'] * 100:.1f}%)" + (f" — {s1['reason']}" if s1["reason"] else ""))
            print("Running Stage 2 (FDI tooth detection)...")
            print("   -> " + (f"{st2['n_teeth']} teeth: " + (", ".join(t["fdi_display"] for t in res["teeth"]) or "none")
                              if st2["status"] == "RAN" else f"{st2['status']} — {st2.get('reason', '')}"))
            print("Running lesion localization...")
            print("   -> " + (f"{loc['n_regions']} region(s) at score >= {loc['score_threshold_used']:.2f} (validated "
                              f"{loc['validated_threshold']:.2f}), {loc['n_valid_crops']} valid crop(s)" if loc["status"] == "RAN"
                              else f"{loc['status']} — {loc.get('reason', '')}"))
            print("Running Stage 3 (G.V. Black) + Stage 4 (ICDAS) on each lesion...")
            print("   -> " + (res["stage3"]["status"] + (f" — {res['stage3'].get('reason')}" if res["stage3"].get("reason") else "")))
            for les in res["lesions"]:
                print("      " + lesion_label(les))
            crops = res.pop("_crops", [])
            if show and crops:
                ims, tts = [], []
                for c, l in zip(crops, res["lesions"]):
                    ims += [c["gvb_crop"], c["icdas_crop"]]
                    tts += [f"L{l['lesion']} (tooth {l['fdi_tooth'] or '?'}) G.V. Black input", f"L{l['lesion']} ICDAS input"]
                show_examples(ims, tts, "Lesion crops used for Stage 3 and Stage 4", ncols=6)
        lines = final_report_lines(res, patient_info, risk)
        print("\n" + "\n".join(lines))
        if rgb is not None:
            _show_table(teeth_table(res), "\nStage 2 — detected teeth:")
            _show_table(lesion_table(res), "\nPer-lesion results:")
            if show: show_annotated_report(rgb, res, f"{name} — {res['overall']}")
            base = os.path.join(PREDICTION_DIR, stem(name) + "_assessment")
            print("Saved:", ", ".join(save_assessment(res, rgb, lines, base)))
        res["habit_risk"] = risk; res["patient"] = patient_info
        out["images"][name] = res
    return out

print(f"Ready: predict_complete_dental_image()  | uploads are saved in {UPLOAD_DIR}")

In [ ]:
# ============================================================
# SECTION 82.1 — RUN THE COMPLETE ASSESSMENT (questionnaire -> upload -> Stage 1 -> 2 -> localization -> 3 -> 4 -> report)
# ============================================================
results = predict_complete_dental_image(score_threshold=None, ask_questions=True)